In [ ]:
import os
from google.colab import drive
import rasterio
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# Step 1. Export unique China grid points for GEE water sampling
# ============================================================

import os
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
SEASON_FP = os.path.join(PATCHED_DIR, "china_seasonal_response_panel_patched.parquet")

OUT_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
os.makedirs(OUT_DIR, exist_ok=True)

season = pd.read_parquet(SEASON_FP, columns=["grid_id", "lon", "lat", "region_id"])

grid = (
    season[["grid_id", "lon", "lat", "region_id"]]
    .drop_duplicates("grid_id")
    .dropna()
    .copy()
)

grid["lon"] = pd.to_numeric(grid["lon"], errors="coerce")
grid["lat"] = pd.to_numeric(grid["lat"], errors="coerce")
grid = grid.dropna(subset=["lon", "lat"])

grid_fp = os.path.join(OUT_DIR, "china_grid_points_for_water_sampling.csv")
grid.to_csv(grid_fp, index=False)

print("Unique grids:", len(grid))
print("Saved:", grid_fp)
display(grid.head())

In [ ]:
import ee

PROJECT_ID = "ee-abao9474"  # 例如 "my-gee-project-123456"

ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

print("Earth Engine initialized with project:", PROJECT_ID)

In [ ]:
# ============================================================
# Step 2. Sample JRC water metrics using GEE Python API
# No manual CSV upload to GEE is needed.
#
# Input:
#   china_grid_points_for_water_sampling.csv
#
# Output to Google Drive folder:
#   China_Aquatic_Buffering_WaterMetrics/
#     water_metrics_chunk_000.csv
#     water_metrics_chunk_001.csv
#     ...
# ============================================================

import os
import math
import pandas as pd
import ee

# ----------------------------
# 0. Initialize Earth Engine
# ----------------------------
import ee

PROJECT_ID = "ee-abao9474"  # 改成你的 Google Cloud Project ID

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine initialized with existing credentials.")
except Exception as e:
    print("Need authentication or project permission. Error:")
    print(e)
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine authenticated and initialized.")

print("Using project:", PROJECT_ID)

# ----------------------------
# 1. Paths
# ----------------------------
PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
GRID_FP = os.path.join(AQUA_DIR, "china_grid_points_for_water_sampling.csv")

grid = pd.read_csv(GRID_FP)

print("Grid points:", len(grid))
display(grid.head())

# ----------------------------
# 2. Load JRC Global Surface Water v1.4
# ----------------------------
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

occurrence = gsw.select("occurrence").rename("water_occurrence")
seasonality = gsw.select("seasonality").rename("water_seasonality")
transition = gsw.select("transition").rename("water_transition")

# Permanent water:
# occurrence >= 80% and appears in >=10 months
perm_water = occurrence.gte(80).And(seasonality.gte(10)).selfMask()

# Seasonal water:
# appears in 1-9 months
seasonal_water = seasonality.gte(1).And(seasonality.lt(10)).selfMask()

# Any water:
# occurrence >= 10%
any_water = occurrence.gte(10).selfMask()

# Distance transform.
# fastDistanceTransform gives squared pixel distance; sqrt converts to pixel distance.
# JRC is Landsat-derived ~30 m, so multiply by 30 for meters.
dist_perm_m = (
    perm_water
    .fastDistanceTransform(1024)
    .sqrt()
    .multiply(30)
    .rename("dist_perm_water_m")
)

dist_seasonal_m = (
    seasonal_water
    .fastDistanceTransform(1024)
    .sqrt()
    .multiply(30)
    .rename("dist_seasonal_water_m")
)

dist_any_m = (
    any_water
    .fastDistanceTransform(1024)
    .sqrt()
    .multiply(30)
    .rename("dist_any_water_m")
)

water_img = (
    occurrence
    .addBands(seasonality)
    .addBands(transition)
    .addBands(dist_perm_m)
    .addBands(dist_seasonal_m)
    .addBands(dist_any_m)
)

# ----------------------------
# 3. Convert one pandas chunk to ee.FeatureCollection
# ----------------------------
def df_to_fc(df):
    feats = []

    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])

        props = {
            "grid_id": str(r["grid_id"]),
            "lon": float(r["lon"]),
            "lat": float(r["lat"]),
            "region_id": str(r["region_id"]) if "region_id" in df.columns else ""
        }

        feats.append(ee.Feature(geom, props))

    return ee.FeatureCollection(feats)

# ----------------------------
# 4. Export in chunks
# ----------------------------
CHUNK_SIZE = 5000
DRIVE_FOLDER = "China_Aquatic_Buffering_WaterMetrics"

n_chunks = math.ceil(len(grid) / CHUNK_SIZE)
print("Total chunks:", n_chunks)

tasks = []

for i in range(n_chunks):
    sub = grid.iloc[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE].copy()
    fc = df_to_fc(sub)

    sampled = water_img.sampleRegions(
        collection=fc,
        properties=["grid_id", "lon", "lat", "region_id"],
        scale=300,
        geometries=False,
        tileScale=4
    )

    desc = f"water_metrics_chunk_{i:03d}"

    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=desc,
        fileFormat="CSV"
    )

    task.start()
    tasks.append(task)

    print("Started:", desc, "rows:", len(sub))

print("\nAll tasks started.")
print("Go to Google Drive folder after export finishes:")
print(DRIVE_FOLDER)

In [ ]:
# ============================================================
# Check Earth Engine export task status
# ============================================================

import ee
import pandas as pd

PROJECT_ID = "ee-abao9474"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

tasks = ee.batch.Task.list()

rows = []
for t in tasks:
    status = t.status()
    desc = status.get("description", "")

    if desc.startswith("water_metrics_chunk_"):
        rows.append({
            "description": desc,
            "state": status.get("state"),
            "creation_timestamp_ms": status.get("creation_timestamp_ms"),
            "update_timestamp_ms": status.get("update_timestamp_ms"),
            "error_message": status.get("error_message", "")
        })

task_df = pd.DataFrame(rows).sort_values("description")
display(task_df)

print(task_df["state"].value_counts())

In [ ]:
# ============================================================
# Check exported CSV files in Google Drive
# ============================================================

import os
import glob
import pandas as pd

WATER_CHUNK_DIR = "/content/drive/MyDrive/China_Aquatic_Buffering_WaterMetrics"

files = sorted(glob.glob(os.path.join(WATER_CHUNK_DIR, "water_metrics_chunk_*.csv")))

print("Folder exists:", os.path.exists(WATER_CHUNK_DIR))
print("CSV files:", len(files))

for f in files:
    print(os.path.basename(f), os.path.getsize(f) / 1024, "KB")

if len(files) > 0:
    sample = pd.read_csv(files[0], nrows=5)
    display(sample)

In [ ]:
# ============================================================
# Merge exported water metric chunks directly from Drive
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

WATER_CHUNK_DIR = "/content/drive/MyDrive/China_Aquatic_Buffering_WaterMetrics"

files = sorted(glob.glob(os.path.join(WATER_CHUNK_DIR, "water_metrics_chunk_*.csv")))

print("Water metric chunk files:", len(files))
for f in files[:5]:
    print(os.path.basename(f))

if len(files) == 0:
    raise FileNotFoundError(
        f"No water metric CSVs found in {WATER_CHUNK_DIR}. "
        "Please wait until Earth Engine export tasks finish."
    )

parts = []
for fp in files:
    df = pd.read_csv(fp)
    df["source_file"] = os.path.basename(fp)
    parts.append(df)

water = pd.concat(parts, ignore_index=True)

print("Raw merged shape:", water.shape)
print("Columns:", water.columns.tolist())

# Remove duplicated rows if any
if "grid_id" not in water.columns:
    raise ValueError("grid_id column not found in exported CSV. Please check GEE export output.")

water = water.drop_duplicates(subset=["grid_id"])

# Numeric conversion
num_cols = [
    "lon", "lat",
    "water_occurrence", "water_seasonality", "water_transition",
    "dist_perm_water_m", "dist_seasonal_water_m", "dist_any_water_m"
]

for c in num_cols:
    if c in water.columns:
        water[c] = pd.to_numeric(water[c], errors="coerce")

# Convert distance to km
for c in ["dist_perm_water_m", "dist_seasonal_water_m", "dist_any_water_m"]:
    if c in water.columns:
        water[c.replace("_m", "_km")] = water[c] / 1000.0

# Cap distance to 35 km for analysis convenience
for c in ["dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km"]:
    if c in water.columns:
        water[c] = water[c].clip(lower=0, upper=35)

out_fp = os.path.join(AQUA_DIR, "china_grid_water_metrics.csv")
water.to_csv(out_fp, index=False)

print("Merged water metrics:", water.shape)
print("Unique grids:", water["grid_id"].nunique())
print("Saved:", out_fp)

display(water.head())
display(water[[
    c for c in [
        "water_occurrence", "water_seasonality", "water_transition",
        "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km"
    ] if c in water.columns
]].describe().T)

In [ ]:
# ============================================================
# Check water metrics coverage against original grid points
# ============================================================

import os
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

GRID_FP = os.path.join(AQUA_DIR, "china_grid_points_for_water_sampling.csv")
WATER_FP = os.path.join(AQUA_DIR, "china_grid_water_metrics.csv")

grid = pd.read_csv(GRID_FP)
water = pd.read_csv(WATER_FP)

check = grid[["grid_id", "lon", "lat", "region_id"]].merge(
    water[["grid_id"]],
    on="grid_id",
    how="left",
    indicator=True
)

display(check["_merge"].value_counts())

missing = check[check["_merge"] == "left_only"]
print("Missing water metrics grids:", len(missing))

if len(missing) > 0:
    display(missing.head())

In [ ]:
# ============================================================
# Merge water metrics with seasonal response panel
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

SEASON_FP = os.path.join(PATCHED_DIR, "china_seasonal_response_panel_patched.parquet")
WATER_FP = os.path.join(AQUA_DIR, "china_grid_water_metrics.csv")

season = pd.read_parquet(SEASON_FP)
water = pd.read_csv(WATER_FP)

water_keep = [
    "grid_id",
    "water_occurrence",
    "water_seasonality",
    "water_transition",
    "dist_perm_water_km",
    "dist_seasonal_water_km",
    "dist_any_water_km"
]
water_keep = [c for c in water_keep if c in water.columns]

df = season.merge(
    water[water_keep],
    on="grid_id",
    how="left",
    validate="many_to_one"
)

# Construct Heat-Dryness Stress Index
for c in ["summer_VPD_z", "summer_LST_Day_z", "summer_SM_rootproxy_z", "summer_Precip_z"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["HDSI"] = np.nanmean(
    np.vstack([
        df["summer_VPD_z"].values,
        df["summer_LST_Day_z"].values,
        -df["summer_SM_rootproxy_z"].values,
        -df["summer_Precip_z"].values
    ]),
    axis=0
)

# Distance bands
dist_col = "dist_any_water_km"

bins = [0, 1, 3, 5, 10, 20, 35.0001]
labels = ["0-1km", "1-3km", "3-5km", "5-10km", "10-20km", ">20km"]

df["water_dist_band"] = pd.cut(
    df[dist_col],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

df["near_water_5km"] = (df[dist_col] < 5).astype(int)
df["near_water_10km"] = (df[dist_col] < 10).astype(int)
df["aquatic_transition_20km"] = (df[dist_col] < 20).astype(int)

# Water context
df["water_context"] = "far_or_no_water"
df.loc[df["dist_perm_water_km"] < 5, "water_context"] = "near_permanent_water"
df.loc[
    (df["dist_perm_water_km"] >= 5) & (df["dist_seasonal_water_km"] < 5),
    "water_context"
] = "near_seasonal_water"

# Negative vegetation response
df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)

out_fp = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel.parquet")
df.to_parquet(out_fp, index=False)

print("Aquatic buffering panel:", df.shape)
print("Saved:", out_fp)

print("Water metric valid fractions:")
display(df[water_keep + ["HDSI", "water_dist_band", "water_context"]].notna().mean().reset_index(name="valid_frac"))

display(df[[
    "grid_id", "year", "lon", "lat", "HDSI",
    "dist_any_water_km", "water_dist_band", "water_context",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05"
]].head())

In [ ]:
# ============================================================
# Step 2 FIXED. Re-export JRC water metrics for ALL grid points
# Fix: unmask water bands and distance bands before sampleRegions
# ============================================================

import os
import math
import pandas as pd
import ee

PROJECT_ID = "ee-abao9474"

try:
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine initialized with existing credentials.")
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("Earth Engine authenticated and initialized.")

print("Using project:", PROJECT_ID)

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
GRID_FP = os.path.join(AQUA_DIR, "china_grid_points_for_water_sampling.csv")

grid = pd.read_csv(GRID_FP)

print("Grid points:", len(grid))
display(grid.head())

# ----------------------------
# JRC Global Surface Water
# ----------------------------
gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater")

# Critical fix: unmask to 0, so non-water pixels are still sampled.
occurrence = gsw.select("occurrence").unmask(0).rename("water_occurrence")
seasonality = gsw.select("seasonality").unmask(0).rename("water_seasonality")
transition = gsw.select("transition").unmask(0).rename("water_transition")

# Define target water masks as 1/0 images, not selfMask images.
perm_water = occurrence.gte(80).And(seasonality.gte(10)).rename("is_perm_water")
seasonal_water = seasonality.gte(1).And(seasonality.lt(10)).rename("is_seasonal_water")
any_water = occurrence.gte(10).rename("is_any_water")

MAX_PIXELS = 1024
PIXEL_SIZE_M = 30
MAX_DIST_M = MAX_PIXELS * PIXEL_SIZE_M

def distance_to_water(mask_img, out_name):
    # Input is 1 for target water, 0 elsewhere.
    # fastDistanceTransform returns squared pixel distance by default.
    dist = (
        mask_img
        .unmask(0)
        .fastDistanceTransform(MAX_PIXELS)
        .sqrt()
        .multiply(PIXEL_SIZE_M)
        .rename(out_name)
    )
    return dist.unmask(MAX_DIST_M).min(MAX_DIST_M)

dist_perm_m = distance_to_water(perm_water, "dist_perm_water_m")
dist_seasonal_m = distance_to_water(seasonal_water, "dist_seasonal_water_m")
dist_any_m = distance_to_water(any_water, "dist_any_water_m")

water_img = (
    occurrence
    .addBands(seasonality)
    .addBands(transition)
    .addBands(perm_water.unmask(0).rename("is_perm_water"))
    .addBands(seasonal_water.unmask(0).rename("is_seasonal_water"))
    .addBands(any_water.unmask(0).rename("is_any_water"))
    .addBands(dist_perm_m)
    .addBands(dist_seasonal_m)
    .addBands(dist_any_m)
    .toFloat()
)

def df_to_fc(df):
    feats = []

    for _, r in df.iterrows():
        geom = ee.Geometry.Point([float(r["lon"]), float(r["lat"])])
        props = {
            "grid_id": str(r["grid_id"]),
            "lon": float(r["lon"]),
            "lat": float(r["lat"]),
            "region_id": str(r["region_id"]) if "region_id" in df.columns else ""
        }
        feats.append(ee.Feature(geom, props))

    return ee.FeatureCollection(feats)

CHUNK_SIZE = 5000
DRIVE_FOLDER = "China_Aquatic_Buffering_WaterMetrics_v2"

n_chunks = math.ceil(len(grid) / CHUNK_SIZE)
print("Total chunks:", n_chunks)

for i in range(n_chunks):
    sub = grid.iloc[i * CHUNK_SIZE : (i + 1) * CHUNK_SIZE].copy()
    fc = df_to_fc(sub)

    sampled = water_img.sampleRegions(
        collection=fc,
        properties=["grid_id", "lon", "lat", "region_id"],
        scale=300,
        geometries=False,
        tileScale=4
    )

    desc = f"water_metrics_v2_chunk_{i:03d}"

    task = ee.batch.Export.table.toDrive(
        collection=sampled,
        description=desc,
        folder=DRIVE_FOLDER,
        fileNamePrefix=desc,
        fileFormat="CSV"
    )

    task.start()
    print("Started:", desc, "rows:", len(sub))

print("\nAll v2 tasks started.")
print("After export finishes, use Drive folder:")
print(DRIVE_FOLDER)

In [ ]:
# ============================================================
# Check v2 Earth Engine export task status
# ============================================================

import ee
import pandas as pd

PROJECT_ID = "ee-abao9474"

try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

tasks = ee.batch.Task.list()

rows = []
for t in tasks:
    status = t.status()
    desc = status.get("description", "")

    if desc.startswith("water_metrics_v2_chunk_"):
        rows.append({
            "description": desc,
            "state": status.get("state"),
            "error_message": status.get("error_message", "")
        })

task_df = pd.DataFrame(rows).sort_values("description")
display(task_df)
print(task_df["state"].value_counts())

In [ ]:
# ============================================================
# Merge v2 exported water metric chunks and check coverage
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

GRID_FP = os.path.join(AQUA_DIR, "china_grid_points_for_water_sampling.csv")
WATER_CHUNK_DIR = "/content/drive/MyDrive/China_Aquatic_Buffering_WaterMetrics_v2"

files = sorted(glob.glob(os.path.join(WATER_CHUNK_DIR, "water_metrics_v2_chunk_*.csv")))

print("v2 CSV files:", len(files))
for f in files:
    print(os.path.basename(f), os.path.getsize(f) / 1024, "KB")

if len(files) != 12:
    print("[WARN] Expected 12 files. Some exports may still be running.")

parts = []
for fp in files:
    df = pd.read_csv(fp)
    df["source_file"] = os.path.basename(fp)
    parts.append(df)

water = pd.concat(parts, ignore_index=True)
print("Raw merged rows:", len(water))
print("Columns:", water.columns.tolist())

water = water.drop_duplicates(subset=["grid_id"])
print("Unique water grids:", water["grid_id"].nunique())

num_cols = [
    "lon", "lat",
    "water_occurrence", "water_seasonality", "water_transition",
    "is_perm_water", "is_seasonal_water", "is_any_water",
    "dist_perm_water_m", "dist_seasonal_water_m", "dist_any_water_m"
]

for c in num_cols:
    if c in water.columns:
        water[c] = pd.to_numeric(water[c], errors="coerce")

for c in ["dist_perm_water_m", "dist_seasonal_water_m", "dist_any_water_m"]:
    if c in water.columns:
        water[c.replace("_m", "_km")] = water[c] / 1000.0

# Keep the original distance cap explicit: 1024 * 30 m = 30.72 km
for c in ["dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km"]:
    if c in water.columns:
        water[c] = water[c].clip(lower=0, upper=30.72)

out_fp = os.path.join(AQUA_DIR, "china_grid_water_metrics_v2.csv")
water.to_csv(out_fp, index=False)

print("\nSaved:", out_fp)
display(water.head())

check_cols = [
    "water_occurrence", "water_seasonality", "water_transition",
    "is_perm_water", "is_seasonal_water", "is_any_water",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km"
]
check_cols = [c for c in check_cols if c in water.columns]

print("\nValid fractions:")
display(water[check_cols].notna().mean().reset_index(name="valid_frac"))

print("\nSummary:")
display(water[check_cols].describe().T)

# Coverage check against original 55,418 grid points
grid = pd.read_csv(GRID_FP)

coverage = grid[["grid_id", "lon", "lat", "region_id"]].merge(
    water[["grid_id"]],
    on="grid_id",
    how="left",
    indicator=True
)

print("\nCoverage:")
display(coverage["_merge"].value_counts())

missing = coverage[coverage["_merge"] == "left_only"]
print("Missing water metrics grids:", len(missing))
if len(missing) > 0:
    display(missing.head())

In [ ]:
# ============================================================
# Step 1. Merge v2 water metrics with seasonal response panel
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

SEASON_FP = os.path.join(PATCHED_DIR, "china_seasonal_response_panel_patched.parquet")
WATER_FP = os.path.join(AQUA_DIR, "china_grid_water_metrics_v2.csv")

season = pd.read_parquet(SEASON_FP)
water = pd.read_csv(WATER_FP)

water_keep = [
    "grid_id",
    "water_occurrence",
    "water_seasonality",
    "water_transition",
    "is_perm_water",
    "is_seasonal_water",
    "is_any_water",
    "dist_perm_water_km",
    "dist_seasonal_water_km",
    "dist_any_water_km"
]

water_keep = [c for c in water_keep if c in water.columns]

df = season.merge(
    water[water_keep],
    on="grid_id",
    how="left",
    validate="many_to_one"
)

# ----------------------------
# Build Heat-Dryness Stress Index
# HDSI high = hotter/drier condition
# ----------------------------
for c in ["summer_VPD_z", "summer_LST_Day_z", "summer_SM_rootproxy_z", "summer_Precip_z"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["HDSI"] = np.nanmean(
    np.vstack([
        df["summer_VPD_z"].values,
        df["summer_LST_Day_z"].values,
        -df["summer_SM_rootproxy_z"].values,
        -df["summer_Precip_z"].values
    ]),
    axis=0
)

# ----------------------------
# Distance bands
# ----------------------------
dist_col = "dist_any_water_km"

bins = [0, 0.1, 0.25, 0.5, 1, 2, 5]
labels = ["0-0.1km", "0.1-0.25km", "0.25-0.5km", "0.5-1km", "1-2km", "2-5km"]

df["water_dist_band"] = pd.cut(
    df[dist_col],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

# Since your dist_any_water_km max is about 4.32 km,
# we use 0.5 / 1 / 2 km as main thresholds.
df["near_water_0p5km"] = (df[dist_col] < 0.5).astype(int)
df["near_water_1km"] = (df[dist_col] < 1.0).astype(int)
df["near_water_2km"] = (df[dist_col] < 2.0).astype(int)

# Water context
df["water_context"] = "far_or_no_water"
df.loc[df["dist_perm_water_km"] < 1.0, "water_context"] = "near_permanent_water"
df.loc[
    (df["dist_perm_water_km"] >= 1.0) & (df["dist_seasonal_water_km"] < 1.0),
    "water_context"
] = "near_seasonal_water"

# Negative vegetation response
df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)

out_fp = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
df.to_parquet(out_fp, index=False)

print("Aquatic buffering panel v2:", df.shape)
print("Saved:", out_fp)

print("\nWater metric valid fractions:")
display(df[water_keep + ["HDSI", "water_dist_band", "water_context"]].notna().mean().reset_index(name="valid_frac"))

display(df[[
    "grid_id", "year", "lon", "lat",
    "HDSI",
    "dist_any_water_km", "water_dist_band", "water_context",
    "water_occurrence", "water_seasonality",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05"
]].head())

In [ ]:
# ============================================================
# Step 2. Check distance-band sample size
# ============================================================

import os
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[df["valid_model_both_y"]].copy()

print("Valid samples:", len(d))
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year"].dropna().unique()))

band_count = (
    d.groupby("water_dist_band", observed=False)
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        dist_mean=("dist_any_water_km", "mean"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(band_count)

out_fp = os.path.join(AQUA_DIR, "distance_band_sample_response_summary_v2.csv")
band_count.to_csv(out_fp, index=False)
print("Saved:", out_fp)

context_count = (
    d.groupby("water_context")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(context_count)

out_fp = os.path.join(AQUA_DIR, "water_context_response_summary_v2.csv")
context_count.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 3. Distance-band response under low vs high heat-dryness stress
# ============================================================

import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["water_dist_band"].notna()
].copy()

q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["stress_group"] = "middle"
d.loc[d["HDSI"] <= q25, "stress_group"] = "low_HDSI"
d.loc[d["HDSI"] >= q75, "stress_group"] = "high_HDSI"

stress_band = (
    d[d["stress_group"].isin(["low_HDSI", "high_HDSI"])]
    .groupby(["stress_group", "water_dist_band"], observed=False)
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(stress_band)

out_fp = os.path.join(AQUA_DIR, "distance_band_response_low_high_HDSI_v2.csv")
stress_band.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# Also compute near vs far difference
near_far = d.copy()
near_far["near_group"] = "far_1km_plus"
near_far.loc[near_far["dist_any_water_km"] < 1.0, "near_group"] = "near_0_1km"

near_far_summary = (
    near_far[near_far["stress_group"].isin(["low_HDSI", "high_HDSI"])]
    .groupby(["stress_group", "near_group"])
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(near_far_summary)

out_fp = os.path.join(AQUA_DIR, "near_far_response_low_high_HDSI_v2.csv")
near_far_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 4. Plot first aquatic buffering figures
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

band = pd.read_csv(os.path.join(AQUA_DIR, "distance_band_sample_response_summary_v2.csv"))
stress_band = pd.read_csv(os.path.join(AQUA_DIR, "distance_band_response_low_high_HDSI_v2.csv"))

# Mean anomaly by distance band
plt.figure(figsize=(8, 5))
plt.plot(band["water_dist_band"], band["EVI_z_mean"], marker="o", label="EVI")
plt.plot(band["water_dist_band"], band["LAI_z_mean"], marker="o", label="LAI")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Distance to surface water")
plt.ylabel("Mean late-summer anomaly (z-score)")
plt.title("Vegetation anomaly along aquatic proximity gradient")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
out_fp = os.path.join(FIG_DIR, "distance_band_mean_EVI_LAI_anomaly_v2.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fp)

# Negative response risk by distance band
plt.figure(figsize=(8, 5))
plt.plot(band["water_dist_band"], band["EVI_neg_rate"], marker="o", label="EVI negative risk")
plt.plot(band["water_dist_band"], band["LAI_neg_rate"], marker="o", label="LAI negative risk")
plt.xlabel("Distance to surface water")
plt.ylabel("P(anomaly z < -0.5)")
plt.title("Negative vegetation response risk along aquatic proximity gradient")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
out_fp = os.path.join(FIG_DIR, "distance_band_negative_response_risk_v2.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fp)

# High vs low HDSI
for metric, ylabel in [
    ("EVI_z_mean", "Mean EVI anomaly"),
    ("LAI_z_mean", "Mean LAI anomaly"),
    ("EVI_neg_rate", "EVI negative response risk"),
    ("LAI_neg_rate", "LAI negative response risk"),
]:
    plt.figure(figsize=(8, 5))

    for group in ["low_HDSI", "high_HDSI"]:
        sub = stress_band[stress_band["stress_group"] == group]
        plt.plot(sub["water_dist_band"], sub[metric], marker="o", label=group)

    if "mean" in metric:
        plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel("Distance to surface water")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} under low vs high heat-dryness stress")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, f"{metric}_low_vs_high_HDSI_v2.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 5. Spatial map of distance to surface water
# ============================================================

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"
WATER_FP = os.path.join(AQUA_DIR, "china_grid_water_metrics_v2.csv")

try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")

china = china.dissolve().reset_index(drop=True)

water = pd.read_csv(WATER_FP)

gdf = gpd.GeoDataFrame(
    water,
    geometry=gpd.points_from_xy(water["lon"], water["lat"]),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(9, 7))
china.boundary.plot(ax=ax, linewidth=0.7, color="black")

gdf.plot(
    ax=ax,
    column="dist_any_water_km",
    markersize=1.0,
    cmap="viridis_r",
    vmin=0,
    vmax=5,
    legend=True,
    legend_kwds={"shrink": 0.65, "label": "Distance to surface water (km)"}
)

ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Distance to JRC surface water")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linewidth=0.4)
plt.tight_layout()

out_fp = os.path.join(FIG_DIR, "map_distance_to_surface_water_v2.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 6 FIXED. Formal test of aquatic buffering and buffering failure
# Distance variable: dist_any_water_km
# Fix: convert LC/year categorical variables to statsmodels-safe string dtype
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# Keep valid samples
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_any_water_km"].notna()
].copy()

# ----------------------------
# Prepare numeric variables
# ----------------------------
num_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_any_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_any"] = np.log1p(d["dist_any_water_km"])

# Center variables
d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_any_c"] = d["log_dist_any"] - d["log_dist_any"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

# Statsmodels-safe categorical variables
d["LC_Type1_int"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_any_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_int", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_int"] = d["LC_Type1_int"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)

# Ensure binary targets are int
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Model samples:", len(d))
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year_cat"].unique()))
print("LC classes:", sorted(d["LC_Type1_int"].unique())[:20])

# ----------------------------
# OLS models
# ----------------------------
ols_formulas = {
    "EVI_continuous_anywater": (
        "late_summer_EVI_z ~ HDSI_c * log_dist_any_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_int) + C(year_cat)"
    ),
    "LAI_continuous_anywater": (
        "late_summer_LAI_z ~ HDSI_c * log_dist_any_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_int) + C(year_cat)"
    ),
}

ols_rows = []

for name, formula in ols_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.ols(formula, data=d).fit(cov_type="HC3")
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]:
        ols_rows.append({
            "model": name,
            "term": term,
            "coef": model.params.get(term, np.nan),
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "r2": model.rsquared
        })

ols_df = pd.DataFrame(ols_rows)
display(ols_df)

ols_out = os.path.join(AQUA_DIR, "formal_ols_anywater_key_terms.csv")
ols_df.to_csv(ols_out, index=False)
print("Saved:", ols_out)

# ----------------------------
# Logistic models
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_logit = d.sample(MAX_N, random_state=42).copy()
else:
    d_logit = d.copy()

logit_formulas = {
    "EVI_negative_anywater": (
        "EVI_neg_z05 ~ HDSI_c * log_dist_any_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_int) + C(year_cat)"
    ),
    "LAI_negative_anywater": (
        "LAI_neg_z05 ~ HDSI_c * log_dist_any_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_int) + C(year_cat)"
    ),
}

logit_rows = []

for name, formula in logit_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.logit(formula, data=d_logit).fit(maxiter=200, disp=False)
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]:
        coef = model.params.get(term, np.nan)
        logit_rows.append({
            "model": name,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "pseudo_r2": model.prsquared
        })

logit_df = pd.DataFrame(logit_rows)
display(logit_df)

logit_out = os.path.join(AQUA_DIR, "formal_logit_anywater_key_terms.csv")
logit_df.to_csv(logit_out, index=False)
print("Saved:", logit_out)

In [ ]:
# ============================================================
# Step 7 FIXED. Formal test using distance to permanent water
# Fix: convert LC/year categorical variables to statsmodels-safe string dtype
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# ----------------------------
# 1. Keep valid samples
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

# ----------------------------
# 2. Numeric conversion
# ----------------------------
num_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    if c in d.columns:
        d[c] = pd.to_numeric(d[c], errors="coerce")

# ----------------------------
# 3. Construct model variables
# ----------------------------
d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

# IMPORTANT:
# statsmodels / patsy does not like pandas nullable Int64Dtype.
# Convert categorical controls to normal string categories.
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Model samples:", len(d))
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year_cat"].unique()))
print("LC classes:", sorted(d["LC_Type1_cat"].unique())[:20])

print("\nPermanent water distance summary:")
display(d["dist_perm_water_km"].describe())

# ----------------------------
# 4. OLS models
# ----------------------------
ols_formulas = {
    "EVI_continuous_permwater": (
        "late_summer_EVI_z ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
    "LAI_continuous_permwater": (
        "late_summer_LAI_z ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
}

ols_rows = []

for name, formula in ols_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.ols(formula, data=d).fit(cov_type="HC3")
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        ols_rows.append({
            "model": name,
            "term": term,
            "coef": model.params.get(term, np.nan),
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "r2": model.rsquared
        })

ols_df = pd.DataFrame(ols_rows)
display(ols_df)

ols_out = os.path.join(AQUA_DIR, "formal_ols_permwater_key_terms.csv")
ols_df.to_csv(ols_out, index=False)
print("Saved:", ols_out)

# ----------------------------
# 5. Logistic models
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_logit = d.sample(MAX_N, random_state=42).copy()
else:
    d_logit = d.copy()

logit_formulas = {
    "EVI_negative_permwater": (
        "EVI_neg_z05 ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
    "LAI_negative_permwater": (
        "LAI_neg_z05 ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
}

logit_rows = []

for name, formula in logit_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.logit(formula, data=d_logit).fit(maxiter=200, disp=False)
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        coef = model.params.get(term, np.nan)
        logit_rows.append({
            "model": name,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "pseudo_r2": model.prsquared
        })

logit_df = pd.DataFrame(logit_rows)
display(logit_df)

logit_out = os.path.join(AQUA_DIR, "formal_logit_permwater_key_terms.csv")
logit_df.to_csv(logit_out, index=False)
print("Saved:", logit_out)

In [ ]:
# ============================================================
# Step 8 FIXED. Marginal effect curves:
# Predicted negative risk along distance gradient under low/mid/high HDSI
# Using dist_any_water_km
#
# Fix:
#   Convert LC_Type1 and year to statsmodels-safe string categories:
#   LC_Type1_cat, year_cat
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# ----------------------------
# 1. Prepare data
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_any_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_any_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_any"] = np.log1p(d["dist_any_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_any_c"] = d["log_dist_any"] - d["log_dist_any"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

# Critical fix: use normal string categories
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_any_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Model samples:", len(d))
print("LC dtype:", d["LC_Type1_cat"].dtype)
print("Year dtype:", d["year_cat"].dtype)
print("LC classes:", sorted(d["LC_Type1_cat"].unique())[:20])
print("Years:", sorted(d["year_cat"].unique()))

# ----------------------------
# 2. Fit logistic models
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

formula_evi = (
    "EVI_neg_z05 ~ HDSI_c * log_dist_any_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

formula_lai = (
    "LAI_neg_z05 ~ HDSI_c * log_dist_any_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

model_evi = smf.logit(formula_evi, data=d_fit).fit(maxiter=200, disp=False)
model_lai = smf.logit(formula_lai, data=d_fit).fit(maxiter=200, disp=False)

print("\nEVI model key terms:")
print(model_evi.params[["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]])
print(model_evi.pvalues[["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]])

print("\nLAI model key terms:")
print(model_lai.params[["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]])
print(model_lai.pvalues[["HDSI_c", "log_dist_any_c", "HDSI_c:log_dist_any_c"]])

# ----------------------------
# 3. Prediction grid
# ----------------------------
dist_values = np.linspace(0, 4.5, 120)
log_dist_values = np.log1p(dist_values)

HDSI_levels = {
    "Low HDSI": d["HDSI"].quantile(0.25),
    "Median HDSI": d["HDSI"].quantile(0.50),
    "High HDSI": d["HDSI"].quantile(0.75),
}

# Use modal LC and modal year as reference controls
base_lc = str(d["LC_Type1_cat"].mode().iloc[0])
base_year = str(d["year_cat"].mode().iloc[0])

pred_rows = []

for level_name, hdsi_value in HDSI_levels.items():
    for dist, log_dist in zip(dist_values, log_dist_values):
        pred_rows.append({
            "dist_any_water_km": dist,
            "log_dist_any": log_dist,
            "HDSI_c": hdsi_value - d["HDSI"].mean(),
            "log_dist_any_c": log_dist - d["log_dist_any"].mean(),
            "Tair_c": 0.0,
            "elevation_c": 0.0,
            "slope_c": 0.0,
            "LC_Type1_cat": base_lc,
            "year_cat": base_year,
            "HDSI_level": level_name
        })

pred_df = pd.DataFrame(pred_rows)

pred_df["EVI_neg_prob"] = model_evi.predict(pred_df)
pred_df["LAI_neg_prob"] = model_lai.predict(pred_df)

out_pred = os.path.join(AQUA_DIR, "marginal_predictions_anywater_logit.csv")
pred_df.to_csv(out_pred, index=False)
print("Saved prediction table:", out_pred)

# ----------------------------
# 4. Plot marginal curves
# ----------------------------
for target_col, ylabel, out_name in [
    ("EVI_neg_prob", "Predicted EVI negative-response risk", "marginal_EVI_negative_risk_anywater.png"),
    ("LAI_neg_prob", "Predicted LAI negative-response risk", "marginal_LAI_negative_risk_anywater.png"),
]:
    plt.figure(figsize=(8, 5))

    for level_name in HDSI_levels.keys():
        sub = pred_df[pred_df["HDSI_level"] == level_name]
        plt.plot(sub["dist_any_water_km"], sub[target_col], linewidth=2, label=level_name)

    plt.xlabel("Distance to any surface water (km)")
    plt.ylabel(ylabel)
    plt.title(ylabel + " along aquatic proximity gradient")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 9. Descriptive statistics by distance to permanent water
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

df = pd.read_parquet(PANEL_FP)

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

# Permanent water distance bands
perm_bins = [0, 0.5, 1, 2, 5, 10, 20]
perm_labels = ["0-0.5km", "0.5-1km", "1-2km", "2-5km", "5-10km", ">10km"]

d["perm_dist_band"] = pd.cut(
    d["dist_perm_water_km"],
    bins=perm_bins,
    labels=perm_labels,
    include_lowest=True,
    right=False
)

# HDSI groups
q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["stress_group"] = "middle"
d.loc[d["HDSI"] <= q25, "stress_group"] = "low_HDSI"
d.loc[d["HDSI"] >= q75, "stress_group"] = "high_HDSI"

# Overall distance-band summary
perm_band_summary = (
    d.groupby("perm_dist_band", observed=False)
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        dist_mean=("dist_perm_water_km", "mean"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(perm_band_summary)

out_fp = os.path.join(AQUA_DIR, "permwater_distance_band_response_summary_v2.csv")
perm_band_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# Low vs high HDSI by permanent water distance
perm_stress_summary = (
    d[d["stress_group"].isin(["low_HDSI", "high_HDSI"])]
    .groupby(["stress_group", "perm_dist_band"], observed=False)
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(perm_stress_summary)

out_fp = os.path.join(AQUA_DIR, "permwater_distance_band_response_low_high_HDSI_v2.csv")
perm_stress_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# Plot overall anomaly
plt.figure(figsize=(8, 5))
plt.plot(perm_band_summary["perm_dist_band"], perm_band_summary["EVI_z_mean"], marker="o", label="EVI")
plt.plot(perm_band_summary["perm_dist_band"], perm_band_summary["LAI_z_mean"], marker="o", label="LAI")
plt.axhline(0, linestyle="--", linewidth=1)
plt.xlabel("Distance to permanent water")
plt.ylabel("Mean late-summer anomaly (z-score)")
plt.title("Vegetation anomaly along permanent-water proximity gradient")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
out_fp = os.path.join(FIG_DIR, "permwater_distance_band_mean_EVI_LAI_anomaly_v2.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fp)

# Plot negative risk
plt.figure(figsize=(8, 5))
plt.plot(perm_band_summary["perm_dist_band"], perm_band_summary["EVI_neg_rate"], marker="o", label="EVI negative risk")
plt.plot(perm_band_summary["perm_dist_band"], perm_band_summary["LAI_neg_rate"], marker="o", label="LAI negative risk")
plt.xlabel("Distance to permanent water")
plt.ylabel("P(anomaly z < -0.5)")
plt.title("Negative vegetation response risk along permanent-water proximity gradient")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
out_fp = os.path.join(FIG_DIR, "permwater_distance_band_negative_response_risk_v2.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fp)

# Plot low vs high HDSI
for metric, ylabel in [
    ("EVI_z_mean", "Mean EVI anomaly"),
    ("LAI_z_mean", "Mean LAI anomaly"),
    ("EVI_neg_rate", "EVI negative response risk"),
    ("LAI_neg_rate", "LAI negative response risk"),
]:
    plt.figure(figsize=(8, 5))

    for group in ["low_HDSI", "high_HDSI"]:
        sub = perm_stress_summary[perm_stress_summary["stress_group"] == group]
        plt.plot(sub["perm_dist_band"], sub[metric], marker="o", label=group)

    if "mean" in metric:
        plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel("Distance to permanent water")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} under low vs high heat-dryness stress")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, f"permwater_{metric}_low_vs_high_HDSI_v2.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 10. Formal test using distance to permanent water
# OLS + Logit
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# ----------------------------
# Prepare data
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

# Statsmodels-safe categorical variables
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Model samples:", len(d))
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year_cat"].unique()))
print("LC classes:", sorted(d["LC_Type1_cat"].unique()))
print("\nPermanent water distance summary:")
display(d["dist_perm_water_km"].describe())

# ----------------------------
# OLS
# ----------------------------
ols_formulas = {
    "EVI_continuous_permwater": (
        "late_summer_EVI_z ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
    "LAI_continuous_permwater": (
        "late_summer_LAI_z ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
}

ols_rows = []

for name, formula in ols_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.ols(formula, data=d).fit(cov_type="HC3")
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        ols_rows.append({
            "model": name,
            "term": term,
            "coef": model.params.get(term, np.nan),
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "r2": model.rsquared
        })

ols_df = pd.DataFrame(ols_rows)
display(ols_df)

ols_out = os.path.join(AQUA_DIR, "formal_ols_permwater_key_terms.csv")
ols_df.to_csv(ols_out, index=False)
print("Saved:", ols_out)

# ----------------------------
# Logit
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_logit = d.sample(MAX_N, random_state=42).copy()
else:
    d_logit = d.copy()

logit_formulas = {
    "EVI_negative_permwater": (
        "EVI_neg_z05 ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
    "LAI_negative_permwater": (
        "LAI_neg_z05 ~ HDSI_c * log_dist_perm_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    ),
}

logit_rows = []

for name, formula in logit_formulas.items():
    print("\n" + "=" * 100)
    print(name)
    print(formula)

    model = smf.logit(formula, data=d_logit).fit(maxiter=200, disp=False)
    print(model.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        coef = model.params.get(term, np.nan)
        logit_rows.append({
            "model": name,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "se": model.bse.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "pseudo_r2": model.prsquared
        })

logit_df = pd.DataFrame(logit_rows)
display(logit_df)

logit_out = os.path.join(AQUA_DIR, "formal_logit_permwater_key_terms.csv")
logit_df.to_csv(logit_out, index=False)
print("Saved:", logit_out)

In [ ]:
# ============================================================
# Step 11. Marginal effect curves using distance to permanent water
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# ----------------------------
# Prepare data
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Model samples:", len(d))
print("Permanent water distance range:", d["dist_perm_water_km"].min(), d["dist_perm_water_km"].max())

# ----------------------------
# Fit logit models
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

formula_evi = (
    "EVI_neg_z05 ~ HDSI_c * log_dist_perm_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

formula_lai = (
    "LAI_neg_z05 ~ HDSI_c * log_dist_perm_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

model_evi = smf.logit(formula_evi, data=d_fit).fit(maxiter=200, disp=False)
model_lai = smf.logit(formula_lai, data=d_fit).fit(maxiter=200, disp=False)

print("\nEVI model key terms:")
print(model_evi.params[["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]])
print(model_evi.pvalues[["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]])

print("\nLAI model key terms:")
print(model_lai.params[["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]])
print(model_lai.pvalues[["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]])

# ----------------------------
# Prediction grid
# ----------------------------
dist_values = np.linspace(0, 15.5, 160)
log_dist_values = np.log1p(dist_values)

HDSI_levels = {
    "Low HDSI": d["HDSI"].quantile(0.25),
    "Median HDSI": d["HDSI"].quantile(0.50),
    "High HDSI": d["HDSI"].quantile(0.75),
}

base_lc = str(d["LC_Type1_cat"].mode().iloc[0])
base_year = str(d["year_cat"].mode().iloc[0])

pred_rows = []

for level_name, hdsi_value in HDSI_levels.items():
    for dist, log_dist in zip(dist_values, log_dist_values):
        pred_rows.append({
            "dist_perm_water_km": dist,
            "log_dist_perm": log_dist,
            "HDSI_c": hdsi_value - d["HDSI"].mean(),
            "log_dist_perm_c": log_dist - d["log_dist_perm"].mean(),
            "Tair_c": 0.0,
            "elevation_c": 0.0,
            "slope_c": 0.0,
            "LC_Type1_cat": base_lc,
            "year_cat": base_year,
            "HDSI_level": level_name
        })

pred_df = pd.DataFrame(pred_rows)

pred_df["EVI_neg_prob"] = model_evi.predict(pred_df)
pred_df["LAI_neg_prob"] = model_lai.predict(pred_df)

out_pred = os.path.join(AQUA_DIR, "marginal_predictions_permwater_logit.csv")
pred_df.to_csv(out_pred, index=False)
print("Saved prediction table:", out_pred)

# ----------------------------
# Plot
# ----------------------------
for target_col, ylabel, out_name in [
    ("EVI_neg_prob", "Predicted EVI negative-response risk", "marginal_EVI_negative_risk_permwater.png"),
    ("LAI_neg_prob", "Predicted LAI negative-response risk", "marginal_LAI_negative_risk_permwater.png"),
]:
    plt.figure(figsize=(8, 5))

    for level_name in HDSI_levels.keys():
        sub = pred_df[pred_df["HDSI_level"] == level_name]
        plt.plot(sub["dist_perm_water_km"], sub[target_col], linewidth=2, label=level_name)

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(ylabel + " along permanent-water proximity gradient")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 12. Compare permanent-water vs seasonal-water contexts
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering"
)
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna()
].copy()

# Define water context more explicitly
d["water_context2"] = "far_from_water"

d.loc[d["dist_perm_water_km"] < 1.0, "water_context2"] = "near_permanent_water"

d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "water_context2"
] = "near_seasonal_water"

# Optional: only compare three clear groups
d = d[d["water_context2"].isin([
    "near_permanent_water",
    "near_seasonal_water",
    "far_from_water"
])].copy()

q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["stress_group"] = "middle"
d.loc[d["HDSI"] <= q25, "stress_group"] = "low_HDSI"
d.loc[d["HDSI"] >= q75, "stress_group"] = "high_HDSI"

context_summary = (
    d.groupby(["stress_group", "water_context2"])
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
)

display(context_summary)

out_fp = os.path.join(AQUA_DIR, "water_context_low_high_HDSI_summary_v2.csv")
context_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# Plot context comparison under high/low HDSI
plot_df = context_summary[context_summary["stress_group"].isin(["low_HDSI", "high_HDSI"])].copy()

for metric, ylabel in [
    ("EVI_neg_rate", "EVI negative response risk"),
    ("LAI_neg_rate", "LAI negative response risk"),
    ("EVI_z_mean", "Mean EVI anomaly"),
    ("LAI_z_mean", "Mean LAI anomaly"),
]:
    plt.figure(figsize=(8, 5))

    for group in ["low_HDSI", "high_HDSI"]:
        sub = plot_df[plot_df["stress_group"] == group]
        plt.plot(sub["water_context2"], sub[metric], marker="o", linewidth=2, label=group)

    if "mean" in metric:
        plt.axhline(0, linestyle="--", linewidth=1)

    plt.xlabel("Water context")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} across water contexts")
    plt.xticks(rotation=20)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, f"water_context_{metric}_low_high_HDSI_v2.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

# Formal logit with water context interaction
num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "water_context2",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d_model = d.dropna(subset=model_vars).copy()

d_model["LC_Type1_cat"] = d_model["LC_Type1_cat"].astype(int).astype(str)
d_model["year_cat"] = d_model["year_cat"].astype(int).astype(str)
d_model["EVI_neg_z05"] = d_model["EVI_neg_z05"].astype(int)
d_model["LAI_neg_z05"] = d_model["LAI_neg_z05"].astype(int)

MAX_N = 200000
if len(d_model) > MAX_N:
    d_fit = d_model.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d_model.copy()

context_logit_rows = []

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    formula = (
        f"{target} ~ HDSI_c * C(water_context2) "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    )

    print("\n" + "=" * 100)
    print(target)
    print(formula)

    model = smf.logit(formula, data=d_fit).fit(maxiter=200, disp=False)
    print(model.summary().tables[1])

    for term in model.params.index:
        if "HDSI_c" in term or "water_context2" in term:
            coef = model.params[term]
            context_logit_rows.append({
                "target": target,
                "term": term,
                "coef_logit": coef,
                "odds_ratio": np.exp(coef),
                "se": model.bse[term],
                "pvalue": model.pvalues[term],
                "n": int(model.nobs),
                "pseudo_r2": model.prsquared
            })

context_logit_df = pd.DataFrame(context_logit_rows)
display(context_logit_df)

out_fp = os.path.join(AQUA_DIR, "formal_logit_water_context_interaction_v2.csv")
context_logit_df.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 13. Consolidate key statistical results
# ============================================================

import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

files = {
    "anywater_logit": os.path.join(AQUA_DIR, "formal_logit_anywater_key_terms.csv"),
    "permwater_logit": os.path.join(AQUA_DIR, "formal_logit_permwater_key_terms.csv"),
    "permwater_ols": os.path.join(AQUA_DIR, "formal_ols_permwater_key_terms.csv"),
    "context_logit": os.path.join(AQUA_DIR, "formal_logit_water_context_interaction_v2.csv"),
}

tables = []

for name, fp in files.items():
    if os.path.exists(fp):
        tmp = pd.read_csv(fp)
        tmp["source"] = name
        tables.append(tmp)
        print("Loaded:", name, tmp.shape)
    else:
        print("[Missing]", name, fp)

combined = pd.concat(tables, ignore_index=True)

out_fp = os.path.join(AQUA_DIR, "final_key_statistical_results_combined.csv")
combined.to_csv(out_fp, index=False)

display(combined)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 14 FIXED. Cluster-robust standard errors by grid_id
# Main robustness test for permanent-water distance
#
# Fix:
#   For statsmodels LogitResults, use cov_type='cluster'
#   directly inside .fit(), instead of get_robustcov_results().
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

# ----------------------------
# 1. Prepare data
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "grid_id",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Full model samples:", len(d))
print("Full unique grids:", d["grid_id"].nunique())

# ----------------------------
# 2. Sample for speed, same as previous logit
# ----------------------------
MAX_N = 200000

if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

print("Fitted samples:", len(d_fit))
print("Cluster grids:", d_fit["grid_id"].nunique())

# ----------------------------
# 3. Cluster-robust Logit
# ----------------------------
formula_template = (
    "{target} ~ HDSI_c * log_dist_perm_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

rows = []

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    formula = formula_template.format(target=target)

    print("\n" + "=" * 100)
    print(target)
    print(formula)

    # Key fix: cluster robust covariance specified inside fit()
    model_cl = smf.logit(formula, data=d_fit).fit(
        maxiter=200,
        disp=False,
        cov_type="cluster",
        cov_kwds={"groups": d_fit["grid_id"]}
    )

    print(model_cl.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        coef = model_cl.params.get(term, np.nan)
        se = model_cl.bse.get(term, np.nan)
        pval = model_cl.pvalues.get(term, np.nan)

        rows.append({
            "target": target,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "cluster_se_grid": se,
            "cluster_pvalue_grid": pval,
            "n": int(model_cl.nobs),
            "n_clusters": d_fit["grid_id"].nunique(),
            "pseudo_r2": model_cl.prsquared
        })

cluster_df = pd.DataFrame(rows)
display(cluster_df)

out_fp = os.path.join(AQUA_DIR, "robust_cluster_grid_logit_permwater_key_terms.csv")
cluster_df.to_csv(out_fp, index=False)

print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 15. Aquatic Buffering Effect and Buffering Failure Risk
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

# Stress groups
q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["stress_group"] = "middle"
d.loc[d["HDSI"] <= q25, "stress_group"] = "low_HDSI"
d.loc[d["HDSI"] >= q75, "stress_group"] = "high_HDSI"

# Distance groups based on permanent water
d["perm_near_far"] = "middle"
d.loc[d["dist_perm_water_km"] < 1.0, "perm_near_far"] = "near_perm_lt1km"
d.loc[d["dist_perm_water_km"] >= 5.0, "perm_near_far"] = "far_perm_ge5km"

use = d[d["perm_near_far"].isin(["near_perm_lt1km", "far_perm_ge5km"])].copy()

summary = (
    use.groupby(["stress_group", "perm_near_far"])
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        EVI_z_mean=("late_summer_EVI_z", "mean"),
        LAI_z_mean=("late_summer_LAI_z", "mean")
    )
    .reset_index()
)

display(summary)

# Compute ABE for each stress group
abe_rows = []

for stress in ["low_HDSI", "middle", "high_HDSI"]:
    sub = summary[summary["stress_group"] == stress]

    if set(sub["perm_near_far"]) >= {"near_perm_lt1km", "far_perm_ge5km"}:
        near = sub[sub["perm_near_far"] == "near_perm_lt1km"].iloc[0]
        far = sub[sub["perm_near_far"] == "far_perm_ge5km"].iloc[0]

        abe_rows.append({
            "stress_group": stress,
            "EVI_ABE_far_minus_near": far["EVI_neg_rate"] - near["EVI_neg_rate"],
            "LAI_ABE_far_minus_near": far["LAI_neg_rate"] - near["LAI_neg_rate"],
            "EVI_BFR_near": near["EVI_neg_rate"],
            "LAI_BFR_near": near["LAI_neg_rate"],
            "EVI_far_risk": far["EVI_neg_rate"],
            "LAI_far_risk": far["LAI_neg_rate"],
            "near_rows": int(near["rows"]),
            "far_rows": int(far["rows"])
        })

abe = pd.DataFrame(abe_rows)

display(abe)

summary_fp = os.path.join(AQUA_DIR, "permwater_near_far_stress_summary_v2.csv")
abe_fp = os.path.join(AQUA_DIR, "aquatic_buffering_effect_ABE_BFR_v2.csv")

summary.to_csv(summary_fp, index=False)
abe.to_csv(abe_fp, index=False)

print("Saved:", summary_fp)
print("Saved:", abe_fp)

In [ ]:
# ============================================================
# Step 16. Spatial maps of buffering failure frequency
# ============================================================

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")

china = china.dissolve().reset_index(drop=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

h75 = d["HDSI"].quantile(0.75)

# Define partial buffering failure:
# near permanent water but still negative under high HDSI
d["near_perm_1km"] = d["dist_perm_water_km"] < 1.0
d["high_HDSI"] = d["HDSI"] >= h75

d["EVI_buffering_failure"] = (
    d["near_perm_1km"] &
    d["high_HDSI"] &
    (d["late_summer_EVI_z"] < -0.5)
).astype(int)

d["LAI_buffering_failure"] = (
    d["near_perm_1km"] &
    d["high_HDSI"] &
    (d["late_summer_LAI_z"] < -0.5)
).astype(int)

bf_grid = (
    d.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        EVI_buffering_failure_freq=("EVI_buffering_failure", "mean"),
        LAI_buffering_failure_freq=("LAI_buffering_failure", "mean"),
        high_HDSI_freq=("high_HDSI", "mean"),
        near_perm_1km=("near_perm_1km", "max"),
        dist_perm_water_km=("dist_perm_water_km", "mean")
    )
)

# Only map near-permanent-water grids for buffering failure
bf_near = bf_grid[bf_grid["near_perm_1km"]].copy()

def plot_map(df_plot, value_col, title, out_name, cmap="magma", vmin=0, vmax=0.5):
    gdf = gpd.GeoDataFrame(
        df_plot.copy(),
        geometry=gpd.points_from_xy(df_plot["lon"], df_plot["lat"]),
        crs="EPSG:4326"
    )

    fig, ax = plt.subplots(figsize=(9, 7))
    china.boundary.plot(ax=ax, linewidth=0.7, color="black")

    gdf.plot(
        ax=ax,
        column=value_col,
        markersize=1.2,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        legend=True,
        legend_kwds={"shrink": 0.65}
    )

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.2, linewidth=0.4)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

plot_map(
    bf_near,
    "EVI_buffering_failure_freq",
    "Frequency of EVI buffering failure near permanent water",
    "map_EVI_buffering_failure_near_permwater_v2.png",
    cmap="magma",
    vmin=0,
    vmax=0.5
)

plot_map(
    bf_near,
    "LAI_buffering_failure_freq",
    "Frequency of LAI buffering failure near permanent water",
    "map_LAI_buffering_failure_near_permwater_v2.png",
    cmap="magma",
    vmin=0,
    vmax=0.5
)

out_fp = os.path.join(AQUA_DIR, "buffering_failure_grid_frequency_v2.csv")
bf_grid.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 17. Cluster-robust Logit by grid_id
# Main model: permanent-water distance
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "grid_id",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Full samples:", len(d))
print("Unique grids:", d["grid_id"].nunique())

MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

print("Fitted samples:", len(d_fit))
print("Cluster grids:", d_fit["grid_id"].nunique())

formula_template = (
    "{target} ~ HDSI_c * log_dist_perm_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

rows = []

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    formula = formula_template.format(target=target)

    print("\n" + "=" * 100)
    print(target)
    print(formula)

    model_cl = smf.logit(formula, data=d_fit).fit(
        maxiter=200,
        disp=False,
        cov_type="cluster",
        cov_kwds={"groups": d_fit["grid_id"]}
    )

    print(model_cl.summary().tables[1])

    for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
        coef = model_cl.params.get(term, np.nan)
        rows.append({
            "target": target,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "cluster_se_grid": model_cl.bse.get(term, np.nan),
            "cluster_pvalue_grid": model_cl.pvalues.get(term, np.nan),
            "n": int(model_cl.nobs),
            "n_clusters": d_fit["grid_id"].nunique(),
            "pseudo_r2": model_cl.prsquared
        })

cluster_df = pd.DataFrame(rows)
display(cluster_df)

out_fp = os.path.join(AQUA_DIR, "robust_cluster_grid_logit_permwater_key_terms.csv")
cluster_df.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 18. Fine continuous distance-response curves with 95% CI
# Permanent water distance, model-based uncertainty
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import patsy

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_v2")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])
d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()
d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

formula_rhs = (
    "HDSI_c * log_dist_perm_c "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

formula_evi = "EVI_neg_z05 ~ " + formula_rhs
formula_lai = "LAI_neg_z05 ~ " + formula_rhs

model_evi = smf.logit(formula_evi, data=d_fit).fit(maxiter=200, disp=False)
model_lai = smf.logit(formula_lai, data=d_fit).fit(maxiter=200, disp=False)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def make_prediction_curve(model, target_name, n_draws=1000):
    # Fine but smooth distance grid:
    # 0-5 km every 0.05 km, 5-16 km every 0.25 km
    dist_values = np.r_[np.arange(0, 5.05, 0.05), np.arange(5.25, 16.01, 0.25)]
    log_dist_values = np.log1p(dist_values)

    hdsi_levels = {
        "Low HDSI": d["HDSI"].quantile(0.25),
        "Median HDSI": d["HDSI"].quantile(0.50),
        "High HDSI": d["HDSI"].quantile(0.75)
    }

    base_lc = str(d["LC_Type1_cat"].mode().iloc[0])
    base_year = str(d["year_cat"].mode().iloc[0])

    pred_rows = []

    for level_name, hdsi_value in hdsi_levels.items():
        for dist, log_dist in zip(dist_values, log_dist_values):
            pred_rows.append({
                "dist_perm_water_km": dist,
                "log_dist_perm": log_dist,
                "HDSI_c": hdsi_value - d["HDSI"].mean(),
                "log_dist_perm_c": log_dist - d["log_dist_perm"].mean(),
                "Tair_c": 0.0,
                "elevation_c": 0.0,
                "slope_c": 0.0,
                "LC_Type1_cat": base_lc,
                "year_cat": base_year,
                "HDSI_level": level_name
            })

    pred_df = pd.DataFrame(pred_rows)

    # Build design matrix using model design_info
    design_info = model.model.data.design_info
    Xp = patsy.build_design_matrices([design_info], pred_df, return_type="dataframe")[0]

    beta = model.params.values
    cov = model.cov_params().values

    eta = np.asarray(Xp) @ beta
    pred = sigmoid(eta)

    rng = np.random.default_rng(42)
    beta_draws = rng.multivariate_normal(beta, cov, size=n_draws)
    pred_draws = sigmoid(np.asarray(Xp) @ beta_draws.T)

    pred_df[f"{target_name}_prob"] = pred
    pred_df[f"{target_name}_lower"] = np.percentile(pred_draws, 2.5, axis=1)
    pred_df[f"{target_name}_upper"] = np.percentile(pred_draws, 97.5, axis=1)

    return pred_df

evi_curve = make_prediction_curve(model_evi, "EVI", n_draws=1000)
lai_curve = make_prediction_curve(model_lai, "LAI", n_draws=1000)

curve_df = evi_curve.merge(
    lai_curve[[
        "dist_perm_water_km", "HDSI_level",
        "LAI_prob", "LAI_lower", "LAI_upper"
    ]],
    on=["dist_perm_water_km", "HDSI_level"],
    how="left"
)

out_fp = os.path.join(AQUA_DIR, "fine_marginal_curves_permwater_with_CI.csv")
curve_df.to_csv(out_fp, index=False)
print("Saved:", out_fp)

def plot_curve(df_curve, prob_col, lower_col, upper_col, ylabel, out_name):
    plt.figure(figsize=(8.5, 5.5))

    for level in ["Low HDSI", "Median HDSI", "High HDSI"]:
        sub = df_curve[df_curve["HDSI_level"] == level].copy()
        plt.plot(sub["dist_perm_water_km"], sub[prob_col], linewidth=2, label=level)
        plt.fill_between(
            sub["dist_perm_water_km"],
            sub[lower_col],
            sub[upper_col],
            alpha=0.18
        )

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(ylabel + " along fine permanent-water distance gradient")
    plt.xlim(0, 16)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_plot = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_plot, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_plot)

plot_curve(
    curve_df,
    "EVI_prob", "EVI_lower", "EVI_upper",
    "Predicted EVI negative-response risk",
    "fine_CI_EVI_negative_risk_permwater.png"
)

plot_curve(
    curve_df,
    "LAI_prob", "LAI_lower", "LAI_upper",
    "Predicted LAI negative-response risk",
    "fine_CI_LAI_negative_risk_permwater.png"
)

In [ ]:
# ============================================================
# Step 19. Fine empirical distance bins with binomial CI
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["stress_group"] = "middle"
d.loc[d["HDSI"] <= q25, "stress_group"] = "low_HDSI"
d.loc[d["HDSI"] >= q75, "stress_group"] = "high_HDSI"

# Fine bins:
# 0-2 km every 0.25 km
# 2-5 km every 0.5 km
# 5-16 km every 1 km
bins = list(np.arange(0, 2.0001, 0.25)) + list(np.arange(2.5, 5.0001, 0.5)) + list(np.arange(6, 17, 1))
bins = sorted(set([round(x, 4) for x in bins]))

labels = []
for a, b in zip(bins[:-1], bins[1:]):
    labels.append(f"{a:g}-{b:g}km")

d["perm_dist_fine_bin"] = pd.cut(
    d["dist_perm_water_km"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

def binom_ci(p, n):
    se = np.sqrt(p * (1 - p) / max(n, 1))
    return max(0, p - 1.96 * se), min(1, p + 1.96 * se)

rows = []

for (stress, b), g in d.dropna(subset=["perm_dist_fine_bin"]).groupby(["stress_group", "perm_dist_fine_bin"], observed=False):
    n = len(g)
    evi_p = g["EVI_neg_z05"].mean()
    lai_p = g["LAI_neg_z05"].mean()

    evi_lo, evi_hi = binom_ci(evi_p, n)
    lai_lo, lai_hi = binom_ci(lai_p, n)

    rows.append({
        "stress_group": stress,
        "perm_dist_fine_bin": str(b),
        "rows": n,
        "grids": g["grid_id"].nunique(),
        "dist_mean": g["dist_perm_water_km"].mean(),
        "EVI_neg_rate": evi_p,
        "EVI_ci_low": evi_lo,
        "EVI_ci_high": evi_hi,
        "LAI_neg_rate": lai_p,
        "LAI_ci_low": lai_lo,
        "LAI_ci_high": lai_hi,
        "EVI_z_mean": g["late_summer_EVI_z"].mean(),
        "LAI_z_mean": g["late_summer_LAI_z"].mean()
    })

fine_bins = pd.DataFrame(rows)

out_fp = os.path.join(AQUA_DIR, "fine_distance_bin_empirical_response_permwater.csv")
fine_bins.to_csv(out_fp, index=False)

display(fine_bins.head(20))
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 20. Regional heterogeneity analysis
# Permanent-water buffering effect by macro-region
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"]

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

d["macro_region"] = d.apply(assign_macro_region, axis=1)

region_summary = (
    d.groupby("macro_region")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        dist_perm_mean=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(region_summary)

region_summary.to_csv(
    os.path.join(AQUA_DIR, "macro_region_summary_v2.csv"),
    index=False
)

# Prepare model variables
num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "macro_region",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "log_dist_perm",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

rows = []

for region in region_summary["macro_region"]:
    sub = d[d["macro_region"] == region].copy()

    if len(sub) < 5000:
        print("[Skip small region]", region, len(sub))
        continue

    # Center within region
    sub["HDSI_c"] = sub["HDSI"] - sub["HDSI"].mean()
    sub["log_dist_perm_c"] = sub["log_dist_perm"] - sub["log_dist_perm"].mean()
    sub["Tair_c"] = sub["summer_Tair_z"] - sub["summer_Tair_z"].mean()
    sub["elevation_c"] = sub["elevation"] - sub["elevation"].mean()
    sub["slope_c"] = sub["slope"] - sub["slope"].mean()

    # sample for speed if necessary
    if len(sub) > 60000:
        sub_fit = sub.sample(60000, random_state=42)
    else:
        sub_fit = sub

    for target in ["EVI_neg_z05", "LAI_neg_z05"]:
        if sub_fit[target].nunique() < 2:
            continue

        formula = (
            f"{target} ~ HDSI_c * log_dist_perm_c "
            "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
        )

        print("\n", region, target, "n=", len(sub_fit))

        try:
            model = smf.logit(formula, data=sub_fit).fit(maxiter=200, disp=False)

            for term in ["HDSI_c", "log_dist_perm_c", "HDSI_c:log_dist_perm_c"]:
                coef = model.params.get(term, np.nan)
                rows.append({
                    "macro_region": region,
                    "target": target,
                    "term": term,
                    "coef_logit": coef,
                    "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
                    "pvalue": model.pvalues.get(term, np.nan),
                    "n": int(model.nobs),
                    "pseudo_r2": model.prsquared
                })

        except Exception as e:
            print("[Model failed]", region, target, e)

region_effects = pd.DataFrame(rows)
display(region_effects)

out_fp = os.path.join(AQUA_DIR, "regional_heterogeneity_logit_permwater_v2.csv")
region_effects.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 24. Fallback human proxy from land-cover type
# If NTL is not ready
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

for c in ["LC_Type1", "HDSI", "dist_perm_water_km", "summer_Tair_z", "elevation", "slope", "year"]:
    d[c] = pd.to_numeric(d[c], errors="coerce")

lc_round = d["LC_Type1"].round()

# MODIS IGBP common classes:
# 12 croplands, 13 urban/built-up, 14 cropland/natural vegetation mosaic
d["human_lc_proxy"] = lc_round.isin([12, 13, 14]).astype(int)

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])
d["HDSI_c"] = d["HDSI"] - d["HDSI"].mean()
d["log_dist_perm_c"] = d["log_dist_perm"] - d["log_dist_perm"].mean()
d["human_c"] = d["human_lc_proxy"] - d["human_lc_proxy"].mean()
d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()
d["LC_Type1_cat"] = lc_round
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "grid_id", "EVI_neg_z05", "LAI_neg_z05",
    "HDSI_c", "log_dist_perm_c", "human_c",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

print("Human LC proxy rate:", d["human_lc_proxy"].mean())

MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

rows = []

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    formula = (
        f"{target} ~ HDSI_c * log_dist_perm_c * human_c "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    )

    print("\n" + "=" * 100)
    print(target)

    model = smf.logit(formula, data=d_fit).fit(
        maxiter=200,
        disp=False,
        cov_type="cluster",
        cov_kwds={"groups": d_fit["grid_id"]}
    )

    key_terms = [
        "HDSI_c",
        "log_dist_perm_c",
        "human_c",
        "HDSI_c:log_dist_perm_c",
        "HDSI_c:human_c",
        "log_dist_perm_c:human_c",
        "HDSI_c:log_dist_perm_c:human_c"
    ]

    for term in key_terms:
        coef = model.params.get(term, np.nan)
        rows.append({
            "target": target,
            "term": term,
            "coef_logit": coef,
            "odds_ratio": np.exp(coef) if pd.notna(coef) else np.nan,
            "cluster_se_grid": model.bse.get(term, np.nan),
            "cluster_pvalue_grid": model.pvalues.get(term, np.nan),
            "n": int(model.nobs),
            "n_clusters": d_fit["grid_id"].nunique(),
            "pseudo_r2": model.prsquared
        })

human_lc_df = pd.DataFrame(rows)
display(human_lc_df)

out_fp = os.path.join(AQUA_DIR, "formal_logit_human_LCproxy_threeway_permwater_v2.csv")
human_lc_df.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 21. Inspect whether NTL / human activity variables exist
# ============================================================

import os
import glob
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

df = pd.read_parquet(PANEL_FP)

keywords = ["ntl", "light", "night", "viirs", "urban", "human", "pop", "imperv"]

matches = []
for c in df.columns:
    cl = c.lower()
    if any(k in cl for k in keywords):
        matches.append(c)

print("Potential human activity columns in panel:")
print(matches)

# Also inspect known light raw path if it exists
RAW_LIGHT_DIR = "/content/drive/MyDrive/GEE_NH_0_65_LightRaw_TargetRegions"

print("\nRAW_LIGHT_DIR exists:", os.path.exists(RAW_LIGHT_DIR))

if os.path.exists(RAW_LIGHT_DIR):
    files = []
    for ext in ["*.tif", "*.tiff", "*.csv", "*.parquet"]:
        files.extend(glob.glob(os.path.join(RAW_LIGHT_DIR, "**", ext), recursive=True))

    print("Found light-related files:", len(files))
    for f in files[:30]:
        print(f)

In [ ]:
# ============================================================
# Step 21A. Inspect LightRaw CSV structure
# ============================================================

import os
import glob
import re
import pandas as pd

RAW_LIGHT_DIR = "/content/drive/MyDrive/GEE_NH_0_65_LightRaw_TargetRegions"

files = sorted(glob.glob(os.path.join(RAW_LIGHT_DIR, "*.csv")))

print("Total light CSV files:", len(files))

# Parse file metadata
rows = []
for fp in files:
    name = os.path.basename(fp)
    m = re.search(r"light_(static|dynamic)_y(\d{4})_(R_\d+_\d+)\.csv", name)
    rows.append({
        "file": name,
        "path": fp,
        "type": m.group(1) if m else None,
        "year": int(m.group(2)) if m else None,
        "region_id": m.group(3) if m else None,
        "size_kb": os.path.getsize(fp) / 1024
    })

inv = pd.DataFrame(rows)
display(inv.head(20))
display(inv.groupby(["type", "year"]).size().reset_index(name="n_files").tail(30))
display(inv.groupby("type").agg(n=("file", "count"), size_mb=("size_kb", lambda x: x.sum()/1024)).reset_index())

# Inspect representative files
for typ in ["static", "dynamic"]:
    sub = inv[inv["type"] == typ].copy()
    if len(sub) == 0:
        continue

    fp = sub.iloc[0]["path"]
    print("\n" + "="*120)
    print("Inspect:", os.path.basename(fp))

    tmp = pd.read_csv(fp, nrows=10)
    print("Shape first 10 rows:", tmp.shape)
    print("Columns:")
    print(tmp.columns.tolist())
    display(tmp.head())

In [ ]:
# ============================================================
# Step 21B. Scan columns from 2018-2023 light CSVs
# ============================================================

import os
import glob
import re
import pandas as pd
from collections import Counter, defaultdict

RAW_LIGHT_DIR = "/content/drive/MyDrive/GEE_NH_0_65_LightRaw_TargetRegions"

files = sorted(glob.glob(os.path.join(RAW_LIGHT_DIR, "*.csv")))

records = []
col_counter = Counter()
sample_cols = {}

for fp in files:
    name = os.path.basename(fp)
    m = re.search(r"light_(static|dynamic)_y(\d{4})_(R_\d+_\d+)\.csv", name)
    if not m:
        continue

    typ = m.group(1)
    year = int(m.group(2))
    region_id = m.group(3)

    if year < 2018 or year > 2023:
        continue

    try:
        tmp = pd.read_csv(fp, nrows=5)
        cols = tmp.columns.tolist()
        for c in cols:
            col_counter[c] += 1

        records.append({
            "file": name,
            "type": typ,
            "year": year,
            "region_id": region_id,
            "n_cols": len(cols),
            "cols": cols
        })

        key = (typ, year)
        if key not in sample_cols:
            sample_cols[key] = cols

    except Exception as e:
        records.append({
            "file": name,
            "type": typ,
            "year": year,
            "region_id": region_id,
            "error": str(e)
        })

scan = pd.DataFrame(records)
print("Scanned files:", len(scan))
display(scan[["file", "type", "year", "region_id", "n_cols"]].head(30))

col_df = pd.DataFrame(
    [{"column": k, "count": v} for k, v in col_counter.items()]
).sort_values("count", ascending=False)

display(col_df)

print("\nSample columns by type/year:")
for key, cols in sample_cols.items():
    print(key, cols)

In [ ]:
# ============================================================
# Fix Google Drive "Transport endpoint is not connected"
# ============================================================

import os
import shutil
from google.colab import drive

# Try to unmount first
try:
    drive.flush_and_unmount()
    print("Drive flushed and unmounted.")
except Exception as e:
    print("flush_and_unmount failed or not needed:", e)

# Remove broken mount folder if possible
try:
    if os.path.exists("/content/drive"):
        shutil.rmtree("/content/drive", ignore_errors=True)
        print("Removed old /content/drive mount folder.")
except Exception as e:
    print("Could not remove /content/drive:", e)

# Remount
drive.mount("/content/drive", force_remount=True)

print("Drive remounted.")
print("Check MyDrive exists:", os.path.exists("/content/drive/MyDrive"))

In [ ]:
# ============================================================
# Step 21C. Build NTL table from existing LightRaw CSVs
# Match by lon/lat rounded grid_id
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
RAW_LIGHT_DIR = "/content/drive/MyDrive/GEE_NH_0_65_LightRaw_TargetRegions"

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
GRID_FP = os.path.join(AQUA_DIR, "china_grid_points_for_water_sampling.csv")

panel = pd.read_parquet(PANEL_FP)
grid = pd.read_csv(GRID_FP)

# Existing grid ids are like g_100.05_21.75
grid_keep = grid[["grid_id", "lon", "lat", "region_id"]].copy()
grid_keep["lon_round"] = grid_keep["lon"].round(2)
grid_keep["lat_round"] = grid_keep["lat"].round(2)

files = sorted(glob.glob(os.path.join(RAW_LIGHT_DIR, "*.csv")))

# Prefer dynamic files for year-varying NTL
target_files = []
for fp in files:
    name = os.path.basename(fp)
    m = re.search(r"light_(static|dynamic)_y(\d{4})_(R_\d+_\d+)\.csv", name)
    if not m:
        continue
    typ = m.group(1)
    year = int(m.group(2))
    region_id = m.group(3)

    if 2018 <= year <= 2023 and typ == "dynamic":
        target_files.append((fp, year, region_id, typ))

# If no dynamic, fallback to static
if len(target_files) == 0:
    for fp in files:
        name = os.path.basename(fp)
        m = re.search(r"light_(static|dynamic)_y(\d{4})_(R_\d+_\d+)\.csv", name)
        if not m:
            continue
        typ = m.group(1)
        year = int(m.group(2))
        region_id = m.group(3)
        if 2018 <= year <= 2023 and typ == "static":
            target_files.append((fp, year, region_id, typ))

print("Target light files:", len(target_files))
for x in target_files[:10]:
    print(os.path.basename(x[0]), x[1], x[2], x[3])

def guess_cols(cols):
    lower = {c.lower(): c for c in cols}

    # lon / lat candidates
    lon_candidates = ["lon", "longitude", "x", "center_lon", "grid_lon"]
    lat_candidates = ["lat", "latitude", "y", "center_lat", "grid_lat"]

    lon_col = None
    lat_col = None

    for c in lon_candidates:
        if c in lower:
            lon_col = lower[c]
            break

    for c in lat_candidates:
        if c in lower:
            lat_col = lower[c]
            break

    # light candidates
    light_keywords = [
        "ntl", "light", "avg_rad", "radiance", "viirs", "night",
        "mean", "b1", "value"
    ]

    exclude_keywords = [
        "lon", "lat", "year", "region", "grid", "row", "col",
        "system", "index", "id"
    ]

    light_candidates = []
    for c in cols:
        cl = c.lower()
        if any(k in cl for k in light_keywords) and not any(e in cl for e in exclude_keywords):
            light_candidates.append(c)

    # fallback: numeric columns excluding coordinates/id
    return lon_col, lat_col, light_candidates

parts = []

for fp, year, region_id, typ in target_files:
    try:
        tmp = pd.read_csv(fp)
    except Exception as e:
        print("[READ FAIL]", os.path.basename(fp), e)
        continue

    lon_col, lat_col, light_candidates = guess_cols(tmp.columns.tolist())

    print("\nFile:", os.path.basename(fp))
    print("shape:", tmp.shape)
    print("lon_col:", lon_col, "lat_col:", lat_col)
    print("light_candidates:", light_candidates)

    if lon_col is None or lat_col is None:
        print("[SKIP] No lon/lat columns found.")
        continue

    # Choose light column
    if len(light_candidates) == 0:
        # fallback numeric columns
        numeric_cols = tmp.select_dtypes(include=[np.number]).columns.tolist()
        bad = {lon_col, lat_col}
        numeric_cols = [c for c in numeric_cols if c not in bad and "year" not in c.lower()]
        print("Fallback numeric cols:", numeric_cols)
        if len(numeric_cols) == 0:
            print("[SKIP] No candidate light column.")
            continue
        light_col = numeric_cols[0]
    else:
        # choose first candidate; after inspection we can refine
        light_col = light_candidates[0]

    use = tmp[[lon_col, lat_col, light_col]].copy()
    use = use.rename(columns={lon_col: "lon", lat_col: "lat", light_col: "NTL_raw"})
    use["year"] = year
    use["region_id_file"] = region_id
    use["source_type"] = typ
    use["source_file"] = os.path.basename(fp)

    use["lon_round"] = pd.to_numeric(use["lon"], errors="coerce").round(2)
    use["lat_round"] = pd.to_numeric(use["lat"], errors="coerce").round(2)
    use["NTL_raw"] = pd.to_numeric(use["NTL_raw"], errors="coerce")

    parts.append(use)

if len(parts) == 0:
    raise RuntimeError("No usable light files found. Please inspect columns from Step 21A/21B.")

light_raw = pd.concat(parts, ignore_index=True)
print("\nRaw light rows:", light_raw.shape)
display(light_raw.head())

# Match to current China grid by rounded lon/lat
light_grid = light_raw.merge(
    grid_keep[["grid_id", "lon_round", "lat_round"]],
    on=["lon_round", "lat_round"],
    how="inner"
)

print("Matched light rows:", light_grid.shape)
print("Matched unique grids:", light_grid["grid_id"].nunique())

# Aggregate if duplicates
ntl = (
    light_grid
    .groupby(["grid_id", "year"], as_index=False)
    .agg(
        NTL=("NTL_raw", "mean"),
        n_light_records=("NTL_raw", "count")
    )
)

ntl["NTL"] = ntl["NTL"].clip(lower=0)
ntl["log_NTL"] = np.log1p(ntl["NTL"])

out_ntl = os.path.join(AQUA_DIR, "china_grid_year_NTL_from_existing_LightRaw_2018_2023.csv")
ntl.to_csv(out_ntl, index=False)

print("NTL table:", ntl.shape)
print("Saved:", out_ntl)
display(ntl.head())
display(ntl["NTL"].describe())

# Merge with panel
panel["year"] = pd.to_numeric(panel["year"], errors="coerce").astype(int)

panel_ntl = panel.merge(
    ntl,
    on=["grid_id", "year"],
    how="left",
    validate="many_to_one"
)

out_panel = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2_with_existing_NTL.parquet")
panel_ntl.to_parquet(out_panel, index=False)

print("Panel with existing NTL:", panel_ntl.shape)
print("NTL valid frac:", panel_ntl["NTL"].notna().mean())
print("Saved:", out_panel)

display(panel_ntl[["grid_id", "year", "lon", "lat", "NTL", "log_NTL", "HDSI", "dist_perm_water_km"]].head())

In [ ]:
# ============================================================
# Step A FIXED. Nonlinear spline logit model for aquatic buffering
# Fix:
#   1) Explicit lower_bound / upper_bound for bs()
#   2) Prediction distance constrained within observed training range
# Output:
#   nonlinear_spline_permwater_predictions.csv
#   nonlinear_spline_EVI_permwater_CI.png
#   nonlinear_spline_LAI_permwater_CI.png
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_threshold")
os.makedirs(FIG_DIR, exist_ok=True)

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
df = pd.read_parquet(PANEL_FP)

# ----------------------------
# 1. Prepare data
# ----------------------------
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

num_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]

for c in num_vars:
    d[c] = pd.to_numeric(d[c], errors="coerce")

d["log_dist_perm"] = np.log1p(d["dist_perm_water_km"])

d["Tair_c"] = d["summer_Tair_z"] - d["summer_Tair_z"].mean()
d["elevation_c"] = d["elevation"] - d["elevation"].mean()
d["slope_c"] = d["slope"] - d["slope"].mean()

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round()
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce")

model_vars = [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "log_dist_perm", "dist_perm_water_km",
    "Tair_c", "elevation_c", "slope_c",
    "LC_Type1_cat", "year_cat"
]

d = d.dropna(subset=model_vars).copy()

d["LC_Type1_cat"] = d["LC_Type1_cat"].astype(int).astype(str)
d["year_cat"] = d["year_cat"].astype(int).astype(str)
d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)

# ----------------------------
# 2. Sample for fitting
# ----------------------------
MAX_N = 200000
if len(d) > MAX_N:
    d_fit = d.sample(MAX_N, random_state=42).copy()
else:
    d_fit = d.copy()

print("Fit samples:", len(d_fit))
print("Unique grids:", d_fit["grid_id"].nunique())
print("Observed permanent-water distance range:")
print(d_fit["dist_perm_water_km"].min(), d_fit["dist_perm_water_km"].max())

# ----------------------------
# 3. HDSI groups
# ----------------------------
q25 = d_fit["HDSI"].quantile(0.25)
q75 = d_fit["HDSI"].quantile(0.75)

d_fit["HDSI_group"] = "mid"
d_fit.loc[d_fit["HDSI"] <= q25, "HDSI_group"] = "low"
d_fit.loc[d_fit["HDSI"] >= q75, "HDSI_group"] = "high"

d_fit["HDSI_group"] = pd.Categorical(
    d_fit["HDSI_group"],
    categories=["low", "mid", "high"],
    ordered=True
)

# ----------------------------
# 4. Explicit spline bounds
# ----------------------------
log_lower = float(d_fit["log_dist_perm"].min())
log_upper = float(d_fit["log_dist_perm"].max())

dist_lower = float(d_fit["dist_perm_water_km"].min())
dist_upper = float(d_fit["dist_perm_water_km"].max())

print("Spline log-distance bounds:", log_lower, log_upper)
print("Spline distance bounds:", dist_lower, dist_upper)

# Important:
# Use explicit lower_bound and upper_bound to avoid prediction outside knots.
spline_term = (
    f"bs(log_dist_perm, df=5, degree=3, "
    f"lower_bound={log_lower}, upper_bound={log_upper})"
)

formula_rhs = (
    f"{spline_term} * C(HDSI_group) "
    "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
)

# ----------------------------
# 5. Fit models
# ----------------------------
models = {}

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    formula = f"{target} ~ {formula_rhs}"
    print("\n" + "=" * 100)
    print(target)
    print(formula)

    model = smf.logit(formula, data=d_fit).fit(maxiter=300, disp=False)
    models[target] = model
    print(model.summary().tables[1])

# ----------------------------
# 6. Prediction curves
# ----------------------------
# Fine curve:
# 0-5 km every 0.05 km, then 5 km to observed max every 0.25 km.
dmax = min(16.0, dist_upper)

dist_values = np.r_[
    np.arange(0, min(5.0, dmax) + 1e-6, 0.05),
    np.arange(5.25, dmax + 1e-6, 0.25)
]

# Make sure all prediction distances are within bounds
dist_values = dist_values[(dist_values >= dist_lower) & (dist_values <= dist_upper)]
dist_values = np.unique(np.round(dist_values, 6))

log_dist_values = np.log1p(dist_values)

base_lc = str(d_fit["LC_Type1_cat"].mode().iloc[0])
base_year = str(d_fit["year_cat"].mode().iloc[0])

pred_rows = []

for group in ["low", "mid", "high"]:
    for dist, logd in zip(dist_values, log_dist_values):
        pred_rows.append({
            "dist_perm_water_km": dist,
            "log_dist_perm": logd,
            "HDSI_group": group,
            "Tair_c": 0.0,
            "elevation_c": 0.0,
            "slope_c": 0.0,
            "LC_Type1_cat": base_lc,
            "year_cat": base_year
        })

pred_df = pd.DataFrame(pred_rows)
pred_df["HDSI_group"] = pd.Categorical(
    pred_df["HDSI_group"],
    categories=["low", "mid", "high"],
    ordered=True
)

# ----------------------------
# 7. Prediction with CI
# ----------------------------
for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    model = models[target]

    sf = model.get_prediction(pred_df).summary_frame(alpha=0.05)

    pred_df[f"{target}_prob"] = sf["mean"].values
    pred_df[f"{target}_lower"] = sf["mean_ci_lower"].values
    pred_df[f"{target}_upper"] = sf["mean_ci_upper"].values

out_csv = os.path.join(AQUA_DIR, "nonlinear_spline_permwater_predictions.csv")
pred_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

# ----------------------------
# 8. Plot
# ----------------------------
def plot_spline(target, ylabel, out_name):
    prob_col = f"{target}_prob"
    low_col = f"{target}_lower"
    high_col = f"{target}_upper"

    plt.figure(figsize=(8.5, 5.5))

    label_map = {
        "low": "Low HDSI",
        "mid": "Median HDSI",
        "high": "High HDSI"
    }

    for group in ["low", "mid", "high"]:
        sub = pred_df[pred_df["HDSI_group"] == group].copy()
        x = sub["dist_perm_water_km"].values.astype(float)
        y = sub[prob_col].values.astype(float)
        lo = sub[low_col].values.astype(float)
        hi = sub[high_col].values.astype(float)

        plt.plot(x, y, linewidth=2.2, label=label_map[group])
        plt.fill_between(x, lo, hi, alpha=0.18)

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(ylabel + " along nonlinear aquatic-buffering gradient")
    plt.xlim(0, dmax)
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

plot_spline(
    "EVI_neg_z05",
    "Predicted EVI negative-response risk",
    "nonlinear_spline_EVI_permwater_CI.png"
)

plot_spline(
    "LAI_neg_z05",
    "Predicted LAI negative-response risk",
    "nonlinear_spline_LAI_permwater_CI.png"
)

In [ ]:
# ============================================================
# Step A Prediction FIX.
# Manually compute predicted probability and 95% CI
# This avoids summary_frame column-name issues.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import patsy

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_threshold")
os.makedirs(FIG_DIR, exist_ok=True)

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def add_prediction_ci(model, pred_df, prefix):
    """
    Compute predicted probability and 95% CI manually.
    Works for statsmodels logit models with formula design matrices.
    """
    design_info = model.model.data.design_info

    Xp = patsy.build_design_matrices(
        [design_info],
        pred_df,
        return_type="dataframe"
    )[0]

    # Align columns with model params
    Xp = Xp[model.params.index]

    beta = model.params.values
    cov = model.cov_params().values

    eta = np.asarray(Xp) @ beta

    # Variance of linear predictor: diag(X cov X')
    eta_var = np.einsum("ij,jk,ik->i", np.asarray(Xp), cov, np.asarray(Xp))
    eta_se = np.sqrt(np.maximum(eta_var, 0))

    eta_low = eta - 1.96 * eta_se
    eta_high = eta + 1.96 * eta_se

    pred_df[f"{prefix}_prob"] = sigmoid(eta)
    pred_df[f"{prefix}_lower"] = sigmoid(eta_low)
    pred_df[f"{prefix}_upper"] = sigmoid(eta_high)

    return pred_df

# Add prediction intervals
pred_df = add_prediction_ci(models["EVI_neg_z05"], pred_df, "EVI_neg_z05")
pred_df = add_prediction_ci(models["LAI_neg_z05"], pred_df, "LAI_neg_z05")

out_csv = os.path.join(AQUA_DIR, "nonlinear_spline_permwater_predictions.csv")
pred_df.to_csv(out_csv, index=False)
print("Saved:", out_csv)

display(pred_df.head())
print(pred_df.columns.tolist())

# ============================================================
# Plot nonlinear spline curves
# ============================================================

def plot_spline(target, ylabel, out_name):
    prob_col = f"{target}_prob"
    low_col = f"{target}_lower"
    high_col = f"{target}_upper"

    plt.figure(figsize=(8.5, 5.5))

    label_map = {
        "low": "Low HDSI",
        "mid": "Median HDSI",
        "high": "High HDSI"
    }

    for group in ["low", "mid", "high"]:
        sub = pred_df[pred_df["HDSI_group"].astype(str) == group].copy()

        x = sub["dist_perm_water_km"].values.astype(float)
        y = sub[prob_col].values.astype(float)
        lo = sub[low_col].values.astype(float)
        hi = sub[high_col].values.astype(float)

        plt.plot(x, y, linewidth=2.2, label=label_map[group])
        plt.fill_between(x, lo, hi, alpha=0.18)

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(ylabel + " along nonlinear aquatic-buffering gradient")
    plt.xlim(0, pred_df["dist_perm_water_km"].max())
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, out_name)
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

plot_spline(
    "EVI_neg_z05",
    "Predicted EVI negative-response risk",
    "nonlinear_spline_EVI_permwater_CI.png"
)

plot_spline(
    "LAI_neg_z05",
    "Predicted LAI negative-response risk",
    "nonlinear_spline_LAI_permwater_CI.png"
)

In [ ]:
# ============================================================
# Step B. Extract effective aquatic buffering distance thresholds
# From nonlinear spline prediction curves
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

PRED_FP = os.path.join(AQUA_DIR, "nonlinear_spline_permwater_predictions.csv")
pred = pd.read_csv(PRED_FP)

def extract_thresholds(curve, prob_col, dist_col="dist_perm_water_km", risk_cut=0.35):
    c = curve.sort_values(dist_col).copy()
    x = c[dist_col].values.astype(float)
    y = c[prob_col].values.astype(float)

    # 去除 nan
    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 5:
        return {
            "P_near": np.nan,
            "P_far": np.nan,
            "EABD50": np.nan,
            "EABD_slope": np.nan,
            "EABD_risk035": np.nan,
            "max_slope": np.nan
        }

    p_near = y[0]
    p_far = y[-1]

    # EABD50：达到风险差异一半的位置
    target = p_near + 0.5 * (p_far - p_near)

    if p_far >= p_near:
        idx50 = np.where(y >= target)[0]
    else:
        idx50 = np.where(y <= target)[0]

    eabd50 = x[idx50[0]] if len(idx50) > 0 else np.nan

    # EABD_slope：风险增速最大处
    dydx = np.gradient(y, x)
    idx_slope = np.nanargmax(dydx)
    eabd_slope = x[idx_slope]
    max_slope = dydx[idx_slope]

    # 风险超过 0.35 的位置
    idx_risk = np.where(y >= risk_cut)[0]
    eabd_risk = x[idx_risk[0]] if len(idx_risk) > 0 else np.nan

    return {
        "P_near": p_near,
        "P_far": p_far,
        "EABD50": eabd50,
        "EABD_slope": eabd_slope,
        "EABD_risk035": eabd_risk,
        "max_slope": max_slope
    }

rows = []

for target, prob_col in [
    ("EVI", "EVI_neg_z05_prob"),
    ("LAI", "LAI_neg_z05_prob")
]:
    for group in ["low", "mid", "high"]:
        sub = pred[pred["HDSI_group"] == group].copy()
        out = extract_thresholds(sub, prob_col, risk_cut=0.35)
        out["target"] = target
        out["HDSI_group"] = group
        rows.append(out)

thresholds = pd.DataFrame(rows)
thresholds = thresholds[[
    "target", "HDSI_group",
    "P_near", "P_far",
    "EABD50", "EABD_slope", "EABD_risk035",
    "max_slope"
]]

display(thresholds)

out_fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_HDSI_permwater.csv")
thresholds.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step C. Interannual shifts in effective aquatic buffering distance
# Fit spline models by year and extract EABD thresholds
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d0 = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

for c in [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]:
    d0[c] = pd.to_numeric(d0[c], errors="coerce")

d0["log_dist_perm"] = np.log1p(d0["dist_perm_water_km"])
d0["LC_Type1_cat"] = pd.to_numeric(d0["LC_Type1"], errors="coerce").round()

d0 = d0.dropna(subset=[
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "log_dist_perm",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1_cat", "year"
]).copy()

d0["LC_Type1_cat"] = d0["LC_Type1_cat"].astype(int).astype(str)
d0["EVI_neg_z05"] = d0["EVI_neg_z05"].astype(int)
d0["LAI_neg_z05"] = d0["LAI_neg_z05"].astype(int)
d0["year"] = d0["year"].astype(int)

def extract_thresholds_from_xy(x, y, risk_cut=0.35):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 5:
        return np.nan, np.nan, np.nan

    p_near = y[0]
    p_far = y[-1]
    target = p_near + 0.5 * (p_far - p_near)

    if p_far >= p_near:
        idx50 = np.where(y >= target)[0]
    else:
        idx50 = np.where(y <= target)[0]
    eabd50 = x[idx50[0]] if len(idx50) > 0 else np.nan

    dydx = np.gradient(y, x)
    eabd_slope = x[np.nanargmax(dydx)]

    idx_risk = np.where(y >= risk_cut)[0]
    eabd_risk = x[idx_risk[0]] if len(idx_risk) > 0 else np.nan

    return eabd50, eabd_slope, eabd_risk

years = sorted(d0["year"].unique())
rows = []

dist_values = np.r_[np.arange(0, 5.05, 0.05), np.arange(5.25, 16.01, 0.25)]
log_dist_values = np.log1p(dist_values)

for year in years:
    dy = d0[d0["year"] == year].copy()

    if len(dy) < 5000:
        continue

    # 年份内中心化
    dy["HDSI_c"] = dy["HDSI"] - dy["HDSI"].mean()
    dy["Tair_c"] = dy["summer_Tair_z"] - dy["summer_Tair_z"].mean()
    dy["elevation_c"] = dy["elevation"] - dy["elevation"].mean()
    dy["slope_c"] = dy["slope"] - dy["slope"].mean()

    # HDSI 分组
    q25 = dy["HDSI"].quantile(0.25)
    q75 = dy["HDSI"].quantile(0.75)
    dy["HDSI_group"] = "mid"
    dy.loc[dy["HDSI"] <= q25, "HDSI_group"] = "low"
    dy.loc[dy["HDSI"] >= q75, "HDSI_group"] = "high"
    dy["HDSI_group"] = pd.Categorical(dy["HDSI_group"], categories=["low", "mid", "high"], ordered=True)

    # 样本太大则抽样
    if len(dy) > 60000:
        dy_fit = dy.sample(60000, random_state=42)
    else:
        dy_fit = dy

    base_lc = str(dy_fit["LC_Type1_cat"].mode().iloc[0])

    pred_base = []
    for group in ["low", "mid", "high"]:
        for dist, logd in zip(dist_values, log_dist_values):
            pred_base.append({
                "dist_perm_water_km": dist,
                "log_dist_perm": logd,
                "HDSI_group": group,
                "Tair_c": 0.0,
                "elevation_c": 0.0,
                "slope_c": 0.0,
                "LC_Type1_cat": base_lc
            })
    pred_df = pd.DataFrame(pred_base)
    pred_df["HDSI_group"] = pd.Categorical(pred_df["HDSI_group"], categories=["low", "mid", "high"], ordered=True)

    formula_rhs = (
        "bs(log_dist_perm, df=5, degree=3) * C(HDSI_group) "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat)"
    )

    for target in ["EVI_neg_z05", "LAI_neg_z05"]:
        if dy_fit[target].nunique() < 2:
            continue

        try:
            model = smf.logit(f"{target} ~ {formula_rhs}", data=dy_fit).fit(maxiter=300, disp=False)
            pred_df[f"{target}_prob"] = model.predict(pred_df)

            for group in ["low", "mid", "high"]:
                sub = pred_df[pred_df["HDSI_group"] == group]
                eabd50, eabd_slope, eabd_risk = extract_thresholds_from_xy(
                    sub["dist_perm_water_km"],
                    sub[f"{target}_prob"],
                    risk_cut=0.35
                )

                rows.append({
                    "year": year,
                    "target": "EVI" if target == "EVI_neg_z05" else "LAI",
                    "HDSI_group": group,
                    "EABD50": eabd50,
                    "EABD_slope": eabd_slope,
                    "EABD_risk035": eabd_risk,
                    "year_HDSI_mean": dy["HDSI"].mean(),
                    "year_HDSI_q75": dy["HDSI"].quantile(0.75),
                    "n": len(dy_fit)
                })

        except Exception as e:
            print("[FAILED]", year, target, e)

year_thresholds = pd.DataFrame(rows)
display(year_thresholds)

out_fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_year_HDSI_permwater.csv")
year_thresholds.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step D. Regional EABD thresholds
# Nonlinear spline curves by macro-region
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d0 = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"]

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

d0["macro_region"] = d0.apply(assign_macro_region, axis=1)

for c in [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]:
    d0[c] = pd.to_numeric(d0[c], errors="coerce")

d0["log_dist_perm"] = np.log1p(d0["dist_perm_water_km"])
d0["LC_Type1_cat"] = pd.to_numeric(d0["LC_Type1"], errors="coerce").round()
d0["year_cat"] = pd.to_numeric(d0["year"], errors="coerce")

d0 = d0.dropna(subset=[
    "macro_region",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "log_dist_perm",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1_cat", "year_cat"
]).copy()

d0["LC_Type1_cat"] = d0["LC_Type1_cat"].astype(int).astype(str)
d0["year_cat"] = d0["year_cat"].astype(int).astype(str)
d0["EVI_neg_z05"] = d0["EVI_neg_z05"].astype(int)
d0["LAI_neg_z05"] = d0["LAI_neg_z05"].astype(int)

region_summary = (
    d0.groupby("macro_region")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        dist_perm_mean=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(region_summary)
region_summary.to_csv(os.path.join(AQUA_DIR, "macro_region_summary_threshold_v2.csv"), index=False)

def extract_thresholds_from_xy(x, y, risk_cut=0.35):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 5:
        return np.nan, np.nan, np.nan

    p_near = y[0]
    p_far = y[-1]
    target = p_near + 0.5 * (p_far - p_near)

    if p_far >= p_near:
        idx50 = np.where(y >= target)[0]
    else:
        idx50 = np.where(y <= target)[0]
    eabd50 = x[idx50[0]] if len(idx50) > 0 else np.nan

    dydx = np.gradient(y, x)
    eabd_slope = x[np.nanargmax(dydx)]

    idx_risk = np.where(y >= risk_cut)[0]
    eabd_risk = x[idx_risk[0]] if len(idx_risk) > 0 else np.nan

    return eabd50, eabd_slope, eabd_risk

dist_values = np.r_[np.arange(0, 5.05, 0.05), np.arange(5.25, 16.01, 0.25)]
log_dist_values = np.log1p(dist_values)

rows = []

for region in region_summary["macro_region"]:
    sub = d0[d0["macro_region"] == region].copy()

    if len(sub) < 8000:
        print("[Skip small region]", region, len(sub))
        continue

    sub["Tair_c"] = sub["summer_Tair_z"] - sub["summer_Tair_z"].mean()
    sub["elevation_c"] = sub["elevation"] - sub["elevation"].mean()
    sub["slope_c"] = sub["slope"] - sub["slope"].mean()

    q25 = sub["HDSI"].quantile(0.25)
    q75 = sub["HDSI"].quantile(0.75)
    sub["HDSI_group"] = "mid"
    sub.loc[sub["HDSI"] <= q25, "HDSI_group"] = "low"
    sub.loc[sub["HDSI"] >= q75, "HDSI_group"] = "high"
    sub["HDSI_group"] = pd.Categorical(sub["HDSI_group"], categories=["low", "mid", "high"], ordered=True)

    if len(sub) > 60000:
        sub_fit = sub.sample(60000, random_state=42)
    else:
        sub_fit = sub

    base_lc = str(sub_fit["LC_Type1_cat"].mode().iloc[0])
    base_year = str(sub_fit["year_cat"].mode().iloc[0])

    pred_rows = []
    for group in ["low", "mid", "high"]:
        for dist, logd in zip(dist_values, log_dist_values):
            pred_rows.append({
                "dist_perm_water_km": dist,
                "log_dist_perm": logd,
                "HDSI_group": group,
                "Tair_c": 0.0,
                "elevation_c": 0.0,
                "slope_c": 0.0,
                "LC_Type1_cat": base_lc,
                "year_cat": base_year
            })

    pred_df = pd.DataFrame(pred_rows)
    pred_df["HDSI_group"] = pd.Categorical(pred_df["HDSI_group"], categories=["low", "mid", "high"], ordered=True)

    formula_rhs = (
        "bs(log_dist_perm, df=5, degree=3) * C(HDSI_group) "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    )

    for target in ["EVI_neg_z05", "LAI_neg_z05"]:
        if sub_fit[target].nunique() < 2:
            continue

        try:
            model = smf.logit(f"{target} ~ {formula_rhs}", data=sub_fit).fit(maxiter=300, disp=False)
            pred_df["prob"] = model.predict(pred_df)

            for group in ["low", "mid", "high"]:
                pp = pred_df[pred_df["HDSI_group"] == group]
                eabd50, eabd_slope, eabd_risk = extract_thresholds_from_xy(
                    pp["dist_perm_water_km"],
                    pp["prob"],
                    risk_cut=0.35
                )

                rows.append({
                    "macro_region": region,
                    "target": "EVI" if target == "EVI_neg_z05" else "LAI",
                    "HDSI_group": group,
                    "EABD50": eabd50,
                    "EABD_slope": eabd_slope,
                    "EABD_risk035": eabd_risk,
                    "region_HDSI_mean": sub["HDSI"].mean(),
                    "region_dist_perm_mean": sub["dist_perm_water_km"].mean(),
                    "n": len(sub_fit)
                })

        except Exception as e:
            print("[FAILED]", region, target, e)

region_thresholds = pd.DataFrame(rows)
display(region_thresholds)

out_fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater.csv")
region_thresholds.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step E. Climate adaptation priority zones
# Based on high heat-dryness exposure + frequent buffering failure
# ============================================================

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_threshold")
os.makedirs(FIG_DIR, exist_ok=True)

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

h75 = d["HDSI"].quantile(0.75)

d["high_HDSI"] = (d["HDSI"] >= h75).astype(int)
d["near_perm_1km"] = (d["dist_perm_water_km"] < 1.0).astype(int)
d["near_perm_2km"] = (d["dist_perm_water_km"] < 2.0).astype(int)

# 缓冲失效：近永久水体 + 高 HDSI + 植被负响应
d["EVI_failure_near_perm"] = (
    (d["near_perm_1km"] == 1) &
    (d["high_HDSI"] == 1) &
    (d["EVI_neg_z05"] == 1)
).astype(int)

d["LAI_failure_near_perm"] = (
    (d["near_perm_1km"] == 1) &
    (d["high_HDSI"] == 1) &
    (d["LAI_neg_z05"] == 1)
).astype(int)

grid_risk = (
    d.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        HDSI_mean=("HDSI", "mean"),
        high_HDSI_freq=("high_HDSI", "mean"),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        near_perm_1km=("near_perm_1km", "max"),
        near_perm_2km=("near_perm_2km", "max"),
        EVI_failure_freq=("EVI_failure_near_perm", "mean"),
        LAI_failure_freq=("LAI_failure_near_perm", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
)

# 构建适应优先指数：
# 高热旱频率 + 近水体仍失效 + 植被负响应
for col in ["high_HDSI_freq", "EVI_failure_freq", "LAI_failure_freq", "EVI_neg_rate", "LAI_neg_rate"]:
    v = grid_risk[col]
    grid_risk[col + "_z01"] = (v - v.min()) / (v.max() - v.min() + 1e-9)

grid_risk["adaptation_priority_index"] = (
    0.30 * grid_risk["high_HDSI_freq_z01"] +
    0.25 * grid_risk["EVI_failure_freq_z01"] +
    0.25 * grid_risk["LAI_failure_freq_z01"] +
    0.10 * grid_risk["EVI_neg_rate_z01"] +
    0.10 * grid_risk["LAI_neg_rate_z01"]
)

out_csv = os.path.join(AQUA_DIR, "adaptation_priority_grid_risk_v2.csv")
grid_risk.to_csv(out_csv, index=False)
print("Saved:", out_csv)
display(grid_risk.head())

# 只画近永久水体 2km 内的适应优先区
plot_df = grid_risk[grid_risk["near_perm_2km"] == 1].copy()

try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
china = china.dissolve().reset_index(drop=True)

gdf = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(plot_df["lon"], plot_df["lat"]),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(9, 7))
china.boundary.plot(ax=ax, linewidth=0.7, color="black")

gdf.plot(
    ax=ax,
    column="adaptation_priority_index",
    markersize=1.4,
    cmap="magma",
    vmin=0,
    vmax=1,
    legend=True,
    legend_kwds={"shrink": 0.65, "label": "Adaptation priority index"}
)

ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Climate adaptation priority near permanent water")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linewidth=0.4)
plt.tight_layout()

out_fp = os.path.join(FIG_DIR, "map_adaptation_priority_near_permwater.png")
plt.savefig(out_fp, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step B2. Robust EABD extraction
# Add gradient-strength filter to avoid meaningless thresholds
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

PRED_FP = os.path.join(AQUA_DIR, "nonlinear_spline_permwater_predictions.csv")
pred = pd.read_csv(PRED_FP)

def first_crossing_distance(x, y, risk_cut):
    idx = np.where(y >= risk_cut)[0]
    return x[idx[0]] if len(idx) > 0 else np.nan

def extract_thresholds_robust(curve, prob_col, dist_col="dist_perm_water_km", min_delta=0.05):
    c = curve.sort_values(dist_col).copy()

    x = c[dist_col].values.astype(float)
    y = c[prob_col].values.astype(float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 5:
        return None

    p_near = y[0]
    p_far = y[-1]
    delta = p_far - p_near

    dydx = np.gradient(y, x)
    idx_slope = np.nanargmax(dydx)

    # 判断是否存在清晰缓冲衰减
    clear_gradient = delta >= min_delta

    if clear_gradient:
        target = p_near + 0.5 * delta
        idx50 = np.where(y >= target)[0]
        eabd50 = x[idx50[0]] if len(idx50) > 0 else np.nan
    else:
        eabd50 = np.nan

    return {
        "P_near": p_near,
        "P_far": p_far,
        "delta_P_far_near": delta,
        "clear_buffering_gradient": clear_gradient,
        "EABD50": eabd50,
        "EABD_slope": x[idx_slope],
        "max_slope": dydx[idx_slope],
        "EABD_risk030": first_crossing_distance(x, y, 0.30),
        "EABD_risk035": first_crossing_distance(x, y, 0.35),
        "EABD_risk040": first_crossing_distance(x, y, 0.40)
    }

rows = []

for target, prob_col in [
    ("EVI", "EVI_neg_z05_prob"),
    ("LAI", "LAI_neg_z05_prob")
]:
    for group in ["low", "mid", "high"]:
        sub = pred[pred["HDSI_group"].astype(str) == group].copy()
        out = extract_thresholds_robust(sub, prob_col, min_delta=0.05)

        if out is not None:
            out["target"] = target
            out["HDSI_group"] = group
            rows.append(out)

thr = pd.DataFrame(rows)

thr = thr[[
    "target", "HDSI_group",
    "P_near", "P_far", "delta_P_far_near",
    "clear_buffering_gradient",
    "EABD50", "EABD_slope", "max_slope",
    "EABD_risk030", "EABD_risk035", "EABD_risk040"
]]

display(thr)

out_fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_HDSI_permwater_robust.csv")
thr.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step D FIXED. Regional EABD thresholds with region-specific spline bounds
# ============================================================

import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d0 = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"]

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

d0["macro_region"] = d0.apply(assign_macro_region, axis=1)

for c in [
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1", "year"
]:
    d0[c] = pd.to_numeric(d0[c], errors="coerce")

d0["log_dist_perm"] = np.log1p(d0["dist_perm_water_km"])
d0["LC_Type1_cat"] = pd.to_numeric(d0["LC_Type1"], errors="coerce").round()
d0["year_cat"] = pd.to_numeric(d0["year"], errors="coerce")

d0 = d0.dropna(subset=[
    "macro_region",
    "EVI_neg_z05", "LAI_neg_z05",
    "HDSI", "dist_perm_water_km", "log_dist_perm",
    "summer_Tair_z", "elevation", "slope",
    "LC_Type1_cat", "year_cat"
]).copy()

d0["LC_Type1_cat"] = d0["LC_Type1_cat"].astype(int).astype(str)
d0["year_cat"] = d0["year_cat"].astype(int).astype(str)
d0["EVI_neg_z05"] = d0["EVI_neg_z05"].astype(int)
d0["LAI_neg_z05"] = d0["LAI_neg_z05"].astype(int)

region_summary = (
    d0.groupby("macro_region")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        dist_min=("dist_perm_water_km", "min"),
        dist_max=("dist_perm_water_km", "max"),
        dist_mean=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
    .reset_index()
    .sort_values("rows", ascending=False)
)

display(region_summary)
region_summary.to_csv(
    os.path.join(AQUA_DIR, "macro_region_summary_threshold_fixed.csv"),
    index=False
)

def first_crossing_distance(x, y, risk_cut):
    idx = np.where(y >= risk_cut)[0]
    return x[idx[0]] if len(idx) > 0 else np.nan

def extract_thresholds_from_xy(x, y, min_delta=0.05):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]

    if len(x) < 5:
        return None

    p_near = y[0]
    p_far = y[-1]
    delta = p_far - p_near

    dydx = np.gradient(y, x)
    idx_slope = np.nanargmax(dydx)

    clear_gradient = delta >= min_delta

    if clear_gradient:
        target = p_near + 0.5 * delta
        idx50 = np.where(y >= target)[0]
        eabd50 = x[idx50[0]] if len(idx50) > 0 else np.nan
    else:
        eabd50 = np.nan

    return {
        "P_near": p_near,
        "P_far": p_far,
        "delta_P_far_near": delta,
        "clear_buffering_gradient": clear_gradient,
        "EABD50": eabd50,
        "EABD_slope": x[idx_slope],
        "max_slope": dydx[idx_slope],
        "EABD_risk030": first_crossing_distance(x, y, 0.30),
        "EABD_risk035": first_crossing_distance(x, y, 0.35),
        "EABD_risk040": first_crossing_distance(x, y, 0.40)
    }

rows = []

for region in region_summary["macro_region"]:
    sub = d0[d0["macro_region"] == region].copy()

    if len(sub) < 8000:
        print("[Skip small region]", region, len(sub))
        continue

    print("\n" + "=" * 120)
    print("Region:", region, "n=", len(sub))

    sub["Tair_c"] = sub["summer_Tair_z"] - sub["summer_Tair_z"].mean()
    sub["elevation_c"] = sub["elevation"] - sub["elevation"].mean()
    sub["slope_c"] = sub["slope"] - sub["slope"].mean()

    q25 = sub["HDSI"].quantile(0.25)
    q75 = sub["HDSI"].quantile(0.75)

    sub["HDSI_group"] = "mid"
    sub.loc[sub["HDSI"] <= q25, "HDSI_group"] = "low"
    sub.loc[sub["HDSI"] >= q75, "HDSI_group"] = "high"

    sub["HDSI_group"] = pd.Categorical(
        sub["HDSI_group"],
        categories=["low", "mid", "high"],
        ordered=True
    )

    if len(sub) > 60000:
        sub_fit = sub.sample(60000, random_state=42).copy()
    else:
        sub_fit = sub.copy()

    # region-specific spline bounds
    log_lower = float(sub_fit["log_dist_perm"].min())
    log_upper = float(sub_fit["log_dist_perm"].max())
    dist_lower = float(sub_fit["dist_perm_water_km"].min())
    dist_upper = float(sub_fit["dist_perm_water_km"].max())

    if log_upper <= log_lower or dist_upper <= dist_lower:
        print("[Skip invalid distance range]", region)
        continue

    spline_term = (
        f"bs(log_dist_perm, df=4, degree=3, "
        f"lower_bound={log_lower}, upper_bound={log_upper})"
    )

    formula_rhs = (
        f"{spline_term} * C(HDSI_group) "
        "+ Tair_c + elevation_c + slope_c + C(LC_Type1_cat) + C(year_cat)"
    )

    # prediction grid within region-specific observed range
    dmax = dist_upper
    dmin = dist_lower

    if dmax <= 5:
        dist_values = np.arange(dmin, dmax + 1e-6, 0.05)
    else:
        dist_values = np.r_[
            np.arange(dmin, min(5.0, dmax) + 1e-6, 0.05),
            np.arange(5.25, dmax + 1e-6, 0.25)
        ]

    dist_values = np.unique(np.round(dist_values, 6))
    log_dist_values = np.log1p(dist_values)

    base_lc = str(sub_fit["LC_Type1_cat"].mode().iloc[0])
    base_year = str(sub_fit["year_cat"].mode().iloc[0])

    pred_rows = []
    for group in ["low", "mid", "high"]:
        for dist, logd in zip(dist_values, log_dist_values):
            pred_rows.append({
                "dist_perm_water_km": dist,
                "log_dist_perm": logd,
                "HDSI_group": group,
                "Tair_c": 0.0,
                "elevation_c": 0.0,
                "slope_c": 0.0,
                "LC_Type1_cat": base_lc,
                "year_cat": base_year
            })

    pred_df = pd.DataFrame(pred_rows)
    pred_df["HDSI_group"] = pd.Categorical(
        pred_df["HDSI_group"],
        categories=["low", "mid", "high"],
        ordered=True
    )

    for target in ["EVI_neg_z05", "LAI_neg_z05"]:
        if sub_fit[target].nunique() < 2:
            print("[Skip one-class target]", region, target)
            continue

        try:
            formula = f"{target} ~ {formula_rhs}"
            model = smf.logit(formula, data=sub_fit).fit(maxiter=300, disp=False)

            pred_df["prob"] = model.predict(pred_df)

            for group in ["low", "mid", "high"]:
                pp = pred_df[pred_df["HDSI_group"].astype(str) == group].copy()

                out = extract_thresholds_from_xy(
                    pp["dist_perm_water_km"],
                    pp["prob"],
                    min_delta=0.05
                )

                if out is None:
                    continue

                out.update({
                    "macro_region": region,
                    "target": "EVI" if target == "EVI_neg_z05" else "LAI",
                    "HDSI_group": group,
                    "region_n": len(sub_fit),
                    "region_dist_min": dist_lower,
                    "region_dist_max": dist_upper,
                    "region_HDSI_mean": sub["HDSI"].mean()
                })

                rows.append(out)

        except Exception as e:
            print("[FAILED]", region, target, e)

region_thresholds = pd.DataFrame(rows)

display(region_thresholds)

out_fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
region_thresholds.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
thr = pd.read_csv(fp)

# 只保留有清晰缓冲梯度的结果
clear = thr[thr["clear_buffering_gradient"] == True].copy()

# 优先看 high HDSI
clear_high = clear[clear["HDSI_group"] == "high"].copy()

clear_high = clear_high.sort_values(["target", "EABD50"])

display(clear_high[[
    "macro_region", "target", "HDSI_group",
    "P_near", "P_far", "delta_P_far_near",
    "EABD50", "EABD_risk035", "EABD_risk040",
    "region_dist_max", "region_n"
]])

out_fp = os.path.join(AQUA_DIR, "interpretable_EABD_high_HDSI_by_region.csv")
clear_high.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# 汇总每个指标的阈值范围
summary = (
    clear_high
    .groupby("target")
    .agg(
        n_region=("macro_region", "nunique"),
        EABD50_min=("EABD50", "min"),
        EABD50_median=("EABD50", "median"),
        EABD50_max=("EABD50", "max"),
        delta_mean=("delta_P_far_near", "mean")
    )
    .reset_index()
)

display(summary)

out_fp = os.path.join(AQUA_DIR, "interpretable_EABD_high_HDSI_summary.csv")
summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FIG_DIR = os.path.join(AQUA_DIR, "figs_threshold")
os.makedirs(FIG_DIR, exist_ok=True)

fp = os.path.join(AQUA_DIR, "interpretable_EABD_high_HDSI_by_region.csv")
clear_high = pd.read_csv(fp)

for target in ["EVI", "LAI"]:
    sub = clear_high[clear_high["target"] == target].copy()
    sub = sub.sort_values("EABD50")

    plt.figure(figsize=(8, 5))
    plt.barh(sub["macro_region"], sub["EABD50"])
    plt.xlabel("Effective aquatic buffering distance, EABD50 (km)")
    plt.ylabel("Macro-region")
    plt.title(f"Regional EABD50 under high HDSI ({target})")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, f"regional_EABD50_high_HDSI_{target}.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 1. Clean interpretable EABD results
# Focus on clear buffering gradients only
# ============================================================

import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
os.makedirs(OUT_DIR, exist_ok=True)

fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
thr = pd.read_csv(fp)

print("Raw threshold table:", thr.shape)
display(thr.head())

# Keep only interpretable gradients
clear = thr[thr["clear_buffering_gradient"] == True].copy()

# Main focus: high HDSI
clear_high = clear[clear["HDSI_group"] == "high"].copy()

# Useful interpretation columns
cols = [
    "macro_region", "target", "HDSI_group",
    "P_near", "P_far", "delta_P_far_near",
    "EABD50", "EABD_slope", "EABD_risk030", "EABD_risk035", "EABD_risk040",
    "max_slope", "region_dist_max", "region_n", "region_HDSI_mean"
]
cols = [c for c in cols if c in clear_high.columns]

clear_high = clear_high[cols].sort_values(["target", "EABD50"])

display(clear_high)

out_fp = os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_by_region.csv")
clear_high.to_csv(out_fp, index=False)
print("Saved:", out_fp)

# Summary by target
summary = (
    clear_high
    .groupby("target")
    .agg(
        n_region=("macro_region", "nunique"),
        EABD50_min=("EABD50", "min"),
        EABD50_median=("EABD50", "median"),
        EABD50_max=("EABD50", "max"),
        delta_mean=("delta_P_far_near", "mean"),
        delta_median=("delta_P_far_near", "median")
    )
    .reset_index()
)

display(summary)

out_fp = os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_summary.csv")
summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 2. Plot regional EABD50 under high HDSI
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

fp = os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_by_region.csv")
clear_high = pd.read_csv(fp)

for target in ["EVI", "LAI"]:
    sub = clear_high[clear_high["target"] == target].copy()
    sub = sub.sort_values("EABD50")

    plt.figure(figsize=(8.5, 5.2))
    plt.barh(sub["macro_region"], sub["EABD50"])
    plt.xlabel("Effective aquatic buffering distance, EABD50 (km)")
    plt.ylabel("Macro-region")
    plt.title(f"Regional aquatic buffering threshold under high HDSI ({target})")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()

    out_fp = os.path.join(FIG_DIR, f"regional_EABD50_high_HDSI_{target}.png")
    plt.savefig(out_fp, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 3. Sensitivity of interpretable threshold to min_delta
# min_delta = 0.03 / 0.05 / 0.10
# ============================================================

import os
import numpy as np
import pandas as pd

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")

fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
thr = pd.read_csv(fp)

rows = []

for min_delta in [0.03, 0.05, 0.10]:
    tmp = thr.copy()
    tmp["clear_delta_sens"] = tmp["delta_P_far_near"] >= min_delta

    high = tmp[
        (tmp["HDSI_group"] == "high") &
        (tmp["clear_delta_sens"])
    ].copy()

    for target in ["EVI", "LAI"]:
        sub = high[high["target"] == target].copy()

        rows.append({
            "min_delta": min_delta,
            "target": target,
            "n_region_clear": sub["macro_region"].nunique(),
            "EABD50_min": sub["EABD50"].min(),
            "EABD50_median": sub["EABD50"].median(),
            "EABD50_max": sub["EABD50"].max(),
            "delta_mean": sub["delta_P_far_near"].mean()
        })

sens = pd.DataFrame(rows)
display(sens)

out_fp = os.path.join(OUT_DIR, "EABD_threshold_sensitivity_min_delta.csv")
sens.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 4. Risk-threshold distance summary
# EABD_risk030 / 035 / 040 under high HDSI
# ============================================================

import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")

fp = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
thr = pd.read_csv(fp)

high_clear = thr[
    (thr["HDSI_group"] == "high") &
    (thr["clear_buffering_gradient"] == True)
].copy()

risk_cols = ["EABD_risk030", "EABD_risk035", "EABD_risk040"]

rows = []

for target in ["EVI", "LAI"]:
    sub = high_clear[high_clear["target"] == target].copy()

    for rc in risk_cols:
        vals = sub[rc].dropna()
        rows.append({
            "target": target,
            "risk_threshold": rc,
            "n_region_available": vals.shape[0],
            "distance_min": vals.min() if len(vals) else np.nan,
            "distance_median": vals.median() if len(vals) else np.nan,
            "distance_max": vals.max() if len(vals) else np.nan
        })

risk_summary = pd.DataFrame(rows)
display(risk_summary)

out_fp = os.path.join(OUT_DIR, "risk_threshold_distance_summary_high_HDSI.csv")
risk_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Step 5. Empirical bin validation of distance-response curves
# No model, direct observed risk by fine distance bins
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["HDSI_group"] = "mid"
d.loc[d["HDSI"] <= q25, "HDSI_group"] = "low"
d.loc[d["HDSI"] >= q75, "HDSI_group"] = "high"

# Fine bins: 0-2 every 0.25 km; 2-5 every 0.5 km; 5-16 every 1 km
bins = list(np.arange(0, 2.0001, 0.25)) + list(np.arange(2.5, 5.0001, 0.5)) + list(np.arange(6, 17, 1))
bins = sorted(set([round(x, 4) for x in bins]))

labels = []
midpoints = []
for a, b in zip(bins[:-1], bins[1:]):
    labels.append(f"{a:g}-{b:g}")
    midpoints.append((a + b) / 2)

bin_mid = dict(zip(labels, midpoints))

d["dist_bin_fine"] = pd.cut(
    d["dist_perm_water_km"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

rows = []

for (group, b), g in d.dropna(subset=["dist_bin_fine"]).groupby(["HDSI_group", "dist_bin_fine"], observed=False):
    n = len(g)
    if n == 0:
        continue

    evi_p = g["EVI_neg_z05"].mean()
    lai_p = g["LAI_neg_z05"].mean()

    def ci(p, n):
        se = np.sqrt(p * (1 - p) / max(n, 1))
        return max(0, p - 1.96 * se), min(1, p + 1.96 * se)

    evi_lo, evi_hi = ci(evi_p, n)
    lai_lo, lai_hi = ci(lai_p, n)

    rows.append({
        "HDSI_group": group,
        "dist_bin_fine": str(b),
        "dist_mid_km": bin_mid[str(b)],
        "rows": n,
        "grids": g["grid_id"].nunique(),
        "EVI_neg_rate": evi_p,
        "EVI_low": evi_lo,
        "EVI_high": evi_hi,
        "LAI_neg_rate": lai_p,
        "LAI_low": lai_lo,
        "LAI_high": lai_hi
    })

emp = pd.DataFrame(rows)

out_fp = os.path.join(OUT_DIR, "empirical_fine_distance_bins_by_HDSI.csv")
emp.to_csv(out_fp, index=False)
print("Saved:", out_fp)
display(emp.head())

# Plot empirical high HDSI response
for target, ycol, lo_col, hi_col, ylabel in [
    ("EVI", "EVI_neg_rate", "EVI_low", "EVI_high", "Observed EVI negative-response risk"),
    ("LAI", "LAI_neg_rate", "LAI_low", "LAI_high", "Observed LAI negative-response risk")
]:
    plt.figure(figsize=(8.5, 5.2))

    for group in ["low", "mid", "high"]:
        sub = emp[emp["HDSI_group"] == group].copy()
        sub = sub.sort_values("dist_mid_km")
        plt.plot(sub["dist_mid_km"], sub[ycol], marker="o", linewidth=1.8, label=group)
        plt.fill_between(sub["dist_mid_km"], sub[lo_col], sub[hi_col], alpha=0.15)

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(f"Empirical distance-response curve by HDSI group ({target})")
    plt.grid(alpha=0.3)
    plt.legend(title="HDSI group")
    plt.tight_layout()

    out_fig = os.path.join(FIG_DIR, f"empirical_fine_distance_curve_{target}.png")
    plt.savefig(out_fig, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fig)

In [ ]:
# ============================================================
# Step 6. Regional empirical validation under high HDSI
# Observed negative-response risk by distance bins in each region
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"))

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"]
    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

d["macro_region"] = d.apply(assign_macro_region, axis=1)

q75 = d["HDSI"].quantile(0.75)
dh = d[d["HDSI"] >= q75].copy()

# Distance bins for high-HDSI regional validation
bins = [0, 0.5, 1, 2, 3, 5, 8, 12, 16]
labels = [f"{a:g}-{b:g}" for a, b in zip(bins[:-1], bins[1:])]
midpoints = [(a+b)/2 for a, b in zip(bins[:-1], bins[1:])]
mid_map = dict(zip(labels, midpoints))

dh["dist_bin"] = pd.cut(
    dh["dist_perm_water_km"],
    bins=bins,
    labels=labels,
    include_lowest=True,
    right=False
)

rows = []

for (region, b), g in dh.dropna(subset=["dist_bin"]).groupby(["macro_region", "dist_bin"], observed=False):
    if len(g) < 30:
        continue
    rows.append({
        "macro_region": region,
        "dist_bin": str(b),
        "dist_mid_km": mid_map[str(b)],
        "rows": len(g),
        "grids": g["grid_id"].nunique(),
        "EVI_neg_rate": g["EVI_neg_z05"].mean(),
        "LAI_neg_rate": g["LAI_neg_z05"].mean()
    })

reg_emp = pd.DataFrame(rows)

out_fp = os.path.join(OUT_DIR, "regional_empirical_high_HDSI_distance_bins.csv")
reg_emp.to_csv(out_fp, index=False)
print("Saved:", out_fp)
display(reg_emp.head())

# Plot selected regions where threshold was clear
selected_regions = [
    "North_China", "Northeast", "Northwest_Arid",
    "Tibetan_Plateau", "Southwest", "South_China"
]

for target, ycol, ylabel in [
    ("EVI", "EVI_neg_rate", "Observed EVI negative-response risk"),
    ("LAI", "LAI_neg_rate", "Observed LAI negative-response risk")
]:
    plt.figure(figsize=(9, 5.5))

    for region in selected_regions:
        sub = reg_emp[reg_emp["macro_region"] == region].copy()
        if len(sub) == 0:
            continue
        sub = sub.sort_values("dist_mid_km")
        plt.plot(sub["dist_mid_km"], sub[ycol], marker="o", linewidth=1.8, label=region)

    plt.xlabel("Distance to permanent water (km)")
    plt.ylabel(ylabel)
    plt.title(f"Regional empirical distance-response under high HDSI ({target})")
    plt.grid(alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()

    out_fig = os.path.join(FIG_DIR, f"regional_empirical_high_HDSI_{target}.png")
    plt.savefig(out_fig, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", out_fig)

In [ ]:
# ============================================================
# Step 7. Adaptation priority with regional EABD context
# High HDSI exposure + buffering failure + region-level short EABD
# ============================================================

import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

risk_fp = os.path.join(AQUA_DIR, "adaptation_priority_grid_risk_v2.csv")
risk = pd.read_csv(risk_fp)

thr_fp = os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_by_region.csv")
if os.path.exists(thr_fp):
    thr = pd.read_csv(thr_fp)
else:
    thr = pd.read_csv(os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv"))
    thr = thr[(thr["HDSI_group"] == "high") & (thr["clear_buffering_gradient"] == True)].copy()

def assign_macro_region_from_lonlat(lon, lat, elev=None):
    if elev is not None and pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

risk["macro_region"] = risk.apply(
    lambda r: assign_macro_region_from_lonlat(r["lon"], r["lat"]),
    axis=1
)

# Region-level EABD: use median across available targets
region_eabd = (
    thr.groupby("macro_region", as_index=False)
    .agg(
        region_EABD50_median=("EABD50", "median"),
        region_EABD50_min=("EABD50", "min"),
        n_threshold=("target", "count")
    )
)

risk2 = risk.merge(region_eabd, on="macro_region", how="left")

# Short EABD means buffering distance is limited; higher priority
v = risk2["region_EABD50_median"]
risk2["short_EABD_score"] = 1 - (v - v.min()) / (v.max() - v.min() + 1e-9)
risk2["short_EABD_score"] = risk2["short_EABD_score"].fillna(0)

# Improved adaptation priority
risk2["adaptation_priority_thresholded"] = (
    0.30 * risk2["high_HDSI_freq_z01"] +
    0.20 * risk2["EVI_failure_freq_z01"] +
    0.20 * risk2["LAI_failure_freq_z01"] +
    0.15 * risk2["short_EABD_score"] +
    0.075 * risk2["EVI_neg_rate_z01"] +
    0.075 * risk2["LAI_neg_rate_z01"]
)

out_fp = os.path.join(OUT_DIR, "adaptation_priority_with_region_EABD.csv")
risk2.to_csv(out_fp, index=False)
print("Saved:", out_fp)
display(risk2.head())

# Plot near permanent-water areas only
plot_df = risk2[risk2["near_perm_2km"] == 1].copy()

try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
china = china.dissolve().reset_index(drop=True)

gdf = gpd.GeoDataFrame(
    plot_df,
    geometry=gpd.points_from_xy(plot_df["lon"], plot_df["lat"]),
    crs="EPSG:4326"
)

fig, ax = plt.subplots(figsize=(9, 7))
china.boundary.plot(ax=ax, linewidth=0.7, color="black")

gdf.plot(
    ax=ax,
    column="adaptation_priority_thresholded",
    markersize=1.4,
    cmap="magma",
    vmin=0,
    vmax=1,
    legend=True,
    legend_kwds={"shrink": 0.65, "label": "Adaptation priority index"}
)

ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Climate adaptation priority considering aquatic-buffering thresholds")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.2, linewidth=0.4)
plt.tight_layout()

out_fig = os.path.join(FIG_DIR, "map_adaptation_priority_with_EABD.png")
plt.savefig(out_fig, dpi=300, bbox_inches="tight")
plt.show()
print("Saved:", out_fig)

In [ ]:
# ============================================================
# Step 8. Create manuscript-ready result summary tables
# ============================================================

import os
import pandas as pd
import numpy as np

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")

files = {
    "EABD_high_HDSI_region": os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_by_region.csv"),
    "EABD_high_HDSI_summary": os.path.join(OUT_DIR, "interpretable_EABD_high_HDSI_summary.csv"),
    "EABD_min_delta_sensitivity": os.path.join(OUT_DIR, "EABD_threshold_sensitivity_min_delta.csv"),
    "risk_threshold_summary": os.path.join(OUT_DIR, "risk_threshold_distance_summary_high_HDSI.csv"),
    "empirical_bins": os.path.join(OUT_DIR, "empirical_fine_distance_bins_by_HDSI.csv"),
    "regional_empirical_bins": os.path.join(OUT_DIR, "regional_empirical_high_HDSI_distance_bins.csv"),
    "adaptation_priority_EABD": os.path.join(OUT_DIR, "adaptation_priority_with_region_EABD.csv"),
}

for name, fp in files.items():
    print(name, os.path.exists(fp), fp)

# Save selected compact summaries
summary_tables = {}

for name, fp in files.items():
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        summary_tables[name] = df

# Main compact table
main_rows = []

if "EABD_high_HDSI_region" in summary_tables:
    eabd = summary_tables["EABD_high_HDSI_region"]
    for target in ["EVI", "LAI"]:
        sub = eabd[eabd["target"] == target]
        main_rows.append({
            "result": f"Regional EABD50 under high HDSI ({target})",
            "n_regions": sub["macro_region"].nunique(),
            "min_km": sub["EABD50"].min(),
            "median_km": sub["EABD50"].median(),
            "max_km": sub["EABD50"].max(),
            "mean_delta_risk": sub["delta_P_far_near"].mean()
        })

main_summary = pd.DataFrame(main_rows)
display(main_summary)

out_fp = os.path.join(OUT_DIR, "manuscript_main_result_summary.csv")
main_summary.to_csv(out_fp, index=False)
print("Saved:", out_fp)

In [ ]:
# ============================================================
# Nature-style figures for aquatic buffering threshold study
# Figures:
#   Fig1: Conceptual and methodological framework
#   Fig2: Spatial overview of HDSI, water distance, and vegetation risk
#   Fig3: Nonlinear spline curves + EABD thresholds
#   Fig4: Empirical validation of distance-response curves
#   Fig5: Regional EABD threshold heterogeneity
#   Fig6: Buffering failure and adaptation priority
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec

try:
    import geopandas as gpd
except Exception:
    gpd = None

# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
THRESH_DIR = os.path.join(AQUA_DIR, "threshold_deep_analysis")
FIG_DIR = os.path.join(AQUA_DIR, "nature_style_figures")
os.makedirs(FIG_DIR, exist_ok=True)

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
PRED_FP = os.path.join(AQUA_DIR, "nonlinear_spline_permwater_predictions.csv")
EABD_HDSI_FP1 = os.path.join(AQUA_DIR, "EABD_thresholds_by_HDSI_permwater_robust.csv")
EABD_HDSI_FP2 = os.path.join(AQUA_DIR, "EABD_thresholds_by_HDSI_permwater.csv")
EABD_REGION_FP = os.path.join(AQUA_DIR, "EABD_thresholds_by_macro_region_permwater_fixed.csv")
ADAPT_FP1 = os.path.join(THRESH_DIR, "adaptation_priority_with_region_EABD.csv")
ADAPT_FP2 = os.path.join(AQUA_DIR, "adaptation_priority_grid_risk_v2.csv")
EMP_FP = os.path.join(THRESH_DIR, "empirical_fine_distance_bins_by_HDSI.csv")

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

# ============================================================
# 1. Nature-like style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Nature-like muted palette
COLORS = {
    "blue": "#3B6EA8",
    "light_blue": "#9BBBD8",
    "orange": "#E69F00",
    "vermillion": "#D55E00",
    "green": "#4E9A51",
    "teal": "#009E73",
    "purple": "#7E62A3",
    "gray": "#666666",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
    "red": "#B2182B",
    "yellow": "#F0E442"
}

HDSI_COLORS = {
    "low": COLORS["blue"],
    "mid": COLORS["orange"],
    "high": COLORS["red"],
    "Low HDSI": COLORS["blue"],
    "Median HDSI": COLORS["orange"],
    "High HDSI": COLORS["red"],
}

# Custom colormaps
cmap_hdry = LinearSegmentedColormap.from_list(
    "hdry",
    ["#f7fbff", "#c6dbef", "#6baed6", "#fd8d3c", "#bd0026"]
)

cmap_dist = LinearSegmentedColormap.from_list(
    "dist",
    ["#08306b", "#2171b5", "#6baed6", "#c6dbef", "#f7f7f7"]
)

cmap_priority = LinearSegmentedColormap.from_list(
    "priority",
    ["#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b", "#67001f"]
)

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def read_china_boundary():
    if gpd is None:
        return None
    if not os.path.exists(CHINA_BOUNDARY_PATH):
        print("[WARN] China boundary not found:", CHINA_BOUNDARY_PATH)
        return None
    try:
        china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
    except Exception:
        china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
    china = china.dissolve().reset_index(drop=True)
    return china

def make_point_gdf(df, lon="lon", lat="lat"):
    if gpd is None:
        return None
    return gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon], df[lat]),
        crs="EPSG:4326"
    )

# ============================================================
# 2. Load data
# ============================================================

panel = pd.read_parquet(PANEL_FP)

pred = pd.read_csv(PRED_FP) if os.path.exists(PRED_FP) else None

if os.path.exists(EABD_HDSI_FP1):
    eabd_hdsi = pd.read_csv(EABD_HDSI_FP1)
elif os.path.exists(EABD_HDSI_FP2):
    eabd_hdsi = pd.read_csv(EABD_HDSI_FP2)
else:
    eabd_hdsi = None

if os.path.exists(EABD_REGION_FP):
    eabd_region = pd.read_csv(EABD_REGION_FP)
else:
    eabd_region = None

if os.path.exists(ADAPT_FP1):
    adapt = pd.read_csv(ADAPT_FP1)
elif os.path.exists(ADAPT_FP2):
    adapt = pd.read_csv(ADAPT_FP2)
else:
    adapt = None

china = read_china_boundary()

print("Panel:", panel.shape)
print("Pred:", None if pred is None else pred.shape)
print("EABD by HDSI:", None if eabd_hdsi is None else eabd_hdsi.shape)
print("EABD region:", None if eabd_region is None else eabd_region.shape)
print("Adapt:", None if adapt is None else adapt.shape)

# ============================================================
# 3. Helper: create empirical bins if missing
# ============================================================

def build_empirical_bins(panel, out_fp):
    d = panel[
        panel["valid_model_both_y"] &
        panel["HDSI"].notna() &
        panel["dist_perm_water_km"].notna()
    ].copy()

    q25 = d["HDSI"].quantile(0.25)
    q75 = d["HDSI"].quantile(0.75)

    d["HDSI_group"] = "mid"
    d.loc[d["HDSI"] <= q25, "HDSI_group"] = "low"
    d.loc[d["HDSI"] >= q75, "HDSI_group"] = "high"

    bins = list(np.arange(0, 2.0001, 0.25)) + list(np.arange(2.5, 5.0001, 0.5)) + list(np.arange(6, 17, 1))
    bins = sorted(set([round(x, 4) for x in bins]))

    labels = []
    midpoints = []
    for a, b in zip(bins[:-1], bins[1:]):
        labels.append(f"{a:g}-{b:g}")
        midpoints.append((a + b) / 2)

    mid_map = dict(zip(labels, midpoints))

    d["dist_bin_fine"] = pd.cut(
        d["dist_perm_water_km"],
        bins=bins,
        labels=labels,
        include_lowest=True,
        right=False
    )

    rows = []
    for (group, b), g in d.dropna(subset=["dist_bin_fine"]).groupby(["HDSI_group", "dist_bin_fine"], observed=False):
        n = len(g)
        if n == 0:
            continue

        def ci(p, n):
            se = np.sqrt(p * (1 - p) / max(n, 1))
            return max(0, p - 1.96 * se), min(1, p + 1.96 * se)

        evi_p = g["EVI_neg_z05"].mean()
        lai_p = g["LAI_neg_z05"].mean()
        evi_lo, evi_hi = ci(evi_p, n)
        lai_lo, lai_hi = ci(lai_p, n)

        rows.append({
            "HDSI_group": group,
            "dist_bin_fine": str(b),
            "dist_mid_km": mid_map[str(b)],
            "rows": n,
            "grids": g["grid_id"].nunique(),
            "EVI_neg_rate": evi_p,
            "EVI_low": evi_lo,
            "EVI_high": evi_hi,
            "LAI_neg_rate": lai_p,
            "LAI_low": lai_lo,
            "LAI_high": lai_hi,
        })

    emp = pd.DataFrame(rows)
    os.makedirs(os.path.dirname(out_fp), exist_ok=True)
    emp.to_csv(out_fp, index=False)
    return emp

if os.path.exists(EMP_FP):
    emp = pd.read_csv(EMP_FP)
else:
    emp = build_empirical_bins(panel, EMP_FP)

print("Empirical bins:", emp.shape)

# ============================================================
# Figure 1. Conceptual and methodological framework
# ============================================================

def draw_arrow(ax, x1, y1, x2, y2, color="#555555"):
    ax.annotate(
        "",
        xy=(x2, y2), xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", lw=1.4, color=color, shrinkA=5, shrinkB=5)
    )

def rounded_box(ax, xy, w, h, text, fc, ec="#333333"):
    box = patches.FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        fc=fc, ec=ec, lw=0.8
    )
    ax.add_patch(box)
    ax.text(
        xy[0] + w/2, xy[1] + h/2,
        text,
        ha="center", va="center",
        fontsize=8.5,
        color="#222222",
        linespacing=1.25
    )

fig = plt.figure(figsize=(11, 6))
ax = fig.add_axes([0, 0, 1, 1])
ax.axis("off")

# Background title
ax.text(
    0.05, 0.93,
    "Aquatic buffering of vegetation under heat–dryness stress",
    fontsize=15,
    fontweight="bold",
    ha="left",
    va="top"
)
ax.text(
    0.05, 0.885,
    "From climate stress and water proximity to nonlinear thresholds and adaptation priority",
    fontsize=9.5,
    color=COLORS["gray"],
    ha="left",
    va="top"
)

# Boxes
rounded_box(ax, (0.06, 0.62), 0.18, 0.16,
            "Climate stress\nVPD ↑  LST ↑\nSoil moisture ↓\nPrecipitation ↓",
            "#FDE0DD")

rounded_box(ax, (0.06, 0.34), 0.18, 0.16,
            "Aquatic context\nDistance to\npermanent water\nWater stability",
            "#D9EAF7")

rounded_box(ax, (0.32, 0.49), 0.20, 0.18,
            "Vegetation response\nLate-summer EVI\nLate-summer LAI\nNegative response risk",
            "#E5F5E0")

rounded_box(ax, (0.60, 0.58), 0.18, 0.15,
            "Nonlinear model\nSpline-logit curves\nLow / mid / high HDSI",
            "#FEEBC8")

rounded_box(ax, (0.60, 0.34), 0.18, 0.15,
            "EABD thresholds\nBuffering distance\nRisk crossing distance\nRegional differences",
            "#E6E6FA")

rounded_box(ax, (0.83, 0.46), 0.13, 0.16,
            "Adaptation\npriority\nNear-water failure\nunder extremes",
            "#FDD0A2")

# Arrows
draw_arrow(ax, 0.24, 0.70, 0.32, 0.58, COLORS["red"])
draw_arrow(ax, 0.24, 0.42, 0.32, 0.55, COLORS["blue"])
draw_arrow(ax, 0.52, 0.58, 0.60, 0.65, COLORS["gray"])
draw_arrow(ax, 0.52, 0.54, 0.60, 0.42, COLORS["gray"])
draw_arrow(ax, 0.78, 0.65, 0.83, 0.56, COLORS["gray"])
draw_arrow(ax, 0.78, 0.42, 0.83, 0.50, COLORS["gray"])

# Small conceptual curve
ax2 = fig.add_axes([0.34, 0.18, 0.30, 0.18])
x = np.linspace(0, 15, 200)
y_low = 0.20 + 0.04*np.log1p(x) + 0.03*np.sin(x/2)
y_high = 0.35 + 0.18*(1 - np.exp(-x/5)) + 0.03*np.sin(x/1.8)
ax2.plot(x, y_low, color=COLORS["blue"], lw=2, label="Low HDSI")
ax2.plot(x, y_high, color=COLORS["red"], lw=2, label="High HDSI")
ax2.axvline(5, color=COLORS["dark"], lw=0.8, ls="--")
ax2.text(5.2, 0.55, "EABD", fontsize=8, color=COLORS["dark"])
ax2.set_xlabel("Distance to permanent water")
ax2.set_ylabel("Negative-response risk")
ax2.set_ylim(0.15, 0.65)
ax2.legend(frameon=False, loc="lower right")
clean_ax(ax2)

savefig(fig, "Fig1_conceptual_framework")
plt.show()

# ============================================================
# Figure 2. Spatial overview
# ============================================================

# Grid-level summaries
d = panel[
    panel["valid_model_both_y"] &
    panel["HDSI"].notna() &
    panel["dist_perm_water_km"].notna()
].copy()

grid_summary = (
    d.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        HDSI_mean=("HDSI", "mean"),
        high_HDSI_freq=("HDSI", lambda x: (x >= d["HDSI"].quantile(0.75)).mean()),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean")
    )
)

gdf = make_point_gdf(grid_summary) if gpd is not None else None

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()

map_specs = [
    ("HDSI_mean", "Mean heat–dryness stress", cmap_hdry, None, None),
    ("dist_perm_water_km", "Distance to permanent water (km)", cmap_dist, 0, 10),
    ("EVI_neg_rate", "EVI negative-response frequency", cmap_priority, 0, 0.6),
    ("LAI_neg_rate", "LAI negative-response frequency", cmap_priority, 0, 0.6),
]

for i, (col, title, cmap, vmin, vmax) in enumerate(map_specs):
    ax = axes[i]
    if china is not None:
        china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")
    if gdf is not None:
        gdf.plot(
            ax=ax,
            column=col,
            markersize=1.0,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            legend=True,
            legend_kwds={"shrink": 0.62}
        )
    else:
        sc = ax.scatter(
            grid_summary["lon"], grid_summary["lat"],
            c=grid_summary[col],
            s=1,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )
        plt.colorbar(sc, ax=ax, shrink=0.62)

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)
    panel_label(ax, chr(65+i))

plt.tight_layout()
savefig(fig, "Fig2_spatial_overview")
plt.show()

# ============================================================
# Figure 3. Nonlinear spline curves + national EABD
# ============================================================

if pred is not None:
    fig = plt.figure(figsize=(11, 7))
    gs = GridSpec(2, 3, figure=fig, width_ratios=[1.2, 1.2, 0.9], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

    # Curve panels
    curve_specs = [
        ("EVI_neg_z05", "Predicted EVI negative-response risk", 0, 0),
        ("LAI_neg_z05", "Predicted LAI negative-response risk", 1, 0),
    ]

    for target, ylabel, row, col in curve_specs:
        ax = fig.add_subplot(gs[row, 0:2])

        for group in ["low", "mid", "high"]:
            sub = pred[pred["HDSI_group"].astype(str) == group].copy()
            sub = sub.sort_values("dist_perm_water_km")
            color = HDSI_COLORS[group]

            x = sub["dist_perm_water_km"].astype(float)
            y = sub[f"{target}_prob"].astype(float)
            lo = sub[f"{target}_lower"].astype(float)
            hi = sub[f"{target}_upper"].astype(float)

            ax.plot(x, y, lw=2.2, color=color, label=f"{group.capitalize()} HDSI")
            ax.fill_between(x, lo, hi, color=color, alpha=0.16, lw=0)

        ax.set_xlabel("Distance to permanent water (km)")
        ax.set_ylabel(ylabel)
        ax.set_xlim(0, pred["dist_perm_water_km"].max())
        ax.set_ylim(0, max(0.8, ax.get_ylim()[1]))
        ax.legend(frameon=False, loc="upper left", ncol=3)
        clean_ax(ax)
        panel_label(ax, "A" if row == 0 else "C")

    # EABD table-like panel
    ax_tbl = fig.add_subplot(gs[:, 2])
    ax_tbl.axis("off")
    panel_label(ax_tbl, "B", x=-0.02, y=1.02)

    ax_tbl.text(
        0.0, 0.98,
        "Effective aquatic\nbuffering distance",
        fontsize=11,
        fontweight="bold",
        va="top"
    )

    if eabd_hdsi is not None:
        tab = eabd_hdsi.copy()
        if "clear_buffering_gradient" in tab.columns:
            tab_show = tab[(tab["clear_buffering_gradient"] == True)].copy()
        else:
            tab_show = tab.copy()

        tab_show = tab_show[tab_show["HDSI_group"].astype(str).isin(["low", "mid", "high"])]

        y0 = 0.83
        ax_tbl.text(0.00, y0, "Target", fontweight="bold")
        ax_tbl.text(0.30, y0, "HDSI", fontweight="bold")
        ax_tbl.text(0.58, y0, "EABD50", fontweight="bold")
        ax_tbl.text(0.82, y0, "ΔRisk", fontweight="bold")

        y = y0 - 0.07
        for _, r in tab_show.iterrows():
            ax_tbl.text(0.00, y, str(r["target"]))
            ax_tbl.text(0.30, y, str(r["HDSI_group"]))
            ax_tbl.text(0.58, y, f"{r['EABD50']:.2f}" if pd.notna(r["EABD50"]) else "—")
            ax_tbl.text(0.82, y, f"{r['delta_P_far_near']:.2f}" if "delta_P_far_near" in r else "")
            y -= 0.06

    ax_tbl.text(
        0.0, 0.20,
        "EABD50 indicates the distance\nat which half of the near-to-far\nrisk increase is reached.\nOnly clear positive gradients\nshould be interpreted as\nbuffering thresholds.",
        fontsize=8,
        color=COLORS["gray"],
        va="top"
    )

    savefig(fig, "Fig3_nonlinear_threshold_curves")
    plt.show()

# ============================================================
# Figure 4. Empirical validation curves
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)

for ax, target, ycol, lo_col, hi_col, ylabel, label in [
    (axes[0], "EVI", "EVI_neg_rate", "EVI_low", "EVI_high", "Observed EVI negative-response risk", "A"),
    (axes[1], "LAI", "LAI_neg_rate", "LAI_low", "LAI_high", "Observed LAI negative-response risk", "B")
]:
    for group in ["low", "mid", "high"]:
        sub = emp[emp["HDSI_group"].astype(str) == group].copy().sort_values("dist_mid_km")
        color = HDSI_COLORS[group]
        ax.plot(sub["dist_mid_km"], sub[ycol], marker="o", ms=3.0, lw=1.6, color=color, label=f"{group.capitalize()} HDSI")
        ax.fill_between(sub["dist_mid_km"], sub[lo_col], sub[hi_col], color=color, alpha=0.12)

    ax.set_xlabel("Distance to permanent water (km)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Empirical validation ({target})")
    ax.set_xlim(0, 16)
    ax.legend(frameon=False)
    clean_ax(ax)
    panel_label(ax, label)

plt.tight_layout()
savefig(fig, "Fig4_empirical_validation_curves")
plt.show()

# ============================================================
# Figure 5. Regional EABD heterogeneity
# ============================================================

if eabd_region is not None:
    er = eabd_region.copy()

    if "clear_buffering_gradient" in er.columns:
        er_clear = er[(er["clear_buffering_gradient"] == True) & (er["HDSI_group"].astype(str) == "high")].copy()
    else:
        er_clear = er[er["HDSI_group"].astype(str) == "high"].copy()

    # Only meaningful EABD50
    er_clear = er_clear[pd.notna(er_clear["EABD50"])].copy()

    fig = plt.figure(figsize=(11, 6.5))
    gs = GridSpec(2, 2, figure=fig, width_ratios=[1.25, 1], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

    # A. EVI lollipop
    for idx, target in enumerate(["EVI", "LAI"]):
        ax = fig.add_subplot(gs[idx, 0])
        sub = er_clear[er_clear["target"] == target].copy().sort_values("EABD50")
        y = np.arange(len(sub))

        ax.hlines(y, 0, sub["EABD50"], color="#BDBDBD", lw=1.4)
        ax.scatter(
            sub["EABD50"], y,
            s=70 + 260 * np.clip(sub["delta_P_far_near"], 0, None),
            color=COLORS["purple"] if target == "LAI" else COLORS["green"],
            edgecolor="white",
            linewidth=0.8,
            zorder=3
        )
        ax.set_yticks(y)
        ax.set_yticklabels(sub["macro_region"])
        ax.set_xlabel("EABD50 under high HDSI (km)")
        ax.set_title(f"Regional threshold ({target})")
        ax.set_xlim(0, max(11, sub["EABD50"].max() + 1 if len(sub) else 11))
        clean_ax(ax)
        panel_label(ax, "A" if idx == 0 else "C")

    # B. Heatmap region × target
    ax = fig.add_subplot(gs[:, 1])
    pivot = er_clear.pivot_table(
        index="macro_region",
        columns="target",
        values="EABD50",
        aggfunc="median"
    )

    # Sort by average threshold
    pivot["mean"] = pivot.mean(axis=1)
    pivot = pivot.sort_values("mean", ascending=False)
    heat = pivot.drop(columns=["mean"])

    im = ax.imshow(heat.values, cmap="YlGnBu", aspect="auto", vmin=0, vmax=np.nanmax(heat.values))
    ax.set_xticks(np.arange(len(heat.columns)))
    ax.set_xticklabels(heat.columns)
    ax.set_yticks(np.arange(len(heat.index)))
    ax.set_yticklabels(heat.index)
    ax.set_title("Median EABD50 by region and response")
    panel_label(ax, "B", x=-0.12, y=1.02)

    for i in range(heat.shape[0]):
        for j in range(heat.shape[1]):
            val = heat.values[i, j]
            txt = "—" if np.isnan(val) else f"{val:.1f}"
            ax.text(j, i, txt, ha="center", va="center", fontsize=8, color="#111111")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("EABD50 (km)")

    plt.tight_layout()
    savefig(fig, "Fig5_regional_EABD_heterogeneity")
    plt.show()

# ============================================================
# Figure 6. Buffering failure and adaptation priority
# ============================================================

if adapt is not None:
    ad = adapt.copy()

    # Make sure required columns exist
    priority_col = "adaptation_priority_thresholded" if "adaptation_priority_thresholded" in ad.columns else "adaptation_priority_index"
    failure_col = "EVI_failure_freq" if "EVI_failure_freq" in ad.columns else None

    gdf_ad = make_point_gdf(ad) if gpd is not None else None

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

    # A. EVI failure frequency
    ax = axes[0]
    if china is not None:
        china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

    plot_df = ad.copy()
    if "near_perm_2km" in plot_df.columns:
        plot_df = plot_df[plot_df["near_perm_2km"] == 1].copy()

    gdf_plot = make_point_gdf(plot_df) if gpd is not None else None

    if gdf_plot is not None and failure_col is not None:
        gdf_plot.plot(
            ax=ax,
            column=failure_col,
            markersize=1.2,
            cmap=cmap_priority,
            vmin=0,
            vmax=max(0.5, plot_df[failure_col].quantile(0.98)),
            legend=True,
            legend_kwds={"shrink": 0.65}
        )
    else:
        sc = ax.scatter(
            plot_df["lon"], plot_df["lat"],
            c=plot_df[failure_col],
            s=1,
            cmap=cmap_priority
        )
        plt.colorbar(sc, ax=ax, shrink=0.65)

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title("Near-water buffering failure")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)
    panel_label(ax, "A")

    # B. Adaptation priority
    ax = axes[1]
    if china is not None:
        china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

    if gdf_plot is not None:
        gdf_plot.plot(
            ax=ax,
            column=priority_col,
            markersize=1.2,
            cmap=cmap_priority,
            vmin=0,
            vmax=1,
            legend=True,
            legend_kwds={"shrink": 0.65}
        )
    else:
        sc = ax.scatter(
            plot_df["lon"], plot_df["lat"],
            c=plot_df[priority_col],
            s=1,
            cmap=cmap_priority,
            vmin=0,
            vmax=1
        )
        plt.colorbar(sc, ax=ax, shrink=0.65)

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title("Climate adaptation priority")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)
    panel_label(ax, "B")

    plt.tight_layout()
    savefig(fig, "Fig6_buffering_failure_adaptation_priority")
    plt.show()

print("\nAll Nature-style figures saved in:")
print(FIG_DIR)

In [ ]:
# ============================================================
# Aquatic-buffering nonlinear failure framework
# Main outputs:
#   Fig1: Framework + model performance
#   Fig2: HDSI-distance nonlinear failure risk surfaces
#   Fig3: Fragility curves by aquatic-distance band
#   Fig4: Failure boundary and acceleration zone
#   Fig5: Buffered / sensitive / failed state diagnosis
#   Fig6: Buffering resilience and adaptation priority maps
#
# Core idea:
#   We do NOT use counterfactual replacement.
#   We model observed exposure-response patterns and diagnose
#   nonlinear failure boundaries of aquatic buffering.
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, brier_score_loss,
    log_loss, accuracy_score
)
from sklearn.isotonic import IsotonicRegression
from sklearn.mixture import GaussianMixture

try:
    import geopandas as gpd
except Exception:
    gpd = None


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "nonlinear_failure_framework")
FIG_DIR = os.path.join(OUT_DIR, "figs")
TAB_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

print("Panel file:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Nature-like visual style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300,
})

COLORS = {
    "blue": "#3B6EA8",
    "light_blue": "#9BBBD8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "teal": "#009E73",
    "purple": "#7E62A3",
    "gray": "#666666",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
    "brown": "#8C510A"
}

cmap_failure = LinearSegmentedColormap.from_list(
    "failure",
    ["#f7fbff", "#c6dbef", "#6baed6", "#fdae61", "#d7191c"]
)

cmap_accel = LinearSegmentedColormap.from_list(
    "accel",
    ["#f7f7f7", "#d9f0d3", "#78c679", "#238443", "#004529"]
)

cmap_priority = LinearSegmentedColormap.from_list(
    "priority",
    ["#f7f7f7", "#fddbc7", "#ef8a62", "#b2182b", "#67001f"]
)

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, f"{name}.png")
    pdf = os.path.join(FIG_DIR, f"{name}.pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def read_china_boundary():
    if gpd is None:
        return None
    if not os.path.exists(CHINA_BOUNDARY_PATH):
        print("[WARN] China boundary not found.")
        return None
    try:
        china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
    except Exception:
        china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
    china = china.dissolve().reset_index(drop=True)
    return china

def make_point_gdf(df, lon="lon", lat="lat"):
    if gpd is None:
        return None
    return gpd.GeoDataFrame(
        df.copy(),
        geometry=gpd.points_from_xy(df[lon], df[lat]),
        crs="EPSG:4326"
    )


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

# ----------------------------
# Ensure required variables exist
# ----------------------------

# HDSI if not already existing
if "HDSI" not in df.columns:
    for c in ["summer_VPD_z", "summer_LST_Day_z", "summer_SM_rootproxy_z", "summer_Precip_z"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values
        ]),
        axis=0
    )

# negative responses if not existing
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)

# macro region
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)

# keep valid modelling samples
use = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna() &
    df["EVI_neg_z05"].notna() &
    df["LAI_neg_z05"].notna()
].copy()

# numeric conversion
num_convert = [
    "HDSI", "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition",
    "spring_EVI_z", "spring_LAI_z",
    "elevation", "slope", "lon", "lat",
    "year", "LC_Type1",
    "EVI_neg_z05", "LAI_neg_z05"
]
for c in num_convert:
    if c in use.columns:
        use[c] = pd.to_numeric(use[c], errors="coerce")

use["log_dist_perm"] = np.log1p(use["dist_perm_water_km"])
use["LC_Type1_cat"] = pd.to_numeric(use["LC_Type1"], errors="coerce").round().astype("Int64").astype(str)
use["year_cat"] = pd.to_numeric(use["year"], errors="coerce").astype("Int64").astype(str)
use["macro_region"] = use["macro_region"].astype(str)

# aquatic distance bands
dist_bins = [0, 0.5, 1, 2, 5, 10, 20]
dist_labels = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]
use["dist_band_perm"] = pd.cut(
    use["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
).astype(str)

# HDSI groups
q25 = use["HDSI"].quantile(0.25)
q75 = use["HDSI"].quantile(0.75)
use["HDSI_group"] = "mid"
use.loc[use["HDSI"] <= q25, "HDSI_group"] = "low"
use.loc[use["HDSI"] >= q75, "HDSI_group"] = "high"

print("Model use:", use.shape)
print("Years:", sorted(use["year"].dropna().unique()))
print("Grids:", use["grid_id"].nunique())
print("EVI neg rate:", use["EVI_neg_z05"].mean())
print("LAI neg rate:", use["LAI_neg_z05"].mean())


# ============================================================
# 3. Machine-learning risk model
# ============================================================

# Compact, interpretable feature set:
# HDSI is used as the exposure index, instead of including all climate components.
num_features = [
    "HDSI",
    "log_dist_perm",
    "dist_perm_water_km",
    "dist_seasonal_water_km",
    "dist_any_water_km",
    "water_occurrence",
    "water_seasonality",
    "spring_EVI_z",
    "spring_LAI_z",
    "elevation",
    "slope",
    "lon",
    "lat"
]

num_features = [c for c in num_features if c in use.columns]

cat_features = [
    "LC_Type1_cat",
    "macro_region",
    "year_cat"
]

cat_features = [c for c in cat_features if c in use.columns]

feature_cols = num_features + cat_features
print("Numeric features:", num_features)
print("Categorical features:", cat_features)

model_data = use.dropna(subset=["year", "EVI_neg_z05", "LAI_neg_z05"]).copy()

# Temporal validation:
# train: 2018-2021; validation: 2022; test: 2023
train_df = model_data[model_data["year"].between(2018, 2021)].copy()
val_df = model_data[model_data["year"] == 2022].copy()
test_df = model_data[model_data["year"] == 2023].copy()

# fallback if year split is incomplete
if len(val_df) == 0 or len(test_df) == 0:
    train_df, test_df = train_test_split(model_data, test_size=0.2, random_state=42, stratify=model_data["EVI_neg_z05"])
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df["EVI_neg_z05"])

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))

# robust OneHotEncoder compatibility
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
        ]), num_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe),
        ]), cat_features)
    ],
    remainder="drop"
)

def make_hgb_model():
    clf = HistGradientBoostingClassifier(
        max_iter=350,
        learning_rate=0.045,
        max_leaf_nodes=31,
        min_samples_leaf=40,
        l2_regularization=0.05,
        random_state=42
    )
    pipe = Pipeline([
        ("preprocess", preprocess),
        ("clf", clf)
    ])
    return pipe

def fit_calibrated_model(target_col):
    X_train = train_df[feature_cols]
    y_train = train_df[target_col].astype(int)

    X_val = val_df[feature_cols]
    y_val = val_df[target_col].astype(int)

    X_test = test_df[feature_cols]
    y_test = test_df[target_col].astype(int)

    model = make_hgb_model()
    model.fit(X_train, y_train)

    # probability calibration using validation year
    val_raw = model.predict_proba(X_val)[:, 1]
    iso = IsotonicRegression(out_of_bounds="clip")
    iso.fit(val_raw, y_val)

    def predict_calibrated(X):
        raw = model.predict_proba(X)[:, 1]
        return iso.transform(raw)

    test_prob = predict_calibrated(X_test)
    test_pred = (test_prob >= 0.5).astype(int)

    metrics = {
        "target": target_col,
        "n_train": len(train_df),
        "n_val": len(val_df),
        "n_test": len(test_df),
        "test_auc": roc_auc_score(y_test, test_prob),
        "test_pr_auc": average_precision_score(y_test, test_prob),
        "test_brier": brier_score_loss(y_test, test_prob),
        "test_logloss": log_loss(y_test, np.clip(test_prob, 1e-6, 1-1e-6)),
        "test_accuracy": accuracy_score(y_test, test_pred),
        "test_event_rate": y_test.mean()
    }

    return model, iso, predict_calibrated, metrics

models = {}
calibrators = {}
predictors = {}
metrics_rows = []

for target in ["EVI_neg_z05", "LAI_neg_z05"]:
    print("\nFitting model:", target)
    model, iso, pred_fun, metrics = fit_calibrated_model(target)
    models[target] = model
    calibrators[target] = iso
    predictors[target] = pred_fun
    metrics_rows.append(metrics)
    print(metrics)

metrics_df = pd.DataFrame(metrics_rows)
metrics_fp = os.path.join(TAB_DIR, "model_temporal_validation_metrics.csv")
metrics_df.to_csv(metrics_fp, index=False)
display(metrics_df)
print("Saved:", metrics_fp)


# ============================================================
# 4. Predict risk for all observations
# ============================================================

def predict_in_chunks(pred_fun, X, chunk=50000):
    outs = []
    for i in range(0, len(X), chunk):
        outs.append(pred_fun(X.iloc[i:i+chunk]))
    return np.concatenate(outs)

all_X = model_data[feature_cols]
model_data["EVI_pred_risk"] = predict_in_chunks(predictors["EVI_neg_z05"], all_X)
model_data["LAI_pred_risk"] = predict_in_chunks(predictors["LAI_neg_z05"], all_X)
model_data["joint_pred_risk"] = 0.5 * (model_data["EVI_pred_risk"] + model_data["LAI_pred_risk"])

pred_obs_fp = os.path.join(TAB_DIR, "all_observations_predicted_failure_risk.parquet")
model_data.to_parquet(pred_obs_fp, index=False)
print("Saved:", pred_obs_fp)


# ============================================================
# 5. Response-surface grid: HDSI × distance
# ============================================================

# Build a standardized diagnostic grid.
# Other variables are fixed to representative values.
# This is an exposure-response diagnostic surface, not a counterfactual design.

def representative_row(df):
    row = {}
    for c in num_features:
        row[c] = df[c].median()
    for c in cat_features:
        row[c] = df[c].mode().iloc[0]
    return row

base = representative_row(model_data)

hdsi_min = model_data["HDSI"].quantile(0.01)
hdsi_max = model_data["HDSI"].quantile(0.99)
dist_min = 0.0
dist_max = min(16.0, model_data["dist_perm_water_km"].max())

hdsi_values = np.linspace(hdsi_min, hdsi_max, 100)
dist_values = np.linspace(dist_min, dist_max, 120)

surface_rows = []
for h in hdsi_values:
    for dval in dist_values:
        r = base.copy()
        r["HDSI"] = h
        r["dist_perm_water_km"] = dval
        r["log_dist_perm"] = np.log1p(dval)
        surface_rows.append(r)

surface_df = pd.DataFrame(surface_rows)

# Make sure all features exist
for c in feature_cols:
    if c not in surface_df.columns:
        surface_df[c] = base.get(c, np.nan)

surface_df["EVI_risk"] = predictors["EVI_neg_z05"](surface_df[feature_cols])
surface_df["LAI_risk"] = predictors["LAI_neg_z05"](surface_df[feature_cols])
surface_df["joint_risk"] = 0.5 * (surface_df["EVI_risk"] + surface_df["LAI_risk"])

surface_fp = os.path.join(TAB_DIR, "diagnostic_HDSI_distance_risk_surface.csv")
surface_df.to_csv(surface_fp, index=False)
print("Saved:", surface_fp)

# reshape helpers
def surface_matrix(df, value_col):
    z = df.pivot_table(index="HDSI", columns="dist_perm_water_km", values=value_col).sort_index()
    X = z.columns.values
    Y = z.index.values
    Z = z.values
    return X, Y, Z

Xdist, Yhdsi, Zevi = surface_matrix(surface_df, "EVI_risk")
_, _, Zlai = surface_matrix(surface_df, "LAI_risk")
_, _, Zjoint = surface_matrix(surface_df, "joint_risk")


# ============================================================
# 6. Failure boundary and acceleration zone
# ============================================================

def extract_failure_boundaries(surface_df, risk_col, thresholds=[0.30, 0.40, 0.50]):
    rows = []
    for h in sorted(surface_df["HDSI"].unique()):
        sub = surface_df[surface_df["HDSI"] == h].sort_values("dist_perm_water_km")
        x = sub["dist_perm_water_km"].values
        y = sub[risk_col].values

        for thr in thresholds:
            idx = np.where(y >= thr)[0]
            boundary_dist = x[idx[0]] if len(idx) > 0 else np.nan
            rows.append({
                "risk_col": risk_col,
                "HDSI": h,
                "risk_threshold": thr,
                "FBD_km": boundary_dist
            })

        dydx = np.gradient(y, x)
        imax = np.nanargmax(dydx)
        rows.append({
            "risk_col": risk_col,
            "HDSI": h,
            "risk_threshold": "max_acceleration",
            "FBD_km": x[imax],
            "max_dRisk_dDist": dydx[imax]
        })

    return pd.DataFrame(rows)

boundary_tables = []
for risk_col in ["EVI_risk", "LAI_risk", "joint_risk"]:
    boundary_tables.append(extract_failure_boundaries(surface_df, risk_col))

boundary_df = pd.concat(boundary_tables, ignore_index=True)
boundary_fp = os.path.join(TAB_DIR, "failure_boundary_distance_by_HDSI.csv")
boundary_df.to_csv(boundary_fp, index=False)
print("Saved:", boundary_fp)

# acceleration surface: derivative along distance
dZdist_evi = np.gradient(Zevi, Xdist, axis=1)
dZdist_lai = np.gradient(Zlai, Xdist, axis=1)
dZdist_joint = np.gradient(Zjoint, Xdist, axis=1)


# ============================================================
# 7. Empirical fragility curves
# ============================================================

# Fragility curve:
# HDSI increases -> observed / predicted failure probability.
# Compare distance bands.

frag = model_data.copy()

# Use fixed quantile bins for HDSI
frag["HDSI_bin"] = pd.qcut(frag["HDSI"], q=12, duplicates="drop")
frag["HDSI_mid"] = frag["HDSI_bin"].apply(lambda x: x.mid if pd.notna(x) else np.nan).astype(float)

frag_rows = []
for (band, hb), g in frag.dropna(subset=["HDSI_bin"]).groupby(["dist_band_perm", "HDSI_bin"], observed=False):
    if len(g) < 50:
        continue
    frag_rows.append({
        "dist_band_perm": band,
        "HDSI_mid": g["HDSI"].median(),
        "rows": len(g),
        "grids": g["grid_id"].nunique(),
        "EVI_obs_fail": g["EVI_neg_z05"].mean(),
        "LAI_obs_fail": g["LAI_neg_z05"].mean(),
        "EVI_pred_fail": g["EVI_pred_risk"].mean(),
        "LAI_pred_fail": g["LAI_pred_risk"].mean(),
        "joint_pred_fail": g["joint_pred_risk"].mean()
    })

frag_df = pd.DataFrame(frag_rows)
frag_fp = os.path.join(TAB_DIR, "fragility_curves_by_distance_band.csv")
frag_df.to_csv(frag_fp, index=False)
print("Saved:", frag_fp)


# ============================================================
# 8. Grid-level BRI and state clustering
# ============================================================

# Define high HDSI at observation level
h75 = model_data["HDSI"].quantile(0.75)
model_data["high_HDSI"] = (model_data["HDSI"] >= h75).astype(int)
model_data["near_perm_2km"] = (model_data["dist_perm_water_km"] < 2.0).astype(int)

# Failure under high HDSI
model_data["observed_joint_failure"] = (
    (model_data["EVI_neg_z05"].astype(int) + model_data["LAI_neg_z05"].astype(int)) >= 1
).astype(int)

grid = (
    model_data.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        HDSI_mean=("HDSI", "mean"),
        high_HDSI_freq=("high_HDSI", "mean"),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        water_occurrence=("water_occurrence", "mean") if "water_occurrence" in model_data.columns else ("HDSI", "mean"),
        water_seasonality=("water_seasonality", "mean") if "water_seasonality" in model_data.columns else ("HDSI", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        observed_joint_failure_rate=("observed_joint_failure", "mean"),
        EVI_pred_risk_mean=("EVI_pred_risk", "mean"),
        LAI_pred_risk_mean=("LAI_pred_risk", "mean"),
        joint_pred_risk_mean=("joint_pred_risk", "mean"),
        near_perm_2km=("near_perm_2km", "max"),
        macro_region=("macro_region", lambda x: x.mode().iloc[0] if len(x.mode()) else x.iloc[0])
    )
)

# High-HDSI risk by grid
high_subset = model_data[model_data["high_HDSI"] == 1].copy()
grid_high = (
    high_subset.groupby("grid_id", as_index=False)
    .agg(
        high_EVI_pred_risk=("EVI_pred_risk", "mean"),
        high_LAI_pred_risk=("LAI_pred_risk", "mean"),
        high_joint_pred_risk=("joint_pred_risk", "mean"),
        high_observed_failure=("observed_joint_failure", "mean")
    )
)

grid = grid.merge(grid_high, on="grid_id", how="left")
grid["high_joint_pred_risk"] = grid["high_joint_pred_risk"].fillna(grid["joint_pred_risk_mean"])

# Buffering Resilience Index:
# high value = low failure risk under high-HDSI episodes
grid["BRI"] = 1 - grid["high_joint_pred_risk"]
grid["BRI"] = grid["BRI"].clip(0, 1)

# Failure-priority score:
# high heat exposure + high risk + low resilience
for c in ["high_HDSI_freq", "high_joint_pred_risk", "observed_joint_failure_rate"]:
    v = grid[c]
    grid[c + "_z01"] = (v - v.min()) / (v.max() - v.min() + 1e-9)

grid["low_BRI_z01"] = 1 - grid["BRI"]

grid["adaptation_priority"] = (
    0.35 * grid["high_HDSI_freq_z01"] +
    0.35 * grid["high_joint_pred_risk_z01"] +
    0.20 * grid["low_BRI_z01"] +
    0.10 * grid["observed_joint_failure_rate_z01"]
)

# State clustering:
# buffered / sensitive / failed
cluster_features = [
    "high_HDSI_freq",
    "dist_perm_water_km",
    "water_occurrence",
    "water_seasonality",
    "EVI_neg_rate",
    "LAI_neg_rate",
    "high_joint_pred_risk",
    "BRI"
]
cluster_features = [c for c in cluster_features if c in grid.columns]

Xc = grid[cluster_features].copy()
Xc = Xc.replace([np.inf, -np.inf], np.nan)
Xc = Xc.fillna(Xc.median())

# standardize
Xc_std = (Xc - Xc.mean()) / (Xc.std() + 1e-9)

gmm = GaussianMixture(n_components=3, random_state=42, covariance_type="full")
grid["state_cluster"] = gmm.fit_predict(Xc_std)

# label clusters by risk and resilience
cent = grid.groupby("state_cluster").agg(
    high_joint_pred_risk=("high_joint_pred_risk", "mean"),
    BRI=("BRI", "mean"),
    high_HDSI_freq=("high_HDSI_freq", "mean"),
    observed_joint_failure_rate=("observed_joint_failure_rate", "mean")
).reset_index()

# Highest risk = failed; lowest risk/highest BRI = buffered; remaining = sensitive
failed_cluster = cent.sort_values("high_joint_pred_risk", ascending=False).iloc[0]["state_cluster"]
buffered_cluster = cent.sort_values(["high_joint_pred_risk", "BRI"], ascending=[True, False]).iloc[0]["state_cluster"]

def label_state(c):
    if c == failed_cluster:
        return "Failed"
    if c == buffered_cluster:
        return "Buffered"
    return "Sensitive"

grid["buffering_state"] = grid["state_cluster"].apply(label_state)

state_summary = grid.groupby("buffering_state").agg(
    grids=("grid_id", "count"),
    HDSI_mean=("HDSI_mean", "mean"),
    high_HDSI_freq=("high_HDSI_freq", "mean"),
    dist_perm_water_km=("dist_perm_water_km", "mean"),
    EVI_neg_rate=("EVI_neg_rate", "mean"),
    LAI_neg_rate=("LAI_neg_rate", "mean"),
    high_joint_pred_risk=("high_joint_pred_risk", "mean"),
    BRI=("BRI", "mean"),
    adaptation_priority=("adaptation_priority", "mean")
).reset_index()

grid_fp = os.path.join(TAB_DIR, "grid_level_buffering_resilience_state.csv")
grid.to_csv(grid_fp, index=False)
state_fp = os.path.join(TAB_DIR, "buffering_state_summary.csv")
state_summary.to_csv(state_fp, index=False)

display(state_summary)
print("Saved:", grid_fp)
print("Saved:", state_fp)


# ============================================================
# 9. Figures
# ============================================================

china = read_china_boundary()

def plot_china_points(ax, data, col, cmap, title, vmin=None, vmax=None, size=1.0):
    if china is not None:
        china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")
    if gpd is not None:
        gdf = make_point_gdf(data)
        gdf.plot(
            ax=ax,
            column=col,
            markersize=size,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            legend=True,
            legend_kwds={"shrink": 0.62}
        )
    else:
        sc = ax.scatter(
            data["lon"], data["lat"],
            c=data[col],
            s=size,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )
        plt.colorbar(sc, ax=ax, shrink=0.62)
    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)


# ----------------------------
# Figure 1: framework + performance
# ----------------------------

fig = plt.figure(figsize=(11, 6.2))
gs = GridSpec(2, 3, figure=fig, width_ratios=[1.1, 1.1, 1.0], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

# A: conceptual diagram
ax = fig.add_subplot(gs[:, 0])
ax.axis("off")
panel_label(ax, "A", x=-0.08, y=1.02)

def rounded_box(ax, xy, w, h, text, fc):
    box = patches.FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        fc=fc, ec="#333333", lw=0.8
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=8.3)

def arrow(ax, a, b):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.2, color="#555555"))

rounded_box(ax, (0.08, 0.76), 0.75, 0.12, "Heat–dryness exposure\n(HDSI)", "#FEE0D2")
rounded_box(ax, (0.08, 0.56), 0.75, 0.12, "Aquatic buffering context\npermanent-water distance + stability", "#D9EAF7")
rounded_box(ax, (0.08, 0.36), 0.75, 0.12, "Vegetation response\nEVI / LAI negative risk", "#E5F5E0")
rounded_box(ax, (0.08, 0.16), 0.75, 0.12, "Failure boundary + resilience state\nBuffered / Sensitive / Failed", "#E6E6FA")
arrow(ax, (0.45, 0.76), (0.45, 0.68))
arrow(ax, (0.45, 0.56), (0.45, 0.48))
arrow(ax, (0.45, 0.36), (0.45, 0.28))
ax.text(0.05, 0.96, "Nonlinear aquatic-buffering\nfailure framework",
        fontsize=11, fontweight="bold", va="top")

# B: model metrics
ax = fig.add_subplot(gs[0, 1])
panel_label(ax, "B")
met = metrics_df.copy()
x = np.arange(len(met))
width = 0.35
ax.bar(x - width/2, met["test_auc"], width, label="ROC-AUC", color=COLORS["blue"])
ax.bar(x + width/2, met["test_pr_auc"], width, label="PR-AUC", color=COLORS["orange"])
ax.set_xticks(x)
ax.set_xticklabels(["EVI", "LAI"])
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Temporal validation")
ax.legend(frameon=False)
clean_ax(ax)

# C: calibration/Brier
ax = fig.add_subplot(gs[0, 2])
panel_label(ax, "C")
ax.bar(["EVI", "LAI"], met["test_brier"], color=[COLORS["green"], COLORS["purple"]])
ax.set_ylabel("Brier score")
ax.set_title("Probability calibration error")
clean_ax(ax)

# D: state summary
ax = fig.add_subplot(gs[1, 1:])
panel_label(ax, "D")
state_order = ["Buffered", "Sensitive", "Failed"]
ss = state_summary.set_index("buffering_state").reindex(state_order).reset_index()
x = np.arange(len(ss))
ax.bar(x - 0.25, ss["high_joint_pred_risk"], width=0.25, color=COLORS["red"], label="High-HDSI risk")
ax.bar(x, ss["BRI"], width=0.25, color=COLORS["blue"], label="BRI")
ax.bar(x + 0.25, ss["adaptation_priority"], width=0.25, color=COLORS["orange"], label="Priority")
ax.set_xticks(x)
ax.set_xticklabels(ss["buffering_state"])
ax.set_ylim(0, 1)
ax.set_ylabel("Index")
ax.set_title("Diagnosed buffering states")
ax.legend(frameon=False, ncol=3)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig1_framework_model_performance")
plt.show()


# ----------------------------
# Figure 2: nonlinear risk surfaces
# ----------------------------

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)

for ax, Z, title, label in [
    (axes[0], Zevi, "EVI failure risk surface", "A"),
    (axes[1], Zlai, "LAI failure risk surface", "B")
]:
    im = ax.contourf(Xdist, Yhdsi, Z, levels=np.linspace(0, 1, 21), cmap=cmap_failure, vmin=0, vmax=1)
    cs = ax.contour(Xdist, Yhdsi, Z, levels=[0.30, 0.40, 0.50], colors="#222222", linewidths=0.8)
    ax.clabel(cs, fmt={0.30:"30%", 0.40:"40%", 0.50:"50%"}, fontsize=7)
    ax.set_xlabel("Distance to permanent water (km)")
    ax.set_ylabel("Heat–dryness stress index (HDSI)")
    ax.set_title(title)
    panel_label(ax, label)
    clean_ax(ax)

cbar = fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, pad=0.02)
cbar.set_label("Predicted failure probability")

plt.tight_layout()
savefig(fig, "Fig2_HDSI_distance_failure_surfaces")
plt.show()


# ----------------------------
# Figure 3: fragility curves
# ----------------------------

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True, sharey=True)

band_order = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]
band_colors = {
    "0-0.5": "#08306B",
    "0.5-1": "#2171B5",
    "1-2": "#6BAED6",
    "2-5": "#FDAE61",
    "5-10": "#F46D43",
    ">10": "#A50026"
}

for ax, target, pred_col, obs_col, title, label in [
    (axes[0], "EVI", "EVI_pred_fail", "EVI_obs_fail", "EVI fragility curves", "A"),
    (axes[1], "LAI", "LAI_pred_fail", "LAI_obs_fail", "LAI fragility curves", "B")
]:
    for band in band_order:
        sub = frag_df[frag_df["dist_band_perm"] == band].sort_values("HDSI_mid")
        if len(sub) == 0:
            continue
        ax.plot(
            sub["HDSI_mid"], sub[pred_col],
            lw=2.0, color=band_colors.get(band, "#999999"),
            label=f"{band} km"
        )
        ax.scatter(
            sub["HDSI_mid"], sub[obs_col],
            s=12, color=band_colors.get(band, "#999999"),
            alpha=0.45
        )

    ax.set_xlabel("Heat–dryness stress index (HDSI)")
    ax.set_ylabel("Failure probability")
    ax.set_title(title)
    ax.set_ylim(0, 1)
    panel_label(ax, label)
    clean_ax(ax)

axes[1].legend(frameon=False, title="Distance band", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig3_fragility_curves_by_distance")
plt.show()


# ----------------------------
# Figure 4: failure boundary and acceleration zone
# ----------------------------

fig = plt.figure(figsize=(11, 6.2))
gs = GridSpec(2, 2, figure=fig, wspace=0.35, hspace=0.35)

# A: boundary curves
ax = fig.add_subplot(gs[0, 0])
bd = boundary_df[(boundary_df["risk_col"] == "joint_risk") & (boundary_df["risk_threshold"].isin([0.30, 0.40, 0.50]))].copy()
for thr, color in zip([0.30, 0.40, 0.50], [COLORS["blue"], COLORS["orange"], COLORS["red"]]):
    sub = bd[bd["risk_threshold"] == thr].copy()
    ax.plot(sub["HDSI"], sub["FBD_km"], color=color, lw=2.0, label=f"Risk>{int(thr*100)}%")
ax.set_xlabel("HDSI")
ax.set_ylabel("Failure boundary distance (km)")
ax.set_title("Joint failure boundary")
ax.legend(frameon=False)
panel_label(ax, "A")
clean_ax(ax)

# B: acceleration boundary
ax = fig.add_subplot(gs[0, 1])
acc = boundary_df[(boundary_df["risk_col"] == "joint_risk") & (boundary_df["risk_threshold"] == "max_acceleration")].copy()
ax.plot(acc["HDSI"], acc["FBD_km"], color=COLORS["green"], lw=2.2)
ax.set_xlabel("HDSI")
ax.set_ylabel("Distance of maximum acceleration (km)")
ax.set_title("Failure acceleration boundary")
panel_label(ax, "B")
clean_ax(ax)

# C/D: acceleration surfaces
for ax, dZ, title, label in [
    (fig.add_subplot(gs[1, 0]), dZdist_evi, "EVI failure acceleration zone", "C"),
    (fig.add_subplot(gs[1, 1]), dZdist_lai, "LAI failure acceleration zone", "D")
]:
    im = ax.contourf(Xdist, Yhdsi, dZ, levels=20, cmap=cmap_accel)
    ax.set_xlabel("Distance to permanent water (km)")
    ax.set_ylabel("HDSI")
    ax.set_title(title)
    panel_label(ax, label)
    clean_ax(ax)
    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("dRisk / dDistance")

plt.tight_layout()
savefig(fig, "Fig4_failure_boundary_acceleration")
plt.show()


# ----------------------------
# Figure 5: risk states
# ----------------------------

fig = plt.figure(figsize=(11, 6.2))
gs = GridSpec(2, 2, figure=fig, width_ratios=[1.3, 1], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

# A: state map
ax = fig.add_subplot(gs[:, 0])
if china is not None:
    china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

state_color = {"Buffered": COLORS["blue"], "Sensitive": COLORS["orange"], "Failed": COLORS["red"]}
for state, color in state_color.items():
    sub = grid[grid["buffering_state"] == state]
    ax.scatter(sub["lon"], sub["lat"], s=1.2, color=color, label=state, alpha=0.85)

ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Aquatic-buffering risk states")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(frameon=False, markerscale=4, loc="lower left")
ax.grid(alpha=0.15, lw=0.3)
panel_label(ax, "A")

# B: state radar-like profile using bars
ax = fig.add_subplot(gs[0, 1])
ss = state_summary.set_index("buffering_state").reindex(["Buffered", "Sensitive", "Failed"]).reset_index()
x = np.arange(len(ss))
ax.bar(x - 0.2, ss["high_HDSI_freq"], width=0.2, color=COLORS["orange"], label="High-HDSI freq")
ax.bar(x, ss["high_joint_pred_risk"], width=0.2, color=COLORS["red"], label="Risk")
ax.bar(x + 0.2, ss["BRI"], width=0.2, color=COLORS["blue"], label="BRI")
ax.set_xticks(x)
ax.set_xticklabels(ss["buffering_state"], rotation=0)
ax.set_ylim(0, 1)
ax.set_title("State profiles")
ax.legend(frameon=False)
panel_label(ax, "B")
clean_ax(ax)

# C: state composition by macro-region
ax = fig.add_subplot(gs[1, 1])
comp = (
    grid.groupby(["macro_region", "buffering_state"])
    .size()
    .reset_index(name="n")
)
comp["frac"] = comp.groupby("macro_region")["n"].transform(lambda x: x / x.sum())
pivot = comp.pivot_table(index="macro_region", columns="buffering_state", values="frac", fill_value=0)
pivot = pivot.reindex(columns=["Buffered", "Sensitive", "Failed"])
pivot = pivot.sort_values("Failed", ascending=True)

bottom = np.zeros(len(pivot))
for state in ["Buffered", "Sensitive", "Failed"]:
    vals = pivot[state].values
    ax.barh(pivot.index, vals, left=bottom, color=state_color[state], label=state)
    bottom += vals

ax.set_xlabel("Fraction of grids")
ax.set_title("Regional state composition")
ax.legend(frameon=False, ncol=1, bbox_to_anchor=(1.02, 1), loc="upper left")
panel_label(ax, "C")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_buffering_state_diagnosis")
plt.show()


# ----------------------------
# Figure 6: BRI and adaptation priority maps
# ----------------------------

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

plot_china_points(
    axes[0], grid, "BRI",
    cmap=LinearSegmentedColormap.from_list("bri", ["#67001f", "#f7f7f7", "#053061"]),
    title="Buffering Resilience Index (BRI)",
    vmin=0, vmax=1, size=1.1
)
panel_label(axes[0], "A")

plot_china_points(
    axes[1], grid, "adaptation_priority",
    cmap=cmap_priority,
    title="Adaptation priority from failure risk and low resilience",
    vmin=0, vmax=1, size=1.1
)
panel_label(axes[1], "B")

plt.tight_layout()
savefig(fig, "Fig6_BRI_adaptation_priority")
plt.show()


# ============================================================
# 10. Export compact interpretation tables
# ============================================================

# Failure boundary compact summary
boundary_compact = (
    boundary_df[
        (boundary_df["risk_col"].isin(["EVI_risk", "LAI_risk", "joint_risk"])) &
        (boundary_df["risk_threshold"].isin([0.30, 0.40, 0.50]))
    ]
    .groupby(["risk_col", "risk_threshold"])
    .agg(
        FBD_min=("FBD_km", "min"),
        FBD_median=("FBD_km", "median"),
        FBD_max=("FBD_km", "max"),
        available=("FBD_km", lambda x: x.notna().sum())
    )
    .reset_index()
)

boundary_compact_fp = os.path.join(TAB_DIR, "compact_failure_boundary_summary.csv")
boundary_compact.to_csv(boundary_compact_fp, index=False)
display(boundary_compact)

# State compact summary
display(state_summary)

print("\nAll outputs saved under:")
print(OUT_DIR)

In [ ]:
# ============================================================
# Main numerical diagnostics for nonlinear aquatic-buffering framework
# Goal:
#   Summarize what results are reliable, where effects are strong,
#   which regions matter, and whether failure-boundary results are usable.
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import geopandas as gpd
except Exception:
    gpd = None

# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
FRAME_DIR = os.path.join(AQUA_DIR, "nonlinear_failure_framework")
TAB_DIR = os.path.join(FRAME_DIR, "tables")
FIG_DIR = os.path.join(FRAME_DIR, "diagnostic_figs_for_interpretation")
OUT_DIR = os.path.join(FRAME_DIR, "main_result_diagnostics")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
PRED_OBS_FP = os.path.join(TAB_DIR, "all_observations_predicted_failure_risk.parquet")
GRID_STATE_FP = os.path.join(TAB_DIR, "grid_level_buffering_resilience_state.csv")
STATE_SUMMARY_FP = os.path.join(TAB_DIR, "buffering_state_summary.csv")
METRICS_FP = os.path.join(TAB_DIR, "model_temporal_validation_metrics.csv")
SURFACE_FP = os.path.join(TAB_DIR, "diagnostic_HDSI_distance_risk_surface.csv")
BOUNDARY_FP = os.path.join(TAB_DIR, "failure_boundary_distance_by_HDSI.csv")
FRAGILITY_FP = os.path.join(TAB_DIR, "fragility_curves_by_distance_band.csv")

CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

print("Check files:")
for fp in [PANEL_FP, PRED_OBS_FP, GRID_STATE_FP, STATE_SUMMARY_FP, METRICS_FP, SURFACE_FP, BOUNDARY_FP, FRAGILITY_FP]:
    print(os.path.basename(fp), os.path.exists(fp), fp)

# ============================================================
# 1. Style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222"
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2, linewidth=0.4)

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

def q(x, p):
    return x.quantile(p)

# ============================================================
# 2. Load data
# ============================================================

panel = pd.read_parquet(PANEL_FP)
pred_obs = pd.read_parquet(PRED_OBS_FP)
grid_state = pd.read_csv(GRID_STATE_FP)
state_summary = pd.read_csv(STATE_SUMMARY_FP)
metrics = pd.read_csv(METRICS_FP)
surface = pd.read_csv(SURFACE_FP)
boundary = pd.read_csv(BOUNDARY_FP)
fragility = pd.read_csv(FRAGILITY_FP)

print("\nLoaded:")
print("panel:", panel.shape)
print("pred_obs:", pred_obs.shape)
print("grid_state:", grid_state.shape)
print("surface:", surface.shape)

# Ensure macro region
if "macro_region" not in pred_obs.columns:
    pred_obs["macro_region"] = pred_obs.apply(assign_macro_region, axis=1)

if "macro_region" not in grid_state.columns:
    grid_state["macro_region"] = grid_state.apply(assign_macro_region, axis=1)

# Ensure joint observed failure
if "observed_joint_failure" not in pred_obs.columns:
    pred_obs["observed_joint_failure"] = (
        (pred_obs["EVI_neg_z05"].astype(int) + pred_obs["LAI_neg_z05"].astype(int)) >= 1
    ).astype(int)

# Ensure high HDSI group
h75 = pred_obs["HDSI"].quantile(0.75)
h25 = pred_obs["HDSI"].quantile(0.25)

pred_obs["HDSI_group"] = "mid"
pred_obs.loc[pred_obs["HDSI"] <= h25, "HDSI_group"] = "low"
pred_obs.loc[pred_obs["HDSI"] >= h75, "HDSI_group"] = "high"

# Distance bands
dist_bins = [0, 0.5, 1, 2, 5, 10, 20]
dist_labels = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]

pred_obs["dist_band_perm"] = pd.cut(
    pred_obs["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
).astype(str)

# ============================================================
# 3. Main overview
# ============================================================

overview_rows = []

overview_rows.append({"metric": "panel_rows", "value": len(panel)})
overview_rows.append({"metric": "model_rows", "value": len(pred_obs)})
overview_rows.append({"metric": "unique_grids", "value": pred_obs["grid_id"].nunique()})
overview_rows.append({"metric": "years", "value": ",".join(map(str, sorted(pred_obs["year"].dropna().astype(int).unique())))})
overview_rows.append({"metric": "EVI_neg_rate", "value": pred_obs["EVI_neg_z05"].mean()})
overview_rows.append({"metric": "LAI_neg_rate", "value": pred_obs["LAI_neg_z05"].mean()})
overview_rows.append({"metric": "joint_failure_rate", "value": pred_obs["observed_joint_failure"].mean()})
overview_rows.append({"metric": "HDSI_q25", "value": h25})
overview_rows.append({"metric": "HDSI_q75", "value": h75})
overview_rows.append({"metric": "dist_perm_mean_km", "value": pred_obs["dist_perm_water_km"].mean()})
overview_rows.append({"metric": "dist_perm_median_km", "value": pred_obs["dist_perm_water_km"].median()})
overview_rows.append({"metric": "dist_perm_p90_km", "value": pred_obs["dist_perm_water_km"].quantile(0.90)})
overview_rows.append({"metric": "dist_perm_max_km", "value": pred_obs["dist_perm_water_km"].max()})

overview = pd.DataFrame(overview_rows)
display(overview)

overview.to_csv(os.path.join(OUT_DIR, "1_main_overview.csv"), index=False)

# ============================================================
# 4. Region overview: what do regions look like?
# ============================================================

region_overview = (
    pred_obs
    .groupby("macro_region")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        HDSI_p75=("HDSI", lambda x: q(x, 0.75)),
        dist_perm_mean=("dist_perm_water_km", "mean"),
        dist_perm_median=("dist_perm_water_km", "median"),
        dist_perm_p90=("dist_perm_water_km", lambda x: q(x, 0.90)),
        water_occurrence_mean=("water_occurrence", "mean") if "water_occurrence" in pred_obs.columns else ("HDSI", "mean"),
        water_seasonality_mean=("water_seasonality", "mean") if "water_seasonality" in pred_obs.columns else ("HDSI", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        joint_failure_rate=("observed_joint_failure", "mean"),
        EVI_pred_risk=("EVI_pred_risk", "mean"),
        LAI_pred_risk=("LAI_pred_risk", "mean"),
        joint_pred_risk=("joint_pred_risk", "mean")
    )
    .reset_index()
    .sort_values("joint_failure_rate", ascending=False)
)

display(region_overview)
region_overview.to_csv(os.path.join(OUT_DIR, "2_region_overview.csv"), index=False)

# ============================================================
# 5. Region × state composition
# ============================================================

region_state = (
    grid_state
    .groupby(["macro_region", "buffering_state"])
    .size()
    .reset_index(name="grids")
)

region_state["region_total_grids"] = region_state.groupby("macro_region")["grids"].transform("sum")
region_state["state_fraction"] = region_state["grids"] / region_state["region_total_grids"]

region_state_wide = (
    region_state
    .pivot_table(index="macro_region", columns="buffering_state", values="state_fraction", fill_value=0)
    .reset_index()
)

for c in ["Buffered", "Sensitive", "Failed"]:
    if c not in region_state_wide.columns:
        region_state_wide[c] = 0

region_state_wide = region_state_wide.sort_values("Failed", ascending=False)

display(region_state_wide)
region_state_wide.to_csv(os.path.join(OUT_DIR, "3_region_state_composition.csv"), index=False)

# ============================================================
# 6. Region risk-resilience summary
# ============================================================

region_resilience = (
    grid_state
    .groupby("macro_region")
    .agg(
        grids=("grid_id", "count"),
        HDSI_mean=("HDSI_mean", "mean"),
        high_HDSI_freq=("high_HDSI_freq", "mean"),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_rate", "mean"),
        LAI_neg_rate=("LAI_neg_rate", "mean"),
        high_joint_pred_risk=("high_joint_pred_risk", "mean"),
        BRI=("BRI", "mean"),
        adaptation_priority=("adaptation_priority", "mean")
    )
    .reset_index()
    .sort_values("adaptation_priority", ascending=False)
)

display(region_resilience)
region_resilience.to_csv(os.path.join(OUT_DIR, "4_region_risk_resilience.csv"), index=False)

# ============================================================
# 7. Fragility summary by distance band and HDSI
# ============================================================

# Summarize how much risk rises from low HDSI to high HDSI in each distance band.
frag_summary_rows = []

for band in dist_labels:
    sub = fragility[fragility["dist_band_perm"] == band].copy()
    if len(sub) == 0:
        continue

    sub = sub.sort_values("HDSI_mid")
    low = sub.iloc[0]
    high = sub.iloc[-1]

    frag_summary_rows.append({
        "dist_band_perm": band,
        "HDSI_low_mid": low["HDSI_mid"],
        "HDSI_high_mid": high["HDSI_mid"],
        "EVI_pred_low": low["EVI_pred_fail"],
        "EVI_pred_high": high["EVI_pred_fail"],
        "EVI_fragility_increase": high["EVI_pred_fail"] - low["EVI_pred_fail"],
        "LAI_pred_low": low["LAI_pred_fail"],
        "LAI_pred_high": high["LAI_pred_fail"],
        "LAI_fragility_increase": high["LAI_pred_fail"] - low["LAI_pred_fail"],
        "joint_pred_low": low["joint_pred_fail"],
        "joint_pred_high": high["joint_pred_fail"],
        "joint_fragility_increase": high["joint_pred_fail"] - low["joint_pred_fail"]
    })

frag_summary = pd.DataFrame(frag_summary_rows)
display(frag_summary)

frag_summary.to_csv(os.path.join(OUT_DIR, "5_distance_band_fragility_summary.csv"), index=False)

# ============================================================
# 8. Failure boundary diagnostics
# ============================================================

# The current FBD values are suspicious if many are 0.
# Diagnose: how often boundary exists, how often boundary=0, and risk ranges.

boundary_diag = (
    boundary[
        boundary["risk_threshold"].isin([0.30, 0.40, 0.50])
    ]
    .groupby(["risk_col", "risk_threshold"])
    .agg(
        n=("FBD_km", "size"),
        available=("FBD_km", lambda x: x.notna().sum()),
        available_frac=("FBD_km", lambda x: x.notna().mean()),
        zero_boundary_frac=("FBD_km", lambda x: (x.fillna(-999) == 0).mean()),
        FBD_min=("FBD_km", "min"),
        FBD_median=("FBD_km", "median"),
        FBD_max=("FBD_km", "max")
    )
    .reset_index()
)

display(boundary_diag)
boundary_diag.to_csv(os.path.join(OUT_DIR, "6_failure_boundary_diagnostics.csv"), index=False)

# ============================================================
# 9. Surface gradient diagnostics
# ============================================================

def surface_diagnostics(surface, risk_col):
    rows = []
    for h, g in surface.groupby("HDSI"):
        g = g.sort_values("dist_perm_water_km")
        x = g["dist_perm_water_km"].values.astype(float)
        y = g[risk_col].values.astype(float)

        if len(x) < 5:
            continue

        dydx = np.gradient(y, x)
        rows.append({
            "risk_col": risk_col,
            "HDSI": h,
            "risk_min": np.nanmin(y),
            "risk_max": np.nanmax(y),
            "risk_delta_far_near": y[-1] - y[0],
            "max_positive_gradient": np.nanmax(dydx),
            "distance_at_max_gradient": x[np.nanargmax(dydx)],
            "risk_at_near0": y[0],
            "risk_at_farmax": y[-1]
        })
    return pd.DataFrame(rows)

surf_diag = pd.concat([
    surface_diagnostics(surface, "EVI_risk"),
    surface_diagnostics(surface, "LAI_risk"),
    surface_diagnostics(surface, "joint_risk")
], ignore_index=True)

surf_summary = (
    surf_diag
    .groupby("risk_col")
    .agg(
        n_HDSI=("HDSI", "count"),
        mean_delta_far_near=("risk_delta_far_near", "mean"),
        median_delta_far_near=("risk_delta_far_near", "median"),
        mean_max_gradient=("max_positive_gradient", "mean"),
        median_distance_max_gradient=("distance_at_max_gradient", "median"),
        risk_min_overall=("risk_min", "min"),
        risk_max_overall=("risk_max", "max")
    )
    .reset_index()
)

display(surf_summary)
surf_diag.to_csv(os.path.join(OUT_DIR, "7_surface_gradient_diagnostics_full.csv"), index=False)
surf_summary.to_csv(os.path.join(OUT_DIR, "7_surface_gradient_diagnostics_summary.csv"), index=False)

# ============================================================
# 10. Model metrics diagnosis
# ============================================================

metrics_diag = metrics.copy()

def judge_auc(x):
    if x >= 0.75:
        return "good"
    if x >= 0.65:
        return "moderate"
    if x >= 0.60:
        return "weak"
    return "poor"

metrics_diag["AUC_quality"] = metrics_diag["test_auc"].apply(judge_auc)
metrics_diag["PR_AUC_over_event_rate"] = metrics_diag["test_pr_auc"] / metrics_diag["test_event_rate"]

display(metrics_diag)
metrics_diag.to_csv(os.path.join(OUT_DIR, "8_model_metrics_diagnosis.csv"), index=False)

# ============================================================
# 11. Water-context summary
# ============================================================

# Permanent/seasonal/far groups
wc = pred_obs.copy()
wc["water_context_simple"] = "far_or_weak_water"
wc.loc[wc["dist_perm_water_km"] < 1.0, "water_context_simple"] = "near_permanent_lt1km"
wc.loc[
    (wc["dist_perm_water_km"] >= 1.0) &
    (wc["dist_seasonal_water_km"] < 1.0),
    "water_context_simple"
] = "near_seasonal_lt1km"

water_context = (
    wc
    .groupby(["water_context_simple", "HDSI_group"])
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        dist_perm_mean=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        joint_failure_rate=("observed_joint_failure", "mean"),
        EVI_pred_risk=("EVI_pred_risk", "mean"),
        LAI_pred_risk=("LAI_pred_risk", "mean"),
        joint_pred_risk=("joint_pred_risk", "mean")
    )
    .reset_index()
)

display(water_context)
water_context.to_csv(os.path.join(OUT_DIR, "9_water_context_summary.csv"), index=False)

# ============================================================
# 12. Warning flags
# ============================================================

warnings_list = []

# Model quality flags
for _, r in metrics_diag.iterrows():
    if r["test_auc"] < 0.60:
        warnings_list.append({
            "issue": "low_model_auc",
            "target": r["target"],
            "detail": f"AUC={r['test_auc']:.3f}; model may not support strong ML-based conclusions."
        })

# Boundary flags
for _, r in boundary_diag.iterrows():
    if r["zero_boundary_frac"] > 0.5:
        warnings_list.append({
            "issue": "many_zero_failure_boundaries",
            "target": f"{r['risk_col']}_{r['risk_threshold']}",
            "detail": f"zero_boundary_frac={r['zero_boundary_frac']:.2f}; FBD may be hard to interpret."
        })

# Surface gradient flags
for _, r in surf_summary.iterrows():
    if abs(r["mean_delta_far_near"]) < 0.02:
        warnings_list.append({
            "issue": "weak_distance_gradient_on_surface",
            "target": r["risk_col"],
            "detail": f"mean_delta_far_near={r['mean_delta_far_near']:.3f}; distance gradient may be weak."
        })

warning_flags = pd.DataFrame(warnings_list)
display(warning_flags)
warning_flags.to_csv(os.path.join(OUT_DIR, "10_key_warning_flags.csv"), index=False)

# ============================================================
# 13. Diagnostic figures
# ============================================================

# Figure A: Model metrics
fig, axes = plt.subplots(1, 3, figsize=(11, 3.6))

x = np.arange(len(metrics_diag))
axes[0].bar(x, metrics_diag["test_auc"], color=[COL["blue"], COL["purple"]])
axes[0].axhline(0.6, color=COL["red"], linestyle="--", linewidth=1)
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_diag["target"].str.replace("_neg_z05", ""))
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("AUC")
axes[0].set_title("Temporal test AUC")
clean_ax(axes[0])

axes[1].bar(x, metrics_diag["test_pr_auc"], color=[COL["orange"], COL["green"]])
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_diag["target"].str.replace("_neg_z05", ""))
axes[1].set_ylabel("PR-AUC")
axes[1].set_title("Temporal test PR-AUC")
clean_ax(axes[1])

axes[2].bar(x, metrics_diag["test_brier"], color=[COL["gray"], COL["gray"]])
axes[2].set_xticks(x)
axes[2].set_xticklabels(metrics_diag["target"].str.replace("_neg_z05", ""))
axes[2].set_ylabel("Brier score")
axes[2].set_title("Probability error")
clean_ax(axes[2])

plt.tight_layout()
savefig(fig, "diagnostic_model_metrics")
plt.show()

# Figure B: Region risk/resilience
plot_reg = region_resilience.sort_values("adaptation_priority", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5.2))
y = np.arange(len(plot_reg))

ax.hlines(y, 0, plot_reg["adaptation_priority"], color="#BBBBBB", linewidth=1.2)
ax.scatter(plot_reg["adaptation_priority"], y, s=70, color=COL["red"], label="Adaptation priority")
ax.scatter(plot_reg["BRI"], y, s=45, color=COL["blue"], label="BRI")
ax.set_yticks(y)
ax.set_yticklabels(plot_reg["macro_region"])
ax.set_xlabel("Index value")
ax.set_title("Regional risk, resilience and adaptation priority")
ax.legend(frameon=False)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "diagnostic_region_risk_resilience")
plt.show()

# Figure C: Region state composition
plot_state = region_state_wide.sort_values("Failed", ascending=True)
fig, ax = plt.subplots(figsize=(8, 5.2))

bottom = np.zeros(len(plot_state))
state_cols = {
    "Buffered": COL["blue"],
    "Sensitive": COL["orange"],
    "Failed": COL["red"]
}

for state in ["Buffered", "Sensitive", "Failed"]:
    vals = plot_state[state].values
    ax.barh(plot_state["macro_region"], vals, left=bottom, color=state_cols[state], label=state)
    bottom += vals

ax.set_xlabel("Fraction of grids")
ax.set_title("Buffering states by region")
ax.legend(frameon=False, loc="lower right")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "diagnostic_region_state_composition")
plt.show()

# Figure D: Fragility summary by distance band
fig, ax = plt.subplots(figsize=(8, 4.8))

x = np.arange(len(frag_summary))
ax.plot(x, frag_summary["EVI_fragility_increase"], marker="o", color=COL["green"], label="EVI")
ax.plot(x, frag_summary["LAI_fragility_increase"], marker="o", color=COL["purple"], label="LAI")
ax.plot(x, frag_summary["joint_fragility_increase"], marker="o", color=COL["red"], label="Joint")
ax.set_xticks(x)
ax.set_xticklabels(frag_summary["dist_band_perm"], rotation=30)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_ylabel("Risk increase from low to high HDSI")
ax.set_xlabel("Distance to permanent water band (km)")
ax.set_title("Fragility increase across aquatic-distance bands")
ax.legend(frameon=False)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "diagnostic_fragility_increase_by_distance")
plt.show()

# Figure E: Surface gradient diagnostic
fig, ax = plt.subplots(figsize=(8, 4.8))

for risk_col, color in [("EVI_risk", COL["green"]), ("LAI_risk", COL["purple"]), ("joint_risk", COL["red"])]:
    sub = surf_diag[surf_diag["risk_col"] == risk_col].sort_values("HDSI")
    ax.plot(sub["HDSI"], sub["risk_delta_far_near"], color=color, linewidth=2, label=risk_col)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xlabel("HDSI")
ax.set_ylabel("Risk difference: far distance - near distance")
ax.set_title("Does risk increase with distance on diagnostic surface?")
ax.legend(frameon=False)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "diagnostic_surface_distance_gradient")
plt.show()

print("\nAll diagnostic outputs saved to:")
print(OUT_DIR)
print(FIG_DIR)

In [ ]:
# ============================================================
# Reproduce main results for aquatic buffering and vegetation fragility
# Main framework:
#   1. Heat–dryness stress: HDSI
#   2. Vegetation negative response: EVI_neg / LAI_neg / joint_failure
#   3. Aquatic context:
#        near permanent water
#        near seasonal water
#        far or weak water
#   4. Distance-band fragility:
#        Risk(high HDSI) - Risk(low HDSI)
#   5. Regional heterogeneity
#   6. Bootstrap 95% CI
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import geopandas as gpd
except Exception:
    gpd = None

# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_DIR = os.path.join(AQUA_DIR, "_main_results_reproduced")
FIG_DIR = os.path.join(OUT_DIR, "figs")
TAB_DIR = os.path.join(OUT_DIR, "tables")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Nature-like plotting style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300,
})

COL = {
    "blue": "#3B6EA8",
    "light_blue": "#9BBBD8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "teal": "#009E73",
    "purple": "#7E62A3",
    "gray": "#666666",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
}

CONTEXT_COLORS = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

HDSI_COLORS = {
    "low": COL["blue"],
    "mid": COL["orange"],
    "high": COL["red"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def read_china_boundary():
    if gpd is None:
        return None
    if not os.path.exists(CHINA_BOUNDARY_PATH):
        print("[WARN] China boundary not found.")
        return None
    try:
        china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
    except Exception:
        china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
    china = china.dissolve().reset_index(drop=True)
    return china


# ============================================================
# 2. Load data and build core variables
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

# Convert important variables
num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Build HDSI if missing
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Build negative response variables if missing
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce").astype("Int64")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce").astype("Int64")

df["joint_failure"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Main valid sample
d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna() &
    df["dist_seasonal_water_km"].notna() &
    df["EVI_neg_z05"].notna() &
    df["LAI_neg_z05"].notna()
].copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["joint_failure"] = d["joint_failure"].astype(int)

# HDSI groups
q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)

d["HDSI_group"] = "mid"
d.loc[d["HDSI"] <= q25, "HDSI_group"] = "low"
d.loc[d["HDSI"] >= q75, "HDSI_group"] = "high"

d["HDSI_group"] = pd.Categorical(
    d["HDSI_group"],
    categories=["low", "mid", "high"],
    ordered=True
)

# Water context classification
# Priority:
#   1. near permanent water < 1 km
#   2. near seasonal water < 1 km but not near permanent
#   3. far or weak water
d["water_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "water_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "water_context"
] = "near_seasonal_lt1km"

d["water_context"] = pd.Categorical(
    d["water_context"],
    categories=["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"],
    ordered=True
)

# Permanent-water distance bands
dist_bins = [0, 0.5, 1, 2, 5, 10, 20]
dist_labels = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]

d["dist_band_perm"] = pd.cut(
    d["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
)

d["dist_band_perm"] = pd.Categorical(
    d["dist_band_perm"],
    categories=dist_labels,
    ordered=True
)

# Macro regions
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in d.columns:
    d["macro_region"] = d.apply(assign_macro_region, axis=1)
else:
    d["macro_region"] = d["macro_region"].astype(str)

print("Valid sample:", d.shape)
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year"].dropna().astype(int).unique()))
print("HDSI Q25/Q75:", q25, q75)
print("EVI neg rate:", d["EVI_neg_z05"].mean())
print("LAI neg rate:", d["LAI_neg_z05"].mean())
print("Joint failure rate:", d["joint_failure"].mean())


# ============================================================
# 3. Bootstrap helpers
# ============================================================

def bootstrap_mean_ci(values, n_boot=500, seed=42):
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan
    mean = np.mean(values)
    if n == 1:
        return mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        sample = rng.choice(values, size=n, replace=True)
        boots.append(np.mean(sample))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi

def bootstrap_diff_ci(a, b, n_boot=500, seed=42):
    """
    Difference: mean(a) - mean(b)
    """
    a = np.asarray(a)
    b = np.asarray(b)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    diff = np.mean(a) - np.mean(b)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots.append(np.mean(aa) - np.mean(bb))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff, lo, hi


# ============================================================
# 4. Data overview table
# ============================================================

overview_rows = [
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].dropna().astype(int).unique())))},
    {"metric": "HDSI_Q25", "value": q25},
    {"metric": "HDSI_Q75", "value": q75},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "joint_failure_rate", "value": d["joint_failure"].mean()},
    {"metric": "dist_perm_mean_km", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_median_km", "value": d["dist_perm_water_km"].median()},
    {"metric": "dist_perm_p90_km", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max_km", "value": d["dist_perm_water_km"].max()},
]

overview = pd.DataFrame(overview_rows)
overview.to_csv(os.path.join(TAB_DIR, "main_overview.csv"), index=False)
display(overview)


# ============================================================
# 5. Main result 1:
#    Water context × HDSI group × response risk + CI
# ============================================================

response_cols = {
    "EVI": "EVI_neg_z05",
    "LAI": "LAI_neg_z05",
    "Joint": "joint_failure"
}

rows = []

for context in d["water_context"].cat.categories:
    for hgroup in d["HDSI_group"].cat.categories:
        sub = d[
            (d["water_context"] == context) &
            (d["HDSI_group"] == hgroup)
        ].copy()

        for resp_name, col in response_cols.items():
            mean, lo, hi = bootstrap_mean_ci(sub[col].values, n_boot=500, seed=42)
            rows.append({
                "water_context": context,
                "HDSI_group": hgroup,
                "response": resp_name,
                "risk": mean,
                "ci_low": lo,
                "ci_high": hi,
                "rows": len(sub),
                "grids": sub["grid_id"].nunique()
            })

water_context_risk = pd.DataFrame(rows)
water_context_risk.to_csv(os.path.join(TAB_DIR, "water_context_HDSI_risk_with_CI.csv"), index=False)
display(water_context_risk.head(20))


# ============================================================
# 6. Main result 2:
#    High-HDSI water context contrasts + CI
# ============================================================

contrast_rows = []

high = d[d["HDSI_group"] == "high"].copy()

for resp_name, col in response_cols.items():
    near_perm = high[high["water_context"] == "near_permanent_lt1km"][col].values
    near_season = high[high["water_context"] == "near_seasonal_lt1km"][col].values
    far_weak = high[high["water_context"] == "far_or_weak_water"][col].values

    diff_far, lo_far, hi_far = bootstrap_diff_ci(far_weak, near_perm, n_boot=500, seed=42)
    diff_season, lo_season, hi_season = bootstrap_diff_ci(near_season, near_perm, n_boot=500, seed=43)

    contrast_rows.append({
        "response": resp_name,
        "contrast": "far_or_weak - near_permanent",
        "risk_difference": diff_far,
        "ci_low": lo_far,
        "ci_high": hi_far,
        "n_group1": len(far_weak),
        "n_group2": len(near_perm)
    })

    contrast_rows.append({
        "response": resp_name,
        "contrast": "near_seasonal - near_permanent",
        "risk_difference": diff_season,
        "ci_low": lo_season,
        "ci_high": hi_season,
        "n_group1": len(near_season),
        "n_group2": len(near_perm)
    })

high_context_contrasts = pd.DataFrame(contrast_rows)
high_context_contrasts.to_csv(os.path.join(TAB_DIR, "high_HDSI_water_context_contrasts_with_CI.csv"), index=False)
display(high_context_contrasts)


# ============================================================
# 7. Main result 3:
#    Distance band × HDSI group risk + CI
# ============================================================

rows = []

for band in dist_labels:
    for hgroup in ["low", "mid", "high"]:
        sub = d[
            (d["dist_band_perm"].astype(str) == band) &
            (d["HDSI_group"].astype(str) == hgroup)
        ].copy()

        for resp_name, col in response_cols.items():
            mean, lo, hi = bootstrap_mean_ci(sub[col].values, n_boot=500, seed=42)
            rows.append({
                "dist_band_perm": band,
                "HDSI_group": hgroup,
                "response": resp_name,
                "risk": mean,
                "ci_low": lo,
                "ci_high": hi,
                "rows": len(sub),
                "grids": sub["grid_id"].nunique()
            })

dist_band_risk = pd.DataFrame(rows)
dist_band_risk.to_csv(os.path.join(TAB_DIR, "distance_band_HDSI_risk_with_CI.csv"), index=False)
display(dist_band_risk.head(20))


# ============================================================
# 8. Main result 4:
#    Fragility increase = Risk(high HDSI) - Risk(low HDSI)
#    by permanent-water distance band
# ============================================================

frag_rows = []

for band in dist_labels:
    sub_low = d[
        (d["dist_band_perm"].astype(str) == band) &
        (d["HDSI_group"].astype(str) == "low")
    ].copy()

    sub_high = d[
        (d["dist_band_perm"].astype(str) == band) &
        (d["HDSI_group"].astype(str) == "high")
    ].copy()

    for resp_name, col in response_cols.items():
        diff, lo, hi = bootstrap_diff_ci(
            sub_high[col].values,
            sub_low[col].values,
            n_boot=500,
            seed=42
        )

        frag_rows.append({
            "dist_band_perm": band,
            "response": resp_name,
            "fragility_increase": diff,
            "ci_low": lo,
            "ci_high": hi,
            "risk_high": np.mean(sub_high[col].values) if len(sub_high) else np.nan,
            "risk_low": np.mean(sub_low[col].values) if len(sub_low) else np.nan,
            "n_high": len(sub_high),
            "n_low": len(sub_low)
        })

fragility = pd.DataFrame(frag_rows)
fragility.to_csv(os.path.join(TAB_DIR, "fragility_increase_by_distance_band_with_CI.csv"), index=False)
display(fragility)


# ============================================================
# 9. Main result 5:
#    Aquatic Fragility Buffering, AFB
#    AFB = Fragility_far_or_weak - Fragility_near_permanent
# ============================================================

afb_rows = []

for resp_name, col in response_cols.items():
    near = d[d["water_context"] == "near_permanent_lt1km"]
    far = d[d["water_context"] == "far_or_weak_water"]
    season = d[d["water_context"] == "near_seasonal_lt1km"]

    near_low = near[near["HDSI_group"] == "low"][col].values
    near_high = near[near["HDSI_group"] == "high"][col].values

    far_low = far[far["HDSI_group"] == "low"][col].values
    far_high = far[far["HDSI_group"] == "high"][col].values

    season_low = season[season["HDSI_group"] == "low"][col].values
    season_high = season[season["HDSI_group"] == "high"][col].values

    rng = np.random.default_rng(42)
    boots_far = []
    boots_season = []

    for _ in range(500):
        np_low = rng.choice(near_low, size=len(near_low), replace=True)
        np_high = rng.choice(near_high, size=len(near_high), replace=True)

        fw_low = rng.choice(far_low, size=len(far_low), replace=True)
        fw_high = rng.choice(far_high, size=len(far_high), replace=True)

        ns_low = rng.choice(season_low, size=len(season_low), replace=True)
        ns_high = rng.choice(season_high, size=len(season_high), replace=True)

        frag_near = np.mean(np_high) - np.mean(np_low)
        frag_far = np.mean(fw_high) - np.mean(fw_low)
        frag_season = np.mean(ns_high) - np.mean(ns_low)

        boots_far.append(frag_far - frag_near)
        boots_season.append(frag_season - frag_near)

    afb_far = np.mean(boots_far)
    afb_far_lo, afb_far_hi = np.percentile(boots_far, [2.5, 97.5])

    afb_season = np.mean(boots_season)
    afb_season_lo, afb_season_hi = np.percentile(boots_season, [2.5, 97.5])

    afb_rows.append({
        "response": resp_name,
        "contrast": "far_or_weak fragility - near_permanent fragility",
        "AFB": afb_far,
        "ci_low": afb_far_lo,
        "ci_high": afb_far_hi
    })

    afb_rows.append({
        "response": resp_name,
        "contrast": "near_seasonal fragility - near_permanent fragility",
        "AFB": afb_season,
        "ci_low": afb_season_lo,
        "ci_high": afb_season_hi
    })

afb = pd.DataFrame(afb_rows)
afb.to_csv(os.path.join(TAB_DIR, "aquatic_fragility_buffering_AFB_with_CI.csv"), index=False)
display(afb)


# ============================================================
# 10. Regional results:
#     Region-level risk, fragility, and AFB
# ============================================================

region_summary = (
    d.groupby("macro_region")
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        HDSI_mean=("HDSI", "mean"),
        HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
        dist_perm_mean=("dist_perm_water_km", "mean"),
        dist_perm_median=("dist_perm_water_km", "median"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        joint_failure_rate=("joint_failure", "mean")
    )
    .reset_index()
    .sort_values("joint_failure_rate", ascending=False)
)

region_summary.to_csv(os.path.join(TAB_DIR, "region_summary.csv"), index=False)
display(region_summary)

# Region water context high-HDSI risks
rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()

    for context in d["water_context"].cat.categories:
        sub = dr[
            (dr["water_context"] == context) &
            (dr["HDSI_group"] == "high")
        ].copy()

        for resp_name, col in response_cols.items():
            mean, lo, hi = bootstrap_mean_ci(sub[col].values, n_boot=400, seed=42)
            rows.append({
                "macro_region": region,
                "water_context": context,
                "HDSI_group": "high",
                "response": resp_name,
                "risk": mean,
                "ci_low": lo,
                "ci_high": hi,
                "rows": len(sub),
                "grids": sub["grid_id"].nunique()
            })

region_high_context = pd.DataFrame(rows)
region_high_context.to_csv(os.path.join(TAB_DIR, "region_high_HDSI_water_context_risk_with_CI.csv"), index=False)
display(region_high_context.head(20))

# Region AFB for joint failure
region_afb_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()

    near = dr[dr["water_context"] == "near_permanent_lt1km"]
    far = dr[dr["water_context"] == "far_or_weak_water"]

    if len(near) < 100 or len(far) < 100:
        continue

    for resp_name, col in response_cols.items():
        near_low = near[near["HDSI_group"] == "low"][col].values
        near_high = near[near["HDSI_group"] == "high"][col].values
        far_low = far[far["HDSI_group"] == "low"][col].values
        far_high = far[far["HDSI_group"] == "high"][col].values

        if min(len(near_low), len(near_high), len(far_low), len(far_high)) < 30:
            continue

        rng = np.random.default_rng(42)
        boots = []

        for _ in range(400):
            nl = rng.choice(near_low, size=len(near_low), replace=True)
            nh = rng.choice(near_high, size=len(near_high), replace=True)
            fl = rng.choice(far_low, size=len(far_low), replace=True)
            fh = rng.choice(far_high, size=len(far_high), replace=True)

            frag_near = np.mean(nh) - np.mean(nl)
            frag_far = np.mean(fh) - np.mean(fl)
            boots.append(frag_far - frag_near)

        region_afb_rows.append({
            "macro_region": region,
            "response": resp_name,
            "AFB_far_minus_near": np.mean(boots),
            "ci_low": np.percentile(boots, 2.5),
            "ci_high": np.percentile(boots, 97.5),
            "n_near": len(near),
            "n_far": len(far)
        })

region_afb = pd.DataFrame(region_afb_rows)
region_afb.to_csv(os.path.join(TAB_DIR, "region_AFB_far_minus_near_with_CI.csv"), index=False)
display(region_afb)


# ============================================================
# 11. Figure 1:
#     Data overview and framework
# ============================================================

fig = plt.figure(figsize=(11, 7))
gs = fig.add_gridspec(2, 3, width_ratios=[1.1, 1, 1], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

# Panel A: framework
ax = fig.add_subplot(gs[:, 0])
ax.axis("off")
panel_label(ax, "A", x=-0.04, y=1.02)

def rounded_box(ax, xy, w, h, text, fc):
    box = mpl.patches.FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.02,rounding_size=0.025",
        fc=fc, ec="#333333", lw=0.8
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=8.5, linespacing=1.25)

def arrow(ax, a, b):
    ax.annotate("", xy=b, xytext=a, arrowprops=dict(arrowstyle="->", lw=1.2, color="#555555"))

rounded_box(ax, (0.10, 0.78), 0.75, 0.12, "Compound heat–dryness\nHDSI", "#FEE0D2")
rounded_box(ax, (0.10, 0.58), 0.75, 0.12, "Aquatic context\npermanent / seasonal / far-weak", "#D9EAF7")
rounded_box(ax, (0.10, 0.38), 0.75, 0.12, "Vegetation response\nEVI, LAI, joint failure", "#E5F5E0")
rounded_box(ax, (0.10, 0.18), 0.75, 0.12, "Fragility analysis\nRisk(high) - Risk(low)", "#E6E6FA")

arrow(ax, (0.48, 0.78), (0.48, 0.70))
arrow(ax, (0.48, 0.58), (0.48, 0.50))
arrow(ax, (0.48, 0.38), (0.48, 0.30))

ax.text(0.08, 0.96, "Aquatic modulation of\nvegetation fragility", fontsize=12, fontweight="bold", va="top")

# Panel B: HDSI distribution
ax = fig.add_subplot(gs[0, 1])
panel_label(ax, "B")
ax.hist(d["HDSI"], bins=50, color=COL["light_blue"], edgecolor="white")
ax.axvline(q25, color=COL["blue"], linestyle="--", linewidth=1.3, label="Q25")
ax.axvline(q75, color=COL["red"], linestyle="--", linewidth=1.3, label="Q75")
ax.set_xlabel("HDSI")
ax.set_ylabel("Count")
ax.set_title("Heat–dryness stress distribution")
ax.legend(frameon=False)
clean_ax(ax)

# Panel C: permanent water distance distribution
ax = fig.add_subplot(gs[0, 2])
panel_label(ax, "C")
ax.hist(d["dist_perm_water_km"], bins=50, color=COL["blue"], alpha=0.75, edgecolor="white")
ax.set_xlim(0, 16)
ax.set_xlabel("Distance to permanent water (km)")
ax.set_ylabel("Count")
ax.set_title("Permanent-water distance")
clean_ax(ax)

# Panel D: overall negative response rates
ax = fig.add_subplot(gs[1, 1])
panel_label(ax, "D")
vals = [
    d["EVI_neg_z05"].mean(),
    d["LAI_neg_z05"].mean(),
    d["joint_failure"].mean()
]
labels = ["EVI", "LAI", "Joint"]
colors = [COL["green"], COL["purple"], COL["red"]]
ax.bar(labels, vals, color=colors)
ax.set_ylim(0, max(vals) * 1.35)
ax.set_ylabel("Negative-response rate")
ax.set_title("Overall vegetation negative response")
clean_ax(ax)

# Panel E: regional joint failure rate
ax = fig.add_subplot(gs[1, 2])
panel_label(ax, "E")
rs = region_summary.sort_values("joint_failure_rate")
ax.barh(rs["macro_region"], rs["joint_failure_rate"], color=COL["orange"])
ax.set_xlabel("Joint failure rate")
ax.set_title("Regional risk background")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig1_data_framework_overview")
plt.show()


# ============================================================
# 12. Figure 2:
#     High-HDSI risk by water context
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(11, 4.2), sharey=True)

context_labels = {
    "near_permanent_lt1km": "Near permanent\n<1 km",
    "near_seasonal_lt1km": "Near seasonal\n<1 km",
    "far_or_weak_water": "Far / weak\nwater"
}

for ax, resp_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = water_context_risk[
        (water_context_risk["HDSI_group"] == "high") &
        (water_context_risk["response"] == resp_name)
    ].copy()

    sub["x"] = np.arange(len(sub))
    colors = [CONTEXT_COLORS[c] for c in sub["water_context"]]

    ax.bar(sub["x"], sub["risk"], color=colors, width=0.65)
    yerr = np.vstack([
        sub["risk"] - sub["ci_low"],
        sub["ci_high"] - sub["risk"]
    ])
    ax.errorbar(sub["x"], sub["risk"], yerr=yerr, fmt="none", color="#222222", linewidth=1, capsize=3)

    ax.set_xticks(sub["x"])
    ax.set_xticklabels([context_labels[c] for c in sub["water_context"]])
    ax.set_title(f"{resp_name} risk under high HDSI")
    ax.set_ylabel("Negative-response risk")
    ax.set_ylim(0, max(0.55, sub["ci_high"].max() * 1.25))
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_high_HDSI_water_context_risk")
plt.show()


# ============================================================
# 13. Figure 3:
#     Distance-band fragility increase
# ============================================================

fig, ax = plt.subplots(figsize=(9, 5.2))

x = np.arange(len(dist_labels))
offsets = {"EVI": -0.22, "LAI": 0.0, "Joint": 0.22}
colors = {"EVI": COL["green"], "LAI": COL["purple"], "Joint": COL["red"]}

for resp_name in ["EVI", "LAI", "Joint"]:
    sub = fragility[fragility["response"] == resp_name].copy()
    sub["dist_band_perm"] = pd.Categorical(sub["dist_band_perm"], categories=dist_labels, ordered=True)
    sub = sub.sort_values("dist_band_perm")

    xx = x + offsets[resp_name]
    ax.errorbar(
        xx,
        sub["fragility_increase"],
        yerr=np.vstack([
            sub["fragility_increase"] - sub["ci_low"],
            sub["ci_high"] - sub["fragility_increase"]
        ]),
        fmt="o",
        markersize=6,
        linewidth=1.2,
        capsize=3,
        color=colors[resp_name],
        label=resp_name
    )
    ax.plot(xx, sub["fragility_increase"], color=colors[resp_name], linewidth=1.2, alpha=0.75)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(dist_labels)
ax.set_xlabel("Distance to permanent water (km)")
ax.set_ylabel("Fragility increase\nRisk(high HDSI) - Risk(low HDSI)")
ax.set_title("Vegetation fragility increases with distance from permanent water")
ax.legend(frameon=False, ncol=3)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_distance_band_fragility_increase")
plt.show()


# ============================================================
# 14. Figure 4:
#     Aquatic Fragility Buffering index
# ============================================================

fig, ax = plt.subplots(figsize=(8.5, 4.6))

plot_afb = afb.copy()
plot_afb["label"] = plot_afb["response"] + "\n" + plot_afb["contrast"].str.replace(" fragility - near_permanent fragility", "", regex=False)
plot_afb = plot_afb.sort_values(["response", "contrast"])

x = np.arange(len(plot_afb))
colors = [COL["red"] if "far_or_weak" in c else COL["orange"] for c in plot_afb["contrast"]]

ax.bar(x, plot_afb["AFB"], color=colors, width=0.65)
ax.errorbar(
    x,
    plot_afb["AFB"],
    yerr=np.vstack([
        plot_afb["AFB"] - plot_afb["ci_low"],
        plot_afb["ci_high"] - plot_afb["AFB"]
    ]),
    fmt="none",
    color="#222222",
    linewidth=1,
    capsize=3
)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot_afb["label"], rotation=30, ha="right")
ax.set_ylabel("Aquatic Fragility Buffering, AFB")
ax.set_title("How much more heat–dryness sensitivity occurs away from permanent water?")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig4_AFB_index")
plt.show()


# ============================================================
# 15. Figure 5:
#     Regional heterogeneity
# ============================================================

# Panel A: regional joint failure rate
# Panel B: regional AFB_joint
# Panel C: high-HDSI water-context risk in key regions

fig = plt.figure(figsize=(11, 7))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.15], height_ratios=[1, 1], wspace=0.35, hspace=0.35)

ax = fig.add_subplot(gs[0, 0])
panel_label(ax, "A")
rs = region_summary.sort_values("joint_failure_rate")
ax.barh(rs["macro_region"], rs["joint_failure_rate"], color=COL["orange"])
ax.set_xlabel("Joint failure rate")
ax.set_title("Regional background risk")
clean_ax(ax)

ax = fig.add_subplot(gs[1, 0])
panel_label(ax, "B")
reg_joint = region_afb[region_afb["response"] == "Joint"].copy()
reg_joint = reg_joint.sort_values("AFB_far_minus_near")
y = np.arange(len(reg_joint))
ax.errorbar(
    reg_joint["AFB_far_minus_near"],
    y,
    xerr=np.vstack([
        reg_joint["AFB_far_minus_near"] - reg_joint["ci_low"],
        reg_joint["ci_high"] - reg_joint["AFB_far_minus_near"]
    ]),
    fmt="o",
    color=COL["red"],
    capsize=3
)
ax.axvline(0, color="#333333", linewidth=0.8)
ax.set_yticks(y)
ax.set_yticklabels(reg_joint["macro_region"])
ax.set_xlabel("Regional AFB for joint failure")
ax.set_title("Regional aquatic fragility buffering")
clean_ax(ax)

ax = fig.add_subplot(gs[:, 1])
panel_label(ax, "C")
# high-HDSI joint risk by water context per region
rh = region_high_context[
    (region_high_context["response"] == "Joint") &
    (region_high_context["HDSI_group"] == "high")
].copy()

# keep regions with enough rows in all contexts
pivot = rh.pivot_table(
    index="macro_region",
    columns="water_context",
    values="risk"
)

# order by far/weak risk
pivot = pivot.sort_values("far_or_weak_water", ascending=True)

y = np.arange(len(pivot))
for context, offset in zip(["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"], [-0.2, 0.0, 0.2]):
    vals = pivot[context].values if context in pivot.columns else np.full(len(pivot), np.nan)
    ax.scatter(vals, y + offset, s=45, color=CONTEXT_COLORS[context], label=context_labels[context].replace("\n", " "))

ax.set_yticks(y)
ax.set_yticklabels(pivot.index)
ax.set_xlabel("High-HDSI joint failure risk")
ax.set_title("Water-context risk varies by region")
ax.legend(frameon=False, loc="lower right")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_regional_heterogeneity")
plt.show()


# ============================================================
# 16. Figure 6:
#     Optional map of observed risk and water distance
# ============================================================

china = read_china_boundary()

grid = (
    d.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        HDSI_mean=("HDSI", "mean"),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        EVI_neg_rate=("EVI_neg_z05", "mean"),
        LAI_neg_rate=("LAI_neg_z05", "mean"),
        joint_failure_rate=("joint_failure", "mean")
    )
)

def plot_points(ax, data, col, cmap, title, vmin=None, vmax=None):
    if china is not None:
        china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

    if gpd is not None:
        gdf = gpd.GeoDataFrame(
            data.copy(),
            geometry=gpd.points_from_xy(data["lon"], data["lat"]),
            crs="EPSG:4326"
        )
        gdf.plot(
            ax=ax,
            column=col,
            markersize=1.0,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax,
            legend=True,
            legend_kwds={"shrink": 0.62}
        )
    else:
        sc = ax.scatter(
            data["lon"], data["lat"],
            c=data[col],
            s=1,
            cmap=cmap,
            vmin=vmin,
            vmax=vmax
        )
        plt.colorbar(sc, ax=ax, shrink=0.62)

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)

cmap_risk = mpl.colors.LinearSegmentedColormap.from_list(
    "risk",
    ["#f7fbff", "#c6dbef", "#6baed6", "#fdae61", "#d7191c"]
)
cmap_dist = mpl.colors.LinearSegmentedColormap.from_list(
    "dist",
    ["#08306b", "#2171b5", "#6baed6", "#f7f7f7"]
)

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8))

plot_points(axes[0, 0], grid, "HDSI_mean", cmap_risk, "Mean HDSI")
panel_label(axes[0, 0], "A")

plot_points(axes[0, 1], grid, "dist_perm_water_km", cmap_dist, "Distance to permanent water", vmin=0, vmax=8)
panel_label(axes[0, 1], "B")

plot_points(axes[1, 0], grid, "joint_failure_rate", cmap_risk, "Joint failure frequency", vmin=0, vmax=0.6)
panel_label(axes[1, 0], "C")

plot_points(axes[1, 1], grid, "LAI_neg_rate", cmap_risk, "LAI negative-response frequency", vmin=0, vmax=0.6)
panel_label(axes[1, 1], "D")

plt.tight_layout()
savefig(fig, "Fig6_spatial_patterns")
plt.show()


# ============================================================
# 17. Compact printed summary
# ============================================================

print("\n==================== MAIN RESULT TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== MAIN FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey expected tables:")
print("1) water_context_HDSI_risk_with_CI.csv")
print("2) high_HDSI_water_context_contrasts_with_CI.csv")
print("3) fragility_increase_by_distance_band_with_CI.csv")
print("4) aquatic_fragility_buffering_AFB_with_CI.csv")
print("5) region_AFB_far_minus_near_with_CI.csv")

print("\nKey expected figures:")
print("Fig1_data_framework_overview.png")
print("Fig2_high_HDSI_water_context_risk.png")
print("Fig3_distance_band_fragility_increase.png")
print("Fig4_AFB_index.png")
print("Fig5_regional_heterogeneity.png")
print("Fig6_spatial_patterns.png")

In [ ]:
# ============================================================
# Replace HDSI mean map with more meaningful HDSI diagnostics
# Maps:
#   1. High-HDSI frequency
#   2. HDSI 75th percentile
#   3. HDSI maximum
#   4. Joint failure frequency
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    import geopandas as gpd
except Exception:
    gpd = None

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_DIR = os.path.join(AQUA_DIR, "_main_results_reproduced")
FIG_DIR = os.path.join(OUT_DIR, "figs")
os.makedirs(FIG_DIR, exist_ok=True)

df = pd.read_parquet(PANEL_FP)

for c in [
    "HDSI", "lon", "lat", "dist_perm_water_km",
    "EVI_neg_z05", "LAI_neg_z05", "late_summer_EVI_z", "late_summer_LAI_z"
]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values
        ]),
        axis=0
    )

if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)

df["joint_failure"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

d = df[
    df["valid_model_both_y"] &
    df["HDSI"].notna() &
    df["dist_perm_water_km"].notna()
].copy()

h75_global = d["HDSI"].quantile(0.75)
d["high_HDSI_flag"] = (d["HDSI"] >= h75_global).astype(int)

grid2 = (
    d.groupby(["grid_id", "lon", "lat"], as_index=False)
    .agg(
        HDSI_mean=("HDSI", "mean"),
        HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
        HDSI_max=("HDSI", "max"),
        high_HDSI_freq=("high_HDSI_flag", "mean"),
        dist_perm_water_km=("dist_perm_water_km", "mean"),
        joint_failure_rate=("joint_failure", "mean")
    )
)

print(grid2[["HDSI_mean", "HDSI_q75", "HDSI_max", "high_HDSI_freq"]].describe())

try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
china = china.dissolve().reset_index(drop=True)

gdf = gpd.GeoDataFrame(
    grid2,
    geometry=gpd.points_from_xy(grid2["lon"], grid2["lat"]),
    crs="EPSG:4326"
)

cmap_h = mpl.colors.LinearSegmentedColormap.from_list(
    "hdsi",
    ["#f7fbff", "#c6dbef", "#6baed6", "#fdae61", "#d7191c"]
)

cmap_d = mpl.colors.LinearSegmentedColormap.from_list(
    "dist",
    ["#08306b", "#2171b5", "#6baed6", "#f7f7f7"]
)

fig, axes = plt.subplots(2, 2, figsize=(10.5, 8))

plots = [
    ("high_HDSI_freq", "Frequency of high-HDSI years", cmap_h, 0, 1),
    ("HDSI_q75", "Upper-quartile HDSI intensity", cmap_h, None, None),
    ("dist_perm_water_km", "Distance to permanent water", cmap_d, 0, 8),
    ("joint_failure_rate", "Joint failure frequency", cmap_h, 0, 0.6),
]

for ax, (col, title, cmap, vmin, vmax) in zip(axes.ravel(), plots):
    china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

    # robust color scale for HDSI_q75 if not specified
    if vmin is None:
        vmin = grid2[col].quantile(0.02)
    if vmax is None:
        vmax = grid2[col].quantile(0.98)

    gdf.plot(
        ax=ax,
        column=col,
        markersize=1.0,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        legend=True,
        legend_kwds={"shrink": 0.62}
    )

    ax.set_xlim(70, 138)
    ax.set_ylim(15, 56)
    ax.set_title(title)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(alpha=0.15, lw=0.3)

plt.tight_layout()
out_png = os.path.join(FIG_DIR, "Fig6_spatial_patterns_HDSI_revised.png")
out_pdf = os.path.join(FIG_DIR, "Fig6_spatial_patterns_HDSI_revised.pdf")
plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)

In [ ]:
# ============================================================
# Extended experiments for aquatic buffering under heat–dryness stress
#
# Purpose:
#   Do NOT force one story.
#   Run multiple robust experiments to identify which mechanism is strongest:
#     1) Distance-decay of fragility
#     2) Water-stability effects
#     3) Stress-channel decomposition
#     4) Seasonal propagation: spring -> late summer change
#     5) Matched near-water vs far-water comparison
#
# Main data:
#   /content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/
#     _aquatic_buffering/china_aquatic_buffering_panel_v2.parquet
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import pairwise_distances

try:
    import geopandas as gpd
except Exception:
    gpd = None


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")

PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_DIR = os.path.join(AQUA_DIR, "_extended_mechanism_experiments")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel file:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300,
})

COL = {
    "blue": "#3B6EA8",
    "light_blue": "#9BBBD8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "teal": "#009E73",
    "purple": "#7E62A3",
    "gray": "#666666",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
    "delta_EVI": COL["green"],
    "delta_LAI": COL["purple"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition",
    "is_perm_water", "is_seasonal_water", "is_any_water"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Negative response variables
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce")

df["joint_failure"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Seasonal change
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# Stress-channel definitions
# For SM and Precip, higher stress = lower SM / lower precipitation, so use negative anomaly.
df["SM_deficit_z"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit_z"] = -df["summer_Precip_z"]

# Macro regions
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid data
required = [
    "HDSI", "dist_perm_water_km", "dist_seasonal_water_km",
    "EVI_neg_z05", "LAI_neg_z05",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z"
]

d = df[df["valid_model_both_y"]].copy()
d = d.dropna(subset=[c for c in required if c in d.columns]).copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["joint_failure"] = d["joint_failure"].astype(int)

# HDSI group
HDSI_Q25 = d["HDSI"].quantile(0.25)
HDSI_Q75 = d["HDSI"].quantile(0.75)

d["HDSI_group"] = "mid"
d.loc[d["HDSI"] <= HDSI_Q25, "HDSI_group"] = "low"
d.loc[d["HDSI"] >= HDSI_Q75, "HDSI_group"] = "high"

# Aquatic context
d["water_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "water_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "water_context"
] = "near_seasonal_lt1km"

water_context_order = ["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"]

# Distance bands: detailed and manuscript-friendly
fine_bins = [0, 0.25, 0.5, 1, 2, 3, 5, 7.5, 10, 15, 25]
fine_labels = ["0-0.25", "0.25-0.5", "0.5-1", "1-2", "2-3", "3-5", "5-7.5", "7.5-10", "10-15", ">15"]

d["dist_band_fine"] = pd.cut(
    d["dist_perm_water_km"],
    bins=fine_bins,
    labels=fine_labels,
    include_lowest=True,
    right=False
)

coarse_bins = [0, 0.5, 1, 2, 5, 10, 25]
coarse_labels = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]

d["dist_band_coarse"] = pd.cut(
    d["dist_perm_water_km"],
    bins=coarse_bins,
    labels=coarse_labels,
    include_lowest=True,
    right=False
)

# Water occurrence classes
occ_bins = [-0.1, 0, 5, 25, 75, 100]
occ_labels = ["no_surface_water", "rare_0-5", "low_5-25", "moderate_25-75", "stable_75-100"]

d["water_occurrence_class"] = pd.cut(
    d["water_occurrence"],
    bins=occ_bins,
    labels=occ_labels,
    include_lowest=True,
    right=True
)

print("Valid data:", d.shape)
print("Unique grids:", d["grid_id"].nunique())
print("Years:", sorted(d["year"].dropna().astype(int).unique()))
print("HDSI Q25/Q75:", HDSI_Q25, HDSI_Q75)
print("EVI neg:", d["EVI_neg_z05"].mean())
print("LAI neg:", d["LAI_neg_z05"].mean())
print("Joint:", d["joint_failure"].mean())

overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].dropna().astype(int).unique())))},
    {"metric": "HDSI_Q25", "value": HDSI_Q25},
    {"metric": "HDSI_Q75", "value": HDSI_Q75},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "joint_failure_rate", "value": d["joint_failure"].mean()},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_median", "value": d["dist_perm_water_km"].median()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])
overview.to_csv(os.path.join(TAB_DIR, "00_data_overview.csv"), index=False)
display(overview)


# ============================================================
# 3. Bootstrap helpers
# ============================================================

def boot_mean_ci(x, n_boot=500, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    mean = np.mean(x)
    if len(x) <= 1:
        return mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = np.mean(rng.choice(x, size=len(x), replace=True))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi

def boot_diff_ci(a, b, n_boot=500, seed=42):
    # mean(a) - mean(b)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    diff = np.mean(a) - np.mean(b)
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots[i] = np.mean(aa) - np.mean(bb)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff, lo, hi

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 4. Experiment 1: Distance-decay of fragility
# ============================================================

response_binary = {
    "EVI": "EVI_neg_z05",
    "LAI": "LAI_neg_z05",
    "Joint": "joint_failure",
}

def risk_by_group(data, group_col, group_order, stress_col="HDSI_group", stress_order=["low", "mid", "high"]):
    rows = []
    for g in group_order:
        for s in stress_order:
            sub = data[(data[group_col].astype(str) == str(g)) & (data[stress_col].astype(str) == str(s))]
            for resp, col in response_binary.items():
                m, lo, hi = boot_mean_ci(sub[col].values, n_boot=500, seed=42)
                rows.append({
                    group_col: g,
                    stress_col: s,
                    "response": resp,
                    "risk": m,
                    "ci_low": lo,
                    "ci_high": hi,
                    "rows": len(sub),
                    "grids": sub["grid_id"].nunique()
                })
    return pd.DataFrame(rows)

distance_fine_risk = risk_by_group(d, "dist_band_fine", fine_labels)
save_table(distance_fine_risk, "01_distance_fine_HDSI_risk_CI.csv")

distance_coarse_risk = risk_by_group(d, "dist_band_coarse", coarse_labels)
save_table(distance_coarse_risk, "01_distance_coarse_HDSI_risk_CI.csv")

# Fragility increase: high - low by distance band
frag_rows = []

for band in fine_labels:
    low = d[(d["dist_band_fine"].astype(str) == band) & (d["HDSI_group"] == "low")]
    high = d[(d["dist_band_fine"].astype(str) == band) & (d["HDSI_group"] == "high")]

    for resp, col in response_binary.items():
        diff, lo, hi = boot_diff_ci(high[col].values, low[col].values, n_boot=500, seed=42)
        frag_rows.append({
            "dist_band_fine": band,
            "response": resp,
            "fragility_increase": diff,
            "ci_low": lo,
            "ci_high": hi,
            "risk_high": high[col].mean() if len(high) else np.nan,
            "risk_low": low[col].mean() if len(low) else np.nan,
            "n_high": len(high),
            "n_low": len(low)
        })

frag_distance = pd.DataFrame(frag_rows)
save_table(frag_distance, "01_distance_decay_fragility_increase_CI.csv")
display(frag_distance)


# ============================================================
# 5. Experiment 2: Water stability and permanence
# ============================================================

# Water context × HDSI
water_context_risk = risk_by_group(d, "water_context", water_context_order)
save_table(water_context_risk, "02_water_context_HDSI_risk_CI.csv")

# Water occurrence class × HDSI
occ_order = occ_labels
occ_risk = risk_by_group(d.dropna(subset=["water_occurrence_class"]), "water_occurrence_class", occ_order)
save_table(occ_risk, "02_water_occurrence_HDSI_risk_CI.csv")

# High-HDSI contrasts among water contexts
contrast_rows = []
high = d[d["HDSI_group"] == "high"].copy()

for resp, col in response_binary.items():
    near = high[high["water_context"] == "near_permanent_lt1km"][col].values
    seasonal = high[high["water_context"] == "near_seasonal_lt1km"][col].values
    far = high[high["water_context"] == "far_or_weak_water"][col].values

    diff_far, lo_far, hi_far = boot_diff_ci(far, near, n_boot=500, seed=42)
    diff_season, lo_season, hi_season = boot_diff_ci(seasonal, near, n_boot=500, seed=43)

    contrast_rows.append({
        "response": resp,
        "contrast": "far_or_weak - near_permanent",
        "risk_difference": diff_far,
        "ci_low": lo_far,
        "ci_high": hi_far,
        "n_far_or_weak": len(far),
        "n_near_permanent": len(near)
    })

    contrast_rows.append({
        "response": resp,
        "contrast": "near_seasonal - near_permanent",
        "risk_difference": diff_season,
        "ci_low": lo_season,
        "ci_high": hi_season,
        "n_near_seasonal": len(seasonal),
        "n_near_permanent": len(near)
    })

water_context_contrasts = pd.DataFrame(contrast_rows)
save_table(water_context_contrasts, "02_high_HDSI_water_context_contrasts_CI.csv")
display(water_context_contrasts)


# Aquatic Fragility Buffering:
# AFB = Fragility_other - Fragility_near_permanent
afb_rows = []

for resp, col in response_binary.items():
    near = d[d["water_context"] == "near_permanent_lt1km"]
    seasonal = d[d["water_context"] == "near_seasonal_lt1km"]
    far = d[d["water_context"] == "far_or_weak_water"]

    near_low = near[near["HDSI_group"] == "low"][col].values
    near_high = near[near["HDSI_group"] == "high"][col].values
    season_low = seasonal[seasonal["HDSI_group"] == "low"][col].values
    season_high = seasonal[seasonal["HDSI_group"] == "high"][col].values
    far_low = far[far["HDSI_group"] == "low"][col].values
    far_high = far[far["HDSI_group"] == "high"][col].values

    rng = np.random.default_rng(42)
    b_far = []
    b_season = []

    for i in range(500):
        nl = rng.choice(near_low, size=len(near_low), replace=True)
        nh = rng.choice(near_high, size=len(near_high), replace=True)

        sl = rng.choice(season_low, size=len(season_low), replace=True)
        sh = rng.choice(season_high, size=len(season_high), replace=True)

        fl = rng.choice(far_low, size=len(far_low), replace=True)
        fh = rng.choice(far_high, size=len(far_high), replace=True)

        frag_near = np.mean(nh) - np.mean(nl)
        frag_season = np.mean(sh) - np.mean(sl)
        frag_far = np.mean(fh) - np.mean(fl)

        b_season.append(frag_season - frag_near)
        b_far.append(frag_far - frag_near)

    afb_rows.append({
        "response": resp,
        "contrast": "near_seasonal fragility - near_permanent fragility",
        "AFB": np.mean(b_season),
        "ci_low": np.percentile(b_season, 2.5),
        "ci_high": np.percentile(b_season, 97.5)
    })

    afb_rows.append({
        "response": resp,
        "contrast": "far_or_weak fragility - near_permanent fragility",
        "AFB": np.mean(b_far),
        "ci_low": np.percentile(b_far, 2.5),
        "ci_high": np.percentile(b_far, 97.5)
    })

afb_table = pd.DataFrame(afb_rows)
save_table(afb_table, "02_AFB_water_stability_CI.csv")
display(afb_table)


# ============================================================
# 6. Experiment 3: Stress-channel decomposition
# ============================================================

# For each stress channel, define low/high by Q25/Q75 and compute:
#   Fragility increase = Risk(high stress) - Risk(low stress)
# by water context.
stress_channels = {
    "HDSI": "HDSI",
    "VPD": "summer_VPD_z",
    "LST_Day": "summer_LST_Day_z",
    "SM_deficit": "SM_deficit_z",
    "Precip_deficit": "Precip_deficit_z",
    "Tair": "summer_Tair_z",
}

stress_rows = []
stress_afb_rows = []

for stress_name, stress_col in stress_channels.items():
    if stress_col not in d.columns:
        continue

    tmp = d.dropna(subset=[stress_col]).copy()
    q25 = tmp[stress_col].quantile(0.25)
    q75 = tmp[stress_col].quantile(0.75)

    tmp["stress_group"] = "mid"
    tmp.loc[tmp[stress_col] <= q25, "stress_group"] = "low"
    tmp.loc[tmp[stress_col] >= q75, "stress_group"] = "high"

    for context in water_context_order:
        low = tmp[(tmp["water_context"] == context) & (tmp["stress_group"] == "low")]
        high = tmp[(tmp["water_context"] == context) & (tmp["stress_group"] == "high")]

        for resp, col in response_binary.items():
            diff, lo, hi = boot_diff_ci(high[col].values, low[col].values, n_boot=400, seed=42)
            stress_rows.append({
                "stress": stress_name,
                "stress_col": stress_col,
                "water_context": context,
                "response": resp,
                "fragility_increase": diff,
                "ci_low": lo,
                "ci_high": hi,
                "stress_q25": q25,
                "stress_q75": q75,
                "n_high": len(high),
                "n_low": len(low)
            })

    # AFB for this stress: far/weak minus near-permanent
    for resp, col in response_binary.items():
        near = tmp[tmp["water_context"] == "near_permanent_lt1km"]
        far = tmp[tmp["water_context"] == "far_or_weak_water"]
        seasonal = tmp[tmp["water_context"] == "near_seasonal_lt1km"]

        near_low = near[near["stress_group"] == "low"][col].values
        near_high = near[near["stress_group"] == "high"][col].values
        far_low = far[far["stress_group"] == "low"][col].values
        far_high = far[far["stress_group"] == "high"][col].values
        sea_low = seasonal[seasonal["stress_group"] == "low"][col].values
        sea_high = seasonal[seasonal["stress_group"] == "high"][col].values

        if min(len(near_low), len(near_high), len(far_low), len(far_high)) > 20:
            rng = np.random.default_rng(42)
            boots_far = []
            boots_sea = []

            for i in range(400):
                nl = rng.choice(near_low, size=len(near_low), replace=True)
                nh = rng.choice(near_high, size=len(near_high), replace=True)

                fl = rng.choice(far_low, size=len(far_low), replace=True)
                fh = rng.choice(far_high, size=len(far_high), replace=True)

                sl = rng.choice(sea_low, size=len(sea_low), replace=True)
                sh = rng.choice(sea_high, size=len(sea_high), replace=True)

                frag_near = np.mean(nh) - np.mean(nl)
                frag_far = np.mean(fh) - np.mean(fl)
                frag_sea = np.mean(sh) - np.mean(sl)

                boots_far.append(frag_far - frag_near)
                boots_sea.append(frag_sea - frag_near)

            stress_afb_rows.append({
                "stress": stress_name,
                "response": resp,
                "contrast": "far_or_weak - near_permanent",
                "AFB": np.mean(boots_far),
                "ci_low": np.percentile(boots_far, 2.5),
                "ci_high": np.percentile(boots_far, 97.5)
            })

            stress_afb_rows.append({
                "stress": stress_name,
                "response": resp,
                "contrast": "near_seasonal - near_permanent",
                "AFB": np.mean(boots_sea),
                "ci_low": np.percentile(boots_sea, 2.5),
                "ci_high": np.percentile(boots_sea, 97.5)
            })

stress_fragility = pd.DataFrame(stress_rows)
stress_afb = pd.DataFrame(stress_afb_rows)

save_table(stress_fragility, "03_stress_channel_fragility_by_water_context_CI.csv")
save_table(stress_afb, "03_stress_channel_AFB_CI.csv")

display(stress_afb.head(20))


# ============================================================
# 7. Experiment 4: Seasonal propagation
# ============================================================

# Outcomes:
#   delta_EVI = late_summer_EVI_z - spring_EVI_z
#   delta_LAI = late_summer_LAI_z - spring_LAI_z
#
# More negative means stronger seasonal deterioration.

delta_rows = []

for context in water_context_order:
    low = d[(d["water_context"] == context) & (d["HDSI_group"] == "low")]
    high = d[(d["water_context"] == context) & (d["HDSI_group"] == "high")]

    for resp, col in {"delta_EVI": "delta_EVI", "delta_LAI": "delta_LAI"}.items():
        # High - low; negative means high HDSI causes stronger decline.
        diff, lo, hi = boot_diff_ci(high[col].values, low[col].values, n_boot=500, seed=42)

        m_low, lo_low, hi_low = boot_mean_ci(low[col].values, n_boot=500, seed=42)
        m_high, lo_high, hi_high = boot_mean_ci(high[col].values, n_boot=500, seed=42)

        delta_rows.append({
            "water_context": context,
            "response": resp,
            "delta_high_minus_low": diff,
            "ci_low": lo,
            "ci_high": hi,
            "mean_low_HDSI": m_low,
            "mean_high_HDSI": m_high,
            "low_ci_low": lo_low,
            "low_ci_high": hi_low,
            "high_ci_low": lo_high,
            "high_ci_high": hi_high,
            "n_low": len(low),
            "n_high": len(high)
        })

seasonal_delta = pd.DataFrame(delta_rows)
save_table(seasonal_delta, "04_seasonal_propagation_delta_EVI_LAI_CI.csv")
display(seasonal_delta)

# Seasonal AFB: deterioration_far - deterioration_near
# Here, deterioration effect = mean(delta under high) - mean(delta under low).
# More negative = stronger seasonal deterioration.
# To make index intuitive:
#   seasonal_buffering_loss = deterioration_far - deterioration_near
# If negative, far water deteriorates more than near water.
seasonal_afb_rows = []

for resp, col in {"delta_EVI": "delta_EVI", "delta_LAI": "delta_LAI"}.items():
    near = d[d["water_context"] == "near_permanent_lt1km"]
    far = d[d["water_context"] == "far_or_weak_water"]
    sea = d[d["water_context"] == "near_seasonal_lt1km"]

    near_low = near[near["HDSI_group"] == "low"][col].values
    near_high = near[near["HDSI_group"] == "high"][col].values
    far_low = far[far["HDSI_group"] == "low"][col].values
    far_high = far[far["HDSI_group"] == "high"][col].values
    sea_low = sea[sea["HDSI_group"] == "low"][col].values
    sea_high = sea[sea["HDSI_group"] == "high"][col].values

    rng = np.random.default_rng(42)
    boots_far = []
    boots_sea = []

    for i in range(500):
        nl = rng.choice(near_low, size=len(near_low), replace=True)
        nh = rng.choice(near_high, size=len(near_high), replace=True)
        fl = rng.choice(far_low, size=len(far_low), replace=True)
        fh = rng.choice(far_high, size=len(far_high), replace=True)
        sl = rng.choice(sea_low, size=len(sea_low), replace=True)
        sh = rng.choice(sea_high, size=len(sea_high), replace=True)

        deterioration_near = np.mean(nh) - np.mean(nl)
        deterioration_far = np.mean(fh) - np.mean(fl)
        deterioration_sea = np.mean(sh) - np.mean(sl)

        boots_far.append(deterioration_far - deterioration_near)
        boots_sea.append(deterioration_sea - deterioration_near)

    seasonal_afb_rows.append({
        "response": resp,
        "contrast": "far_or_weak seasonal deterioration - near_permanent",
        "difference": np.mean(boots_far),
        "ci_low": np.percentile(boots_far, 2.5),
        "ci_high": np.percentile(boots_far, 97.5)
    })

    seasonal_afb_rows.append({
        "response": resp,
        "contrast": "near_seasonal seasonal deterioration - near_permanent",
        "difference": np.mean(boots_sea),
        "ci_low": np.percentile(boots_sea, 2.5),
        "ci_high": np.percentile(boots_sea, 97.5)
    })

seasonal_afb = pd.DataFrame(seasonal_afb_rows)
save_table(seasonal_afb, "04_seasonal_propagation_AFB_delta_CI.csv")
display(seasonal_afb)


# ============================================================
# 8. Experiment 5: Matched comparison
# ============================================================

# Treatment:
#   near permanent water < 1 km
# Control:
#   far or weak water: dist_perm >= 2 km and dist_seasonal >= 1 km
#
# Matching controls:
#   elevation, slope, spring_EVI_z, spring_LAI_z, HDSI, LC_Type1, macro_region, year
#
# This does not prove causality, but reduces obvious spatial confounding.

match = d.copy()

match["treat_near_perm"] = np.nan
match.loc[match["dist_perm_water_km"] < 1.0, "treat_near_perm"] = 1
match.loc[
    (match["dist_perm_water_km"] >= 2.0) &
    (match["dist_seasonal_water_km"] >= 1.0),
    "treat_near_perm"
] = 0

match = match.dropna(subset=["treat_near_perm"]).copy()
match["treat_near_perm"] = match["treat_near_perm"].astype(int)

# Use high HDSI subset for main matched comparison
match_high = match[match["HDSI_group"] == "high"].copy()

print("Matching data high HDSI:", match_high.shape)
print(match_high["treat_near_perm"].value_counts())

num_match_features = [
    "elevation", "slope", "spring_EVI_z", "spring_LAI_z",
    "HDSI", "summer_Tair_z", "lon", "lat"
]
num_match_features = [c for c in num_match_features if c in match_high.columns]

cat_match_features = ["LC_Type1", "macro_region", "year"]
cat_match_features = [c for c in cat_match_features if c in match_high.columns]

for c in cat_match_features:
    match_high[c] = match_high[c].astype(str)

# Sample if data too large for matching speed
MAX_MATCH_N = 60000
if len(match_high) > MAX_MATCH_N:
    match_high_s = match_high.sample(MAX_MATCH_N, random_state=42).copy()
else:
    match_high_s = match_high.copy()

treated = match_high_s[match_high_s["treat_near_perm"] == 1].copy()
control = match_high_s[match_high_s["treat_near_perm"] == 0].copy()

print("Treated:", len(treated), "Control:", len(control))

# Preprocess features
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

pre = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_match_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", ohe)
        ]), cat_match_features)
    ],
    remainder="drop"
)

X_all = pre.fit_transform(match_high_s[num_match_features + cat_match_features])
treat_arr = match_high_s["treat_near_perm"].values

X_t = X_all[treat_arr == 1]
X_c = X_all[treat_arr == 0]

control_df = match_high_s[match_high_s["treat_near_perm"] == 0].copy().reset_index(drop=True)
treated_df = match_high_s[match_high_s["treat_near_perm"] == 1].copy().reset_index(drop=True)

# Nearest-neighbor matching with replacement
nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
nn.fit(X_c)
dist, idx = nn.kneighbors(X_t)

matched_control = control_df.iloc[idx.flatten()].copy().reset_index(drop=True)
matched_treated = treated_df.copy().reset_index(drop=True)

matched_pairs = pd.DataFrame({
    "treated_index": matched_treated.index,
    "control_index": idx.flatten(),
    "match_distance": dist.flatten()
})

# Optional caliper: keep relatively close matches
caliper = np.quantile(matched_pairs["match_distance"], 0.90)
keep = matched_pairs["match_distance"] <= caliper

matched_treated = matched_treated.loc[keep.values].reset_index(drop=True)
matched_control = matched_control.loc[keep.values].reset_index(drop=True)
matched_pairs = matched_pairs.loc[keep.values].reset_index(drop=True)

print("Matched pairs kept:", len(matched_pairs))
print("Match distance summary:")
display(matched_pairs["match_distance"].describe())

matched_summary_rows = []

outcomes = {
    "EVI_neg": "EVI_neg_z05",
    "LAI_neg": "LAI_neg_z05",
    "Joint": "joint_failure",
    "delta_EVI": "delta_EVI",
    "delta_LAI": "delta_LAI",
    "late_EVI_z": "late_summer_EVI_z",
    "late_LAI_z": "late_summer_LAI_z"
}

for name, col in outcomes.items():
    tvals = matched_treated[col].values.astype(float)
    cvals = matched_control[col].values.astype(float)

    # Effect defined as near permanent - far/weak.
    # For negative-response risks, negative means near water has lower risk.
    # For delta/late-summer z, positive means near water has better vegetation condition.
    diff, lo, hi = boot_diff_ci(tvals, cvals, n_boot=500, seed=42)

    matched_summary_rows.append({
        "outcome": name,
        "near_perm_mean": np.nanmean(tvals),
        "far_weak_matched_mean": np.nanmean(cvals),
        "near_minus_far": diff,
        "ci_low": lo,
        "ci_high": hi,
        "n_pairs": len(tvals)
    })

matched_summary = pd.DataFrame(matched_summary_rows)
save_table(matched_summary, "05_matched_high_HDSI_near_vs_far_summary_CI.csv")
display(matched_summary)

matched_pairs.to_csv(os.path.join(TAB_DIR, "05_matched_pair_distances.csv"), index=False)


# Balance diagnostics: standardized mean difference before/after
def standardized_mean_diff(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan
    sd = np.sqrt((np.var(a) + np.var(b)) / 2)
    if sd == 0:
        return 0
    return (np.mean(a) - np.mean(b)) / sd

balance_rows = []

for c in num_match_features:
    # Before matching
    tb = treated[c].values
    cb = control[c].values
    smd_before = standardized_mean_diff(tb, cb)

    # After matching
    ta = matched_treated[c].values
    ca = matched_control[c].values
    smd_after = standardized_mean_diff(ta, ca)

    balance_rows.append({
        "variable": c,
        "SMD_before": smd_before,
        "SMD_after": smd_after,
        "abs_SMD_before": abs(smd_before),
        "abs_SMD_after": abs(smd_after)
    })

balance = pd.DataFrame(balance_rows)
save_table(balance, "05_matching_balance_numeric_SMD.csv")
display(balance)


# ============================================================
# 9. Regional decomposition
# ============================================================

# Region × water context × high-HDSI risk
region_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()

    for context in water_context_order:
        sub = dr[(dr["water_context"] == context) & (dr["HDSI_group"] == "high")].copy()

        for resp, col in response_binary.items():
            m, lo, hi = boot_mean_ci(sub[col].values, n_boot=400, seed=42)
            region_rows.append({
                "macro_region": region,
                "water_context": context,
                "response": resp,
                "risk_high_HDSI": m,
                "ci_low": lo,
                "ci_high": hi,
                "rows": len(sub),
                "grids": sub["grid_id"].nunique()
            })

region_high_risk = pd.DataFrame(region_rows)
save_table(region_high_risk, "06_region_high_HDSI_water_context_risk_CI.csv")

# Region AFB: far fragility - near fragility
region_afb_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()

    near = dr[dr["water_context"] == "near_permanent_lt1km"]
    far = dr[dr["water_context"] == "far_or_weak_water"]

    # Skip tiny groups
    if len(near) < 200 or len(far) < 100:
        continue

    for resp, col in response_binary.items():
        near_low = near[near["HDSI_group"] == "low"][col].values
        near_high = near[near["HDSI_group"] == "high"][col].values
        far_low = far[far["HDSI_group"] == "low"][col].values
        far_high = far[far["HDSI_group"] == "high"][col].values

        if min(len(near_low), len(near_high), len(far_low), len(far_high)) < 30:
            continue

        rng = np.random.default_rng(42)
        boots = []
        for i in range(400):
            nl = rng.choice(near_low, size=len(near_low), replace=True)
            nh = rng.choice(near_high, size=len(near_high), replace=True)
            fl = rng.choice(far_low, size=len(far_low), replace=True)
            fh = rng.choice(far_high, size=len(far_high), replace=True)

            frag_near = np.mean(nh) - np.mean(nl)
            frag_far = np.mean(fh) - np.mean(fl)
            boots.append(frag_far - frag_near)

        region_afb_rows.append({
            "macro_region": region,
            "response": resp,
            "AFB_far_minus_near": np.mean(boots),
            "ci_low": np.percentile(boots, 2.5),
            "ci_high": np.percentile(boots, 97.5),
            "n_near": len(near),
            "n_far": len(far)
        })

region_afb = pd.DataFrame(region_afb_rows)
save_table(region_afb, "06_region_AFB_far_minus_near_CI.csv")
display(region_afb)


# ============================================================
# 10. Figures
# ============================================================

# ------------------------------------------------------------
# Fig 1. Distance decay of fragility
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 5.2))

x = np.arange(len(fine_labels))
offset = {"EVI": -0.22, "LAI": 0.0, "Joint": 0.22}

for resp in ["EVI", "LAI", "Joint"]:
    sub = frag_distance[frag_distance["response"] == resp].copy()
    sub["dist_band_fine"] = pd.Categorical(sub["dist_band_fine"], categories=fine_labels, ordered=True)
    sub = sub.sort_values("dist_band_fine")

    xx = x + offset[resp]
    y = sub["fragility_increase"].values
    ylo = sub["ci_low"].values
    yhi = sub["ci_high"].values

    ax.errorbar(
        xx, y,
        yerr=np.vstack([y - ylo, yhi - y]),
        fmt="o",
        color=RESP_COLORS[resp],
        capsize=3,
        linewidth=1.2,
        label=resp
    )
    ax.plot(xx, y, color=RESP_COLORS[resp], linewidth=1.2, alpha=0.75)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(fine_labels, rotation=30, ha="right")
ax.set_xlabel("Distance to permanent water (km)")
ax.set_ylabel("Fragility increase\nRisk(high stress) - Risk(low stress)")
ax.set_title("Distance decay of vegetation fragility under heat–dryness stress")
ax.legend(frameon=False, ncol=3)
clean_ax(ax)
plt.tight_layout()
savefig(fig, "Fig1_distance_decay_fragility")
plt.show()


# ------------------------------------------------------------
# Fig 2. Water stability: high-HDSI risk and AFB
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# A: High-HDSI joint risk by water context
ax = axes[0]
sub = water_context_risk[
    (water_context_risk["HDSI_group"] == "high") &
    (water_context_risk["response"] == "Joint")
].copy()

sub["water_context"] = pd.Categorical(sub["water_context"], categories=water_context_order, ordered=True)
sub = sub.sort_values("water_context")
x = np.arange(len(sub))

context_colors = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

ax.bar(x, sub["risk"], color=[context_colors[c] for c in sub["water_context"]], width=0.65)
ax.errorbar(
    x, sub["risk"],
    yerr=np.vstack([sub["risk"] - sub["ci_low"], sub["ci_high"] - sub["risk"]]),
    fmt="none", color="#222222", capsize=3, linewidth=1
)

ax.set_xticks(x)
ax.set_xticklabels(["Near permanent\n<1 km", "Near seasonal\n<1 km", "Far / weak\nwater"])
ax.set_ylabel("Joint failure risk under high HDSI")
ax.set_title("Permanent water shows the lowest high-stress risk")
panel_label(ax, "A")
clean_ax(ax)

# B: AFB by response
ax = axes[1]
plot_afb = afb_table[afb_table["contrast"].str.contains("far_or_weak")].copy()
plot_afb["response"] = pd.Categorical(plot_afb["response"], categories=["EVI", "LAI", "Joint"], ordered=True)
plot_afb = plot_afb.sort_values("response")
x = np.arange(len(plot_afb))

ax.bar(x, plot_afb["AFB"], color=[RESP_COLORS[r] for r in plot_afb["response"]], width=0.65)
ax.errorbar(
    x, plot_afb["AFB"],
    yerr=np.vstack([plot_afb["AFB"] - plot_afb["ci_low"], plot_afb["ci_high"] - plot_afb["AFB"]]),
    fmt="none", color="#222222", capsize=3, linewidth=1
)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot_afb["response"])
ax.set_ylabel("AFB = Fragility_far/weak - Fragility_near permanent")
ax.set_title("Aquatic fragility buffering")
panel_label(ax, "B")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_water_stability_and_AFB")
plt.show()


# ------------------------------------------------------------
# Fig 3. Stress-channel decomposition
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), sharey=False)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = stress_afb[
        (stress_afb["response"] == resp) &
        (stress_afb["contrast"] == "far_or_weak - near_permanent")
    ].copy()

    stress_order = ["HDSI", "VPD", "LST_Day", "SM_deficit", "Precip_deficit", "Tair"]
    sub["stress"] = pd.Categorical(sub["stress"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress")

    x = np.arange(len(sub))
    y = sub["AFB"].values
    ylo = sub["ci_low"].values
    yhi = sub["ci_high"].values

    ax.bar(x, y, color=RESP_COLORS[resp], width=0.65, alpha=0.85)
    ax.errorbar(x, y, yerr=np.vstack([y - ylo, yhi - y]), fmt="none", color="#222222", capsize=3, linewidth=1)
    ax.axhline(0, color="#333333", linewidth=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress"], rotation=35, ha="right")
    ax.set_ylabel("AFB")
    ax.set_title(f"Stress-channel AFB ({resp})")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_stress_channel_decomposition")
plt.show()


# ------------------------------------------------------------
# Fig 4. Seasonal propagation
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# A: deterioration effect by water context
ax = axes[0]
plot = seasonal_delta.copy()
plot["water_context"] = pd.Categorical(plot["water_context"], categories=water_context_order, ordered=True)
plot = plot.sort_values(["response", "water_context"])

for i, resp in enumerate(["delta_EVI", "delta_LAI"]):
    sub = plot[plot["response"] == resp].copy()
    x = np.arange(len(water_context_order)) + (i - 0.5) * 0.22
    y = sub["delta_high_minus_low"].values
    ylo = sub["ci_low"].values
    yhi = sub["ci_high"].values

    ax.errorbar(
        x, y,
        yerr=np.vstack([y - ylo, yhi - y]),
        fmt="o",
        capsize=3,
        color=RESP_COLORS[resp],
        label=resp.replace("_", " ")
    )

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(np.arange(len(water_context_order)))
ax.set_xticklabels(["Near permanent\n<1 km", "Near seasonal\n<1 km", "Far / weak\nwater"])
ax.set_ylabel("Seasonal deterioration effect\nDelta(high HDSI) - Delta(low HDSI)")
ax.set_title("Seasonal propagation of heat–dryness stress")
ax.legend(frameon=False)
panel_label(ax, "A")
clean_ax(ax)

# B: seasonal AFB
ax = axes[1]
plot = seasonal_afb[seasonal_afb["contrast"].str.contains("far_or_weak")].copy()
plot["response"] = pd.Categorical(plot["response"], categories=["delta_EVI", "delta_LAI"], ordered=True)
plot = plot.sort_values("response")

x = np.arange(len(plot))
y = plot["difference"].values
ylo = plot["ci_low"].values
yhi = plot["ci_high"].values

ax.bar(x, y, color=[RESP_COLORS[r] for r in plot["response"]], width=0.65)
ax.errorbar(x, y, yerr=np.vstack([y - ylo, yhi - y]), fmt="none", color="#222222", capsize=3)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(["delta EVI", "delta LAI"])
ax.set_ylabel("Far/weak - near permanent\nseasonal deterioration difference")
ax.set_title("Does near water weaken seasonal deterioration?")
panel_label(ax, "B")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig4_seasonal_propagation")
plt.show()


# ------------------------------------------------------------
# Fig 5. Matched comparison
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))

# A: matching effects
ax = axes[0]
plot = matched_summary.copy()
plot_order = ["EVI_neg", "LAI_neg", "Joint", "delta_EVI", "delta_LAI", "late_EVI_z", "late_LAI_z"]
plot["outcome"] = pd.Categorical(plot["outcome"], categories=plot_order, ordered=True)
plot = plot.sort_values("outcome")

x = np.arange(len(plot))
y = plot["near_minus_far"].values
ylo = plot["ci_low"].values
yhi = plot["ci_high"].values

colors = []
for out in plot["outcome"].astype(str):
    if "LAI" in out:
        colors.append(COL["purple"])
    elif "EVI" in out:
        colors.append(COL["green"])
    elif "Joint" in out:
        colors.append(COL["red"])
    else:
        colors.append(COL["gray"])

ax.bar(x, y, color=colors, width=0.65)
ax.errorbar(x, y, yerr=np.vstack([y - ylo, yhi - y]), fmt="none", color="#222222", capsize=3)
ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["outcome"], rotation=35, ha="right")
ax.set_ylabel("Matched difference: near permanent - far/weak")
ax.set_title("Matched high-HDSI comparison")
panel_label(ax, "A")
clean_ax(ax)

# B: balance
ax = axes[1]
bal = balance.sort_values("abs_SMD_before")
ypos = np.arange(len(bal))
ax.plot(bal["abs_SMD_before"], ypos, "o", color=COL["gray"], label="Before matching")
ax.plot(bal["abs_SMD_after"], ypos, "o", color=COL["blue"], label="After matching")
ax.axvline(0.1, color=COL["red"], linestyle="--", linewidth=1, label="0.1")
ax.set_yticks(ypos)
ax.set_yticklabels(bal["variable"])
ax.set_xlabel("Absolute standardized mean difference")
ax.set_title("Matching balance")
ax.legend(frameon=False)
panel_label(ax, "B")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_matched_comparison")
plt.show()


# ------------------------------------------------------------
# Fig 6. Regional decomposition
# ------------------------------------------------------------
fig = plt.figure(figsize=(11, 7))
gs = fig.add_gridspec(2, 2, wspace=0.35, hspace=0.35)

# A: regional high-HDSI joint risk by water context
ax = fig.add_subplot(gs[:, 0])
plot = region_high_risk[region_high_risk["response"] == "Joint"].copy()
pivot = plot.pivot_table(index="macro_region", columns="water_context", values="risk_high_HDSI")
pivot = pivot.sort_values("far_or_weak_water", ascending=True)

y = np.arange(len(pivot))
for context, offset in zip(water_context_order, [-0.2, 0, 0.2]):
    if context not in pivot.columns:
        continue
    ax.scatter(
        pivot[context].values,
        y + offset,
        s=45,
        color=context_colors[context],
        label=context.replace("_", " ")
    )

ax.set_yticks(y)
ax.set_yticklabels(pivot.index)
ax.set_xlabel("High-HDSI joint failure risk")
ax.set_title("Regional water-context risk")
ax.legend(frameon=False, fontsize=7)
panel_label(ax, "A")
clean_ax(ax)

# B/C: regional AFB by response
for ax, resp, lab in [
    (fig.add_subplot(gs[0, 1]), "Joint", "B"),
    (fig.add_subplot(gs[1, 1]), "LAI", "C"),
]:
    sub = region_afb[region_afb["response"] == resp].copy()
    sub = sub.sort_values("AFB_far_minus_near")
    y = np.arange(len(sub))
    ax.errorbar(
        sub["AFB_far_minus_near"], y,
        xerr=np.vstack([sub["AFB_far_minus_near"] - sub["ci_low"], sub["ci_high"] - sub["AFB_far_minus_near"]]),
        fmt="o",
        color=RESP_COLORS[resp] if resp in RESP_COLORS else COL["red"],
        capsize=3
    )
    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["macro_region"])
    ax.set_xlabel(f"Regional AFB ({resp})")
    ax.set_title(f"Regional heterogeneity: {resp}")
    panel_label(ax, lab)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig6_regional_decomposition")
plt.show()


# ============================================================
# 11. Optional spatial maps for revised HDSI diagnostics
# ============================================================

try:
    china = None
    if gpd is not None and os.path.exists(CHINA_BOUNDARY_PATH):
        try:
            china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
        except Exception:
            china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")
        china = china.dissolve().reset_index(drop=True)

    d["high_HDSI_flag"] = (d["HDSI"] >= HDSI_Q75).astype(int)

    grid_map = (
        d.groupby(["grid_id", "lon", "lat"], as_index=False)
        .agg(
            high_HDSI_freq=("high_HDSI_flag", "mean"),
            HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
            dist_perm_water_km=("dist_perm_water_km", "mean"),
            joint_failure_rate=("joint_failure", "mean")
        )
    )

    if gpd is not None:
        gdf = gpd.GeoDataFrame(
            grid_map,
            geometry=gpd.points_from_xy(grid_map["lon"], grid_map["lat"]),
            crs="EPSG:4326"
        )

        cmap_h = mpl.colors.LinearSegmentedColormap.from_list(
            "hdsi", ["#f7fbff", "#c6dbef", "#6baed6", "#fdae61", "#d7191c"]
        )
        cmap_d = mpl.colors.LinearSegmentedColormap.from_list(
            "dist", ["#08306b", "#2171b5", "#6baed6", "#f7f7f7"]
        )

        fig, axes = plt.subplots(2, 2, figsize=(10.5, 8))

        plots = [
            ("high_HDSI_freq", "Frequency of high-HDSI years", cmap_h, 0, 1),
            ("HDSI_q75", "Upper-quartile HDSI intensity", cmap_h, None, None),
            ("dist_perm_water_km", "Distance to permanent water", cmap_d, 0, 8),
            ("joint_failure_rate", "Joint failure frequency", cmap_h, 0, 0.6),
        ]

        for ax, (col, title, cmap, vmin, vmax) in zip(axes.ravel(), plots):
            if china is not None:
                china.boundary.plot(ax=ax, linewidth=0.5, color="#333333")

            if vmin is None:
                vmin = grid_map[col].quantile(0.02)
            if vmax is None:
                vmax = grid_map[col].quantile(0.98)

            gdf.plot(
                ax=ax,
                column=col,
                markersize=1.0,
                cmap=cmap,
                vmin=vmin,
                vmax=vmax,
                legend=True,
                legend_kwds={"shrink": 0.62}
            )

            ax.set_xlim(70, 138)
            ax.set_ylim(15, 56)
            ax.set_title(title)
            ax.set_xlabel("Longitude")
            ax.set_ylabel("Latitude")
            ax.grid(alpha=0.15, linewidth=0.3)

        plt.tight_layout()
        savefig(fig, "Fig7_spatial_diagnostics")
        plt.show()

except Exception as e:
    print("Spatial map skipped:", e)


# ============================================================
# 12. Print outputs
# ============================================================

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to send back:")
print("01_distance_decay_fragility_increase_CI.csv")
print("02_AFB_water_stability_CI.csv")
print("03_stress_channel_AFB_CI.csv")
print("04_seasonal_propagation_AFB_delta_CI.csv")
print("05_matched_high_HDSI_near_vs_far_summary_CI.csv")
print("05_matching_balance_numeric_SMD.csv")
print("06_region_AFB_far_minus_near_CI.csv")

In [ ]:
# ============================================================
# Vegetation-loss stress-threshold experiment
#
# Question:
#   Does surface-water proximity increase the heat–dryness threshold
#   required for vegetation to enter a negative-response state?
#
# Core outputs:
#   1. Empirical risk curves by water-distance group
#   2. Adjusted logistic-spline risk curves
#   3. Stress thresholds at P(loss)=0.25 / 0.30 / 0.40 / 0.50
#   4. Threshold shifts between near-water and far-water groups
#   5. Permanent vs seasonal water comparison
#   6. Regional heterogeneity of threshold shifts
#
# Main data:
#   /content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/
#     _aquatic_buffering/china_aquatic_buffering_panel_v2.parquet
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    import geopandas as gpd
except Exception:
    gpd = None


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_DIR = os.path.join(AQUA_DIR, "_stress_threshold_experiments")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel file:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 3,
    "ytick.major.size": 3,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "savefig.dpi": 300,
})

COL = {
    "blue": "#3B6EA8",
    "light_blue": "#9BBBD8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "teal": "#009E73",
    "purple": "#7E62A3",
    "gray": "#666666",
    "light_gray": "#D9D9D9",
    "dark": "#222222",
}

DIST_COLORS = {
    "0-1 km": "#08306B",
    "1-3 km": "#2171B5",
    "3-5 km": "#6BAED6",
    "5-10 km": "#FDAE61",
    "10-25 km": "#F46D43",
    ">25 km": "#A50026",
}

HYDRO_COLORS = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition",
    "is_perm_water", "is_seasonal_water", "is_any_water"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Stress channels
df["SM_deficit_z"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit_z"] = -df["summer_Precip_z"]

# Negative-response events
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce")

df["Joint_neg"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Macro region, fallback if not already exists
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid sample
required_cols = [
    "HDSI", "dist_perm_water_km", "dist_seasonal_water_km",
    "EVI_neg_z05", "LAI_neg_z05",
    "spring_EVI_z", "spring_LAI_z",
    "elevation", "slope", "LC_Type1", "year"
]

valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

d = df[valid_mask].copy()
d = d.dropna(subset=[c for c in required_cols if c in d.columns]).copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["Joint_neg"] = d["Joint_neg"].astype(int)

# Permanent-water distance groups
# Note: Current dataset max dist_perm_water_km is about 15.7 km, so >25 km will likely be empty.
dist_bins = [0, 1, 3, 5, 10, 25, np.inf]
dist_labels = ["0-1 km", "1-3 km", "3-5 km", "5-10 km", "10-25 km", ">25 km"]

d["perm_dist_group"] = pd.cut(
    d["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
)

d["perm_dist_group"] = d["perm_dist_group"].astype(str)

# Hydrological water context groups
d["hydro_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "hydro_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "hydro_context"
] = "near_seasonal_lt1km"

hydro_order = ["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"]

# Water occurrence classes
occ_bins = [-0.1, 0, 5, 25, 75, 100]
occ_labels = ["0", "0-5", "5-25", "25-75", "75-100"]
d["water_occurrence_group"] = pd.cut(
    d["water_occurrence"],
    bins=occ_bins,
    labels=occ_labels,
    include_lowest=True,
    right=True
).astype(str)

# Categorical fixed effects
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round().astype("Int64").astype(str)
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce").astype("Int64").astype(str)

if "region_id" in d.columns:
    d["region_cat"] = d["region_id"].astype(str)
else:
    d["region_cat"] = d["macro_region"].astype(str)

# Overview
overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].dropna().astype(int).unique())))},
    {"metric": "HDSI_Q25", "value": d["HDSI"].quantile(0.25)},
    {"metric": "HDSI_Q75", "value": d["HDSI"].quantile(0.75)},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "Joint_neg_rate", "value": d["Joint_neg"].mean()},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_median", "value": d["dist_perm_water_km"].median()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])

save_table(overview, "00_data_overview.csv")
display(overview)

print("Valid data:", d.shape)
print("Distance group counts:")
display(d["perm_dist_group"].value_counts(dropna=False).sort_index())
print("Hydro context counts:")
display(d["hydro_context"].value_counts())


# ============================================================
# 3. Helper functions
# ============================================================

def boot_mean_ci(x, n_boot=300, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    mean = np.mean(x)
    if len(x) <= 1:
        return mean, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = np.mean(rng.choice(x, size=len(x), replace=True))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi


def threshold_from_curve(x, y, prob_threshold=0.30):
    """
    Extract the x value at which y first reaches prob_threshold.
    If y never reaches the threshold, return NaN.
    If y is already above threshold at the first x, return x[0] with status 'already_above'.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 2:
        return np.nan, "too_few_points"

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if np.nanmax(y) < prob_threshold:
        return np.nan, "not_reached"

    if y[0] >= prob_threshold:
        return x[0], "already_above"

    idx = np.where(y >= prob_threshold)[0]
    if len(idx) == 0:
        return np.nan, "not_reached"

    i = idx[0]
    if i == 0:
        return x[0], "already_above"

    x0, x1 = x[i-1], x[i]
    y0, y1 = y[i-1], y[i]

    if y1 == y0:
        return x1, "crossed_flat"

    # linear interpolation
    x_thr = x0 + (prob_threshold - y0) * (x1 - x0) / (y1 - y0)
    return x_thr, "crossed"


def make_empirical_risk_curve(
    data,
    stress_col,
    outcome_col,
    group_col,
    group_order,
    n_bins=12,
    min_n=50,
    n_boot=300,
    seed=42
):
    """
    Nonparametric empirical risk curve:
    stress is divided into quantile bins; event probability is computed per group.
    """
    tmp = data.dropna(subset=[stress_col, outcome_col, group_col]).copy()

    # Use global stress quantile bins to keep x-axis comparable across water groups
    try:
        tmp["stress_bin"] = pd.qcut(tmp[stress_col], q=n_bins, duplicates="drop")
    except Exception:
        tmp["stress_bin"] = pd.cut(tmp[stress_col], bins=n_bins)

    rows = []

    for g in group_order:
        tg = tmp[tmp[group_col].astype(str) == str(g)].copy()
        if len(tg) == 0:
            continue

        for b, gb in tg.groupby("stress_bin", observed=False):
            if len(gb) < min_n:
                continue

            m, lo, hi = boot_mean_ci(gb[outcome_col].values, n_boot=n_boot, seed=seed)
            rows.append({
                "stress_col": stress_col,
                "outcome_col": outcome_col,
                "group_col": group_col,
                "group": g,
                "stress_bin": str(b),
                "stress_mid": gb[stress_col].median(),
                "stress_mean": gb[stress_col].mean(),
                "risk": m,
                "ci_low": lo,
                "ci_high": hi,
                "rows": len(gb),
                "grids": gb["grid_id"].nunique()
            })

    return pd.DataFrame(rows)


def extract_thresholds_from_curve_table(curve_df, thresholds=[0.25, 0.30, 0.40, 0.50]):
    rows = []

    for (stress_col, outcome_col, group_col, group), g in curve_df.groupby(
        ["stress_col", "outcome_col", "group_col", "group"],
        observed=False
    ):
        g = g.sort_values("stress_mid")
        x = g["stress_mid"].values
        y = g["risk"].values

        for p in thresholds:
            thr, status = threshold_from_curve(x, y, p)
            rows.append({
                "stress_col": stress_col,
                "outcome_col": outcome_col,
                "group_col": group_col,
                "group": group,
                "prob_threshold": p,
                "stress_threshold": thr,
                "status": status,
                "risk_min": np.nanmin(y) if len(y) else np.nan,
                "risk_max": np.nanmax(y) if len(y) else np.nan,
                "n_curve_points": len(g)
            })

    return pd.DataFrame(rows)


def standardize_col(data, col):
    mu = data[col].mean()
    sd = data[col].std()
    if sd == 0 or not np.isfinite(sd):
        sd = 1.0
    return (data[col] - mu) / sd, mu, sd


def fit_adjusted_spline_model(
    data,
    stress_col,
    outcome_col,
    group_col,
    group_order,
    response_name="EVI",
    df_spline=5,
    max_rows=None,
    seed=42
):
    """
    Adjusted logistic spline model:
      outcome ~ bs(stress_z, df=5) * C(group)
                + spring vegetation + elevation + slope
                + C(LC) + C(region) + C(year)

    This estimates adjusted exposure-response curves.
    It does not replace water conditions pixel-by-pixel.
    """

    fit = data.dropna(subset=[
        stress_col, outcome_col, group_col,
        "elevation", "slope", "LC_Type1_cat", "region_cat", "year_cat"
    ]).copy()

    # Keep valid groups
    fit = fit[fit[group_col].astype(str).isin([str(x) for x in group_order])].copy()
    fit[group_col] = pd.Categorical(fit[group_col].astype(str), categories=[str(x) for x in group_order])

    # Spring controls
    if response_name == "EVI":
        control_terms = "spring_EVI_z"
        fit = fit.dropna(subset=["spring_EVI_z"])
    elif response_name == "LAI":
        control_terms = "spring_LAI_z"
        fit = fit.dropna(subset=["spring_LAI_z"])
    else:
        control_terms = "spring_EVI_z + spring_LAI_z"
        fit = fit.dropna(subset=["spring_EVI_z", "spring_LAI_z"])

    # Optional subsampling for speed; default None = use all data
    if max_rows is not None and len(fit) > max_rows:
        fit = fit.sample(max_rows, random_state=seed).copy()

    fit["stress_z"], mu, sd = standardize_col(fit, stress_col)
    fit["elevation_z"], elev_mu, elev_sd = standardize_col(fit, "elevation")
    fit["slope_z"], slope_mu, slope_sd = standardize_col(fit, "slope")

    formula = (
        f"{outcome_col} ~ bs(stress_z, df={df_spline}, include_intercept=False) * C({group_col}) "
        f"+ {control_terms} + elevation_z + slope_z "
        f"+ C(LC_Type1_cat) + C(region_cat) + C(year_cat)"
    )

    print("\nFitting adjusted spline model")
    print("Outcome:", outcome_col)
    print("Stress:", stress_col)
    print("Group:", group_col)
    print("Rows:", len(fit))
    print("Formula:", formula)

    model = smf.glm(
        formula=formula,
        data=fit,
        family=sm.families.Binomial()
    ).fit(maxiter=100)

    return model, {
        "stress_mu": mu,
        "stress_sd": sd,
        "elevation_mu": elev_mu,
        "elevation_sd": elev_sd,
        "slope_mu": slope_mu,
        "slope_sd": slope_sd,
        "fit_rows": len(fit),
        "formula": formula
    }


def predict_adjusted_curves(
    model,
    info,
    data,
    stress_col,
    outcome_col,
    group_col,
    group_order,
    response_name="EVI",
    n_grid=120,
    q_low=0.02,
    q_high=0.98
):
    """
    Predict adjusted risk curves for each group at representative covariates.
    """

    x_min = data[stress_col].quantile(q_low)
    x_max = data[stress_col].quantile(q_high)
    x_grid = np.linspace(x_min, x_max, n_grid)

    # Representative values
    base = {}
    base["elevation"] = data["elevation"].median()
    base["slope"] = data["slope"].median()
    base["elevation_z"] = (base["elevation"] - info["elevation_mu"]) / info["elevation_sd"]
    base["slope_z"] = (base["slope"] - info["slope_mu"]) / info["slope_sd"]

    base["spring_EVI_z"] = data["spring_EVI_z"].median()
    base["spring_LAI_z"] = data["spring_LAI_z"].median()
    base["LC_Type1_cat"] = data["LC_Type1_cat"].mode().iloc[0]
    base["region_cat"] = data["region_cat"].mode().iloc[0]
    base["year_cat"] = data["year_cat"].mode().iloc[0]

    pred_rows = []

    for g in group_order:
        for x in x_grid:
            r = base.copy()
            r[group_col] = str(g)
            r[stress_col] = x
            r["stress_z"] = (x - info["stress_mu"]) / info["stress_sd"]
            pred_rows.append(r)

    new = pd.DataFrame(pred_rows)

    # Important: make group categorical with same group order
    new[group_col] = pd.Categorical(new[group_col].astype(str), categories=[str(x) for x in group_order])

    sf = model.get_prediction(new).summary_frame(alpha=0.05)
    new["risk"] = sf["mean"].values
    new["ci_low"] = sf["mean_ci_lower"].values
    new["ci_high"] = sf["mean_ci_upper"].values

    new["stress_col"] = stress_col
    new["outcome_col"] = outcome_col
    new["group_col"] = group_col
    new["group"] = new[group_col].astype(str)
    new["response_name"] = response_name

    return new


# ============================================================
# 4. Empirical risk curves and thresholds
# ============================================================

stress_channels = {
    "HDSI": "HDSI",
    "VPD": "summer_VPD_z",
    "LST_Day": "summer_LST_Day_z",
    "SM_deficit": "SM_deficit_z",
    "Precip_deficit": "Precip_deficit_z",
    "Tair": "summer_Tair_z",
}

outcomes = {
    "EVI": "EVI_neg_z05",
    "LAI": "LAI_neg_z05",
    "Joint": "Joint_neg",
}

# Main group definitions
group_definitions = {
    "perm_dist_group": {
        "col": "perm_dist_group",
        "order": dist_labels,
        "label": "Permanent-water distance group"
    },
    "hydro_context": {
        "col": "hydro_context",
        "order": hydro_order,
        "label": "Hydrological context"
    },
    "water_occurrence_group": {
        "col": "water_occurrence_group",
        "order": occ_labels,
        "label": "Water occurrence group"
    }
}

# Empirical curves: main analysis with HDSI
empirical_curves = []

for group_key in ["perm_dist_group", "hydro_context", "water_occurrence_group"]:
    group_col = group_definitions[group_key]["col"]
    group_order = group_definitions[group_key]["order"]

    for response_name, outcome_col in outcomes.items():
        curve = make_empirical_risk_curve(
            data=d,
            stress_col="HDSI",
            outcome_col=outcome_col,
            group_col=group_col,
            group_order=group_order,
            n_bins=14,
            min_n=60,
            n_boot=300,
            seed=42
        )
        curve["response_name"] = response_name
        curve["group_key"] = group_key
        empirical_curves.append(curve)

empirical_curves = pd.concat(empirical_curves, ignore_index=True)
save_table(empirical_curves, "01_empirical_HDSI_risk_curves_all_groups.csv")

empirical_thresholds = extract_thresholds_from_curve_table(
    empirical_curves,
    thresholds=[0.25, 0.30, 0.40, 0.50]
)
save_table(empirical_thresholds, "02_empirical_HDSI_thresholds_all_groups.csv")
display(empirical_thresholds.head(20))


# ============================================================
# 5. Adjusted logistic-spline curves and thresholds
# ============================================================

# To keep runtime reasonable, start with HDSI for two most important groupings:
#   1) permanent distance groups
#   2) hydro context groups
#
# You can set MAX_MODEL_ROWS=None to use all rows.
MAX_MODEL_ROWS = None

adjusted_curves_list = []
adjusted_thresholds_list = []
model_summaries = []

for group_key in ["perm_dist_group", "hydro_context"]:
    group_col = group_definitions[group_key]["col"]
    group_order = group_definitions[group_key]["order"]

    # Remove empty groups for model
    valid_group_counts = d[group_col].value_counts()
    group_order_use = [g for g in group_order if valid_group_counts.get(str(g), 0) >= 100]
    print("\nGroup key:", group_key)
    print("Groups used:", group_order_use)

    for response_name, outcome_col in outcomes.items():
        # For speed and clarity, run EVI and LAI as primary; Joint optional but included.
        model, info = fit_adjusted_spline_model(
            data=d,
            stress_col="HDSI",
            outcome_col=outcome_col,
            group_col=group_col,
            group_order=group_order_use,
            response_name=response_name,
            df_spline=5,
            max_rows=MAX_MODEL_ROWS,
            seed=42
        )

        pred = predict_adjusted_curves(
            model=model,
            info=info,
            data=d,
            stress_col="HDSI",
            outcome_col=outcome_col,
            group_col=group_col,
            group_order=group_order_use,
            response_name=response_name,
            n_grid=160,
            q_low=0.02,
            q_high=0.98
        )
        pred["group_key"] = group_key
        adjusted_curves_list.append(pred)

        # Thresholds from adjusted curves
        pred_curve = pred.rename(columns={"HDSI": "stress_mid"})
        thr_rows = []
        for g, gg in pred.groupby("group", observed=False):
            gg = gg.sort_values("HDSI")
            for p in [0.25, 0.30, 0.40, 0.50]:
                thr, status = threshold_from_curve(gg["HDSI"].values, gg["risk"].values, p)
                thr_rows.append({
                    "group_key": group_key,
                    "stress_col": "HDSI",
                    "outcome_col": outcome_col,
                    "response_name": response_name,
                    "group_col": group_col,
                    "group": g,
                    "prob_threshold": p,
                    "stress_threshold": thr,
                    "status": status,
                    "risk_min": gg["risk"].min(),
                    "risk_max": gg["risk"].max(),
                    "n_curve_points": len(gg)
                })

        adjusted_thresholds_list.append(pd.DataFrame(thr_rows))

        model_summaries.append({
            "group_key": group_key,
            "response_name": response_name,
            "outcome_col": outcome_col,
            "stress_col": "HDSI",
            "group_col": group_col,
            "fit_rows": info["fit_rows"],
            "formula": info["formula"],
            "AIC": model.aic,
            "BIC": model.bic if hasattr(model, "bic") else np.nan,
            "converged": getattr(model, "converged", np.nan)
        })

adjusted_curves = pd.concat(adjusted_curves_list, ignore_index=True)
adjusted_thresholds = pd.concat(adjusted_thresholds_list, ignore_index=True)
model_summaries = pd.DataFrame(model_summaries)

save_table(adjusted_curves, "03_adjusted_HDSI_spline_risk_curves.csv")
save_table(adjusted_thresholds, "04_adjusted_HDSI_spline_thresholds.csv")
save_table(model_summaries, "05_adjusted_spline_model_summaries.csv")

display(adjusted_thresholds.head(20))


# ============================================================
# 6. Threshold shifts
# ============================================================

def compute_threshold_shifts(
    threshold_df,
    group_key,
    response_name,
    reference_group,
    comparison_groups,
    prob_thresholds=[0.25, 0.30, 0.40, 0.50],
    source_name="adjusted"
):
    """
    Threshold shift = threshold(reference_group) - threshold(comparison_group)
    Positive value: reference group requires higher stress to reach the same loss risk.
    """

    rows = []
    sub = threshold_df[
        (threshold_df["group_key"] == group_key) &
        (threshold_df["response_name"] == response_name)
    ].copy()

    for p in prob_thresholds:
        ref = sub[
            (sub["group"].astype(str) == str(reference_group)) &
            (sub["prob_threshold"] == p)
        ]

        if len(ref) == 0:
            continue

        ref_thr = ref["stress_threshold"].iloc[0]
        ref_status = ref["status"].iloc[0]

        for cg in comparison_groups:
            cc = sub[
                (sub["group"].astype(str) == str(cg)) &
                (sub["prob_threshold"] == p)
            ]

            if len(cc) == 0:
                continue

            comp_thr = cc["stress_threshold"].iloc[0]
            comp_status = cc["status"].iloc[0]

            rows.append({
                "source": source_name,
                "group_key": group_key,
                "response_name": response_name,
                "reference_group": reference_group,
                "comparison_group": cg,
                "prob_threshold": p,
                "reference_threshold": ref_thr,
                "comparison_threshold": comp_thr,
                "threshold_shift_ref_minus_comp": ref_thr - comp_thr if np.isfinite(ref_thr) and np.isfinite(comp_thr) else np.nan,
                "reference_status": ref_status,
                "comparison_status": comp_status
            })

    return pd.DataFrame(rows)


shift_tables = []

# Permanent distance group: near 0-1 km as reference.
for response_name in ["EVI", "LAI", "Joint"]:
    shift_tables.append(
        compute_threshold_shifts(
            threshold_df=adjusted_thresholds,
            group_key="perm_dist_group",
            response_name=response_name,
            reference_group="0-1 km",
            comparison_groups=["1-3 km", "3-5 km", "5-10 km", "10-25 km", ">25 km"],
            source_name="adjusted_spline"
        )
    )

# Hydro context: near permanent as reference.
for response_name in ["EVI", "LAI", "Joint"]:
    shift_tables.append(
        compute_threshold_shifts(
            threshold_df=adjusted_thresholds,
            group_key="hydro_context",
            response_name=response_name,
            reference_group="near_permanent_lt1km",
            comparison_groups=["near_seasonal_lt1km", "far_or_weak_water"],
            source_name="adjusted_spline"
        )
    )

threshold_shifts = pd.concat(shift_tables, ignore_index=True)
save_table(threshold_shifts, "06_adjusted_threshold_shifts.csv")
display(threshold_shifts)


# Also compute empirical threshold shifts
empirical_shift_tables = []

for response_name in ["EVI", "LAI", "Joint"]:
    empirical_shift_tables.append(
        compute_threshold_shifts(
            threshold_df=empirical_thresholds.assign(response_name=empirical_thresholds["outcome_col"].map({
                "EVI_neg_z05": "EVI",
                "LAI_neg_z05": "LAI",
                "Joint_neg": "Joint"
            })),
            group_key="perm_dist_group",
            response_name=response_name,
            reference_group="0-1 km",
            comparison_groups=["1-3 km", "3-5 km", "5-10 km", "10-25 km", ">25 km"],
            source_name="empirical"
        )
    )

for response_name in ["EVI", "LAI", "Joint"]:
    empirical_shift_tables.append(
        compute_threshold_shifts(
            threshold_df=empirical_thresholds.assign(response_name=empirical_thresholds["outcome_col"].map({
                "EVI_neg_z05": "EVI",
                "LAI_neg_z05": "LAI",
                "Joint_neg": "Joint"
            })),
            group_key="hydro_context",
            response_name=response_name,
            reference_group="near_permanent_lt1km",
            comparison_groups=["near_seasonal_lt1km", "far_or_weak_water"],
            source_name="empirical"
        )
    )

empirical_threshold_shifts = pd.concat(empirical_shift_tables, ignore_index=True)
save_table(empirical_threshold_shifts, "07_empirical_threshold_shifts.csv")
display(empirical_threshold_shifts)


# ============================================================
# 7. Stress-channel threshold shifts
# ============================================================

# This section repeats threshold extraction for individual stress channels using empirical curves.
# This avoids over-fitting many adjusted spline models at once.
stress_channel_curves = []

for stress_name, stress_col in stress_channels.items():
    if stress_col not in d.columns:
        continue

    for response_name, outcome_col in outcomes.items():
        curve = make_empirical_risk_curve(
            data=d,
            stress_col=stress_col,
            outcome_col=outcome_col,
            group_col="hydro_context",
            group_order=hydro_order,
            n_bins=14,
            min_n=60,
            n_boot=300,
            seed=42
        )
        curve["stress_name"] = stress_name
        curve["response_name"] = response_name
        stress_channel_curves.append(curve)

stress_channel_curves = pd.concat(stress_channel_curves, ignore_index=True)
save_table(stress_channel_curves, "08_empirical_stress_channel_risk_curves.csv")

stress_channel_thresholds = extract_thresholds_from_curve_table(
    stress_channel_curves,
    thresholds=[0.25, 0.30, 0.40, 0.50]
)

stress_channel_thresholds["stress_name"] = stress_channel_thresholds["stress_col"].map({
    "HDSI": "HDSI",
    "summer_VPD_z": "VPD",
    "summer_LST_Day_z": "LST_Day",
    "SM_deficit_z": "SM_deficit",
    "Precip_deficit_z": "Precip_deficit",
    "summer_Tair_z": "Tair",
})

stress_channel_thresholds["response_name"] = stress_channel_thresholds["outcome_col"].map({
    "EVI_neg_z05": "EVI",
    "LAI_neg_z05": "LAI",
    "Joint_neg": "Joint"
})

save_table(stress_channel_thresholds, "09_empirical_stress_channel_thresholds.csv")

# Stress-channel threshold shifts: near permanent minus far/weak.
stress_shift_rows = []

for stress_name in stress_channel_thresholds["stress_name"].dropna().unique():
    for response_name in ["EVI", "LAI", "Joint"]:
        sub = stress_channel_thresholds[
            (stress_channel_thresholds["stress_name"] == stress_name) &
            (stress_channel_thresholds["response_name"] == response_name)
        ].copy()

        for p in [0.25, 0.30, 0.40, 0.50]:
            ref = sub[
                (sub["group"] == "near_permanent_lt1km") &
                (sub["prob_threshold"] == p)
            ]
            far = sub[
                (sub["group"] == "far_or_weak_water") &
                (sub["prob_threshold"] == p)
            ]
            sea = sub[
                (sub["group"] == "near_seasonal_lt1km") &
                (sub["prob_threshold"] == p)
            ]

            if len(ref) and len(far):
                ref_thr = ref["stress_threshold"].iloc[0]
                far_thr = far["stress_threshold"].iloc[0]
                stress_shift_rows.append({
                    "stress_name": stress_name,
                    "response_name": response_name,
                    "contrast": "near_permanent - far_or_weak",
                    "prob_threshold": p,
                    "threshold_shift": ref_thr - far_thr if np.isfinite(ref_thr) and np.isfinite(far_thr) else np.nan,
                    "near_threshold": ref_thr,
                    "far_threshold": far_thr,
                    "near_status": ref["status"].iloc[0],
                    "far_status": far["status"].iloc[0],
                })

            if len(ref) and len(sea):
                ref_thr = ref["stress_threshold"].iloc[0]
                sea_thr = sea["stress_threshold"].iloc[0]
                stress_shift_rows.append({
                    "stress_name": stress_name,
                    "response_name": response_name,
                    "contrast": "near_permanent - near_seasonal",
                    "prob_threshold": p,
                    "threshold_shift": ref_thr - sea_thr if np.isfinite(ref_thr) and np.isfinite(sea_thr) else np.nan,
                    "near_threshold": ref_thr,
                    "seasonal_threshold": sea_thr,
                    "near_status": ref["status"].iloc[0],
                    "seasonal_status": sea["status"].iloc[0],
                })

stress_channel_shifts = pd.DataFrame(stress_shift_rows)
save_table(stress_channel_shifts, "10_empirical_stress_channel_threshold_shifts.csv")
display(stress_channel_shifts.head(30))


# ============================================================
# 8. Regional threshold heterogeneity
# ============================================================

# Regional empirical HDSI thresholds by hydro_context.
# This is safer than fitting many regional spline models.
regional_curves = []

min_region_rows = 1000

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()
    if len(dr) < min_region_rows:
        continue

    for response_name, outcome_col in outcomes.items():
        curve = make_empirical_risk_curve(
            data=dr,
            stress_col="HDSI",
            outcome_col=outcome_col,
            group_col="hydro_context",
            group_order=hydro_order,
            n_bins=10,
            min_n=40,
            n_boot=250,
            seed=42
        )
        curve["macro_region"] = region
        curve["response_name"] = response_name
        regional_curves.append(curve)

regional_curves = pd.concat(regional_curves, ignore_index=True)
save_table(regional_curves, "11_regional_empirical_HDSI_risk_curves.csv")

regional_thresholds = extract_thresholds_from_curve_table(
    regional_curves,
    thresholds=[0.25, 0.30, 0.40, 0.50]
)
regional_thresholds["macro_region"] = regional_thresholds["macro_region"] if "macro_region" in regional_thresholds.columns else np.nan

# The groupby extraction above does not preserve macro_region because it is not in group keys.
# Recalculate thresholds preserving region:
reg_thr_rows = []

for (region, response_name, group), g in regional_curves.groupby(
    ["macro_region", "response_name", "group"],
    observed=False
):
    g = g.sort_values("stress_mid")
    for p in [0.25, 0.30, 0.40, 0.50]:
        thr, status = threshold_from_curve(g["stress_mid"].values, g["risk"].values, p)
        reg_thr_rows.append({
            "macro_region": region,
            "response_name": response_name,
            "group": group,
            "prob_threshold": p,
            "HDSI_threshold": thr,
            "status": status,
            "risk_min": g["risk"].min(),
            "risk_max": g["risk"].max(),
            "n_curve_points": len(g)
        })

regional_thresholds = pd.DataFrame(reg_thr_rows)
save_table(regional_thresholds, "12_regional_empirical_HDSI_thresholds.csv")

# Regional threshold shifts: near permanent - far/weak
reg_shift_rows = []
for region in sorted(regional_thresholds["macro_region"].dropna().unique()):
    for response_name in ["EVI", "LAI", "Joint"]:
        sub = regional_thresholds[
            (regional_thresholds["macro_region"] == region) &
            (regional_thresholds["response_name"] == response_name)
        ]

        for p in [0.25, 0.30, 0.40, 0.50]:
            near = sub[(sub["group"] == "near_permanent_lt1km") & (sub["prob_threshold"] == p)]
            far = sub[(sub["group"] == "far_or_weak_water") & (sub["prob_threshold"] == p)]
            sea = sub[(sub["group"] == "near_seasonal_lt1km") & (sub["prob_threshold"] == p)]

            if len(near) and len(far):
                nt = near["HDSI_threshold"].iloc[0]
                ft = far["HDSI_threshold"].iloc[0]
                reg_shift_rows.append({
                    "macro_region": region,
                    "response_name": response_name,
                    "contrast": "near_permanent - far_or_weak",
                    "prob_threshold": p,
                    "threshold_shift": nt - ft if np.isfinite(nt) and np.isfinite(ft) else np.nan,
                    "near_threshold": nt,
                    "far_threshold": ft,
                    "near_status": near["status"].iloc[0],
                    "far_status": far["status"].iloc[0]
                })

            if len(near) and len(sea):
                nt = near["HDSI_threshold"].iloc[0]
                st = sea["HDSI_threshold"].iloc[0]
                reg_shift_rows.append({
                    "macro_region": region,
                    "response_name": response_name,
                    "contrast": "near_permanent - near_seasonal",
                    "prob_threshold": p,
                    "threshold_shift": nt - st if np.isfinite(nt) and np.isfinite(st) else np.nan,
                    "near_threshold": nt,
                    "seasonal_threshold": st,
                    "near_status": near["status"].iloc[0],
                    "seasonal_status": sea["status"].iloc[0]
                })

regional_threshold_shifts = pd.DataFrame(reg_shift_rows)
save_table(regional_threshold_shifts, "13_regional_empirical_HDSI_threshold_shifts.csv")
display(regional_threshold_shifts.head(30))


# ============================================================
# 9. Figures
# ============================================================

# ------------------------------------------------------------
# Figure 1: Adjusted HDSI risk curves by permanent-water distance group
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, outcome_col, label in zip(
    axes,
    ["EVI", "LAI", "Joint"],
    ["EVI_neg_z05", "LAI_neg_z05", "Joint_neg"],
    ["A", "B", "C"]
):
    sub = adjusted_curves[
        (adjusted_curves["group_key"] == "perm_dist_group") &
        (adjusted_curves["response_name"] == response_name)
    ].copy()

    for g in dist_labels:
        sg = sub[sub["group"] == g].sort_values("HDSI")
        if len(sg) == 0:
            continue

        color = DIST_COLORS.get(g, COL["gray"])
        ax.plot(sg["HDSI"], sg["risk"], color=color, lw=2, label=g)
        ax.fill_between(sg["HDSI"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.12, lw=0)

    ax.axhline(0.30, color="#333333", linestyle="--", lw=0.9)
    ax.set_xlabel("HDSI")
    ax.set_title(f"{response_name} loss risk")
    ax.set_ylim(0, 0.75)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Adjusted probability of negative response")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig1_adjusted_HDSI_risk_curves_by_perm_distance")
plt.show()


# ------------------------------------------------------------
# Figure 2: Adjusted HDSI thresholds by distance group
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

target_prob = 0.30

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = adjusted_thresholds[
        (adjusted_thresholds["group_key"] == "perm_dist_group") &
        (adjusted_thresholds["response_name"] == response_name) &
        (adjusted_thresholds["prob_threshold"] == target_prob)
    ].copy()

    sub["group"] = pd.Categorical(sub["group"], categories=dist_labels, ordered=True)
    sub = sub.sort_values("group")

    x = np.arange(len(sub))
    colors = [DIST_COLORS.get(str(g), COL["gray"]) for g in sub["group"]]

    ax.bar(x, sub["stress_threshold"], color=colors, width=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["group"], rotation=35, ha="right")
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_title(f"HDSI threshold at P({response_name} loss)={target_prob}")
    ax.set_ylabel("HDSI threshold")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_adjusted_HDSI_thresholds_by_perm_distance")
plt.show()


# ------------------------------------------------------------
# Figure 3: Threshold shift: near water versus farther groups
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = threshold_shifts[
        (threshold_shifts["group_key"] == "perm_dist_group") &
        (threshold_shifts["response_name"] == response_name) &
        (threshold_shifts["prob_threshold"] == target_prob)
    ].copy()

    comp_order = ["1-3 km", "3-5 km", "5-10 km", "10-25 km", ">25 km"]
    sub["comparison_group"] = pd.Categorical(sub["comparison_group"], categories=comp_order, ordered=True)
    sub = sub.sort_values("comparison_group")

    x = np.arange(len(sub))
    ax.bar(x, sub["threshold_shift_ref_minus_comp"], color=COL["blue"], width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["comparison_group"], rotation=35, ha="right")
    ax.set_title(f"Threshold shift: 0–1 km minus farther groups ({response_name})")
    ax.set_ylabel("HDSI threshold shift")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_adjusted_threshold_shifts_perm_distance")
plt.show()


# ------------------------------------------------------------
# Figure 4: Permanent vs seasonal vs far/weak risk curves
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

hydro_labels = {
    "near_permanent_lt1km": "Near permanent <1 km",
    "near_seasonal_lt1km": "Near seasonal <1 km",
    "far_or_weak_water": "Far / weak water",
}

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = adjusted_curves[
        (adjusted_curves["group_key"] == "hydro_context") &
        (adjusted_curves["response_name"] == response_name)
    ].copy()

    for g in hydro_order:
        sg = sub[sub["group"] == g].sort_values("HDSI")
        if len(sg) == 0:
            continue

        color = HYDRO_COLORS.get(g, COL["gray"])
        ax.plot(sg["HDSI"], sg["risk"], color=color, lw=2, label=hydro_labels[g])
        ax.fill_between(sg["HDSI"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.15, lw=0)

    ax.axhline(0.30, color="#333333", linestyle="--", lw=0.9)
    ax.set_xlabel("HDSI")
    ax.set_title(f"{response_name} loss risk")
    ax.set_ylim(0, 0.75)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Adjusted probability of negative response")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig4_adjusted_HDSI_risk_curves_by_hydro_context")
plt.show()


# ------------------------------------------------------------
# Figure 5: Stress-channel threshold shift
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

stress_order = ["HDSI", "VPD", "LST_Day", "SM_deficit", "Precip_deficit", "Tair"]

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = stress_channel_shifts[
        (stress_channel_shifts["response_name"] == response_name) &
        (stress_channel_shifts["contrast"] == "near_permanent - far_or_weak") &
        (stress_channel_shifts["prob_threshold"] == target_prob)
    ].copy()

    sub["stress_name"] = pd.Categorical(sub["stress_name"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress_name")

    x = np.arange(len(sub))
    ax.bar(x, sub["threshold_shift"], color=RESP_COLORS[response_name], width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress_name"], rotation=35, ha="right")
    ax.set_title(f"Threshold shift by stress channel ({response_name})")
    ax.set_ylabel("Near-permanent threshold − far/weak threshold")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_stress_channel_threshold_shifts")
plt.show()


# ------------------------------------------------------------
# Figure 6: Regional HDSI threshold shifts
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = regional_threshold_shifts[
        (regional_threshold_shifts["response_name"] == response_name) &
        (regional_threshold_shifts["contrast"] == "near_permanent - far_or_weak") &
        (regional_threshold_shifts["prob_threshold"] == target_prob)
    ].copy()

    sub = sub.sort_values("threshold_shift")
    y = np.arange(len(sub))

    ax.scatter(sub["threshold_shift"], y, color=RESP_COLORS[response_name], s=45)
    ax.axvline(0, color="#333333", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["macro_region"])
    ax.set_xlabel("HDSI threshold shift")
    ax.set_title(f"Regional threshold shift ({response_name})")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig6_regional_threshold_shifts")
plt.show()


# ============================================================
# 10. Optional: Regional risk curves for key regions
# ============================================================

key_regions = [
    "Tibetan_Plateau",
    "Southwest",
    "Northwest_Arid",
    "North_China",
    "Northeast"
]

plot_reg = regional_curves[
    (regional_curves["macro_region"].isin(key_regions)) &
    (regional_curves["response_name"] == "Joint")
].copy()

if len(plot_reg) > 0:
    fig, axes = plt.subplots(1, len(key_regions), figsize=(3.2 * len(key_regions), 4), sharey=True)

    if len(key_regions) == 1:
        axes = [axes]

    for ax, region in zip(axes, key_regions):
        sub = plot_reg[plot_reg["macro_region"] == region].copy()

        for g in hydro_order:
            sg = sub[sub["group"] == g].sort_values("stress_mid")
            if len(sg) == 0:
                continue

            color = HYDRO_COLORS.get(g, COL["gray"])
            ax.plot(sg["stress_mid"], sg["risk"], color=color, lw=2, label=hydro_labels[g])
            ax.fill_between(sg["stress_mid"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.12, lw=0)

        ax.axhline(0.30, color="#333333", linestyle="--", lw=0.8)
        ax.set_title(region)
        ax.set_xlabel("HDSI")
        clean_ax(ax)

    axes[0].set_ylabel("Empirical joint failure risk")
    axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    savefig(fig, "Fig7_key_region_empirical_HDSI_risk_curves")
    plt.show()


# ============================================================
# 11. Print outputs
# ============================================================

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("02_empirical_HDSI_thresholds_all_groups.csv")
print("04_adjusted_HDSI_spline_thresholds.csv")
print("06_adjusted_threshold_shifts.csv")
print("10_empirical_stress_channel_threshold_shifts.csv")
print("13_regional_empirical_HDSI_threshold_shifts.csv")

print("\nKey figures to inspect:")
print("Fig1_adjusted_HDSI_risk_curves_by_perm_distance.png")
print("Fig2_adjusted_HDSI_thresholds_by_perm_distance.png")
print("Fig3_adjusted_threshold_shifts_perm_distance.png")
print("Fig4_adjusted_HDSI_risk_curves_by_hydro_context.png")
print("Fig5_stress_channel_threshold_shifts.png")
print("Fig6_regional_threshold_shifts.png")

In [ ]:
# ============================================================
# Mechanism experiments for vegetation-loss stress thresholds
#
# Core question:
#   WHY does surface-water proximity change the HDSI threshold
#   required for vegetation to enter negative-response state?
#
# Mechanism pathways:
#   A. Water-supply pathway:
#      SM_deficit_z, Precip_deficit_z
#   B. Thermal / atmospheric pathway:
#      LST_Day_z, VPD_z, Tair_z
#   C. Spring vegetation state:
#      spring_EVI_z, spring_LAI_z
#
# Outputs:
#   tables/
#     00_data_overview.csv
#     01_threshold_base_models.csv
#     02_threshold_shift_attenuation.csv
#     03_HDSI_to_mediator_curves.csv
#     04_mediator_to_loss_curves.csv
#     05_mediator_thresholds.csv
#     06_regional_threshold_shifts.csv
#
#   figs/
#     Fig1_threshold_curves_HDSI.png
#     Fig2_threshold_attenuation.png
#     Fig3_HDSI_to_mediator_pathways.png
#     Fig4_mediator_to_loss_pathways.png
#     Fig5_regional_threshold_shifts.png
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

import statsmodels.api as sm
import statsmodels.formula.api as smf

# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

OUT_DIR = os.path.join(AQUA_DIR, "_threshold_mechanism_chain")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222",
}

HYDRO_COLORS = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

MED_COLORS = {
    "SM_deficit_z": COL["blue"],
    "Precip_deficit_z": COL["teal"] if "teal" in COL else "#009E73",
    "summer_LST_Day_z": COL["red"],
    "summer_VPD_z": COL["orange"],
    "summer_Tair_z": COL["purple"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

df["SM_deficit_z"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit_z"] = -df["summer_Precip_z"]

if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce")

df["Joint_neg"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Macro regions
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

required = [
    "HDSI", "dist_perm_water_km", "dist_seasonal_water_km",
    "EVI_neg_z05", "LAI_neg_z05",
    "spring_EVI_z", "spring_LAI_z",
    "elevation", "slope", "LC_Type1", "year",
    "summer_VPD_z", "summer_LST_Day_z",
    "SM_deficit_z", "Precip_deficit_z", "summer_Tair_z"
]

d = df[valid_mask].copy()
d = d.dropna(subset=[c for c in required if c in d.columns]).copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["Joint_neg"] = d["Joint_neg"].astype(int)

d["hydro_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "hydro_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "hydro_context"
] = "near_seasonal_lt1km"

hydro_order = ["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"]
d["hydro_context"] = pd.Categorical(d["hydro_context"], categories=hydro_order, ordered=True)

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round().astype("Int64").astype(str)
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce").astype("Int64").astype(str)
d["region_cat"] = d["region_id"].astype(str) if "region_id" in d.columns else d["macro_region"].astype(str)

# standardize variables for stable model terms
def zscore(data, col):
    mu = data[col].mean()
    sd = data[col].std()
    if not np.isfinite(sd) or sd == 0:
        sd = 1
    data[col + "_std"] = (data[col] - mu) / sd
    return mu, sd

scale_info = {}
for c in [
    "HDSI",
    "SM_deficit_z",
    "Precip_deficit_z",
    "summer_LST_Day_z",
    "summer_VPD_z",
    "summer_Tair_z",
    "elevation",
    "slope",
    "spring_EVI_z",
    "spring_LAI_z"
]:
    scale_info[c] = zscore(d, c)

overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].astype(int).unique())))},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "Joint_neg_rate", "value": d["Joint_neg"].mean()},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])
save_table(overview, "00_data_overview.csv")
display(overview)


# ============================================================
# 3. Helper functions
# ============================================================

def threshold_from_curve(x, y, prob_threshold=0.40):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 2:
        return np.nan, "too_few_points"

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if np.nanmax(y) < prob_threshold:
        return np.nan, "not_reached"

    if y[0] >= prob_threshold:
        return x[0], "already_above"

    idx = np.where(y >= prob_threshold)[0]
    if len(idx) == 0:
        return np.nan, "not_reached"

    i = idx[0]
    if i == 0:
        return x[0], "already_above"

    x0, x1 = x[i - 1], x[i]
    y0, y1 = y[i - 1], y[i]

    if y1 == y0:
        return x1, "crossed_flat"

    x_thr = x0 + (prob_threshold - y0) * (x1 - x0) / (y1 - y0)
    return x_thr, "crossed"


def representative_base(data):
    base = {
        "elevation_std": 0.0,
        "slope_std": 0.0,
        "spring_EVI_z_std": 0.0,
        "spring_LAI_z_std": 0.0,
        "LC_Type1_cat": data["LC_Type1_cat"].mode().iloc[0],
        "region_cat": data["region_cat"].mode().iloc[0],
        "year_cat": data["year_cat"].mode().iloc[0],
    }
    return base


def fit_glm_binomial(formula, data, maxiter=100):
    model = smf.glm(
        formula=formula,
        data=data,
        family=sm.families.Binomial()
    ).fit(maxiter=maxiter)
    return model


def predict_loss_curve(
    model,
    data,
    x_var_std,
    x_var_original,
    group_col="hydro_context",
    group_order=hydro_order,
    response_name="EVI",
    n_grid=160,
    q_low=0.02,
    q_high=0.98,
    extra_fixed=None
):
    """
    Predict adjusted loss probability curves across x_var.
    x_var_std is used in model formula.
    x_var_original is the original variable for plotting and threshold extraction.
    """
    x_min = data[x_var_original].quantile(q_low)
    x_max = data[x_var_original].quantile(q_high)
    x_grid = np.linspace(x_min, x_max, n_grid)

    mu, sd = scale_info[x_var_original.replace("_std", "")] if x_var_original.replace("_std", "") in scale_info else (data[x_var_original].mean(), data[x_var_original].std())

    base = representative_base(data)

    # default mediators set to 0 if used
    for c in [
        "HDSI_std", "SM_deficit_z_std", "Precip_deficit_z_std",
        "summer_LST_Day_z_std", "summer_VPD_z_std", "summer_Tair_z_std"
    ]:
        base[c] = 0.0

    if extra_fixed:
        base.update(extra_fixed)

    rows = []
    for g in group_order:
        for x in x_grid:
            r = base.copy()
            r[group_col] = g
            r[x_var_original] = x

            orig_name = x_var_original
            if orig_name in scale_info:
                mu, sd = scale_info[orig_name]
                r[x_var_std] = (x - mu) / sd
            else:
                r[x_var_std] = x

            rows.append(r)

    new = pd.DataFrame(rows)
    new[group_col] = pd.Categorical(new[group_col], categories=group_order, ordered=True)

    sf = model.get_prediction(new).summary_frame(alpha=0.05)
    new["risk"] = sf["mean"].values
    new["ci_low"] = sf["mean_ci_lower"].values
    new["ci_high"] = sf["mean_ci_upper"].values
    new["x_value"] = new[x_var_original]
    new["group"] = new[group_col].astype(str)
    new["response_name"] = response_name
    new["x_var"] = x_var_original

    return new


def extract_threshold_table(curves, prob_thresholds=[0.30, 0.40, 0.50]):
    rows = []
    for (response_name, x_var, group), g in curves.groupby(["response_name", "x_var", "group"], observed=False):
        g = g.sort_values("x_value")
        for p in prob_thresholds:
            thr, status = threshold_from_curve(g["x_value"].values, g["risk"].values, p)
            rows.append({
                "response_name": response_name,
                "x_var": x_var,
                "group": group,
                "prob_threshold": p,
                "threshold": thr,
                "status": status,
                "risk_min": g["risk"].min(),
                "risk_max": g["risk"].max()
            })
    return pd.DataFrame(rows)


def compute_shift(thr, response_name, x_var, p, ref="near_permanent_lt1km", comp="far_or_weak_water"):
    sub = thr[
        (thr["response_name"] == response_name) &
        (thr["x_var"] == x_var) &
        (thr["prob_threshold"] == p)
    ]
    a = sub[sub["group"] == ref]
    b = sub[sub["group"] == comp]
    if len(a) == 0 or len(b) == 0:
        return np.nan, None, None

    at = a["threshold"].iloc[0]
    bt = b["threshold"].iloc[0]
    ast = a["status"].iloc[0]
    bst = b["status"].iloc[0]

    if not np.isfinite(at) or not np.isfinite(bt):
        return np.nan, ast, bst

    return at - bt, ast, bst


# ============================================================
# 4. Step 1: baseline HDSI threshold curves
# ============================================================

outcomes = {
    "EVI": "EVI_neg_z05",
    "LAI": "LAI_neg_z05",
    "Joint": "Joint_neg",
}

base_models = {}
base_curves = []

for response_name, ycol in outcomes.items():
    if response_name == "EVI":
        spring_terms = "spring_EVI_z_std"
    elif response_name == "LAI":
        spring_terms = "spring_LAI_z_std"
    else:
        spring_terms = "spring_EVI_z_std + spring_LAI_z_std"

    formula = (
        f"{ycol} ~ bs(HDSI_std, df=5, include_intercept=False) * C(hydro_context) "
        f"+ {spring_terms} + elevation_std + slope_std "
        f"+ C(LC_Type1_cat) + C(region_cat) + C(year_cat)"
    )

    print("\nFitting base HDSI threshold model:", response_name)
    print(formula)

    model = fit_glm_binomial(formula, d)
    base_models[response_name] = model

    curve = predict_loss_curve(
        model=model,
        data=d,
        x_var_std="HDSI_std",
        x_var_original="HDSI",
        response_name=response_name,
        n_grid=160
    )
    curve["model_block"] = "base_HDSI"
    base_curves.append(curve)

base_curves = pd.concat(base_curves, ignore_index=True)
base_thresholds = extract_threshold_table(base_curves, prob_thresholds=[0.30, 0.40, 0.50])

save_table(base_curves, "01_base_HDSI_threshold_curves.csv")
save_table(base_thresholds, "01_base_HDSI_thresholds.csv")


# ============================================================
# 5. Step 2: attenuation analysis
# ============================================================

# We compare threshold shifts after adding mediator blocks.
# If adding SM variables reduces near-permanent vs far/weak shift,
# water-supply pathway likely explains part of the threshold difference.

model_blocks = {
    "M0_static_only": {
        "extra_terms": "",
        "description": "HDSI × water + static FE; no spring or mediator"
    },
    "M1_plus_spring": {
        "extra_terms": "SPRING",
        "description": "Adds spring vegetation state"
    },
    "M2_plus_water_supply": {
        "extra_terms": "SPRING + SM_deficit_z_std + Precip_deficit_z_std",
        "description": "Adds spring + water-supply mediators"
    },
    "M3_plus_thermal": {
        "extra_terms": "SPRING + summer_LST_Day_z_std + summer_VPD_z_std + summer_Tair_z_std",
        "description": "Adds spring + thermal/atmospheric mediators"
    },
    "M4_full_pathway": {
        "extra_terms": "SPRING + SM_deficit_z_std + Precip_deficit_z_std + summer_LST_Day_z_std + summer_VPD_z_std + summer_Tair_z_std",
        "description": "Adds spring + all mediator variables"
    }
}

atten_curves = []
atten_thresholds_all = []
atten_models_summary = []

for response_name, ycol in outcomes.items():
    for block_name, block in model_blocks.items():

        if response_name == "EVI":
            spring = "spring_EVI_z_std"
        elif response_name == "LAI":
            spring = "spring_LAI_z_std"
        else:
            spring = "spring_EVI_z_std + spring_LAI_z_std"

        extra = block["extra_terms"]
        extra = extra.replace("SPRING", spring)

        if extra.strip() == "":
            extra_formula = ""
        else:
            extra_formula = " + " + extra

        formula = (
            f"{ycol} ~ bs(HDSI_std, df=5, include_intercept=False) * C(hydro_context) "
            f"+ elevation_std + slope_std "
            f"+ C(LC_Type1_cat) + C(region_cat) + C(year_cat)"
            f"{extra_formula}"
        )

        print("\nFitting attenuation model:", response_name, block_name)
        print(formula)

        model = fit_glm_binomial(formula, d)

        curve = predict_loss_curve(
            model=model,
            data=d,
            x_var_std="HDSI_std",
            x_var_original="HDSI",
            response_name=response_name,
            n_grid=160
        )
        curve["model_block"] = block_name
        curve["model_description"] = block["description"]
        atten_curves.append(curve)

        thr = extract_threshold_table(curve, prob_thresholds=[0.30, 0.40, 0.50])
        thr["model_block"] = block_name
        thr["model_description"] = block["description"]
        atten_thresholds_all.append(thr)

        atten_models_summary.append({
            "response_name": response_name,
            "model_block": block_name,
            "description": block["description"],
            "AIC": model.aic,
            "nobs": model.nobs
        })

atten_curves = pd.concat(atten_curves, ignore_index=True)
atten_thresholds = pd.concat(atten_thresholds_all, ignore_index=True)
atten_models_summary = pd.DataFrame(atten_models_summary)

# compute threshold shifts and attenuation
shift_rows = []
for response_name in outcomes.keys():
    for block_name in model_blocks.keys():
        for p in [0.30, 0.40, 0.50]:
            sub = atten_thresholds[
                (atten_thresholds["response_name"] == response_name) &
                (atten_thresholds["model_block"] == block_name) &
                (atten_thresholds["prob_threshold"] == p)
            ].copy()

            for comp in ["near_seasonal_lt1km", "far_or_weak_water"]:
                near = sub[sub["group"] == "near_permanent_lt1km"]
                cc = sub[sub["group"] == comp]

                if len(near) == 0 or len(cc) == 0:
                    continue

                nt = near["threshold"].iloc[0]
                ct = cc["threshold"].iloc[0]
                ns = near["status"].iloc[0]
                cs = cc["status"].iloc[0]

                shift = nt - ct if np.isfinite(nt) and np.isfinite(ct) else np.nan

                shift_rows.append({
                    "response_name": response_name,
                    "model_block": block_name,
                    "comparison": f"near_permanent - {comp}",
                    "prob_threshold": p,
                    "near_threshold": nt,
                    "comparison_threshold": ct,
                    "threshold_shift": shift,
                    "near_status": ns,
                    "comparison_status": cs,
                    "valid_crossed_only": (
                        ns in ["crossed", "crossed_flat"] and
                        cs in ["crossed", "crossed_flat"] and
                        np.isfinite(shift)
                    )
                })

atten_shifts = pd.DataFrame(shift_rows)

# attenuation ratio relative to M0 or M1
def add_attenuation_ratio(tab, baseline_block="M1_plus_spring"):
    out = tab.copy()
    out["attenuation_relative_to"] = baseline_block
    out["attenuation_fraction"] = np.nan

    key_cols = ["response_name", "comparison", "prob_threshold"]

    for keys, g in out.groupby(key_cols):
        base = g[g["model_block"] == baseline_block]
        if len(base) == 0:
            continue
        base_shift = base["threshold_shift"].iloc[0]
        if not np.isfinite(base_shift) or base_shift == 0:
            continue

        idx = g.index
        out.loc[idx, "attenuation_fraction"] = (
            (base_shift - out.loc[idx, "threshold_shift"]) / base_shift
        )

    return out

atten_shifts = add_attenuation_ratio(atten_shifts, baseline_block="M1_plus_spring")

save_table(atten_curves, "02_attenuation_HDSI_curves.csv")
save_table(atten_thresholds, "02_attenuation_HDSI_thresholds.csv")
save_table(atten_shifts, "02_threshold_shift_attenuation.csv")
save_table(atten_models_summary, "02_attenuation_model_summaries.csv")


# ============================================================
# 6. Step 3: HDSI -> mediator pathway
# ============================================================

mediators = {
    "SM_deficit_z": "Water deficit: soil moisture",
    "Precip_deficit_z": "Water supply: precipitation deficit",
    "summer_LST_Day_z": "Surface thermal stress",
    "summer_VPD_z": "Atmospheric dryness",
    "summer_Tair_z": "Air temperature"
}

mediator_models = {}
mediator_curves = []

for med, med_label in mediators.items():
    formula = (
        f"{med}_std ~ bs(HDSI_std, df=5, include_intercept=False) * C(hydro_context) "
        f"+ elevation_std + slope_std "
        f"+ C(LC_Type1_cat) + C(region_cat) + C(year_cat)"
    )

    print("\nFitting HDSI -> mediator model:", med)
    print(formula)

    model = smf.ols(formula, data=d).fit(cov_type="HC3")
    mediator_models[med] = model

    x_grid = np.linspace(d["HDSI"].quantile(0.02), d["HDSI"].quantile(0.98), 160)
    base = representative_base(d)

    rows = []
    for g in hydro_order:
        for x in x_grid:
            r = base.copy()
            r["hydro_context"] = g
            r["HDSI"] = x
            mu, sd = scale_info["HDSI"]
            r["HDSI_std"] = (x - mu) / sd
            rows.append(r)

    new = pd.DataFrame(rows)
    new["hydro_context"] = pd.Categorical(new["hydro_context"], categories=hydro_order, ordered=True)

    pred = model.get_prediction(new).summary_frame(alpha=0.05)
    new["mediator_pred_std"] = pred["mean"].values
    new["ci_low"] = pred["mean_ci_lower"].values
    new["ci_high"] = pred["mean_ci_upper"].values
    new["mediator"] = med
    new["mediator_label"] = med_label
    new["group"] = new["hydro_context"].astype(str)
    new["HDSI_value"] = new["HDSI"]

    mediator_curves.append(new)

mediator_curves = pd.concat(mediator_curves, ignore_index=True)
save_table(mediator_curves, "03_HDSI_to_mediator_curves.csv")


# ============================================================
# 7. Step 4: mediator -> vegetation loss pathway
# ============================================================

mediator_loss_curves = []
mediator_loss_thresholds_all = []
mediator_loss_models_summary = []

for med, med_label in mediators.items():
    med_std = med + "_std"

    for response_name, ycol in outcomes.items():
        if response_name == "EVI":
            spring_terms = "spring_EVI_z_std"
        elif response_name == "LAI":
            spring_terms = "spring_LAI_z_std"
        else:
            spring_terms = "spring_EVI_z_std + spring_LAI_z_std"

        # HDSI_std is included to ask whether mediator matters beyond overall HDSI.
        formula = (
            f"{ycol} ~ bs({med_std}, df=5, include_intercept=False) * C(hydro_context) "
            f"+ HDSI_std + {spring_terms} "
            f"+ elevation_std + slope_std "
            f"+ C(LC_Type1_cat) + C(region_cat) + C(year_cat)"
        )

        print("\nFitting mediator -> loss model:", med, response_name)
        print(formula)

        model = fit_glm_binomial(formula, d)

        curve = predict_loss_curve(
            model=model,
            data=d,
            x_var_std=med_std,
            x_var_original=med,
            response_name=response_name,
            n_grid=160,
            extra_fixed={"HDSI_std": 0.0}
        )
        curve["mediator"] = med
        curve["mediator_label"] = med_label
        mediator_loss_curves.append(curve)

        thr = extract_threshold_table(curve, prob_thresholds=[0.30, 0.40, 0.50])
        thr["mediator"] = med
        thr["mediator_label"] = med_label
        mediator_loss_thresholds_all.append(thr)

        mediator_loss_models_summary.append({
            "mediator": med,
            "response_name": response_name,
            "AIC": model.aic,
            "nobs": model.nobs
        })

mediator_loss_curves = pd.concat(mediator_loss_curves, ignore_index=True)
mediator_loss_thresholds = pd.concat(mediator_loss_thresholds_all, ignore_index=True)
mediator_loss_models_summary = pd.DataFrame(mediator_loss_models_summary)

save_table(mediator_loss_curves, "04_mediator_to_loss_curves.csv")
save_table(mediator_loss_thresholds, "05_mediator_loss_thresholds.csv")
save_table(mediator_loss_models_summary, "05_mediator_loss_model_summaries.csv")


# ============================================================
# 8. Regional threshold shifts
# ============================================================

regional_rows = []

target_p = 0.40

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()

    if len(dr) < 1000:
        continue

    for response_name, ycol in outcomes.items():
        if response_name == "EVI":
            spring_terms = "spring_EVI_z_std"
        elif response_name == "LAI":
            spring_terms = "spring_LAI_z_std"
        else:
            spring_terms = "spring_EVI_z_std + spring_LAI_z_std"

        formula = (
            f"{ycol} ~ bs(HDSI_std, df=4, include_intercept=False) * C(hydro_context) "
            f"+ {spring_terms} + elevation_std + slope_std "
            f"+ C(LC_Type1_cat) + C(year_cat)"
        )

        try:
            model = fit_glm_binomial(formula, dr)
            curve = predict_loss_curve(
                model=model,
                data=dr,
                x_var_std="HDSI_std",
                x_var_original="HDSI",
                response_name=response_name,
                n_grid=120
            )
            thr = extract_threshold_table(curve, prob_thresholds=[target_p])

            near = thr[(thr["group"] == "near_permanent_lt1km") & (thr["prob_threshold"] == target_p)]
            far = thr[(thr["group"] == "far_or_weak_water") & (thr["prob_threshold"] == target_p)]
            sea = thr[(thr["group"] == "near_seasonal_lt1km") & (thr["prob_threshold"] == target_p)]

            if len(near) and len(far):
                regional_rows.append({
                    "macro_region": region,
                    "response_name": response_name,
                    "contrast": "near_permanent - far_or_weak",
                    "prob_threshold": target_p,
                    "near_threshold": near["threshold"].iloc[0],
                    "comparison_threshold": far["threshold"].iloc[0],
                    "threshold_shift": (
                        near["threshold"].iloc[0] - far["threshold"].iloc[0]
                        if np.isfinite(near["threshold"].iloc[0]) and np.isfinite(far["threshold"].iloc[0])
                        else np.nan
                    ),
                    "near_status": near["status"].iloc[0],
                    "comparison_status": far["status"].iloc[0],
                    "n_region": len(dr)
                })

            if len(near) and len(sea):
                regional_rows.append({
                    "macro_region": region,
                    "response_name": response_name,
                    "contrast": "near_permanent - near_seasonal",
                    "prob_threshold": target_p,
                    "near_threshold": near["threshold"].iloc[0],
                    "comparison_threshold": sea["threshold"].iloc[0],
                    "threshold_shift": (
                        near["threshold"].iloc[0] - sea["threshold"].iloc[0]
                        if np.isfinite(near["threshold"].iloc[0]) and np.isfinite(sea["threshold"].iloc[0])
                        else np.nan
                    ),
                    "near_status": near["status"].iloc[0],
                    "comparison_status": sea["status"].iloc[0],
                    "n_region": len(dr)
                })

        except Exception as e:
            print("Regional model failed:", region, response_name, e)

regional_threshold_shifts = pd.DataFrame(regional_rows)
save_table(regional_threshold_shifts, "06_regional_threshold_shifts.csv")


# ============================================================
# 9. Figures
# ============================================================

# Fig 1: HDSI threshold curves
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = base_curves[base_curves["response_name"] == response_name].copy()

    for g in hydro_order:
        sg = sub[sub["group"] == g].sort_values("x_value")
        color = HYDRO_COLORS[g]
        ax.plot(sg["x_value"], sg["risk"], color=color, lw=2, label=g.replace("_", " "))
        ax.fill_between(sg["x_value"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.14, lw=0)

    ax.axhline(0.40, color="#333333", ls="--", lw=0.9)
    ax.set_xlabel("HDSI")
    ax.set_title(f"{response_name} loss risk")
    ax.set_ylim(0, 0.75)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Adjusted probability of negative response")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig1_threshold_curves_HDSI")
plt.show()


# Fig 2: attenuation
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

plot_atten = atten_shifts[
    (atten_shifts["comparison"] == "near_permanent - far_or_weak_water") &
    (atten_shifts["prob_threshold"] == 0.40)
].copy()

block_order = [
    "M0_static_only",
    "M1_plus_spring",
    "M2_plus_water_supply",
    "M3_plus_thermal",
    "M4_full_pathway"
]

block_labels = [
    "Static",
    "+Spring",
    "+Water\nsupply",
    "+Thermal",
    "+Full"
]

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = plot_atten[plot_atten["response_name"] == response_name].copy()
    sub["model_block"] = pd.Categorical(sub["model_block"], categories=block_order, ordered=True)
    sub = sub.sort_values("model_block")

    x = np.arange(len(sub))
    ax.bar(x, sub["threshold_shift"], color=RESP_COLORS[response_name], width=0.65)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(block_labels, rotation=0)
    ax.set_ylabel("HDSI threshold shift\nnear permanent − far/weak")
    ax.set_title(f"Attenuation of threshold shift ({response_name})")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_threshold_attenuation")
plt.show()


# Fig 3: HDSI -> mediator
fig, axes = plt.subplots(1, 5, figsize=(17, 4.2), sharex=True)

mediator_plot_order = [
    "SM_deficit_z",
    "Precip_deficit_z",
    "summer_LST_Day_z",
    "summer_VPD_z",
    "summer_Tair_z"
]

for ax, med, label in zip(axes, mediator_plot_order, ["A", "B", "C", "D", "E"]):
    sub = mediator_curves[mediator_curves["mediator"] == med].copy()

    for g in hydro_order:
        sg = sub[sub["group"] == g].sort_values("HDSI_value")
        color = HYDRO_COLORS[g]
        ax.plot(sg["HDSI_value"], sg["mediator_pred_std"], color=color, lw=2, label=g.replace("_", " "))
        ax.fill_between(sg["HDSI_value"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.12, lw=0)

    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xlabel("HDSI")
    ax.set_title(med)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Adjusted mediator anomaly")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig3_HDSI_to_mediator_pathways")
plt.show()


# Fig 4: mediator -> loss, Joint only
fig, axes = plt.subplots(1, 5, figsize=(17, 4.2), sharey=True)

for ax, med, label in zip(axes, mediator_plot_order, ["A", "B", "C", "D", "E"]):
    sub = mediator_loss_curves[
        (mediator_loss_curves["mediator"] == med) &
        (mediator_loss_curves["response_name"] == "Joint")
    ].copy()

    for g in hydro_order:
        sg = sub[sub["group"] == g].sort_values("x_value")
        color = HYDRO_COLORS[g]
        ax.plot(sg["x_value"], sg["risk"], color=color, lw=2, label=g.replace("_", " "))
        ax.fill_between(sg["x_value"], sg["ci_low"], sg["ci_high"], color=color, alpha=0.12, lw=0)

    ax.axhline(0.40, color="#333333", ls="--", lw=0.8)
    ax.set_xlabel(med)
    ax.set_title(f"{med} → Joint loss")
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Adjusted joint loss probability")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig4_mediator_to_loss_pathways")
plt.show()


# Fig 5: regional threshold shifts
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = regional_threshold_shifts[
        (regional_threshold_shifts["response_name"] == response_name) &
        (regional_threshold_shifts["contrast"] == "near_permanent - far_or_weak")
    ].copy()

    sub = sub.sort_values("threshold_shift")
    y = np.arange(len(sub))

    ax.scatter(sub["threshold_shift"], y, color=RESP_COLORS[response_name], s=45)
    ax.axvline(0, color="#333333", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["macro_region"])
    ax.set_xlabel("HDSI threshold shift")
    ax.set_title(f"Regional threshold shift ({response_name})")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_regional_threshold_shifts")
plt.show()


# ============================================================
# 10. Print key outputs
# ============================================================

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("01_base_HDSI_thresholds.csv")
print("02_threshold_shift_attenuation.csv")
print("03_HDSI_to_mediator_curves.csv")
print("04_mediator_to_loss_curves.csv")
print("05_mediator_loss_thresholds.csv")
print("06_regional_threshold_shifts.csv")

In [ ]:
# ============================================================
# Stratified standardized stress-threshold analysis
#
# Goal:
#   In comparable environmental strata, test whether surface-water
#   proximity increases the stress threshold required for vegetation
#   to enter a negative-response state.
#
# Main idea:
#   Do NOT use representative mean covariates.
#   Do NOT mix all regions blindly.
#   Do NOT put HDSI and its components in the same loss model.
#
# Instead:
#   1. Define comparable strata:
#      year × macro_region × LC_Type1 × elevation_bin × slope_bin × spring_bin
#   2. Within each stratum and stress bin, compute loss risk by water group.
#   3. Standardize / aggregate stratum-level risks across common strata.
#   4. Extract stress threshold at P(loss)=0.40.
#   5. Repeat for HDSI and single stress channels:
#      SM_deficit, LST_Day, VPD, Precip_deficit, Tair.
#
# Outputs:
#   tables/
#     00_data_overview.csv
#     01_strata_coverage.csv
#     02_standardized_risk_curves.csv
#     03_stress_thresholds.csv
#     04_threshold_shifts.csv
#     05_high_exposure_threshold_shifts.csv
#     06_regional_threshold_shifts.csv
#     07_high_HDSI_mediator_state_difference.csv
#
#   figs/
#     Fig1_HDSI_standardized_risk_curves.png
#     Fig2_threshold_shift_by_stress_channel.png
#     Fig3_high_exposure_threshold_shift.png
#     Fig4_regional_threshold_shift.png
#     Fig5_high_HDSI_mediator_state_difference.png
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

OUT_DIR = os.path.join(AQUA_DIR, "_stratified_threshold_analysis")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222",
    "lightgray": "#D9D9D9",
    "teal": "#009E73",
}

HYDRO_COLORS = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

STRESS_COLORS = {
    "HDSI": COL["dark"],
    "SM_deficit": COL["blue"],
    "Precip_deficit": COL["teal"],
    "LST_Day": COL["red"],
    "VPD": COL["orange"],
    "Tair": COL["purple"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Single stress channels
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

# Vegetation loss events
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce")

df["Joint_neg"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Macro-region fallback
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid sample
valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

required = [
    "grid_id", "lon", "lat", "year", "macro_region",
    "HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "EVI_neg_z05", "LAI_neg_z05", "Joint_neg",
    "spring_EVI_z", "spring_LAI_z",
    "elevation", "slope", "LC_Type1"
]

d = df[valid_mask].copy()
d = d.dropna(subset=required).copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["Joint_neg"] = d["Joint_neg"].astype(int)

# Unified hydrological groups
d["hydro_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "hydro_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "hydro_context"
] = "near_seasonal_lt1km"

hydro_order = ["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"]
d["hydro_context"] = pd.Categorical(d["hydro_context"], categories=hydro_order, ordered=True)

# LC category
d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round().astype(int).astype(str)
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce").astype(int).astype(str)

# Overview
overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].astype(int).unique())))},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "Joint_neg_rate", "value": d["Joint_neg"].mean()},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])
save_table(overview, "00_data_overview.csv")
display(overview)


# ============================================================
# 3. Define high-exposure domain
#    This does NOT use vegetation loss outcome.
# ============================================================

# Grid-level HDSI exposure summary
grid_exp = (
    d.groupby("grid_id", as_index=False)
    .agg(
        HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
        HDSI_max=("HDSI", "max"),
        HDSI_mean=("HDSI", "mean")
    )
)

q_exp = grid_exp["HDSI_q75"].quantile(0.75)
grid_exp["high_exposure_grid"] = grid_exp["HDSI_q75"] >= q_exp

d = d.merge(grid_exp[["grid_id", "HDSI_q75", "HDSI_max", "high_exposure_grid"]], on="grid_id", how="left")

print("High exposure grid cutoff HDSI_q75:", q_exp)
print("High exposure rows:", d["high_exposure_grid"].sum(), "of", len(d))


# ============================================================
# 4. Build comparable strata
# ============================================================

def safe_qcut(series, q=4, labels=None):
    try:
        return pd.qcut(series, q=q, labels=labels, duplicates="drop")
    except Exception:
        return pd.cut(series, bins=q, labels=labels)

# Bins should not be too fine; otherwise common strata disappear.
d["elev_bin"] = safe_qcut(d["elevation"], q=4, labels=["E1", "E2", "E3", "E4"]).astype(str)
d["slope_bin"] = safe_qcut(d["slope"], q=4, labels=["S1", "S2", "S3", "S4"]).astype(str)

# Use response-specific spring bin later.
# For joint, use mean of spring EVI and LAI as initial vegetation state.
d["spring_joint_z"] = 0.5 * d["spring_EVI_z"] + 0.5 * d["spring_LAI_z"]

d["spring_EVI_bin"] = safe_qcut(d["spring_EVI_z"], q=4, labels=["V1", "V2", "V3", "V4"]).astype(str)
d["spring_LAI_bin"] = safe_qcut(d["spring_LAI_z"], q=4, labels=["V1", "V2", "V3", "V4"]).astype(str)
d["spring_Joint_bin"] = safe_qcut(d["spring_joint_z"], q=4, labels=["V1", "V2", "V3", "V4"]).astype(str)

# Base strata components
base_strata_cols = [
    "year_cat",
    "macro_region",
    "LC_Type1_cat",
    "elev_bin",
    "slope_bin",
]

# Response-specific strata adds spring state bin.
response_config = {
    "EVI": {
        "outcome": "EVI_neg_z05",
        "spring_bin": "spring_EVI_bin"
    },
    "LAI": {
        "outcome": "LAI_neg_z05",
        "spring_bin": "spring_LAI_bin"
    },
    "Joint": {
        "outcome": "Joint_neg",
        "spring_bin": "spring_Joint_bin"
    },
}

stress_config = {
    "HDSI": "HDSI",
    "SM_deficit": "SM_deficit",
    "Precip_deficit": "Precip_deficit",
    "LST_Day": "LST_Day",
    "VPD": "VPD",
    "Tair": "Tair",
}


# ============================================================
# 5. Helpers: standardized risk curves and threshold extraction
# ============================================================

def make_stress_bins(data, stress_col, n_bins=12):
    """
    Use global quantile bins within the analysis domain.
    """
    x = data[stress_col].copy()
    try:
        bins = pd.qcut(x, q=n_bins, duplicates="drop")
    except Exception:
        bins = pd.cut(x, bins=n_bins)
    return bins

def weighted_mean_and_ci(values, weights=None, n_boot=500, seed=42):
    values = np.asarray(values, dtype=float)
    if weights is None:
        weights = np.ones_like(values, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    values = values[ok]
    weights = weights[ok]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    mean = np.average(values, weights=weights)

    if len(values) == 1:
        return mean, np.nan, np.nan

    rng = np.random.default_rng(seed)
    idx = np.arange(len(values))
    boots = []

    # Bootstrap strata, not individual pixels.
    prob = weights / weights.sum()
    for _ in range(n_boot):
        sidx = rng.choice(idx, size=len(idx), replace=True, p=prob)
        boots.append(np.average(values[sidx], weights=weights[sidx]))

    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi

def threshold_from_curve(x, y, prob_threshold=0.40):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 3:
        return np.nan, "too_few_points"

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if np.nanmax(y) < prob_threshold:
        return np.nan, "not_reached"

    if y[0] >= prob_threshold:
        return x[0], "already_above"

    idx = np.where(y >= prob_threshold)[0]
    if len(idx) == 0:
        return np.nan, "not_reached"

    i = idx[0]
    if i == 0:
        return x[0], "already_above"

    x0, x1 = x[i - 1], x[i]
    y0, y1 = y[i - 1], y[i]

    if y1 == y0:
        return x1, "crossed_flat"

    thr = x0 + (prob_threshold - y0) * (x1 - x0) / (y1 - y0)
    return thr, "crossed"

def build_strata_id(data, strata_cols):
    return data[strata_cols].astype(str).agg("|".join, axis=1)

def standardized_risk_curve(
    data,
    response_name,
    stress_name,
    stress_col,
    outcome_col,
    spring_bin_col,
    hydro_groups=hydro_order,
    n_stress_bins=12,
    min_cell_n=8,
    min_common_strata_per_bin=5,
    require_all_hydro_groups=True,
    n_boot=400,
    seed=42
):
    """
    Stratified standardized risk curve.

    Step:
      - Define strata = year × region × LC × elevation_bin × slope_bin × spring_bin.
      - Define stress bins globally within data.
      - For each stress_bin × stratum × hydro_group, compute event risk.
      - Keep only strata that contain required hydro groups within same stress bin.
      - Aggregate hydro-specific stratum risks using same stratum weights.
    """
    dd = data.dropna(subset=[stress_col, outcome_col, spring_bin_col, "hydro_context"]).copy()

    strata_cols = base_strata_cols + [spring_bin_col]
    dd["strata_id"] = build_strata_id(dd, strata_cols)
    dd["stress_bin"] = make_stress_bins(dd, stress_col, n_bins=n_stress_bins)

    # Cell-level risk: stress_bin × strata × hydro
    cell = (
        dd.groupby(["stress_bin", "strata_id", "hydro_context"], observed=False)
        .agg(
            risk=(outcome_col, "mean"),
            n=(outcome_col, "size"),
            stress_mid=(stress_col, "median"),
            stress_mean=(stress_col, "mean"),
            grid_n=("grid_id", "nunique")
        )
        .reset_index()
    )

    cell = cell[cell["n"] >= min_cell_n].copy()
    cell["hydro_context"] = cell["hydro_context"].astype(str)

    # Determine common strata within each stress bin.
    valid_cells = []
    coverage_rows = []

    for stress_bin, gb in cell.groupby("stress_bin", observed=False):
        # Which hydro groups exist for each stratum?
        counts = gb.pivot_table(
            index="strata_id",
            columns="hydro_context",
            values="n",
            aggfunc="sum",
            fill_value=0
        )

        required_groups = list(hydro_groups)

        if require_all_hydro_groups:
            common = counts.index[(counts[required_groups] > 0).all(axis=1)] if all(g in counts.columns for g in required_groups) else []
        else:
            common = counts.index[(counts[required_groups] > 0).sum(axis=1) >= 2] if any(g in counts.columns for g in required_groups) else []

        common = list(common)

        coverage_rows.append({
            "response": response_name,
            "stress": stress_name,
            "stress_bin": str(stress_bin),
            "n_common_strata": len(common),
            "n_all_strata": counts.shape[0],
            "stress_mid": gb["stress_mid"].median(),
        })

        if len(common) < min_common_strata_per_bin:
            continue

        valid_cells.append(gb[gb["strata_id"].isin(common)].copy())

    if len(valid_cells) == 0:
        return pd.DataFrame(), pd.DataFrame(coverage_rows)

    cell_common = pd.concat(valid_cells, ignore_index=True)

    # Stratum weights: equalize across hydro groups.
    # Weight stratum by total observations across hydro groups in that stress bin.
    stratum_weight = (
        cell_common.groupby(["stress_bin", "strata_id"], observed=False)
        .agg(strata_n=("n", "sum"))
        .reset_index()
    )

    cell_common = cell_common.merge(stratum_weight, on=["stress_bin", "strata_id"], how="left")

    rows = []

    for (stress_bin, hydro), gb in cell_common.groupby(["stress_bin", "hydro_context"], observed=False):
        if hydro not in hydro_groups:
            continue

        m, lo, hi = weighted_mean_and_ci(
            gb["risk"].values,
            weights=gb["strata_n"].values,
            n_boot=n_boot,
            seed=seed
        )

        rows.append({
            "response": response_name,
            "stress": stress_name,
            "stress_col": stress_col,
            "outcome_col": outcome_col,
            "hydro_context": hydro,
            "stress_bin": str(stress_bin),
            "stress_mid": gb["stress_mid"].median(),
            "stress_mean": gb["stress_mean"].mean(),
            "risk": m,
            "ci_low": lo,
            "ci_high": hi,
            "n_cells": len(gb),
            "n_strata": gb["strata_id"].nunique(),
            "n_obs": gb["n"].sum(),
            "n_grids": gb["grid_n"].sum(),
            "spring_bin_col": spring_bin_col,
        })

    curve = pd.DataFrame(rows).sort_values(["hydro_context", "stress_mid"])
    coverage = pd.DataFrame(coverage_rows)

    return curve, coverage

def extract_thresholds(curve, prob_thresholds=[0.40, 0.50]):
    rows = []
    if len(curve) == 0:
        return pd.DataFrame()

    for (response, stress, hydro), g in curve.groupby(["response", "stress", "hydro_context"], observed=False):
        g = g.sort_values("stress_mid")
        for p in prob_thresholds:
            thr, status = threshold_from_curve(g["stress_mid"].values, g["risk"].values, p)
            rows.append({
                "response": response,
                "stress": stress,
                "hydro_context": hydro,
                "prob_threshold": p,
                "threshold": thr,
                "status": status,
                "risk_min": g["risk"].min(),
                "risk_max": g["risk"].max(),
                "n_curve_points": len(g),
                "n_obs_total": g["n_obs"].sum(),
                "n_strata_mean": g["n_strata"].mean(),
            })
    return pd.DataFrame(rows)

def compute_threshold_shifts(thr, ref="near_permanent_lt1km", comps=("near_seasonal_lt1km", "far_or_weak_water")):
    rows = []
    if len(thr) == 0:
        return pd.DataFrame()

    for (response, stress, p), g in thr.groupby(["response", "stress", "prob_threshold"], observed=False):
        refrow = g[g["hydro_context"] == ref]
        if len(refrow) == 0:
            continue

        ref_thr = refrow["threshold"].iloc[0]
        ref_status = refrow["status"].iloc[0]

        for comp in comps:
            comprow = g[g["hydro_context"] == comp]
            if len(comprow) == 0:
                continue

            comp_thr = comprow["threshold"].iloc[0]
            comp_status = comprow["status"].iloc[0]

            shift = ref_thr - comp_thr if np.isfinite(ref_thr) and np.isfinite(comp_thr) else np.nan

            valid_crossed = (
                ref_status in ["crossed", "crossed_flat"] and
                comp_status in ["crossed", "crossed_flat"] and
                np.isfinite(shift)
            )

            rows.append({
                "response": response,
                "stress": stress,
                "prob_threshold": p,
                "reference": ref,
                "comparison": comp,
                "reference_threshold": ref_thr,
                "comparison_threshold": comp_thr,
                "threshold_shift": shift,
                "reference_status": ref_status,
                "comparison_status": comp_status,
                "valid_crossed_only": valid_crossed,
            })

    return pd.DataFrame(rows)


# ============================================================
# 6. Main analysis:
#    standardized risk curves across all comparable strata
# ============================================================

all_curves = []
all_coverage = []

for response_name, rcfg in response_config.items():
    outcome_col = rcfg["outcome"]
    spring_bin_col = rcfg["spring_bin"]

    for stress_name, stress_col in stress_config.items():
        print(f"\nRunning standardized curve: {response_name} | {stress_name}")

        curve, cov = standardized_risk_curve(
            data=d,
            response_name=response_name,
            stress_name=stress_name,
            stress_col=stress_col,
            outcome_col=outcome_col,
            spring_bin_col=spring_bin_col,
            n_stress_bins=12,
            min_cell_n=8,
            min_common_strata_per_bin=5,
            require_all_hydro_groups=True,
            n_boot=400,
            seed=42
        )

        all_curves.append(curve)
        all_coverage.append(cov)

curves = pd.concat(all_curves, ignore_index=True)
coverage = pd.concat(all_coverage, ignore_index=True)

thresholds = extract_thresholds(curves, prob_thresholds=[0.40, 0.50])
shifts = compute_threshold_shifts(thresholds)

save_table(coverage, "01_strata_coverage.csv")
save_table(curves, "02_standardized_risk_curves.csv")
save_table(thresholds, "03_stress_thresholds.csv")
save_table(shifts, "04_threshold_shifts.csv")

display(shifts.head(30))


# ============================================================
# 7. High-exposure-domain analysis
# ============================================================

d_high = d[d["high_exposure_grid"] == True].copy()
print("High exposure subset:", d_high.shape)

high_curves_list = []
high_cov_list = []

for response_name, rcfg in response_config.items():
    outcome_col = rcfg["outcome"]
    spring_bin_col = rcfg["spring_bin"]

    for stress_name, stress_col in stress_config.items():
        print(f"\nRunning HIGH-EXPOSURE curve: {response_name} | {stress_name}")

        curve, cov = standardized_risk_curve(
            data=d_high,
            response_name=response_name,
            stress_name=stress_name,
            stress_col=stress_col,
            outcome_col=outcome_col,
            spring_bin_col=spring_bin_col,
            n_stress_bins=10,
            min_cell_n=6,
            min_common_strata_per_bin=4,
            require_all_hydro_groups=True,
            n_boot=400,
            seed=42
        )

        curve["analysis_domain"] = "high_exposure"
        cov["analysis_domain"] = "high_exposure"

        high_curves_list.append(curve)
        high_cov_list.append(cov)

high_curves = pd.concat(high_curves_list, ignore_index=True)
high_coverage = pd.concat(high_cov_list, ignore_index=True)
high_thresholds = extract_thresholds(high_curves, prob_thresholds=[0.40, 0.50])
high_shifts = compute_threshold_shifts(high_thresholds)

save_table(high_coverage, "05_high_exposure_strata_coverage.csv")
save_table(high_curves, "05_high_exposure_standardized_risk_curves.csv")
save_table(high_thresholds, "05_high_exposure_thresholds.csv")
save_table(high_shifts, "05_high_exposure_threshold_shifts.csv")

display(high_shifts.head(30))


# ============================================================
# 8. Regional analysis
# ============================================================

regional_curve_list = []
regional_cov_list = []

min_region_rows = 3000
regions = sorted(d["macro_region"].dropna().unique())

for region in regions:
    dr = d[d["macro_region"] == region].copy()
    if len(dr) < min_region_rows:
        continue

    for response_name, rcfg in response_config.items():
        outcome_col = rcfg["outcome"]
        spring_bin_col = rcfg["spring_bin"]

        # Use HDSI only for regional threshold first.
        print(f"\nRunning regional curve: {region} | {response_name} | HDSI")

        curve, cov = standardized_risk_curve(
            data=dr,
            response_name=response_name,
            stress_name="HDSI",
            stress_col="HDSI",
            outcome_col=outcome_col,
            spring_bin_col=spring_bin_col,
            n_stress_bins=8,
            min_cell_n=5,
            min_common_strata_per_bin=3,
            require_all_hydro_groups=True,
            n_boot=300,
            seed=42
        )

        curve["macro_region"] = region
        cov["macro_region"] = region

        regional_curve_list.append(curve)
        regional_cov_list.append(cov)

regional_curves = pd.concat(regional_curve_list, ignore_index=True) if regional_curve_list else pd.DataFrame()
regional_coverage = pd.concat(regional_cov_list, ignore_index=True) if regional_cov_list else pd.DataFrame()

regional_thresholds = extract_thresholds(regional_curves, prob_thresholds=[0.40, 0.50]) if len(regional_curves) else pd.DataFrame()

# Add region back to thresholds
if len(regional_curves):
    reg_thr_rows = []
    for (region, response, stress, hydro), g in regional_curves.groupby(
        ["macro_region", "response", "stress", "hydro_context"],
        observed=False
    ):
        g = g.sort_values("stress_mid")
        for p in [0.40, 0.50]:
            thr, status = threshold_from_curve(g["stress_mid"].values, g["risk"].values, p)
            reg_thr_rows.append({
                "macro_region": region,
                "response": response,
                "stress": stress,
                "hydro_context": hydro,
                "prob_threshold": p,
                "threshold": thr,
                "status": status,
                "risk_min": g["risk"].min(),
                "risk_max": g["risk"].max(),
                "n_curve_points": len(g),
                "n_obs_total": g["n_obs"].sum(),
                "n_strata_mean": g["n_strata"].mean(),
            })

    regional_thresholds = pd.DataFrame(reg_thr_rows)

    reg_shift_rows = []
    for (region, response, stress, p), g in regional_thresholds.groupby(
        ["macro_region", "response", "stress", "prob_threshold"],
        observed=False
    ):
        ref = g[g["hydro_context"] == "near_permanent_lt1km"]
        if len(ref) == 0:
            continue

        ref_thr = ref["threshold"].iloc[0]
        ref_status = ref["status"].iloc[0]

        for comp in ["near_seasonal_lt1km", "far_or_weak_water"]:
            cc = g[g["hydro_context"] == comp]
            if len(cc) == 0:
                continue

            comp_thr = cc["threshold"].iloc[0]
            comp_status = cc["status"].iloc[0]

            shift = ref_thr - comp_thr if np.isfinite(ref_thr) and np.isfinite(comp_thr) else np.nan

            reg_shift_rows.append({
                "macro_region": region,
                "response": response,
                "stress": stress,
                "prob_threshold": p,
                "reference": "near_permanent_lt1km",
                "comparison": comp,
                "reference_threshold": ref_thr,
                "comparison_threshold": comp_thr,
                "threshold_shift": shift,
                "reference_status": ref_status,
                "comparison_status": comp_status,
                "valid_crossed_only": (
                    ref_status in ["crossed", "crossed_flat"] and
                    comp_status in ["crossed", "crossed_flat"] and
                    np.isfinite(shift)
                )
            })

    regional_shifts = pd.DataFrame(reg_shift_rows)
else:
    regional_shifts = pd.DataFrame()

save_table(regional_coverage, "06_regional_strata_coverage.csv")
save_table(regional_curves, "06_regional_standardized_HDSI_curves.csv")
save_table(regional_thresholds, "06_regional_HDSI_thresholds.csv")
save_table(regional_shifts, "06_regional_threshold_shifts.csv")

display(regional_shifts.head(30))


# ============================================================
# 9. High-HDSI mediator state difference
#    This checks whether near-water has lower stress states
#    under high HDSI, in comparable strata.
# ============================================================

def standardized_mediator_difference(
    data,
    mediator_col,
    spring_bin_col="spring_Joint_bin",
    high_hsdi_quantile=0.75,
    min_cell_n=8,
    n_boot=400,
    seed=42
):
    """
    Within high-HDSI years, compare mediator states across hydro groups
    using common strata.

    Output:
      standardized mediator mean by hydro group.
      difference near_permanent - far_or_weak.
    """
    q = data["HDSI"].quantile(high_hsdi_quantile)
    dd = data[data["HDSI"] >= q].copy()

    strata_cols = base_strata_cols + [spring_bin_col]
    dd["strata_id"] = build_strata_id(dd, strata_cols)

    cell = (
        dd.groupby(["strata_id", "hydro_context"], observed=False)
        .agg(
            mediator_mean=(mediator_col, "mean"),
            n=(mediator_col, "size"),
            grid_n=("grid_id", "nunique")
        )
        .reset_index()
    )

    cell = cell[cell["n"] >= min_cell_n].copy()
    cell["hydro_context"] = cell["hydro_context"].astype(str)

    counts = cell.pivot_table(
        index="strata_id",
        columns="hydro_context",
        values="n",
        aggfunc="sum",
        fill_value=0
    )

    if not all(g in counts.columns for g in hydro_order):
        return pd.DataFrame(), pd.DataFrame()

    common = counts.index[(counts[hydro_order] > 0).all(axis=1)].tolist()
    cell = cell[cell["strata_id"].isin(common)].copy()

    if len(cell) == 0:
        return pd.DataFrame(), pd.DataFrame()

    stratum_weight = (
        cell.groupby("strata_id", observed=False)
        .agg(strata_n=("n", "sum"))
        .reset_index()
    )
    cell = cell.merge(stratum_weight, on="strata_id", how="left")

    rows = []
    for hydro, gb in cell.groupby("hydro_context", observed=False):
        if hydro not in hydro_order:
            continue

        m, lo, hi = weighted_mean_and_ci(
            gb["mediator_mean"].values,
            weights=gb["strata_n"].values,
            n_boot=n_boot,
            seed=seed
        )

        rows.append({
            "mediator": mediator_col,
            "hydro_context": hydro,
            "mean": m,
            "ci_low": lo,
            "ci_high": hi,
            "n_cells": len(gb),
            "n_strata": gb["strata_id"].nunique(),
            "n_obs": gb["n"].sum(),
        })

    res = pd.DataFrame(rows)

    # Differences
    diff_rows = []
    ref = res[res["hydro_context"] == "near_permanent_lt1km"]
    if len(ref):
        ref_mean = ref["mean"].iloc[0]

        for comp in ["near_seasonal_lt1km", "far_or_weak_water"]:
            cc = res[res["hydro_context"] == comp]
            if len(cc):
                diff_rows.append({
                    "mediator": mediator_col,
                    "contrast": f"near_permanent - {comp}",
                    "difference": ref_mean - cc["mean"].iloc[0],
                    "near_mean": ref_mean,
                    "comparison_mean": cc["mean"].iloc[0],
                    "n_common_strata": cell["strata_id"].nunique(),
                    "HDSI_high_cutoff": q
                })

    diff = pd.DataFrame(diff_rows)
    return res, diff

mediator_state_list = []
mediator_diff_list = []

for med in ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]:
    print("High-HDSI mediator standardized comparison:", med)

    res, diff = standardized_mediator_difference(
        data=d,
        mediator_col=med,
        spring_bin_col="spring_Joint_bin",
        high_hsdi_quantile=0.75,
        min_cell_n=8,
        n_boot=400,
        seed=42
    )

    mediator_state_list.append(res)
    mediator_diff_list.append(diff)

mediator_state = pd.concat(mediator_state_list, ignore_index=True)
mediator_diff = pd.concat(mediator_diff_list, ignore_index=True)

save_table(mediator_state, "07_high_HDSI_mediator_state_by_hydro.csv")
save_table(mediator_diff, "07_high_HDSI_mediator_state_difference.csv")

display(mediator_diff)


# ============================================================
# 10. Figures
# ============================================================

# ------------------------------------------------------------
# Fig 1: HDSI standardized risk curves
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = curves[
        (curves["response"] == response_name) &
        (curves["stress"] == "HDSI")
    ].copy()

    for hydro in hydro_order:
        sg = sub[sub["hydro_context"] == hydro].sort_values("stress_mid")
        if len(sg) == 0:
            continue

        ax.plot(
            sg["stress_mid"],
            sg["risk"],
            color=HYDRO_COLORS[hydro],
            lw=2,
            label=hydro.replace("_", " ")
        )
        ax.fill_between(
            sg["stress_mid"],
            sg["ci_low"],
            sg["ci_high"],
            color=HYDRO_COLORS[hydro],
            alpha=0.14,
            lw=0
        )

    ax.axhline(0.40, color="#333333", lw=0.9, ls="--")
    ax.set_xlabel("HDSI")
    ax.set_title(f"{response_name} loss risk")
    ax.set_ylim(0, 0.75)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Stratified standardized loss risk")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig1_HDSI_standardized_risk_curves")
plt.show()


# ------------------------------------------------------------
# Fig 2: threshold shift by stress channel
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

target_p = 0.40

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = shifts[
        (shifts["response"] == response_name) &
        (shifts["comparison"] == "far_or_weak_water") &
        (shifts["prob_threshold"] == target_p)
    ].copy()

    stress_order = ["HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
    sub["stress"] = pd.Categorical(sub["stress"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress")

    x = np.arange(len(sub))
    colors = [STRESS_COLORS.get(s, COL["gray"]) for s in sub["stress"].astype(str)]

    ax.bar(x, sub["threshold_shift"], color=colors, width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress"], rotation=35, ha="right")
    ax.set_title(f"{response_name}: near permanent − far/weak")
    ax.set_ylabel("Stress threshold shift")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_threshold_shift_by_stress_channel")
plt.show()


# ------------------------------------------------------------
# Fig 3: high-exposure threshold shift
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = high_shifts[
        (high_shifts["response"] == response_name) &
        (high_shifts["comparison"] == "far_or_weak_water") &
        (high_shifts["prob_threshold"] == target_p)
    ].copy()

    stress_order = ["HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
    sub["stress"] = pd.Categorical(sub["stress"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress")

    x = np.arange(len(sub))
    colors = [STRESS_COLORS.get(s, COL["gray"]) for s in sub["stress"].astype(str)]

    ax.bar(x, sub["threshold_shift"], color=colors, width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress"], rotation=35, ha="right")
    ax.set_title(f"High-exposure domain: {response_name}")
    ax.set_ylabel("Stress threshold shift")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_high_exposure_threshold_shift")
plt.show()


# ------------------------------------------------------------
# Fig 4: regional HDSI threshold shift
# ------------------------------------------------------------

if len(regional_shifts) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)

    for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
        sub = regional_shifts[
            (regional_shifts["response"] == response_name) &
            (regional_shifts["stress"] == "HDSI") &
            (regional_shifts["comparison"] == "far_or_weak_water") &
            (regional_shifts["prob_threshold"] == target_p)
        ].copy()

        sub = sub.sort_values("threshold_shift")
        y = np.arange(len(sub))

        ax.scatter(sub["threshold_shift"], y, color=RESP_COLORS[response_name], s=45)
        ax.axvline(0, color="#333333", lw=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["macro_region"])
        ax.set_xlabel("HDSI threshold shift")
        ax.set_title(f"Regional threshold shift ({response_name})")
        panel_label(ax, label)
        clean_ax(ax)

    plt.tight_layout()
    savefig(fig, "Fig4_regional_threshold_shift")
    plt.show()


# ------------------------------------------------------------
# Fig 5: high-HDSI mediator state difference
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.8))

plot = mediator_diff[
    mediator_diff["contrast"] == "near_permanent - far_or_weak_water"
].copy()

stress_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
plot["mediator"] = pd.Categorical(plot["mediator"], categories=stress_order, ordered=True)
plot = plot.sort_values("mediator")

x = np.arange(len(plot))
colors = [STRESS_COLORS.get(s, COL["gray"]) for s in plot["mediator"].astype(str)]

ax.bar(x, plot["difference"], color=colors, width=0.7)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["mediator"], rotation=35, ha="right")
ax.set_ylabel("Standardized mediator difference\nnear permanent − far/weak")
ax.set_title("High-HDSI comparable strata: mediator state difference")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_high_HDSI_mediator_state_difference")
plt.show()


# ============================================================
# 11. Diagnostic summary
# ============================================================

def summarize_valid_shifts(tab, name):
    if len(tab) == 0:
        return pd.DataFrame()

    ss = tab[
        (tab["prob_threshold"] == 0.40) &
        (tab["comparison"] == "far_or_weak_water")
    ].copy()

    out = (
        ss.groupby(["response"], as_index=False)
        .agg(
            n=("threshold_shift", "size"),
            n_valid=("valid_crossed_only", "sum"),
            mean_shift=("threshold_shift", "mean"),
            median_shift=("threshold_shift", "median"),
            positive_frac=("threshold_shift", lambda x: np.mean(np.asarray(x) > 0))
        )
    )
    out["analysis"] = name
    return out

summary = pd.concat([
    summarize_valid_shifts(shifts, "all_comparable_strata"),
    summarize_valid_shifts(high_shifts, "high_exposure_comparable_strata"),
], ignore_index=True)

save_table(summary, "08_threshold_shift_summary.csv")
display(summary)


print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("02_standardized_risk_curves.csv")
print("04_threshold_shifts.csv")
print("05_high_exposure_threshold_shifts.csv")
print("06_regional_threshold_shifts.csv")
print("07_high_HDSI_mediator_state_difference.csv")
print("08_threshold_shift_summary.csv")

In [ ]:
# ============================================================
# Overlap-weighted vegetation-loss threshold analysis
#
# Purpose:
#   Replace over-strict exact-strata analysis with overlap-weighted
#   risk curves. This keeps enough sample size while controlling major
#   background differences.
#
# Core logic:
#   1. Define hydrological groups:
#        near_permanent_lt1km
#        near_seasonal_lt1km
#        far_or_weak_water
#   2. Estimate propensity scores for hydrological group membership
#      using background covariates only:
#        year, macro_region, LC_Type1, elevation, slope,
#        spring_EVI_z, spring_LAI_z, lon, lat
#   3. Compute overlap weights:
#        weight_i = min(ps_i across groups) / ps_observed_group_i
#      This emphasizes comparable samples.
#   4. Build weighted stress-risk curves:
#        Pr(loss = 1) as a function of HDSI / SM_deficit / LST / VPD / Tair
#   5. Extract stress threshold at P(loss)=0.30, 0.40, 0.50.
#   6. Define threshold shift:
#        threshold_near_permanent - threshold_far_or_weak
#
# Outputs:
#   tables/
#     00_data_overview.csv
#     01_propensity_balance_numeric.csv
#     02_weight_distribution.csv
#     03_weighted_risk_curves.csv
#     04_weighted_thresholds.csv
#     05_weighted_threshold_shifts.csv
#     06_high_exposure_weighted_threshold_shifts.csv
#     07_weighted_high_HDSI_mediator_state.csv
#
#   figs/
#     Fig1_balance_diagnostics.png
#     Fig2_weighted_HDSI_risk_curves.png
#     Fig3_threshold_shift_by_stress_channel.png
#     Fig4_high_exposure_threshold_shift.png
#     Fig5_high_HDSI_mediator_state_difference.png
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

OUT_DIR = os.path.join(AQUA_DIR, "_overlap_weighted_threshold_analysis")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222",
    "lightgray": "#D9D9D9",
    "teal": "#009E73",
}

HYDRO_COLORS = {
    "near_permanent_lt1km": COL["blue"],
    "near_seasonal_lt1km": COL["orange"],
    "far_or_weak_water": COL["gray"],
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

STRESS_COLORS = {
    "HDSI": COL["dark"],
    "SM_deficit": COL["blue"],
    "Precip_deficit": COL["teal"],
    "LST_Day": COL["red"],
    "VPD": COL["orange"],
    "Tair": COL["purple"],
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km", "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Single stress channels.
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

# Vegetation loss events.
if "EVI_neg_z05" not in df.columns:
    df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
else:
    df["EVI_neg_z05"] = pd.to_numeric(df["EVI_neg_z05"], errors="coerce")

if "LAI_neg_z05" not in df.columns:
    df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
else:
    df["LAI_neg_z05"] = pd.to_numeric(df["LAI_neg_z05"], errors="coerce")

df["Joint_neg"] = (
    (df["EVI_neg_z05"].fillna(0).astype(int) == 1) |
    (df["LAI_neg_z05"].fillna(0).astype(int) == 1)
).astype(int)

# Macro region fallback.
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid sample.
valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

required = [
    "grid_id", "lon", "lat", "year", "macro_region",
    "HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "EVI_neg_z05", "LAI_neg_z05", "Joint_neg",
    "spring_EVI_z", "spring_LAI_z",
    "elevation", "slope", "LC_Type1"
]

d = df[valid_mask].copy()
d = d.dropna(subset=required).copy()

d["EVI_neg_z05"] = d["EVI_neg_z05"].astype(int)
d["LAI_neg_z05"] = d["LAI_neg_z05"].astype(int)
d["Joint_neg"] = d["Joint_neg"].astype(int)

# Hydrological groups.
d["hydro_context"] = "far_or_weak_water"
d.loc[d["dist_perm_water_km"] < 1.0, "hydro_context"] = "near_permanent_lt1km"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "hydro_context"
] = "near_seasonal_lt1km"

hydro_order = ["near_permanent_lt1km", "near_seasonal_lt1km", "far_or_weak_water"]
d["hydro_context"] = pd.Categorical(d["hydro_context"], categories=hydro_order, ordered=True)

d["LC_Type1_cat"] = pd.to_numeric(d["LC_Type1"], errors="coerce").round().astype(int).astype(str)
d["year_cat"] = pd.to_numeric(d["year"], errors="coerce").astype(int).astype(str)

# Grid-level exposure domain.
grid_exp = (
    d.groupby("grid_id", as_index=False)
    .agg(
        HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
        HDSI_mean=("HDSI", "mean"),
        HDSI_max=("HDSI", "max")
    )
)

q_exp = grid_exp["HDSI_q75"].quantile(0.75)
grid_exp["high_exposure_grid"] = grid_exp["HDSI_q75"] >= q_exp

d = d.merge(grid_exp, on="grid_id", how="left")

overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].astype(int).unique())))},
    {"metric": "EVI_neg_rate", "value": d["EVI_neg_z05"].mean()},
    {"metric": "LAI_neg_rate", "value": d["LAI_neg_z05"].mean()},
    {"metric": "Joint_neg_rate", "value": d["Joint_neg"].mean()},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
    {"metric": "high_exposure_HDSI_q75_cutoff", "value": q_exp},
    {"metric": "high_exposure_rows", "value": int(d["high_exposure_grid"].sum())},
])
save_table(overview, "00_data_overview.csv")
display(overview)


# ============================================================
# 3. Propensity score and overlap weights
# ============================================================

def fit_overlap_weights(
    data,
    group_col="hydro_context",
    group_order=hydro_order,
    weight_col="ow",
    min_ps=0.01,
    max_weight_quantile=0.99,
    verbose=True
):
    """
    Multi-arm overlap-like weights:
        w_i = min_j e_j(x_i) / e_{T_i}(x_i)

    This emphasizes covariate-overlap regions and prevents exact-strata
    fragmentation.
    """

    dd = data.copy()

    # Background covariates only.
    # Do NOT include current-year HDSI or stress variables here.
    num_covars = [
        "elevation", "slope",
        "spring_EVI_z", "spring_LAI_z",
        "lon", "lat"
    ]

    # Grid-level HDSI exposure can be included as background exposure.
    # If you worry it over-controls climate exposure, set INCLUDE_GRID_HDSI_Q75=False.
    INCLUDE_GRID_HDSI_Q75 = True
    if INCLUDE_GRID_HDSI_Q75 and "HDSI_q75" in dd.columns:
        num_covars.append("HDSI_q75")

    cat_covars = [
        "year_cat",
        "macro_region",
        "LC_Type1_cat"
    ]

    # Drop missing.
    model_cols = num_covars + cat_covars + [group_col]
    dd = dd.dropna(subset=model_cols).copy()

    dd[group_col] = pd.Categorical(dd[group_col].astype(str), categories=group_order, ordered=False)
    y = dd[group_col].astype(str).values

    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    pre = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), num_covars),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", ohe)
            ]), cat_covars),
        ],
        remainder="drop"
    )

    clf = LogisticRegression(
        multi_class="multinomial",
        solver="lbfgs",
        max_iter=500,
        C=1.0,
        n_jobs=-1
    )

    pipe = Pipeline([
        ("pre", pre),
        ("clf", clf)
    ])

    pipe.fit(dd[num_covars + cat_covars], y)

    ps = pipe.predict_proba(dd[num_covars + cat_covars])
    classes = list(pipe.named_steps["clf"].classes_)

    ps = np.clip(ps, min_ps, 1.0)
    ps = ps / ps.sum(axis=1, keepdims=True)

    ps_df = pd.DataFrame(ps, columns=[f"ps_{c}" for c in classes], index=dd.index)

    # Observed group propensity.
    obs_ps = np.zeros(len(dd))
    min_ps_each = ps.min(axis=1)

    for c in classes:
        mask = y == c
        obs_ps[mask] = ps[mask, classes.index(c)]

    ow = min_ps_each / obs_ps

    # Trim extreme weights.
    cap = np.nanquantile(ow, max_weight_quantile)
    ow_trim = np.minimum(ow, cap)

    dd = pd.concat([dd, ps_df], axis=1)
    dd[weight_col] = ow_trim
    dd[weight_col + "_raw"] = ow
    dd["ps_observed"] = obs_ps
    dd["ps_min"] = min_ps_each

    if verbose:
        print("\nPropensity classes:", classes)
        print("Rows for weighting:", len(dd))
        print("Weight summary:")
        display(dd[[weight_col, weight_col + "_raw", "ps_observed", "ps_min"]].describe())
        print("Hydro group counts:")
        display(dd[group_col].value_counts())

    return dd, pipe, {
        "num_covars": num_covars,
        "cat_covars": cat_covars,
        "classes": classes,
        "weight_cap": cap
    }

d_w, ps_model, ps_info = fit_overlap_weights(d, weight_col="ow", verbose=True)

weight_summary = (
    d_w.groupby("hydro_context", observed=False)
    .agg(
        rows=("grid_id", "count"),
        grids=("grid_id", "nunique"),
        weight_mean=("ow", "mean"),
        weight_median=("ow", "median"),
        weight_p95=("ow", lambda x: np.quantile(x, 0.95)),
        ps_observed_mean=("ps_observed", "mean"),
        ps_min_mean=("ps_min", "mean"),
    )
    .reset_index()
)
save_table(weight_summary, "02_weight_distribution.csv")
display(weight_summary)


# ============================================================
# 4. Balance diagnostics
# ============================================================

def weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if ok.sum() == 0:
        return np.nan
    return np.average(x[ok], weights=w[ok])

def weighted_var(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)
    ok = np.isfinite(x) & np.isfinite(w) & (w > 0)
    if ok.sum() <= 1:
        return np.nan
    m = np.average(x[ok], weights=w[ok])
    return np.average((x[ok] - m) ** 2, weights=w[ok])

def pairwise_smd(data, var, group_col="hydro_context", weight_col=None, groups=hydro_order):
    rows = []

    for i in range(len(groups)):
        for j in range(i + 1, len(groups)):
            g1, g2 = groups[i], groups[j]
            a = data[data[group_col].astype(str) == g1]
            b = data[data[group_col].astype(str) == g2]

            if len(a) == 0 or len(b) == 0:
                continue

            if weight_col is None:
                m1, m2 = a[var].mean(), b[var].mean()
                v1, v2 = a[var].var(), b[var].var()
            else:
                m1 = weighted_mean(a[var], a[weight_col])
                m2 = weighted_mean(b[var], b[weight_col])
                v1 = weighted_var(a[var], a[weight_col])
                v2 = weighted_var(b[var], b[weight_col])

            pooled = np.sqrt((v1 + v2) / 2)
            smd = (m1 - m2) / pooled if pooled and np.isfinite(pooled) else np.nan

            rows.append({
                "variable": var,
                "group1": g1,
                "group2": g2,
                "smd": smd,
                "abs_smd": abs(smd) if np.isfinite(smd) else np.nan,
                "weighted": weight_col is not None
            })

    return rows

balance_vars = [
    "elevation", "slope",
    "spring_EVI_z", "spring_LAI_z",
    "lon", "lat", "HDSI_q75"
]

bal_rows = []
for v in balance_vars:
    bal_rows.extend(pairwise_smd(d_w, v, weight_col=None))
    bal_rows.extend(pairwise_smd(d_w, v, weight_col="ow"))

balance = pd.DataFrame(bal_rows)

balance_summary = (
    balance.groupby(["variable", "weighted"], as_index=False)
    .agg(
        max_abs_smd=("abs_smd", "max"),
        mean_abs_smd=("abs_smd", "mean")
    )
)

save_table(balance, "01_propensity_balance_numeric_pairwise.csv")
save_table(balance_summary, "01_propensity_balance_numeric_summary.csv")
display(balance_summary)


# ============================================================
# 5. Weighted risk curves and threshold extraction
# ============================================================

response_config = {
    "EVI": "EVI_neg_z05",
    "LAI": "LAI_neg_z05",
    "Joint": "Joint_neg",
}

stress_config = {
    "HDSI": "HDSI",
    "SM_deficit": "SM_deficit",
    "Precip_deficit": "Precip_deficit",
    "LST_Day": "LST_Day",
    "VPD": "VPD",
    "Tair": "Tair",
}

def effective_n(w):
    w = np.asarray(w, dtype=float)
    w = w[np.isfinite(w) & (w > 0)]
    if len(w) == 0:
        return 0
    return (w.sum() ** 2) / np.sum(w ** 2)

def weighted_binomial_ci(p, w):
    neff = effective_n(w)
    if neff <= 1 or not np.isfinite(p):
        return np.nan, np.nan
    se = np.sqrt(max(p * (1 - p), 0) / neff)
    lo = max(0, p - 1.96 * se)
    hi = min(1, p + 1.96 * se)
    return lo, hi

def make_weighted_risk_curves(
    data,
    domain_name,
    weight_col="ow",
    n_bins=14,
    min_bin_n=80,
    stress_config=stress_config,
    response_config=response_config,
):
    rows = []

    for stress_name, stress_col in stress_config.items():
        dd = data.dropna(subset=[stress_col, weight_col]).copy()

        try:
            dd["stress_bin"] = pd.qcut(dd[stress_col], q=n_bins, duplicates="drop")
        except Exception:
            dd["stress_bin"] = pd.cut(dd[stress_col], bins=n_bins)

        for response_name, outcome_col in response_config.items():
            tmp = dd.dropna(subset=[outcome_col]).copy()

            for (hydro, stress_bin), gb in tmp.groupby(["hydro_context", "stress_bin"], observed=False):
                hydro = str(hydro)

                if hydro not in hydro_order:
                    continue

                if len(gb) < min_bin_n:
                    continue

                w = gb[weight_col].values.astype(float)
                y = gb[outcome_col].values.astype(float)

                ok = np.isfinite(w) & np.isfinite(y) & (w > 0)
                if ok.sum() < min_bin_n:
                    continue

                p = np.average(y[ok], weights=w[ok])
                lo, hi = weighted_binomial_ci(p, w[ok])

                rows.append({
                    "domain": domain_name,
                    "response": response_name,
                    "outcome_col": outcome_col,
                    "stress": stress_name,
                    "stress_col": stress_col,
                    "hydro_context": hydro,
                    "stress_bin": str(stress_bin),
                    "stress_mid": np.nanmedian(gb.loc[ok, stress_col]),
                    "stress_mean": np.average(gb.loc[ok, stress_col], weights=w[ok]),
                    "risk": p,
                    "ci_low": lo,
                    "ci_high": hi,
                    "n": int(ok.sum()),
                    "n_eff": effective_n(w[ok]),
                    "w_sum": np.sum(w[ok]),
                    "n_grids": gb.loc[ok, "grid_id"].nunique(),
                })

    out = pd.DataFrame(rows)
    return out.sort_values(["domain", "response", "stress", "hydro_context", "stress_mid"])

def threshold_from_curve(x, y, prob_threshold=0.40):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    ok = np.isfinite(x) & np.isfinite(y)
    x = x[ok]
    y = y[ok]

    if len(x) < 3:
        return np.nan, "too_few_points"

    order = np.argsort(x)
    x = x[order]
    y = y[order]

    if np.nanmax(y) < prob_threshold:
        return np.nan, "not_reached"

    if y[0] >= prob_threshold:
        return x[0], "already_above"

    idx = np.where(y >= prob_threshold)[0]
    if len(idx) == 0:
        return np.nan, "not_reached"

    i = idx[0]
    if i == 0:
        return x[0], "already_above"

    x0, x1 = x[i - 1], x[i]
    y0, y1 = y[i - 1], y[i]

    if y1 == y0:
        return x1, "crossed_flat"

    thr = x0 + (prob_threshold - y0) * (x1 - x0) / (y1 - y0)
    return thr, "crossed"

def extract_thresholds(curves, prob_thresholds=(0.30, 0.40, 0.50)):
    rows = []

    for (domain, response, stress, hydro), g in curves.groupby(
        ["domain", "response", "stress", "hydro_context"],
        observed=False
    ):
        g = g.sort_values("stress_mid")

        for p in prob_thresholds:
            thr, status = threshold_from_curve(g["stress_mid"].values, g["risk"].values, p)

            rows.append({
                "domain": domain,
                "response": response,
                "stress": stress,
                "hydro_context": hydro,
                "prob_threshold": p,
                "threshold": thr,
                "status": status,
                "risk_min": g["risk"].min(),
                "risk_max": g["risk"].max(),
                "n_curve_points": len(g),
                "n_eff_total": g["n_eff"].sum(),
                "n_total": g["n"].sum(),
            })

    return pd.DataFrame(rows)

def compute_threshold_shifts(
    thresholds,
    ref="near_permanent_lt1km",
    comps=("near_seasonal_lt1km", "far_or_weak_water")
):
    rows = []

    for (domain, response, stress, p), g in thresholds.groupby(
        ["domain", "response", "stress", "prob_threshold"],
        observed=False
    ):
        refrow = g[g["hydro_context"] == ref]
        if len(refrow) == 0:
            continue

        ref_thr = refrow["threshold"].iloc[0]
        ref_status = refrow["status"].iloc[0]

        for comp in comps:
            crow = g[g["hydro_context"] == comp]
            if len(crow) == 0:
                continue

            comp_thr = crow["threshold"].iloc[0]
            comp_status = crow["status"].iloc[0]

            shift = ref_thr - comp_thr if np.isfinite(ref_thr) and np.isfinite(comp_thr) else np.nan

            rows.append({
                "domain": domain,
                "response": response,
                "stress": stress,
                "prob_threshold": p,
                "reference": ref,
                "comparison": comp,
                "reference_threshold": ref_thr,
                "comparison_threshold": comp_thr,
                "threshold_shift": shift,
                "reference_status": ref_status,
                "comparison_status": comp_status,
                "valid_crossed_only": (
                    ref_status in ["crossed", "crossed_flat"] and
                    comp_status in ["crossed", "crossed_flat"] and
                    np.isfinite(shift)
                )
            })

    return pd.DataFrame(rows)


# ============================================================
# 6. Run weighted curves: all sample and high-exposure sample
# ============================================================

curves_all = make_weighted_risk_curves(
    d_w,
    domain_name="all_overlap_weighted",
    weight_col="ow",
    n_bins=14,
    min_bin_n=80
)

d_w_high = d_w[d_w["high_exposure_grid"] == True].copy()

curves_high = make_weighted_risk_curves(
    d_w_high,
    domain_name="high_exposure_overlap_weighted",
    weight_col="ow",
    n_bins=10,
    min_bin_n=50
)

curves = pd.concat([curves_all, curves_high], ignore_index=True)

thresholds = extract_thresholds(curves, prob_thresholds=(0.30, 0.40, 0.50))
shifts = compute_threshold_shifts(thresholds)

save_table(curves, "03_weighted_risk_curves.csv")
save_table(thresholds, "04_weighted_thresholds.csv")
save_table(shifts, "05_weighted_threshold_shifts.csv")

display(shifts.head(40))


# ============================================================
# 7. High-HDSI mediator state difference under weights
# ============================================================

def weighted_mediator_state(data, high_q=0.75, weight_col="ow"):
    cutoff = data["HDSI"].quantile(high_q)
    dd = data[data["HDSI"] >= cutoff].copy()

    rows = []
    diffs = []

    mediators = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]

    for med in mediators:
        for hydro in hydro_order:
            gb = dd[dd["hydro_context"].astype(str) == hydro].dropna(subset=[med, weight_col])

            if len(gb) == 0:
                continue

            w = gb[weight_col].values.astype(float)
            x = gb[med].values.astype(float)
            ok = np.isfinite(w) & np.isfinite(x) & (w > 0)

            if ok.sum() == 0:
                continue

            m = np.average(x[ok], weights=w[ok])
            neff = effective_n(w[ok])
            var = np.average((x[ok] - m) ** 2, weights=w[ok])
            se = np.sqrt(var / neff) if neff > 1 else np.nan

            rows.append({
                "mediator": med,
                "hydro_context": hydro,
                "mean": m,
                "ci_low": m - 1.96 * se if np.isfinite(se) else np.nan,
                "ci_high": m + 1.96 * se if np.isfinite(se) else np.nan,
                "n": int(ok.sum()),
                "n_eff": neff,
                "HDSI_cutoff": cutoff,
            })

    state = pd.DataFrame(rows)

    for med in mediators:
        sub = state[state["mediator"] == med]

        near = sub[sub["hydro_context"] == "near_permanent_lt1km"]
        if len(near) == 0:
            continue

        near_mean = near["mean"].iloc[0]

        for comp in ["near_seasonal_lt1km", "far_or_weak_water"]:
            cc = sub[sub["hydro_context"] == comp]
            if len(cc) == 0:
                continue

            diffs.append({
                "mediator": med,
                "contrast": f"near_permanent - {comp}",
                "difference": near_mean - cc["mean"].iloc[0],
                "near_mean": near_mean,
                "comparison_mean": cc["mean"].iloc[0],
                "HDSI_cutoff": cutoff,
            })

    return state, pd.DataFrame(diffs)

med_state, med_diff = weighted_mediator_state(d_w, high_q=0.75, weight_col="ow")

save_table(med_state, "07_weighted_high_HDSI_mediator_state.csv")
save_table(med_diff, "07_weighted_high_HDSI_mediator_state_difference.csv")

display(med_diff)


# ============================================================
# 8. Optional regional weighted HDSI threshold shifts
# ============================================================

regional_curves_list = []

for region in sorted(d_w["macro_region"].dropna().unique()):
    dr = d_w[d_w["macro_region"] == region].copy()

    if len(dr) < 3000:
        continue

    tmp_curves = make_weighted_risk_curves(
        dr,
        domain_name=f"region_{region}",
        weight_col="ow",
        n_bins=8,
        min_bin_n=40,
        stress_config={"HDSI": "HDSI"},
        response_config=response_config
    )

    tmp_curves["macro_region"] = region
    regional_curves_list.append(tmp_curves)

regional_curves = pd.concat(regional_curves_list, ignore_index=True) if regional_curves_list else pd.DataFrame()

if len(regional_curves):
    regional_thresholds = extract_thresholds(regional_curves, prob_thresholds=(0.40, 0.50))
    regional_shifts = compute_threshold_shifts(regional_thresholds)

    # Add macro region from domain name.
    regional_shifts["macro_region"] = regional_shifts["domain"].str.replace("region_", "", regex=False)
else:
    regional_thresholds = pd.DataFrame()
    regional_shifts = pd.DataFrame()

save_table(regional_curves, "06_regional_weighted_HDSI_curves.csv")
save_table(regional_thresholds, "06_regional_weighted_HDSI_thresholds.csv")
save_table(regional_shifts, "06_regional_weighted_HDSI_threshold_shifts.csv")

display(regional_shifts.head(40))


# ============================================================
# 9. Figures
# ============================================================

# ------------------------------------------------------------
# Fig 1: balance diagnostics
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.8))

plot_bal = balance_summary.copy()
plot_bal["weighted_label"] = plot_bal["weighted"].map({False: "Before weighting", True: "After weighting"})

pivot = plot_bal.pivot(index="variable", columns="weighted_label", values="max_abs_smd")
pivot = pivot.sort_values("Before weighting")

y = np.arange(len(pivot))

ax.scatter(pivot["Before weighting"], y, color=COL["gray"], label="Before weighting", s=45)
ax.scatter(pivot["After weighting"], y, color=COL["blue"], label="After weighting", s=45)
ax.axvline(0.1, color=COL["red"], ls="--", lw=1, label="0.1")
ax.set_yticks(y)
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Maximum pairwise absolute SMD")
ax.set_title("Covariate balance across hydrological groups")
ax.legend(frameon=False)
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig1_balance_diagnostics")
plt.show()


# ------------------------------------------------------------
# Fig 2: HDSI weighted risk curves
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = curves[
        (curves["domain"] == "all_overlap_weighted") &
        (curves["response"] == response_name) &
        (curves["stress"] == "HDSI")
    ].copy()

    for hydro in hydro_order:
        sg = sub[sub["hydro_context"] == hydro].sort_values("stress_mid")
        if len(sg) == 0:
            continue

        ax.plot(
            sg["stress_mid"],
            sg["risk"],
            color=HYDRO_COLORS[hydro],
            lw=2,
            label=hydro.replace("_", " ")
        )
        ax.fill_between(
            sg["stress_mid"],
            sg["ci_low"],
            sg["ci_high"],
            color=HYDRO_COLORS[hydro],
            alpha=0.14,
            lw=0
        )

    ax.axhline(0.40, color="#333333", lw=0.9, ls="--")
    ax.set_xlabel("HDSI")
    ax.set_title(f"{response_name} loss risk")
    ax.set_ylim(0, 0.75)
    panel_label(ax, label)
    clean_ax(ax)

axes[0].set_ylabel("Overlap-weighted loss risk")
axes[-1].legend(frameon=False, fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
savefig(fig, "Fig2_weighted_HDSI_risk_curves")
plt.show()


# ------------------------------------------------------------
# Fig 3: threshold shift by stress channel
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

target_p = 0.40

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = shifts[
        (shifts["domain"] == "all_overlap_weighted") &
        (shifts["response"] == response_name) &
        (shifts["comparison"] == "far_or_weak_water") &
        (shifts["prob_threshold"] == target_p)
    ].copy()

    stress_order = ["HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
    sub["stress"] = pd.Categorical(sub["stress"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress")

    x = np.arange(len(sub))
    colors = [STRESS_COLORS.get(s, COL["gray"]) for s in sub["stress"].astype(str)]

    ax.bar(x, sub["threshold_shift"], color=colors, width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress"], rotation=35, ha="right")
    ax.set_title(f"{response_name}: near permanent − far/weak")
    ax.set_ylabel("Stress threshold shift")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_threshold_shift_by_stress_channel")
plt.show()


# ------------------------------------------------------------
# Fig 4: high exposure threshold shift
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)

for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = shifts[
        (shifts["domain"] == "high_exposure_overlap_weighted") &
        (shifts["response"] == response_name) &
        (shifts["comparison"] == "far_or_weak_water") &
        (shifts["prob_threshold"] == target_p)
    ].copy()

    stress_order = ["HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
    sub["stress"] = pd.Categorical(sub["stress"], categories=stress_order, ordered=True)
    sub = sub.sort_values("stress")

    x = np.arange(len(sub))
    colors = [STRESS_COLORS.get(s, COL["gray"]) for s in sub["stress"].astype(str)]

    ax.bar(x, sub["threshold_shift"], color=colors, width=0.7)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(sub["stress"], rotation=35, ha="right")
    ax.set_title(f"High-exposure: {response_name}")
    ax.set_ylabel("Stress threshold shift")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig4_high_exposure_threshold_shift")
plt.show()


# ------------------------------------------------------------
# Fig 5: high-HDSI mediator state difference
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.8))

plot = med_diff[
    med_diff["contrast"] == "near_permanent - far_or_weak_water"
].copy()

stress_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
plot["mediator"] = pd.Categorical(plot["mediator"], categories=stress_order, ordered=True)
plot = plot.sort_values("mediator")

x = np.arange(len(plot))
colors = [STRESS_COLORS.get(s, COL["gray"]) for s in plot["mediator"].astype(str)]

ax.bar(x, plot["difference"], color=colors, width=0.7)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["mediator"], rotation=35, ha="right")
ax.set_ylabel("Weighted mediator difference\nnear permanent − far/weak")
ax.set_title("High-HDSI state difference after overlap weighting")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_high_HDSI_mediator_state_difference")
plt.show()


# ------------------------------------------------------------
# Fig 6: regional HDSI threshold shift
# ------------------------------------------------------------

if len(regional_shifts) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)

    for ax, response_name, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
        sub = regional_shifts[
            (regional_shifts["response"] == response_name) &
            (regional_shifts["stress"] == "HDSI") &
            (regional_shifts["comparison"] == "far_or_weak_water") &
            (regional_shifts["prob_threshold"] == target_p)
        ].copy()

        sub = sub.sort_values("threshold_shift")
        y = np.arange(len(sub))

        ax.scatter(sub["threshold_shift"], y, color=RESP_COLORS[response_name], s=45)
        ax.axvline(0, color="#333333", lw=0.8)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["macro_region"])
        ax.set_xlabel("HDSI threshold shift")
        ax.set_title(f"Regional threshold shift ({response_name})")
        panel_label(ax, label)
        clean_ax(ax)

    plt.tight_layout()
    savefig(fig, "Fig6_regional_HDSI_threshold_shift")
    plt.show()


# ============================================================
# 10. Diagnostic summaries
# ============================================================

summary = (
    shifts[
        (shifts["comparison"] == "far_or_weak_water") &
        (shifts["prob_threshold"] == 0.40)
    ]
    .groupby(["domain", "response"], as_index=False)
    .agg(
        n=("threshold_shift", "size"),
        n_valid=("valid_crossed_only", "sum"),
        mean_shift=("threshold_shift", "mean"),
        median_shift=("threshold_shift", "median"),
        positive_frac=("threshold_shift", lambda x: np.mean(np.asarray(x, dtype=float) > 0))
    )
)

save_table(summary, "08_weighted_threshold_shift_summary.csv")
display(summary)

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("01_propensity_balance_numeric_summary.csv")
print("02_weight_distribution.csv")
print("03_weighted_risk_curves.csv")
print("05_weighted_threshold_shifts.csv")
print("06_regional_weighted_HDSI_threshold_shifts.csv")
print("07_weighted_high_HDSI_mediator_state_difference.csv")
print("08_weighted_threshold_shift_summary.csv")

In [ ]:
# ============================================================
# Simple reproducible analysis:
# Surface-water permanence and vegetation loss risk
# under compound heat–dryness events
#
# Four core results:
#   1. HDSI identifies vegetation loss risk.
#   2. Under high HDSI, loss risk differs among aquatic contexts.
#   3. Aquatic context changes risk amplification from low to high HDSI.
#   4. Regional heterogeneity of aquatic buffering.
#
# Main input:
#   china_aquatic_buffering_panel_v2.parquet
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

OUT_DIR = os.path.join(AQUA_DIR, "_simple_core_results")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))

# ============================================================
# 1. Figure style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222",
    "lightgray": "#D9D9D9",
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

AQUA_COLORS = {
    "near_permanent": COL["blue"],
    "near_seasonal": COL["orange"],
    "other_weak": COL["gray"],
}

AQUA_LABELS = {
    "near_permanent": "Near permanent water",
    "near_seasonal": "Near seasonal water",
    "other_weak": "Other / weak aquatic context",
}

HDSI_LABELS = {
    "low": "Low HDSI",
    "mid": "Mid HDSI",
    "high": "High HDSI",
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp

# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI: compound heat–dryness stress
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Vegetation loss events
df["EVI_loss"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_loss"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
df["Joint_loss"] = ((df["EVI_loss"] == 1) | (df["LAI_loss"] == 1)).astype(int)

# Macro-region fallback
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid sample
valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

required = [
    "grid_id", "lon", "lat", "year", "macro_region",
    "HDSI",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_loss", "LAI_loss", "Joint_loss",
]

d = df[valid_mask].copy()
d = d.dropna(subset=required).copy()

# Aquatic context
# Use neutral language: "other/weak", not "far water"
d["aquatic_context"] = "other_weak"

d.loc[
    d["dist_perm_water_km"] < 1.0,
    "aquatic_context"
] = "near_permanent"

d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "aquatic_context"
] = "near_seasonal"

aquatic_order = ["near_permanent", "near_seasonal", "other_weak"]
d["aquatic_context"] = pd.Categorical(
    d["aquatic_context"],
    categories=aquatic_order,
    ordered=True
)

# HDSI groups
q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)
q90 = d["HDSI"].quantile(0.90)

d["HDSI_group"] = "mid"
d.loc[d["HDSI"] <= q25, "HDSI_group"] = "low"
d.loc[d["HDSI"] >= q75, "HDSI_group"] = "high"

d["HDSI_group"] = pd.Categorical(
    d["HDSI_group"],
    categories=["low", "mid", "high"],
    ordered=True
)

d["HDSI_extreme90"] = (d["HDSI"] >= q90).astype(int)

# Overview
overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].astype(int).unique())))},
    {"metric": "HDSI_Q25", "value": q25},
    {"metric": "HDSI_Q75", "value": q75},
    {"metric": "HDSI_Q90", "value": q90},
    {"metric": "EVI_loss_rate", "value": d["EVI_loss"].mean()},
    {"metric": "LAI_loss_rate", "value": d["LAI_loss"].mean()},
    {"metric": "Joint_loss_rate", "value": d["Joint_loss"].mean()},
    {"metric": "near_permanent_rows", "value": int((d["aquatic_context"] == "near_permanent").sum())},
    {"metric": "near_seasonal_rows", "value": int((d["aquatic_context"] == "near_seasonal").sum())},
    {"metric": "other_weak_rows", "value": int((d["aquatic_context"] == "other_weak").sum())},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])

save_table(overview, "00_data_overview.csv")
display(overview)

print("\nAquatic context counts:")
display(d["aquatic_context"].value_counts())

print("\nHDSI group counts:")
display(d["HDSI_group"].value_counts())

# ============================================================
# 3. Bootstrap helpers
# ============================================================

def boot_mean_ci(x, n_boot=500, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan, np.nan, np.nan

    mean = x.mean()

    if len(x) <= 1:
        return mean, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)

    for i in range(n_boot):
        boots[i] = rng.choice(x, size=len(x), replace=True).mean()

    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi

def boot_diff_ci(a, b, n_boot=500, seed=42):
    # mean(a) - mean(b)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan

    diff = a.mean() - b.mean()

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)

    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots[i] = aa.mean() - bb.mean()

    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff, lo, hi

# ============================================================
# 4. Result 1:
#    HDSI identifies vegetation loss risk
# ============================================================

loss_vars = {
    "EVI": "EVI_loss",
    "LAI": "LAI_loss",
    "Joint": "Joint_loss",
}

r1_rows = []

for group in ["low", "mid", "high"]:
    sub = d[d["HDSI_group"] == group]

    for resp, col in loss_vars.items():
        m, lo, hi = boot_mean_ci(sub[col].values, n_boot=500, seed=42)

        r1_rows.append({
            "HDSI_group": group,
            "response": resp,
            "risk": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

r1 = pd.DataFrame(r1_rows)
save_table(r1, "01_HDSI_group_loss_risk.csv")
display(r1)

# ============================================================
# 5. Result 2:
#    High-HDSI loss risk by aquatic context
# ============================================================

r2_rows = []

high = d[d["HDSI_group"] == "high"].copy()

for aq in aquatic_order:
    sub = high[high["aquatic_context"] == aq]

    for resp, col in loss_vars.items():
        m, lo, hi = boot_mean_ci(sub[col].values, n_boot=500, seed=42)

        r2_rows.append({
            "aquatic_context": aq,
            "response": resp,
            "risk_high_HDSI": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

r2 = pd.DataFrame(r2_rows)

# Risk differences under high HDSI
r2_diff_rows = []

for resp, col in loss_vars.items():
    near = high[high["aquatic_context"] == "near_permanent"][col].values

    for comp in ["near_seasonal", "other_weak"]:
        cc = high[high["aquatic_context"] == comp][col].values

        # positive means comparison has higher risk than near permanent
        diff, lo, hi = boot_diff_ci(cc, near, n_boot=500, seed=42)

        r2_diff_rows.append({
            "response": resp,
            "contrast": f"{comp} - near_permanent",
            "risk_difference": diff,
            "ci_low": lo,
            "ci_high": hi,
            "n_comparison": len(cc),
            "n_near_permanent": len(near)
        })

r2_diff = pd.DataFrame(r2_diff_rows)

save_table(r2, "02_high_HDSI_loss_risk_by_aquatic_context.csv")
save_table(r2_diff, "02_high_HDSI_risk_difference_vs_near_permanent.csv")

display(r2)
display(r2_diff)

# ============================================================
# 6. Result 3:
#    Risk amplification from low to high HDSI
# ============================================================

r3_rows = []

for aq in aquatic_order:
    daq = d[d["aquatic_context"] == aq]

    low = daq[daq["HDSI_group"] == "low"]
    high = daq[daq["HDSI_group"] == "high"]

    for resp, col in loss_vars.items():
        diff, lo, hi = boot_diff_ci(
            high[col].values,
            low[col].values,
            n_boot=500,
            seed=42
        )

        r3_rows.append({
            "aquatic_context": aq,
            "response": resp,
            "fragility_increase_high_minus_low": diff,
            "ci_low": lo,
            "ci_high": hi,
            "risk_low": low[col].mean(),
            "risk_high": high[col].mean(),
            "n_low": len(low),
            "n_high": len(high)
        })

r3 = pd.DataFrame(r3_rows)

# Aquatic fragility buffering:
# comparison fragility - near permanent fragility
afb_rows = []

for resp in ["EVI", "LAI", "Joint"]:
    col = loss_vars[resp]

    near = d[d["aquatic_context"] == "near_permanent"]
    near_low = near[near["HDSI_group"] == "low"][col].values
    near_high = near[near["HDSI_group"] == "high"][col].values

    rng = np.random.default_rng(42)

    for comp in ["near_seasonal", "other_weak"]:
        dc = d[d["aquatic_context"] == comp]
        comp_low = dc[dc["HDSI_group"] == "low"][col].values
        comp_high = dc[dc["HDSI_group"] == "high"][col].values

        if len(near_low) == 0 or len(near_high) == 0 or len(comp_low) == 0 or len(comp_high) == 0:
            afb_rows.append({
                "response": resp,
                "contrast": f"{comp} fragility - near_permanent fragility",
                "AFB": np.nan,
                "ci_low": np.nan,
                "ci_high": np.nan
            })
            continue

        boots = []

        for i in range(500):
            nl = rng.choice(near_low, size=len(near_low), replace=True)
            nh = rng.choice(near_high, size=len(near_high), replace=True)
            cl = rng.choice(comp_low, size=len(comp_low), replace=True)
            ch = rng.choice(comp_high, size=len(comp_high), replace=True)

            frag_near = nh.mean() - nl.mean()
            frag_comp = ch.mean() - cl.mean()

            boots.append(frag_comp - frag_near)

        boots = np.asarray(boots)

        afb_rows.append({
            "response": resp,
            "contrast": f"{comp} fragility - near_permanent fragility",
            "AFB": boots.mean(),
            "ci_low": np.percentile(boots, 2.5),
            "ci_high": np.percentile(boots, 97.5)
        })

afb = pd.DataFrame(afb_rows)

save_table(r3, "03_fragility_increase_by_aquatic_context.csv")
save_table(afb, "03_aquatic_fragility_buffering_AFB.csv")

display(r3)
display(afb)

# ============================================================
# 7. Result 4:
#    Regional heterogeneity
# ============================================================

r4_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region]

    # Require enough sample to avoid noisy region results
    if len(dr) < 1000:
        continue

    for aq in aquatic_order:
        sub = dr[(dr["aquatic_context"] == aq) & (dr["HDSI_group"] == "high")]

        for resp, col in loss_vars.items():
            if len(sub) < 50:
                m, lo, hi = np.nan, np.nan, np.nan
            else:
                m, lo, hi = boot_mean_ci(sub[col].values, n_boot=400, seed=42)

            r4_rows.append({
                "macro_region": region,
                "aquatic_context": aq,
                "response": resp,
                "risk_high_HDSI": m,
                "ci_low": lo,
                "ci_high": hi,
                "n": len(sub),
                "grids": sub["grid_id"].nunique()
            })

r4 = pd.DataFrame(r4_rows)

# Regional risk difference: other_weak - near_permanent under high HDSI
r4_diff_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region]
    dh = dr[dr["HDSI_group"] == "high"]

    if len(dh) < 1000:
        continue

    for resp, col in loss_vars.items():
        near = dh[dh["aquatic_context"] == "near_permanent"][col].values
        other = dh[dh["aquatic_context"] == "other_weak"][col].values
        seasonal = dh[dh["aquatic_context"] == "near_seasonal"][col].values

        for comp_name, comp_values in [
            ("near_seasonal", seasonal),
            ("other_weak", other),
        ]:
            if len(near) < 50 or len(comp_values) < 50:
                diff, lo, hi = np.nan, np.nan, np.nan
            else:
                diff, lo, hi = boot_diff_ci(
                    comp_values,
                    near,
                    n_boot=400,
                    seed=42
                )

            r4_diff_rows.append({
                "macro_region": region,
                "response": resp,
                "contrast": f"{comp_name} - near_permanent",
                "risk_difference_high_HDSI": diff,
                "ci_low": lo,
                "ci_high": hi,
                "n_near_permanent": len(near),
                "n_comparison": len(comp_values)
            })

r4_diff = pd.DataFrame(r4_diff_rows)

save_table(r4, "04_regional_high_HDSI_loss_risk.csv")
save_table(r4_diff, "04_regional_high_HDSI_risk_difference.csv")

display(r4_diff.head(30))

# ============================================================
# 8. Optional: distance-gradient descriptive check
#    Not a main result; only to see whether risk varies with distance.
# ============================================================

dist_bins = [0, 0.5, 1, 2, 5, 10, 25]
dist_labels = ["0-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]

d["dist_perm_band"] = pd.cut(
    d["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
)

dist_rows = []

for band in dist_labels:
    sub = d[(d["dist_perm_band"].astype(str) == band) & (d["HDSI_group"] == "high")]

    for resp, col in loss_vars.items():
        if len(sub) < 50:
            m, lo, hi = np.nan, np.nan, np.nan
        else:
            m, lo, hi = boot_mean_ci(sub[col].values, n_boot=400, seed=42)

        dist_rows.append({
            "dist_perm_band": band,
            "response": resp,
            "risk_high_HDSI": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

dist_risk = pd.DataFrame(dist_rows)
save_table(dist_risk, "05_optional_distance_band_high_HDSI_risk.csv")

# ============================================================
# 9. Figures
# ============================================================

# ------------------------------------------------------------
# Fig 1. HDSI group loss risk
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharey=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = r1[r1["response"] == resp].copy()
    sub["HDSI_group"] = pd.Categorical(sub["HDSI_group"], categories=["low", "mid", "high"], ordered=True)
    sub = sub.sort_values("HDSI_group")

    x = np.arange(len(sub))

    ax.bar(
        x,
        sub["risk"],
        color=RESP_COLORS[resp],
        width=0.65
    )

    ax.errorbar(
        x,
        sub["risk"],
        yerr=np.vstack([
            sub["risk"] - sub["ci_low"],
            sub["ci_high"] - sub["risk"]
        ]),
        fmt="none",
        color="#222222",
        capsize=3,
        linewidth=1
    )

    ax.set_xticks(x)
    ax.set_xticklabels([HDSI_LABELS[g] for g in sub["HDSI_group"].astype(str)])
    ax.set_ylabel("Vegetation loss risk")
    ax.set_title(f"{resp} loss")
    ax.set_ylim(0, max(0.65, sub["ci_high"].max() + 0.05))
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig1_HDSI_identifies_loss_risk")
plt.show()

# ------------------------------------------------------------
# Fig 2. High-HDSI loss risk by aquatic context
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = r2[r2["response"] == resp].copy()
    sub["aquatic_context"] = pd.Categorical(sub["aquatic_context"], categories=aquatic_order, ordered=True)
    sub = sub.sort_values("aquatic_context")

    x = np.arange(len(sub))
    colors = [AQUA_COLORS[a] for a in sub["aquatic_context"].astype(str)]

    ax.bar(
        x,
        sub["risk_high_HDSI"],
        color=colors,
        width=0.65
    )

    ax.errorbar(
        x,
        sub["risk_high_HDSI"],
        yerr=np.vstack([
            sub["risk_high_HDSI"] - sub["ci_low"],
            sub["ci_high"] - sub["risk_high_HDSI"]
        ]),
        fmt="none",
        color="#222222",
        capsize=3,
        linewidth=1
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        ["Near\npermanent", "Near\nseasonal", "Other/\nweak"],
        rotation=0
    )
    ax.set_ylabel("High-HDSI loss risk")
    ax.set_title(f"{resp} loss")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig2_high_HDSI_loss_risk_by_aquatic_context")
plt.show()

# ------------------------------------------------------------
# Fig 3. Fragility increase by aquatic context
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = r3[r3["response"] == resp].copy()
    sub["aquatic_context"] = pd.Categorical(sub["aquatic_context"], categories=aquatic_order, ordered=True)
    sub = sub.sort_values("aquatic_context")

    x = np.arange(len(sub))
    colors = [AQUA_COLORS[a] for a in sub["aquatic_context"].astype(str)]

    ax.bar(
        x,
        sub["fragility_increase_high_minus_low"],
        color=colors,
        width=0.65
    )

    ax.errorbar(
        x,
        sub["fragility_increase_high_minus_low"],
        yerr=np.vstack([
            sub["fragility_increase_high_minus_low"] - sub["ci_low"],
            sub["ci_high"] - sub["fragility_increase_high_minus_low"]
        ]),
        fmt="none",
        color="#222222",
        capsize=3,
        linewidth=1
    )

    ax.axhline(0, color="#333333", linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(["Near\npermanent", "Near\nseasonal", "Other/\nweak"])
    ax.set_ylabel("Risk amplification\nHigh HDSI − Low HDSI")
    ax.set_title(f"{resp} loss")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_risk_amplification_by_aquatic_context")
plt.show()

# ------------------------------------------------------------
# Fig 4. AFB summary
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 4.5))

plot = afb[afb["contrast"].str.contains("other_weak")].copy()
plot["response"] = pd.Categorical(plot["response"], categories=["EVI", "LAI", "Joint"], ordered=True)
plot = plot.sort_values("response")

x = np.arange(len(plot))
colors = [RESP_COLORS[r] for r in plot["response"].astype(str)]

ax.bar(
    x,
    plot["AFB"],
    color=colors,
    width=0.65
)

ax.errorbar(
    x,
    plot["AFB"],
    yerr=np.vstack([
        plot["AFB"] - plot["ci_low"],
        plot["ci_high"] - plot["AFB"]
    ]),
    fmt="none",
    color="#222222",
    capsize=3,
    linewidth=1
)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["response"])
ax.set_ylabel("AFB\nFragility_other/weak − Fragility_near permanent")
ax.set_title("Aquatic fragility buffering")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig4_aquatic_fragility_buffering_AFB")
plt.show()

# ------------------------------------------------------------
# Fig 5. Regional heterogeneity
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharex=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = r4_diff[
        (r4_diff["response"] == resp) &
        (r4_diff["contrast"] == "other_weak - near_permanent")
    ].copy()

    sub = sub.dropna(subset=["risk_difference_high_HDSI"])
    sub = sub.sort_values("risk_difference_high_HDSI")

    y = np.arange(len(sub))

    ax.errorbar(
        sub["risk_difference_high_HDSI"],
        y,
        xerr=np.vstack([
            sub["risk_difference_high_HDSI"] - sub["ci_low"],
            sub["ci_high"] - sub["risk_difference_high_HDSI"]
        ]),
        fmt="o",
        color=RESP_COLORS[resp],
        capsize=3,
        linewidth=1
    )

    ax.axvline(0, color="#333333", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["macro_region"])
    ax.set_xlabel("High-HDSI risk difference\nOther/weak − near permanent")
    ax.set_title(f"{resp} loss")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_regional_heterogeneity_high_HDSI_risk_difference")
plt.show()

# ------------------------------------------------------------
# Fig 6. Optional distance-band check
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = dist_risk[dist_risk["response"] == resp].copy()
    sub["dist_perm_band"] = pd.Categorical(sub["dist_perm_band"], categories=dist_labels, ordered=True)
    sub = sub.sort_values("dist_perm_band")

    x = np.arange(len(sub))

    ax.plot(
        x,
        sub["risk_high_HDSI"],
        marker="o",
        color=RESP_COLORS[resp],
        linewidth=1.6
    )

    ax.fill_between(
        x,
        sub["ci_low"],
        sub["ci_high"],
        color=RESP_COLORS[resp],
        alpha=0.18
    )

    ax.set_xticks(x)
    ax.set_xticklabels(sub["dist_perm_band"], rotation=35, ha="right")
    ax.set_xlabel("Distance to permanent water (km)")
    ax.set_ylabel("High-HDSI loss risk")
    ax.set_title(f"{resp} loss")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig6_optional_distance_band_high_HDSI_risk")
plt.show()

# ============================================================
# 10. Print outputs
# ============================================================

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("01_HDSI_group_loss_risk.csv")
print("02_high_HDSI_loss_risk_by_aquatic_context.csv")
print("02_high_HDSI_risk_difference_vs_near_permanent.csv")
print("03_fragility_increase_by_aquatic_context.csv")
print("03_aquatic_fragility_buffering_AFB.csv")
print("04_regional_high_HDSI_risk_difference.csv")
print("05_optional_distance_band_high_HDSI_risk.csv")

In [ ]:
# ============================================================
# Lightweight mechanism diagnostic:
# Under high HDSI, do aquatic contexts differ in stress components?
#
# Purpose:
#   Explore HOW near permanent water may reduce HDSI-related vegetation loss:
#   water-supply pathway, surface thermal pathway, atmospheric dryness pathway.
#
# This is descriptive mechanism evidence, not causal mediation.
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

MECH_DIR = os.path.join(OUT_DIR, "mechanism_light")
MECH_TAB_DIR = os.path.join(MECH_DIR, "tables")
MECH_FIG_DIR = os.path.join(MECH_DIR, "figs")

os.makedirs(MECH_TAB_DIR, exist_ok=True)
os.makedirs(MECH_FIG_DIR, exist_ok=True)

# Construct stress components if not already available
d["SM_deficit"] = -d["summer_SM_rootproxy_z"]
d["Precip_deficit"] = -d["summer_Precip_z"]
d["LST_Day"] = d["summer_LST_Day_z"]
d["VPD"] = d["summer_VPD_z"]
d["Tair"] = d["summer_Tair_z"]

stress_components = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature"
}

# High HDSI subset
dh = d[d["HDSI_group"] == "high"].copy()

def boot_mean_ci(x, n_boot=500, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    mean = x.mean()
    if len(x) <= 1:
        return mean, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        boots[i] = rng.choice(x, size=len(x), replace=True).mean()
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return mean, lo, hi

def boot_diff_ci(a, b, n_boot=500, seed=42):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    diff = a.mean() - b.mean()
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for i in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots[i] = aa.mean() - bb.mean()
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return diff, lo, hi

# 1. Component means by aquatic context
mech_rows = []

for comp, label in stress_components.items():
    for aq in aquatic_order:
        sub = dh[dh["aquatic_context"] == aq]
        m, lo, hi = boot_mean_ci(sub[comp].values, n_boot=500, seed=42)
        mech_rows.append({
            "component": comp,
            "component_label": label,
            "aquatic_context": aq,
            "mean_high_HDSI": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

mech_state = pd.DataFrame(mech_rows)

# 2. Component differences relative to near permanent
diff_rows = []

for comp, label in stress_components.items():
    near = dh[dh["aquatic_context"] == "near_permanent"][comp].values

    for comp_aq in ["near_seasonal", "other_weak"]:
        cc = dh[dh["aquatic_context"] == comp_aq][comp].values

        # positive means comparison has higher stress component than near permanent
        diff, lo, hi = boot_diff_ci(cc, near, n_boot=500, seed=42)

        diff_rows.append({
            "component": comp,
            "component_label": label,
            "contrast": f"{comp_aq} - near_permanent",
            "difference": diff,
            "ci_low": lo,
            "ci_high": hi,
            "n_comparison": len(cc),
            "n_near_permanent": len(near)
        })

mech_diff = pd.DataFrame(diff_rows)

mech_state.to_csv(os.path.join(MECH_TAB_DIR, "mechanism_high_HDSI_component_means.csv"), index=False)
mech_diff.to_csv(os.path.join(MECH_TAB_DIR, "mechanism_high_HDSI_component_differences.csv"), index=False)

display(mech_state)
display(mech_diff)

# 3. Figure: high-HDSI component differences
plot = mech_diff[mech_diff["contrast"] == "other_weak - near_permanent"].copy()
plot["component"] = pd.Categorical(
    plot["component"],
    categories=["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"],
    ordered=True
)
plot = plot.sort_values("component")

fig, ax = plt.subplots(figsize=(8, 4.6))

x = np.arange(len(plot))
colors = ["#3B6EA8", "#009E73", "#B2182B", "#E69F00", "#7E62A3"]

ax.bar(x, plot["difference"], color=colors, width=0.65)

ax.errorbar(
    x,
    plot["difference"],
    yerr=np.vstack([
        plot["difference"] - plot["ci_low"],
        plot["ci_high"] - plot["difference"]
    ]),
    fmt="none",
    color="#222222",
    capsize=3,
    linewidth=1
)

ax.axhline(0, color="#333333", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["component_label"], rotation=30, ha="right")
ax.set_ylabel("Component difference under high HDSI\nOther/weak − near permanent")
ax.set_title("Environmental stress components during high-HDSI events")

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.grid(alpha=0.18, linewidth=0.4)

plt.tight_layout()

out_png = os.path.join(MECH_FIG_DIR, "Fig_mechanism_high_HDSI_component_difference.png")
out_pdf = os.path.join(MECH_FIG_DIR, "Fig_mechanism_high_HDSI_component_difference.pdf")

plt.savefig(out_png, dpi=300, bbox_inches="tight")
plt.savefig(out_pdf, bbox_inches="tight")
plt.show()

print("Saved:", out_png)
print("Saved:", out_pdf)

In [ ]:
# ============================================================
# Core analysis and visualization:
# Surface-water permanence, distance decay, and vegetation loss
# under compound heat–dryness events
#
# Main outputs:
#   Fig1: HDSI risk gradient + national map
#   Fig2: Distance-decay of fragility increase
#   Fig3: Water stability and risk amplification
#   Fig4: Lightweight mechanism diagnostics
#   Fig5: Seasonal propagation
#   Fig6: Regional boundary conditions
#
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

try:
    from scipy.stats import spearmanr, pearsonr
except Exception:
    spearmanr = None
    pearsonr = None

try:
    import geopandas as gpd
except Exception:
    gpd = None


# ============================================================
# 0. Paths
# ============================================================

PATCHED_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(PATCHED_DIR, "_aquatic_buffering")
PANEL_FP = os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet")

# Optional China boundary, change if needed
CHINA_BOUNDARY_FP = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_DIR = os.path.join(AQUA_DIR, "_final_core_story_results")
TAB_DIR = os.path.join(OUT_DIR, "tables")
FIG_DIR = os.path.join(OUT_DIR, "figs")

os.makedirs(TAB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

print("Panel:", PANEL_FP)
print("Exists:", os.path.exists(PANEL_FP))


# ============================================================
# 1. Plot style
# ============================================================

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 8.5,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "legend.fontsize": 7.5,
    "axes.linewidth": 0.7,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "blue": "#3B6EA8",
    "orange": "#E69F00",
    "red": "#B2182B",
    "green": "#4E9A51",
    "purple": "#7E62A3",
    "gray": "#666666",
    "dark": "#222222",
    "lightgray": "#D9D9D9",
    "teal": "#009E73",
    "brown": "#8C510A"
}

RESP_COLORS = {
    "EVI": COL["green"],
    "LAI": COL["purple"],
    "Joint": COL["red"],
}

AQUA_COLORS = {
    "near_permanent": COL["blue"],
    "near_seasonal": COL["orange"],
    "other_weak": COL["gray"],
}

STRESS_COLORS = {
    "HDSI": COL["dark"],
    "SM_deficit": COL["blue"],
    "Precip_deficit": COL["teal"],
    "LST_Day": COL["red"],
    "VPD": COL["orange"],
    "Tair": COL["purple"],
}

AQUA_LABELS = {
    "near_permanent": "Near permanent",
    "near_seasonal": "Near seasonal",
    "other_weak": "Other / weak",
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.18, linewidth=0.4)

def panel_label(ax, label, x=-0.08, y=1.04):
    ax.text(
        x, y, label,
        transform=ax.transAxes,
        fontsize=12,
        fontweight="bold",
        va="top",
        ha="left"
    )

def savefig(fig, name):
    png = os.path.join(FIG_DIR, name + ".png")
    pdf = os.path.join(FIG_DIR, name + ".pdf")
    fig.savefig(png, bbox_inches="tight", dpi=300)
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png)
    print("Saved:", pdf)

def save_table(df, name):
    fp = os.path.join(TAB_DIR, name)
    df.to_csv(fp, index=False)
    print("Saved:", fp)
    return fp


# ============================================================
# 2. Load and prepare data
# ============================================================

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

num_cols = [
    "lon", "lat", "year",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_VPD_z", "summer_LST_Day_z",
    "summer_SM_rootproxy_z", "summer_Precip_z", "summer_Tair_z",
    "elevation", "slope", "LC_Type1",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "dist_any_water_km",
    "water_occurrence", "water_seasonality", "water_transition"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# HDSI
if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )
else:
    df["HDSI"] = pd.to_numeric(df["HDSI"], errors="coerce")

# Stress components
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

# Vegetation loss
df["EVI_loss"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_loss"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
df["Joint_loss"] = ((df["EVI_loss"] == 1) | (df["LAI_loss"] == 1)).astype(int)

# Seasonal propagation
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# Macro region fallback
def assign_macro_region(row):
    lon = row["lon"]
    lat = row["lat"]
    elev = row["elevation"] if "elevation" in row else np.nan

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

# Valid sample
valid_mask = df["valid_model_both_y"] if "valid_model_both_y" in df.columns else True

required = [
    "grid_id", "lon", "lat", "year", "macro_region",
    "HDSI",
    "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "EVI_loss", "LAI_loss", "Joint_loss",
    "delta_EVI", "delta_LAI"
]

d = df[valid_mask].copy()
d = d.dropna(subset=required).copy()

# Aquatic context
d["aquatic_context"] = "other_weak"
d.loc[d["dist_perm_water_km"] < 1.0, "aquatic_context"] = "near_permanent"
d.loc[
    (d["dist_perm_water_km"] >= 1.0) &
    (d["dist_seasonal_water_km"] < 1.0),
    "aquatic_context"
] = "near_seasonal"

aquatic_order = ["near_permanent", "near_seasonal", "other_weak"]
d["aquatic_context"] = pd.Categorical(d["aquatic_context"], categories=aquatic_order, ordered=True)

# HDSI groups
q25 = d["HDSI"].quantile(0.25)
q75 = d["HDSI"].quantile(0.75)
q90 = d["HDSI"].quantile(0.90)

d["HDSI_group"] = "mid"
d.loc[d["HDSI"] <= q25, "HDSI_group"] = "low"
d.loc[d["HDSI"] >= q75, "HDSI_group"] = "high"
d["HDSI_group"] = pd.Categorical(d["HDSI_group"], categories=["low", "mid", "high"], ordered=True)

# Distance bands
dist_bins = [0, 0.25, 0.5, 1, 2, 3, 5, 7.5, 10, 15, np.inf]
dist_labels = ["0-0.25", "0.25-0.5", "0.5-1", "1-2", "2-3", "3-5", "5-7.5", "7.5-10", "10-15", ">15"]

d["dist_perm_band"] = pd.cut(
    d["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
)

dist_mid = {
    "0-0.25": 0.125,
    "0.25-0.5": 0.375,
    "0.5-1": 0.75,
    "1-2": 1.5,
    "2-3": 2.5,
    "3-5": 4.0,
    "5-7.5": 6.25,
    "7.5-10": 8.75,
    "10-15": 12.5,
    ">15": 17.5,
}

loss_vars = {
    "EVI": "EVI_loss",
    "LAI": "LAI_loss",
    "Joint": "Joint_loss",
}

stress_vars = {
    "HDSI": "HDSI",
    "SM_deficit": "SM_deficit",
    "Precip_deficit": "Precip_deficit",
    "LST_Day": "LST_Day",
    "VPD": "VPD",
    "Tair": "Tair",
}

overview = pd.DataFrame([
    {"metric": "rows", "value": len(d)},
    {"metric": "unique_grids", "value": d["grid_id"].nunique()},
    {"metric": "years", "value": ",".join(map(str, sorted(d["year"].astype(int).unique())))},
    {"metric": "HDSI_Q25", "value": q25},
    {"metric": "HDSI_Q75", "value": q75},
    {"metric": "HDSI_Q90", "value": q90},
    {"metric": "EVI_loss_rate", "value": d["EVI_loss"].mean()},
    {"metric": "LAI_loss_rate", "value": d["LAI_loss"].mean()},
    {"metric": "Joint_loss_rate", "value": d["Joint_loss"].mean()},
    {"metric": "near_permanent_rows", "value": int((d["aquatic_context"] == "near_permanent").sum())},
    {"metric": "near_seasonal_rows", "value": int((d["aquatic_context"] == "near_seasonal").sum())},
    {"metric": "other_weak_rows", "value": int((d["aquatic_context"] == "other_weak").sum())},
    {"metric": "dist_perm_mean", "value": d["dist_perm_water_km"].mean()},
    {"metric": "dist_perm_p90", "value": d["dist_perm_water_km"].quantile(0.90)},
    {"metric": "dist_perm_max", "value": d["dist_perm_water_km"].max()},
])

save_table(overview, "00_data_overview.csv")
display(overview)


# ============================================================
# 3. Cluster bootstrap helpers
#    Bootstrap by grid_id to reduce pseudo-replication concern
# ============================================================

def cluster_boot_mean(data, value_col, cluster_col="grid_id", n_boot=300, seed=42):
    dd = data[[cluster_col, value_col]].dropna()
    if len(dd) == 0:
        return np.nan, np.nan, np.nan

    point = dd[value_col].mean()
    clusters = dd[cluster_col].drop_duplicates().values

    if len(clusters) <= 1:
        return point, np.nan, np.nan

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)

    grouped = {k: v[value_col].values for k, v in dd.groupby(cluster_col)}

    for i in range(n_boot):
        sampled = rng.choice(clusters, size=len(clusters), replace=True)
        vals = np.concatenate([grouped[c] for c in sampled if c in grouped])
        boots[i] = vals.mean() if len(vals) else np.nan

    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return point, lo, hi

def cluster_boot_diff(data_a, data_b, value_col, cluster_col="grid_id", n_boot=300, seed=42):
    # mean(a) - mean(b)
    da = data_a[[cluster_col, value_col]].dropna()
    db = data_b[[cluster_col, value_col]].dropna()

    if len(da) == 0 or len(db) == 0:
        return np.nan, np.nan, np.nan

    point = da[value_col].mean() - db[value_col].mean()

    ca = da[cluster_col].drop_duplicates().values
    cb = db[cluster_col].drop_duplicates().values

    if len(ca) <= 1 or len(cb) <= 1:
        return point, np.nan, np.nan

    ga = {k: v[value_col].values for k, v in da.groupby(cluster_col)}
    gb = {k: v[value_col].values for k, v in db.groupby(cluster_col)}

    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)

    for i in range(n_boot):
        sa = rng.choice(ca, size=len(ca), replace=True)
        sb = rng.choice(cb, size=len(cb), replace=True)
        va = np.concatenate([ga[c] for c in sa if c in ga])
        vb = np.concatenate([gb[c] for c in sb if c in gb])
        boots[i] = va.mean() - vb.mean()

    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return point, lo, hi

def cluster_boot_fragility(data, value_col, low_mask, high_mask, cluster_col="grid_id", n_boot=300, seed=42):
    low = data[low_mask]
    high = data[high_mask]
    return cluster_boot_diff(high, low, value_col, cluster_col=cluster_col, n_boot=n_boot, seed=seed)

def corr_stats(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]

    out = {"pearson_r": np.nan, "pearson_p": np.nan, "spearman_r": np.nan, "spearman_p": np.nan}

    if len(x) >= 3:
        if pearsonr is not None:
            r, p = pearsonr(x, y)
            out["pearson_r"] = r
            out["pearson_p"] = p
        if spearmanr is not None:
            r, p = spearmanr(x, y)
            out["spearman_r"] = r
            out["spearman_p"] = p
    return out


# ============================================================
# 4. Result 1: HDSI identifies vegetation loss risk
# ============================================================

r1_rows = []

for hgroup in ["low", "mid", "high"]:
    sub = d[d["HDSI_group"] == hgroup]

    for resp, col in loss_vars.items():
        m, lo, hi = cluster_boot_mean(sub, col, n_boot=300, seed=42)

        r1_rows.append({
            "HDSI_group": hgroup,
            "response": resp,
            "risk": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

r1 = pd.DataFrame(r1_rows)
save_table(r1, "01_HDSI_group_loss_risk.csv")


# ============================================================
# 5. Result 2: Distance decay of fragility increase
# ============================================================

dist_rows = []

for band in dist_labels:
    db = d[d["dist_perm_band"].astype(str) == band].copy()
    if len(db) == 0:
        continue

    for resp, col in loss_vars.items():
        low_mask = db["HDSI_group"] == "low"
        high_mask = db["HDSI_group"] == "high"

        risk_low, low_lo, low_hi = cluster_boot_mean(db[low_mask], col, n_boot=300, seed=42)
        risk_high, high_lo, high_hi = cluster_boot_mean(db[high_mask], col, n_boot=300, seed=42)
        frag, frag_lo, frag_hi = cluster_boot_fragility(db, col, low_mask, high_mask, n_boot=300, seed=42)

        dist_rows.append({
            "dist_perm_band": band,
            "dist_mid_km": dist_mid.get(band, np.nan),
            "response": resp,
            "risk_low_HDSI": risk_low,
            "risk_low_ci_low": low_lo,
            "risk_low_ci_high": low_hi,
            "risk_high_HDSI": risk_high,
            "risk_high_ci_low": high_lo,
            "risk_high_ci_high": high_hi,
            "fragility_increase": frag,
            "fragility_ci_low": frag_lo,
            "fragility_ci_high": frag_hi,
            "n_total": len(db),
            "n_low": int(low_mask.sum()),
            "n_high": int(high_mask.sum()),
            "grids": db["grid_id"].nunique()
        })

dist_res = pd.DataFrame(dist_rows)

# Correlation of distance mid vs fragility
corr_rows = []
for resp in ["EVI", "LAI", "Joint"]:
    sub = dist_res[(dist_res["response"] == resp) & (dist_res["dist_perm_band"] != ">15")].dropna(subset=["dist_mid_km", "fragility_increase"])
    cs = corr_stats(sub["dist_mid_km"], sub["fragility_increase"])
    corr_rows.append({
        "response": resp,
        **cs
    })

dist_corr = pd.DataFrame(corr_rows)

save_table(dist_res, "02_distance_decay_fragility.csv")
save_table(dist_corr, "02_distance_decay_correlation.csv")


# ============================================================
# 6. Result 3: Water stability and AFB
# ============================================================

stability_rows = []
afb_rows = []

for aq in aquatic_order:
    daq = d[d["aquatic_context"] == aq]

    for resp, col in loss_vars.items():
        low = daq[daq["HDSI_group"] == "low"]
        high = daq[daq["HDSI_group"] == "high"]

        risk_low, low_lo, low_hi = cluster_boot_mean(low, col, n_boot=300, seed=42)
        risk_high, high_lo, high_hi = cluster_boot_mean(high, col, n_boot=300, seed=42)
        frag, frag_lo, frag_hi = cluster_boot_diff(high, low, col, n_boot=300, seed=42)

        stability_rows.append({
            "aquatic_context": aq,
            "response": resp,
            "risk_low_HDSI": risk_low,
            "risk_low_ci_low": low_lo,
            "risk_low_ci_high": low_hi,
            "risk_high_HDSI": risk_high,
            "risk_high_ci_low": high_lo,
            "risk_high_ci_high": high_hi,
            "fragility_increase": frag,
            "fragility_ci_low": frag_lo,
            "fragility_ci_high": frag_hi,
            "n_low": len(low),
            "n_high": len(high),
            "grids": daq["grid_id"].nunique()
        })

stability_res = pd.DataFrame(stability_rows)

# AFB = fragility(comp) - fragility(near_permanent)
for resp, col in loss_vars.items():
    near = d[d["aquatic_context"] == "near_permanent"]
    near_low = near[near["HDSI_group"] == "low"]
    near_high = near[near["HDSI_group"] == "high"]

    for comp in ["near_seasonal", "other_weak"]:
        dc = d[d["aquatic_context"] == comp]
        comp_low = dc[dc["HDSI_group"] == "low"]
        comp_high = dc[dc["HDSI_group"] == "high"]

        # bootstrap fragility difference
        rng = np.random.default_rng(42)
        boot_vals = []

        # prepare cluster dictionaries
        def prep(dd):
            dd = dd[["grid_id", col]].dropna()
            clusters = dd["grid_id"].drop_duplicates().values
            grouped = {k: v[col].values for k, v in dd.groupby("grid_id")}
            return clusters, grouped

        cl_nl, g_nl = prep(near_low)
        cl_nh, g_nh = prep(near_high)
        cl_cl, g_cl = prep(comp_low)
        cl_ch, g_ch = prep(comp_high)

        if min(len(cl_nl), len(cl_nh), len(cl_cl), len(cl_ch)) <= 1:
            afb, lo, hi = np.nan, np.nan, np.nan
        else:
            for i in range(300):
                snl = rng.choice(cl_nl, size=len(cl_nl), replace=True)
                snh = rng.choice(cl_nh, size=len(cl_nh), replace=True)
                scl = rng.choice(cl_cl, size=len(cl_cl), replace=True)
                sch = rng.choice(cl_ch, size=len(cl_ch), replace=True)

                v_nl = np.concatenate([g_nl[c] for c in snl])
                v_nh = np.concatenate([g_nh[c] for c in snh])
                v_cl = np.concatenate([g_cl[c] for c in scl])
                v_ch = np.concatenate([g_ch[c] for c in sch])

                frag_near = v_nh.mean() - v_nl.mean()
                frag_comp = v_ch.mean() - v_cl.mean()
                boot_vals.append(frag_comp - frag_near)

            boot_vals = np.asarray(boot_vals)
            afb = np.nanmean(boot_vals)
            lo, hi = np.nanpercentile(boot_vals, [2.5, 97.5])

        afb_rows.append({
            "response": resp,
            "contrast": f"{comp} fragility - near_permanent fragility",
            "AFB": afb,
            "ci_low": lo,
            "ci_high": hi
        })

afb_res = pd.DataFrame(afb_rows)

save_table(stability_res, "03_water_stability_risk_and_fragility.csv")
save_table(afb_res, "03_aquatic_fragility_buffering_AFB.csv")


# ============================================================
# 7. Result 4: Lightweight mechanism
#    4a. Stress-channel fragility
# ============================================================

stress_frag_rows = []

for stress_name, stress_col in stress_vars.items():
    s25 = d[stress_col].quantile(0.25)
    s75 = d[stress_col].quantile(0.75)

    for aq in aquatic_order:
        daq = d[d["aquatic_context"] == aq]

        low_mask = daq[stress_col] <= s25
        high_mask = daq[stress_col] >= s75

        for resp, col in loss_vars.items():
            risk_low, _, _ = cluster_boot_mean(daq[low_mask], col, n_boot=300, seed=42)
            risk_high, _, _ = cluster_boot_mean(daq[high_mask], col, n_boot=300, seed=42)
            frag, lo, hi = cluster_boot_diff(daq[high_mask], daq[low_mask], col, n_boot=300, seed=42)

            stress_frag_rows.append({
                "stress": stress_name,
                "stress_q25": s25,
                "stress_q75": s75,
                "aquatic_context": aq,
                "response": resp,
                "risk_low_stress": risk_low,
                "risk_high_stress": risk_high,
                "fragility_increase": frag,
                "ci_low": lo,
                "ci_high": hi,
                "n_low": int(low_mask.sum()),
                "n_high": int(high_mask.sum())
            })

stress_frag = pd.DataFrame(stress_frag_rows)

# Stress-channel AFB
stress_afb_rows = []

for stress_name in stress_vars.keys():
    for resp in ["EVI", "LAI", "Joint"]:
        sub = stress_frag[(stress_frag["stress"] == stress_name) & (stress_frag["response"] == resp)]
        near = sub[sub["aquatic_context"] == "near_permanent"]

        if len(near) == 0:
            continue

        near_frag = near["fragility_increase"].iloc[0]

        for comp in ["near_seasonal", "other_weak"]:
            cc = sub[sub["aquatic_context"] == comp]
            if len(cc) == 0:
                continue

            stress_afb_rows.append({
                "stress": stress_name,
                "response": resp,
                "contrast": f"{comp} - near_permanent",
                "AFB_simple": cc["fragility_increase"].iloc[0] - near_frag,
                "fragility_near_permanent": near_frag,
                "fragility_comparison": cc["fragility_increase"].iloc[0]
            })

stress_afb = pd.DataFrame(stress_afb_rows)

save_table(stress_frag, "04_stress_channel_fragility.csv")
save_table(stress_afb, "04_stress_channel_AFB_simple.csv")


# ============================================================
# 8. Result 4b: Seasonal propagation
# ============================================================

season_rows = []
delta_vars = {
    "delta_EVI": "delta_EVI",
    "delta_LAI": "delta_LAI"
}

for aq in aquatic_order:
    daq = d[d["aquatic_context"] == aq]

    for name, col in delta_vars.items():
        low = daq[daq["HDSI_group"] == "low"]
        high = daq[daq["HDSI_group"] == "high"]

        mean_low, low_lo, low_hi = cluster_boot_mean(low, col, n_boot=300, seed=42)
        mean_high, high_lo, high_hi = cluster_boot_mean(high, col, n_boot=300, seed=42)
        deterioration, det_lo, det_hi = cluster_boot_diff(high, low, col, n_boot=300, seed=42)

        season_rows.append({
            "aquatic_context": aq,
            "delta_variable": name,
            "mean_low_HDSI": mean_low,
            "mean_low_ci_low": low_lo,
            "mean_low_ci_high": low_hi,
            "mean_high_HDSI": mean_high,
            "mean_high_ci_low": high_lo,
            "mean_high_ci_high": high_hi,
            "high_minus_low_delta": deterioration,
            "ci_low": det_lo,
            "ci_high": det_hi,
            "n_low": len(low),
            "n_high": len(high)
        })

season_res = pd.DataFrame(season_rows)

# Compare deterioration: comp - near_permanent
season_diff_rows = []

for name, col in delta_vars.items():
    near = season_res[(season_res["aquatic_context"] == "near_permanent") & (season_res["delta_variable"] == name)]
    near_val = near["high_minus_low_delta"].iloc[0]

    for comp in ["near_seasonal", "other_weak"]:
        cc = season_res[(season_res["aquatic_context"] == comp) & (season_res["delta_variable"] == name)]
        if len(cc) == 0:
            continue
        season_diff_rows.append({
            "delta_variable": name,
            "contrast": f"{comp} deterioration - near_permanent deterioration",
            "difference_simple": cc["high_minus_low_delta"].iloc[0] - near_val,
            "near_permanent_deterioration": near_val,
            "comparison_deterioration": cc["high_minus_low_delta"].iloc[0]
        })

season_diff = pd.DataFrame(season_diff_rows)

save_table(season_res, "05_seasonal_propagation_delta.csv")
save_table(season_diff, "05_seasonal_propagation_difference_simple.csv")


# ============================================================
# 9. Result 4c: High-HDSI stress components
# ============================================================

component_rows = []
component_diff_rows = []

components = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature"
}

dh = d[d["HDSI_group"] == "high"].copy()

for comp, label in components.items():
    for aq in aquatic_order:
        sub = dh[dh["aquatic_context"] == aq]
        m, lo, hi = cluster_boot_mean(sub, comp, n_boot=300, seed=42)

        component_rows.append({
            "component": comp,
            "component_label": label,
            "aquatic_context": aq,
            "mean_high_HDSI": m,
            "ci_low": lo,
            "ci_high": hi,
            "n": len(sub),
            "grids": sub["grid_id"].nunique()
        })

    near = dh[dh["aquatic_context"] == "near_permanent"]
    for comp_aq in ["near_seasonal", "other_weak"]:
        cc = dh[dh["aquatic_context"] == comp_aq]
        diff, lo, hi = cluster_boot_diff(cc, near, comp, n_boot=300, seed=42)

        component_diff_rows.append({
            "component": comp,
            "component_label": label,
            "contrast": f"{comp_aq} - near_permanent",
            "difference": diff,
            "ci_low": lo,
            "ci_high": hi,
            "n_comparison": len(cc),
            "n_near_permanent": len(near)
        })

component_state = pd.DataFrame(component_rows)
component_diff = pd.DataFrame(component_diff_rows)

save_table(component_state, "06_high_HDSI_component_state.csv")
save_table(component_diff, "06_high_HDSI_component_difference.csv")


# ============================================================
# 10. Result 5: Regional boundary conditions
# ============================================================

regional_rows = []
regional_afb_rows = []

for region in sorted(d["macro_region"].dropna().unique()):
    dr = d[d["macro_region"] == region].copy()
    if len(dr) < 1000:
        continue

    for aq in aquatic_order:
        daq = dr[dr["aquatic_context"] == aq]

        for resp, col in loss_vars.items():
            low = daq[daq["HDSI_group"] == "low"]
            high = daq[daq["HDSI_group"] == "high"]

            if len(low) < 50 or len(high) < 50:
                frag, lo, hi = np.nan, np.nan, np.nan
                risk_high = np.nan
            else:
                risk_high, _, _ = cluster_boot_mean(high, col, n_boot=200, seed=42)
                frag, lo, hi = cluster_boot_diff(high, low, col, n_boot=200, seed=42)

            regional_rows.append({
                "macro_region": region,
                "aquatic_context": aq,
                "response": resp,
                "risk_high_HDSI": risk_high,
                "fragility_increase": frag,
                "ci_low": lo,
                "ci_high": hi,
                "n_low": len(low),
                "n_high": len(high),
                "grids": daq["grid_id"].nunique()
            })

regional_res = pd.DataFrame(regional_rows)

for region in sorted(regional_res["macro_region"].dropna().unique()):
    for resp in ["EVI", "LAI", "Joint"]:
        sub = regional_res[(regional_res["macro_region"] == region) & (regional_res["response"] == resp)]
        near = sub[sub["aquatic_context"] == "near_permanent"]

        if len(near) == 0:
            continue

        near_frag = near["fragility_increase"].iloc[0]

        for comp in ["near_seasonal", "other_weak"]:
            cc = sub[sub["aquatic_context"] == comp]
            if len(cc) == 0:
                continue

            regional_afb_rows.append({
                "macro_region": region,
                "response": resp,
                "contrast": f"{comp} - near_permanent",
                "AFB_simple": cc["fragility_increase"].iloc[0] - near_frag,
                "near_permanent_fragility": near_frag,
                "comparison_fragility": cc["fragility_increase"].iloc[0],
                "n_comp_high": cc["n_high"].iloc[0],
                "n_near_high": near["n_high"].iloc[0]
            })

regional_afb = pd.DataFrame(regional_afb_rows)

save_table(regional_res, "07_regional_fragility_by_context.csv")
save_table(regional_afb, "07_regional_AFB_simple.csv")


# ============================================================
# 11. National map summaries
# ============================================================

grid_summary = (
    d.groupby("grid_id", as_index=False)
    .agg(
        lon=("lon", "first"),
        lat=("lat", "first"),
        macro_region=("macro_region", "first"),
        aquatic_context=("aquatic_context", "first"),
        dist_perm_water_km=("dist_perm_water_km", "first"),
        HDSI_mean=("HDSI", "mean"),
        HDSI_q75=("HDSI", lambda x: x.quantile(0.75)),
        high_HDSI_freq=("HDSI_group", lambda x: np.mean(x == "high")),
        EVI_loss_rate=("EVI_loss", "mean"),
        LAI_loss_rate=("LAI_loss", "mean"),
        Joint_loss_rate=("Joint_loss", "mean")
    )
)

save_table(grid_summary, "08_grid_level_map_summary.csv")


# ============================================================
# 12. Optional boundary
# ============================================================

china_boundary = None
if gpd is not None and os.path.exists(CHINA_BOUNDARY_FP):
    try:
        china_boundary = gpd.read_file(CHINA_BOUNDARY_FP)
        china_boundary = china_boundary.to_crs("EPSG:4326")
        print("Loaded China boundary:", china_boundary.shape)
    except Exception as e:
        print("Boundary load failed:", e)
        china_boundary = None


# ============================================================
# 13. Figures
# ============================================================

# ------------------------------------------------------------
# Fig 1: HDSI risk gradient + map
# ------------------------------------------------------------

fig = plt.figure(figsize=(13.5, 7))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1.05], width_ratios=[1, 1, 1])

axes_top = [fig.add_subplot(gs[0, i]) for i in range(3)]
ax_map = fig.add_subplot(gs[1, :])

for ax, resp, label in zip(axes_top, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = r1[r1["response"] == resp].copy()
    sub["HDSI_group"] = pd.Categorical(sub["HDSI_group"], categories=["low", "mid", "high"], ordered=True)
    sub = sub.sort_values("HDSI_group")

    x = np.arange(len(sub))
    ax.bar(x, sub["risk"], color=RESP_COLORS[resp], width=0.65)
    ax.errorbar(
        x, sub["risk"],
        yerr=np.vstack([sub["risk"] - sub["ci_low"], sub["ci_high"] - sub["risk"]]),
        fmt="none", color="#222222", capsize=3, linewidth=1
    )
    ax.set_xticks(x)
    ax.set_xticklabels(["Low", "Mid", "High"])
    ax.set_ylabel("Loss risk")
    ax.set_title(f"{resp} loss increases with HDSI")
    panel_label(ax, label)
    clean_ax(ax)

# Map: grid-level high-HDSI frequency
if china_boundary is not None:
    china_boundary.boundary.plot(ax=ax_map, linewidth=0.5, color="#555555")

sc = ax_map.scatter(
    grid_summary["lon"],
    grid_summary["lat"],
    c=grid_summary["high_HDSI_freq"],
    s=3,
    cmap="YlOrRd",
    alpha=0.75,
    linewidths=0
)

ax_map.set_title("D. National distribution of high-HDSI exposure frequency")
ax_map.set_xlabel("Longitude")
ax_map.set_ylabel("Latitude")
clean_ax(ax_map)
cb = plt.colorbar(sc, ax=ax_map, fraction=0.025, pad=0.01)
cb.set_label("High-HDSI frequency")

plt.tight_layout()
savefig(fig, "Fig1_HDSI_risk_gradient_and_exposure_map")
plt.show()


# ------------------------------------------------------------
# Fig 2: Distance decay
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 4.6), sharey=False)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    sub = dist_res[(dist_res["response"] == resp) & (dist_res["dist_perm_band"] != ">15")].copy()
    sub["dist_perm_band"] = pd.Categorical(sub["dist_perm_band"], categories=dist_labels[:-1], ordered=True)
    sub = sub.sort_values("dist_perm_band")

    x = sub["dist_mid_km"].values
    y = sub["fragility_increase"].values

    ax.errorbar(
        x, y,
        yerr=np.vstack([y - sub["fragility_ci_low"].values, sub["fragility_ci_high"].values - y]),
        fmt="o",
        color=RESP_COLORS[resp],
        ecolor="#222222",
        elinewidth=1,
        capsize=3,
        markersize=5,
        label="Distance-band estimate"
    )

    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() >= 3:
        coef = np.polyfit(x[ok], y[ok], deg=1)
        xs = np.linspace(np.nanmin(x[ok]), np.nanmax(x[ok]), 100)
        ys = coef[0] * xs + coef[1]
        ax.plot(xs, ys, color="#222222", lw=1.2, ls="--", label="Linear trend")

        cs = corr_stats(x[ok], y[ok])
        txt = f"Spearman r={cs['spearman_r']:.2f}"
        ax.text(0.04, 0.93, txt, transform=ax.transAxes, ha="left", va="top", fontsize=8)

    ax.set_xlabel("Distance to permanent water (km)")
    ax.set_ylabel("Fragility increase\nRisk(high HDSI) − Risk(low HDSI)")
    ax.set_title(f"{resp}: distance-decay of aquatic buffering")
    panel_label(ax, label)
    clean_ax(ax)
    ax.legend(frameon=False, loc="lower right", fontsize=7)

plt.tight_layout()
savefig(fig, "Fig2_distance_decay_fragility_with_trend")
plt.show()


# ------------------------------------------------------------
# Fig 3: Water stability
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 3, figsize=(14, 8.2), sharey="row")

for j, resp in enumerate(["EVI", "LAI", "Joint"]):
    # High-HDSI risk
    ax = axes[0, j]
    sub = stability_res[stability_res["response"] == resp].copy()
    sub["aquatic_context"] = pd.Categorical(sub["aquatic_context"], categories=aquatic_order, ordered=True)
    sub = sub.sort_values("aquatic_context")

    x = np.arange(len(sub))
    colors = [AQUA_COLORS[a] for a in sub["aquatic_context"].astype(str)]

    ax.bar(x, sub["risk_high_HDSI"], color=colors, width=0.65)
    ax.errorbar(
        x, sub["risk_high_HDSI"],
        yerr=np.vstack([sub["risk_high_HDSI"] - sub["risk_high_ci_low"], sub["risk_high_ci_high"] - sub["risk_high_HDSI"]]),
        fmt="none", color="#222222", capsize=3, linewidth=1
    )
    ax.set_xticks(x)
    ax.set_xticklabels([AQUA_LABELS[a] for a in sub["aquatic_context"].astype(str)], rotation=15, ha="right")
    ax.set_title(f"{resp}: high-HDSI loss risk")
    ax.set_ylabel("Loss risk")
    panel_label(ax, ["A", "B", "C"][j])
    clean_ax(ax)

    # Fragility increase
    ax = axes[1, j]
    ax.bar(x, sub["fragility_increase"], color=colors, width=0.65)
    ax.errorbar(
        x, sub["fragility_increase"],
        yerr=np.vstack([sub["fragility_increase"] - sub["fragility_ci_low"], sub["fragility_ci_high"] - sub["fragility_increase"]]),
        fmt="none", color="#222222", capsize=3, linewidth=1
    )
    ax.axhline(0, color="#333333", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([AQUA_LABELS[a] for a in sub["aquatic_context"].astype(str)], rotation=15, ha="right")
    ax.set_title(f"{resp}: risk amplification")
    ax.set_ylabel("Risk(high) − Risk(low)")
    panel_label(ax, ["D", "E", "F"][j])
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig3_water_stability_high_risk_and_fragility")
plt.show()


# ------------------------------------------------------------
# Fig 4: Lightweight mechanism
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))

# A: stress-channel AFB for other_weak - near_permanent
ax = axes[0]
plot = stress_afb[(stress_afb["contrast"] == "other_weak - near_permanent") & (stress_afb["response"] == "Joint")].copy()
stress_order = ["HDSI", "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
plot["stress"] = pd.Categorical(plot["stress"], categories=stress_order, ordered=True)
plot = plot.sort_values("stress")
x = np.arange(len(plot))
colors = [STRESS_COLORS[s] for s in plot["stress"].astype(str)]
ax.bar(x, plot["AFB_simple"], color=colors, width=0.65)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["stress"], rotation=35, ha="right")
ax.set_ylabel("AFB simple\nFragility(other/weak) − Fragility(near permanent)")
ax.set_title("Joint loss: pathway-specific fragility")
panel_label(ax, "A")
clean_ax(ax)

# B: high-HDSI component difference
ax = axes[1]
plot = component_diff[component_diff["contrast"] == "other_weak - near_permanent"].copy()
plot["component"] = pd.Categorical(plot["component"], categories=["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"], ordered=True)
plot = plot.sort_values("component")
x = np.arange(len(plot))
colors = [STRESS_COLORS.get(s, COL["gray"]) for s in plot["component"].astype(str)]
ax.bar(x, plot["difference"], color=colors, width=0.65)
ax.errorbar(
    x, plot["difference"],
    yerr=np.vstack([plot["difference"] - plot["ci_low"], plot["ci_high"] - plot["difference"]]),
    fmt="none", color="#222222", capsize=3, linewidth=1
)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels(plot["component"], rotation=35, ha="right")
ax.set_ylabel("Component difference\nOther/weak − near permanent")
ax.set_title("High-HDSI stress components")
panel_label(ax, "B")
clean_ax(ax)

# C: seasonal propagation
ax = axes[2]
plot = season_res[season_res["delta_variable"] == "delta_LAI"].copy()
plot["aquatic_context"] = pd.Categorical(plot["aquatic_context"], categories=aquatic_order, ordered=True)
plot = plot.sort_values("aquatic_context")
x = np.arange(len(plot))
colors = [AQUA_COLORS[a] for a in plot["aquatic_context"].astype(str)]
ax.bar(x, plot["high_minus_low_delta"], color=colors, width=0.65)
ax.errorbar(
    x, plot["high_minus_low_delta"],
    yerr=np.vstack([plot["high_minus_low_delta"] - plot["ci_low"], plot["ci_high"] - plot["high_minus_low_delta"]]),
    fmt="none", color="#222222", capsize=3, linewidth=1
)
ax.axhline(0, color="#333333", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([AQUA_LABELS[a] for a in plot["aquatic_context"].astype(str)], rotation=15, ha="right")
ax.set_ylabel("ΔLAI response\nHigh HDSI − Low HDSI")
ax.set_title("Seasonal propagation of LAI deterioration")
panel_label(ax, "C")
clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig4_lightweight_mechanism_pathways")
plt.show()


# ------------------------------------------------------------
# Fig 5: Regional boundary conditions
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharex=True)

for ax, resp, label in zip(axes, ["EVI", "LAI", "Joint"], ["A", "B", "C"]):
    plot = regional_afb[
        (regional_afb["response"] == resp) &
        (regional_afb["contrast"] == "other_weak - near_permanent")
    ].copy()

    plot = plot.dropna(subset=["AFB_simple"])
    plot = plot.sort_values("AFB_simple")
    y = np.arange(len(plot))

    ax.scatter(plot["AFB_simple"], y, color=RESP_COLORS[resp], s=45)
    ax.axvline(0, color="#333333", lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(plot["macro_region"])
    ax.set_xlabel("Regional AFB\nFragility(other/weak) − Fragility(near permanent)")
    ax.set_title(f"{resp}: regional boundary conditions")
    panel_label(ax, label)
    clean_ax(ax)

plt.tight_layout()
savefig(fig, "Fig5_regional_boundary_conditions_AFB")
plt.show()


# ------------------------------------------------------------
# Fig 6: National map of aquatic context and Joint loss
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# A aquatic context
ax = axes[0]
if china_boundary is not None:
    china_boundary.boundary.plot(ax=ax, linewidth=0.5, color="#555555")

for aq in aquatic_order:
    sub = grid_summary[grid_summary["aquatic_context"].astype(str) == aq]
    ax.scatter(
        sub["lon"], sub["lat"],
        s=3,
        color=AQUA_COLORS[aq],
        alpha=0.65,
        linewidths=0,
        label=AQUA_LABELS[aq]
    )

ax.set_title("A. Aquatic context")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend(frameon=False, loc="lower left", fontsize=7)
clean_ax(ax)

# B joint loss rate
ax = axes[1]
if china_boundary is not None:
    china_boundary.boundary.plot(ax=ax, linewidth=0.5, color="#555555")

sc = ax.scatter(
    grid_summary["lon"], grid_summary["lat"],
    c=grid_summary["Joint_loss_rate"],
    s=3,
    cmap="Reds",
    alpha=0.75,
    linewidths=0
)
ax.set_title("B. Grid-level Joint loss rate")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
clean_ax(ax)
cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.01)
cb.set_label("Joint loss rate")

plt.tight_layout()
savefig(fig, "Fig6_national_maps_aquatic_context_and_joint_loss")
plt.show()


# ============================================================
# 14. Print outputs
# ============================================================

print("\n==================== TABLES SAVED ====================")
for fn in sorted(os.listdir(TAB_DIR)):
    print(os.path.join(TAB_DIR, fn))

print("\n==================== FIGURES SAVED ====================")
for fn in sorted(os.listdir(FIG_DIR)):
    print(os.path.join(FIG_DIR, fn))

print("\nKey tables to inspect:")
print("01_HDSI_group_loss_risk.csv")
print("02_distance_decay_fragility.csv")
print("02_distance_decay_correlation.csv")
print("03_water_stability_risk_and_fragility.csv")
print("03_aquatic_fragility_buffering_AFB.csv")
print("04_stress_channel_AFB_simple.csv")
print("05_seasonal_propagation_delta.csv")
print("06_high_HDSI_component_difference.csv")
print("07_regional_AFB_simple.csv")
print("08_grid_level_map_summary.csv")

In [ ]:
# ============================================================
# Replot Figure 4 & Figure 5 + 3 supplementary analyses
# Author: ChatGPT (for Baoling)
# Purpose:
#   1) Redraw Figure 4 with clearer mechanism diagnostics
#   2) Redraw Figure 5 with regional forest plots + CI
#   3) Add threshold sensitivity / regional AFB table / sample-size tables
# ============================================================

import os
import glob
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr

# ------------------------------------------------------------
# 0. Style
# ------------------------------------------------------------
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "normal"
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 600
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Nature-like colors
COLORS = {
    "near_permanent": "#3B6FB6",   # blue
    "near_seasonal": "#E69F00",    # orange
    "other_weak": "#6E6E6E",       # grey
    "EVI": "#4E9F50",              # green
    "LAI": "#7B61A8",              # purple
    "Joint": "#B5162F",            # red
    "low_HDSI": "#6AB0DE",
    "high_HDSI": "#D95F5F"
}

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------
BASE = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
AQUA_DIR = os.path.join(BASE, "_aquatic_buffering")
OUT_DIR = os.path.join(AQUA_DIR, "_improved_fig4_fig5")
FIG_DIR = os.path.join(OUT_DIR, "figs")
TAB_DIR = os.path.join(OUT_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)

CANDIDATE_PANEL = [
    os.path.join(AQUA_DIR, "china_aquatic_buffering_panel_v2.parquet"),
    os.path.join(AQUA_DIR, "china_aquatic_buffering_panel.parquet"),
]

def first_existing(paths):
    for p in paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"None of these files exists:\n" + "\n".join(paths))

PANEL_FP = first_existing(CANDIDATE_PANEL)
print("Using panel:", PANEL_FP)

# ------------------------------------------------------------
# 2. Load data
# ------------------------------------------------------------
df = pd.read_parquet(PANEL_FP)
print("Raw shape:", df.shape)

# ------------------------------------------------------------
# 3. Helper functions
# ------------------------------------------------------------
def pick_first_existing_column(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"Cannot find any of these columns: {candidates}")
    return None

def bootstrap_ci(values, func=np.mean, n_boot=400, alpha=0.05, random_state=42):
    """
    Bootstrap CI for a 1D array.
    """
    values = np.asarray(values)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    est = func(values)
    boots = []
    n = len(values)
    for _ in range(n_boot):
        samp = rng.choice(values, size=n, replace=True)
        boots.append(func(samp))
    lo = np.percentile(boots, 100 * alpha / 2)
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return est, lo, hi

def bootstrap_diff(a, b, func=np.mean, n_boot=400, alpha=0.05, random_state=42):
    """
    Bootstrap CI for difference func(a) - func(b)
    """
    a = np.asarray(a)
    b = np.asarray(b)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(random_state)
    est = func(a) - func(b)
    boots = []
    na, nb = len(a), len(b)
    for _ in range(n_boot):
        sa = rng.choice(a, size=na, replace=True)
        sb = rng.choice(b, size=nb, replace=True)
        boots.append(func(sa) - func(sb))
    lo = np.percentile(boots, 100 * alpha / 2)
    hi = np.percentile(boots, 100 * (1 - alpha / 2))
    return est, lo, hi

def prop_mean(x):
    return np.mean(np.asarray(x).astype(float))

def ensure_binary(series):
    s = pd.Series(series).astype(float)
    return s

def safe_sample(g, n=2500, random_state=42):
    if len(g) <= n:
        return g
    return g.sample(n=n, random_state=random_state)

# ------------------------------------------------------------
# 4. Required columns
# ------------------------------------------------------------
grid_col = pick_first_existing_column(df, ["grid_id"])
year_col = pick_first_existing_column(df, ["year"])
region_col = pick_first_existing_column(
    df,
    ["macro_region", "region_group", "region_name", "region_label", "region_id"]
)

# main columns
need_cols = [
    "HDSI",
    "dist_perm_water_km",
    "dist_seasonal_water_km",
    "late_summer_EVI_z",
    "late_summer_LAI_z",
    "spring_EVI_z",
    "spring_LAI_z",
    "summer_SM_rootproxy_z",
    "summer_Precip_z",
    "summer_LST_Day_z",
    "summer_VPD_z",
    "summer_Tair_z"
]
missing = [c for c in need_cols if c not in df.columns]
if missing:
    raise KeyError("Missing columns:\n" + "\n".join(missing))

# ------------------------------------------------------------
# 5. Core derived variables
# ------------------------------------------------------------
# aquatic context
if "aquatic_context" not in df.columns:
    cond_perm = df["dist_perm_water_km"] < 1.0
    cond_seas = (~cond_perm) & (df["dist_seasonal_water_km"] < 1.0)
    df["aquatic_context"] = np.select(
        [cond_perm, cond_seas],
        ["near_permanent", "near_seasonal"],
        default="other_weak"
    )
else:
    # normalize naming
    mapping = {
        "near permanent": "near_permanent",
        "near_permanent": "near_permanent",
        "near_perm": "near_permanent",
        "near seasonal": "near_seasonal",
        "near_seasonal": "near_seasonal",
        "seasonal": "near_seasonal",
        "other / weak": "other_weak",
        "other_weak": "other_weak",
        "far_or_weak": "other_weak",
        "far_or_no_water": "other_weak",
        "other": "other_weak",
    }
    df["aquatic_context"] = df["aquatic_context"].astype(str).map(lambda x: mapping.get(x, x))

# HDSI quantile groups
q25 = df["HDSI"].quantile(0.25)
q75 = df["HDSI"].quantile(0.75)
df["HDSI_group"] = pd.cut(
    df["HDSI"],
    bins=[-np.inf, q25, q75, np.inf],
    labels=["low_HDSI", "mid_HDSI", "high_HDSI"]
)

# events at -0.5
df["EVI_neg_z05"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_neg_z05"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
df["Joint_neg_z05"] = ((df["EVI_neg_z05"] == 1) & (df["LAI_neg_z05"] == 1)).astype(int)

# seasonal deterioration
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# stress channels: unified sign so larger = worse
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]   # lower SM => larger deficit
df["Precip_deficit"] = -df["summer_Precip_z"]     # lower precip => larger deficit
df["LST_Day"] = df["summer_LST_Day_z"]            # higher is worse
df["VPD"] = df["summer_VPD_z"]                    # higher is worse
df["Tair"] = df["summer_Tair_z"]                  # higher is worse

# keep usable rows
core_cols = [
    grid_col, year_col, region_col, "aquatic_context", "HDSI", "HDSI_group",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "late_summer_EVI_z", "late_summer_LAI_z", "spring_EVI_z", "spring_LAI_z",
    "delta_EVI", "delta_LAI",
    "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "EVI_neg_z05", "LAI_neg_z05", "Joint_neg_z05"
]
d = df[core_cols].copy()
d = d.replace([np.inf, -np.inf], np.nan)

# keep three major groups only
d = d[d["aquatic_context"].isin(["near_permanent", "near_seasonal", "other_weak"])].copy()

print("Usable shape:", d.shape)
print(d["aquatic_context"].value_counts(dropna=False))

# ------------------------------------------------------------
# 6. Sample-size supplementary table
# ------------------------------------------------------------
perm_bins = [0, 0.25, 0.5, 1, 2, 5, 10, np.inf]
perm_labels = ["0-0.25", "0.25-0.5", "0.5-1", "1-2", "2-5", "5-10", ">10"]
d["perm_band"] = pd.cut(d["dist_perm_water_km"], bins=perm_bins, labels=perm_labels, include_lowest=True)

tab_sample = (
    d.groupby(["aquatic_context", "perm_band"])
     .agg(
         n_samples=(grid_col, "size"),
         n_grids=(grid_col, "nunique"),
         mean_HDSI=("HDSI", "mean"),
         mean_joint_risk=("Joint_neg_z05", "mean")
     )
     .reset_index()
)
tab_sample.to_csv(os.path.join(TAB_DIR, "sample_size_by_aquatic_context_and_perm_band.csv"), index=False)

# ------------------------------------------------------------
# 7. Threshold sensitivity supplementary table / figure
# ------------------------------------------------------------
def calc_fragility(sub, event_col):
    low = sub.loc[sub["HDSI_group"] == "low_HDSI", event_col].dropna()
    high = sub.loc[sub["HDSI_group"] == "high_HDSI", event_col].dropna()
    if len(low) == 0 or len(high) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan
    est_low, lo_low, hi_low = bootstrap_ci(low, func=prop_mean)
    est_high, lo_high, hi_high = bootstrap_ci(high, func=prop_mean)
    frag = est_high - est_low
    return est_low, lo_low, hi_low, est_high, lo_high, hi_high, frag

thr_list = [-0.25, -0.5, -0.75, -1.0]
sens_rows = []

for thr in thr_list:
    tmp = d.copy()
    tmp["EVI_evt"] = (tmp["late_summer_EVI_z"] < thr).astype(int)
    tmp["LAI_evt"] = (tmp["late_summer_LAI_z"] < thr).astype(int)
    tmp["Joint_evt"] = ((tmp["EVI_evt"] == 1) & (tmp["LAI_evt"] == 1)).astype(int)

    for metric_name, evt in [("EVI", "EVI_evt"), ("LAI", "LAI_evt"), ("Joint", "Joint_evt")]:
        # near permanent
        g_perm = tmp[tmp["aquatic_context"] == "near_permanent"]
        g_other = tmp[tmp["aquatic_context"] == "other_weak"]

        lowp, _, _, highp, _, _, frag_perm = calc_fragility(g_perm, evt)
        lowo, _, _, higho, _, _, frag_other = calc_fragility(g_other, evt)

        # bootstrap AFB
        low_perm = g_perm.loc[g_perm["HDSI_group"] == "low_HDSI", evt].dropna().values
        high_perm = g_perm.loc[g_perm["HDSI_group"] == "high_HDSI", evt].dropna().values
        low_other = g_other.loc[g_other["HDSI_group"] == "low_HDSI", evt].dropna().values
        high_other = g_other.loc[g_other["HDSI_group"] == "high_HDSI", evt].dropna().values

        rng = np.random.default_rng(42)
        boots = []
        n_boot = 400
        if min(len(low_perm), len(high_perm), len(low_other), len(high_other)) > 0:
            for _ in range(n_boot):
                slp = rng.choice(low_perm, size=len(low_perm), replace=True)
                shp = rng.choice(high_perm, size=len(high_perm), replace=True)
                slo = rng.choice(low_other, size=len(low_other), replace=True)
                sho = rng.choice(high_other, size=len(high_other), replace=True)
                frag_p = shp.mean() - slp.mean()
                frag_o = sho.mean() - slo.mean()
                boots.append(frag_o - frag_p)
            afb_est = np.mean(boots)
            afb_lo = np.percentile(boots, 2.5)
            afb_hi = np.percentile(boots, 97.5)
        else:
            afb_est = afb_lo = afb_hi = np.nan

        sens_rows.append({
            "threshold": thr,
            "metric": metric_name,
            "fragility_near_permanent": frag_perm,
            "fragility_other_weak": frag_other,
            "AFB_other_minus_perm": afb_est,
            "AFB_ci_low": afb_lo,
            "AFB_ci_high": afb_hi,
            "n_perm": len(g_perm),
            "n_other": len(g_other)
        })

sens = pd.DataFrame(sens_rows)
sens.to_csv(os.path.join(TAB_DIR, "threshold_sensitivity_AFB.csv"), index=False)

# figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, metric, color in zip(axes, ["EVI", "LAI", "Joint"], [COLORS["EVI"], COLORS["LAI"], COLORS["Joint"]]):
    ss = sens[sens["metric"] == metric].copy()
    ax.errorbar(
        ss["threshold"], ss["AFB_other_minus_perm"],
        yerr=[ss["AFB_other_minus_perm"] - ss["AFB_ci_low"], ss["AFB_ci_high"] - ss["AFB_other_minus_perm"]],
        fmt="o-", color=color, lw=2, capsize=4
    )
    ax.axhline(0, color="k", lw=1, ls="--")
    ax.set_title(metric)
    ax.set_xlabel("Loss threshold")
axes[0].set_ylabel("AFB = Fragility(other/weak) - Fragility(near permanent)")
fig.suptitle("Threshold sensitivity of aquatic buffering effect", y=1.03, fontsize=16, fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Supp_threshold_sensitivity_AFB.png"), bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "Supp_threshold_sensitivity_AFB.pdf"), bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 8. Figure 4: Improved mechanism diagnostics
# ------------------------------------------------------------
# Use only high/low HDSI for the mechanism story
d_hilo = d[d["HDSI_group"].isin(["low_HDSI", "high_HDSI"])].copy()

# -------------------------
# 8.1 Panel A: high-HDSI raw distributions for stress components
# -------------------------
comp_map = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature"
}
comp_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
ctx_order = ["near_permanent", "near_seasonal", "other_weak"]

high = d_hilo[d_hilo["HDSI_group"] == "high_HDSI"].copy()

long_high = []
for c in comp_order:
    tmp = high[[c, "aquatic_context"]].copy()
    tmp = tmp.rename(columns={c: "value"})
    tmp["component"] = c
    tmp["component_label"] = comp_map[c]
    long_high.append(tmp)
long_high = pd.concat(long_high, axis=0, ignore_index=True).dropna()

# sample for plotting only
long_high_plot = (
    long_high.groupby(["component", "aquatic_context"], group_keys=False)
             .apply(lambda g: safe_sample(g, n=3000, random_state=42))
             .reset_index(drop=True)
)

# stats for panel B
rows_b = []
for c in comp_order:
    sub = high[[c, "aquatic_context"]].dropna().copy()
    x_perm = sub.loc[sub["aquatic_context"] == "near_permanent", c].values
    x_seas = sub.loc[sub["aquatic_context"] == "near_seasonal", c].values
    x_weak = sub.loc[sub["aquatic_context"] == "other_weak", c].values

    est1, lo1, hi1 = bootstrap_diff(x_seas, x_perm, func=np.mean, n_boot=500)
    est2, lo2, hi2 = bootstrap_diff(x_weak, x_perm, func=np.mean, n_boot=500)

    rows_b.append({
        "component": c,
        "component_label": comp_map[c],
        "contrast": "near_seasonal - near_permanent",
        "difference": est1,
        "ci_low": lo1,
        "ci_high": hi1
    })
    rows_b.append({
        "component": c,
        "component_label": comp_map[c],
        "contrast": "other_weak - near_permanent",
        "difference": est2,
        "ci_low": lo2,
        "ci_high": hi2
    })
tab_comp_diff = pd.DataFrame(rows_b)
tab_comp_diff.to_csv(os.path.join(TAB_DIR, "Fig4_component_difference_high_HDSI.csv"), index=False)

# -------------------------
# 8.2 Panel C: delta_LAI distributions in low vs high HDSI
# -------------------------
dlai = d_hilo[["delta_LAI", "aquatic_context", "HDSI_group"]].dropna().copy()
dlai = dlai[dlai["HDSI_group"].isin(["low_HDSI", "high_HDSI"])]

# plot sample only
dlai_plot = (
    dlai.groupby(["aquatic_context", "HDSI_group"], group_keys=False)
        .apply(lambda g: safe_sample(g, n=2500, random_state=42))
        .reset_index(drop=True)
)

# -------------------------
# 8.3 Panel D: risk amplification under low vs high HDSI
# -------------------------
risk_rows = []
for evt, evt_label in [("LAI_neg_z05", "LAI loss"), ("Joint_neg_z05", "Joint loss")]:
    for ctx in ctx_order:
        sub = d_hilo[d_hilo["aquatic_context"] == ctx].copy()
        low_vals = sub.loc[sub["HDSI_group"] == "low_HDSI", evt].dropna().values
        high_vals = sub.loc[sub["HDSI_group"] == "high_HDSI", evt].dropna().values

        est_low, lo_low, hi_low = bootstrap_ci(low_vals, func=prop_mean, n_boot=500)
        est_high, lo_high, hi_high = bootstrap_ci(high_vals, func=prop_mean, n_boot=500)
        amp, amp_lo, amp_hi = bootstrap_diff(high_vals, low_vals, func=prop_mean, n_boot=500)

        risk_rows.append({
            "event": evt,
            "event_label": evt_label,
            "aquatic_context": ctx,
            "p_low": est_low,
            "p_low_ci_low": lo_low,
            "p_low_ci_high": hi_low,
            "p_high": est_high,
            "p_high_ci_low": lo_high,
            "p_high_ci_high": hi_high,
            "amplification": amp,
            "amp_ci_low": amp_lo,
            "amp_ci_high": amp_hi,
            "n_low": len(low_vals),
            "n_high": len(high_vals)
        })
tab_risk_amp = pd.DataFrame(risk_rows)
tab_risk_amp.to_csv(os.path.join(TAB_DIR, "Fig4_risk_amplification.csv"), index=False)

# -------------------------
# 8.4 Draw Figure 4
# -------------------------
fig = plt.figure(figsize=(20, 14))
gs = fig.add_gridspec(2, 2, height_ratios=[1.15, 1], width_ratios=[1.1, 0.9], hspace=0.28, wspace=0.18)

# Panel A
axA = fig.add_subplot(gs[0, 0])
sns.violinplot(
    data=long_high_plot,
    x="value", y="component_label",
    hue="aquatic_context",
    order=[comp_map[c] for c in comp_order],
    hue_order=ctx_order,
    palette=[COLORS[c] for c in ctx_order],
    orient="h",
    cut=0,
    inner=None,
    linewidth=0.8,
    ax=axA
)
# overlay medians
med = (
    long_high.groupby(["component_label", "aquatic_context"])["value"]
             .median()
             .reset_index()
)
sns.pointplot(
    data=med,
    x="value", y="component_label",
    hue="aquatic_context",
    order=[comp_map[c] for c in comp_order],
    hue_order=ctx_order,
    palette=[COLORS[c] for c in ctx_order],
    orient="h",
    dodge=0.45,
    join=False,
    markers="d",
    scale=0.8,
    errorbar=None,
    ax=axA
)
handles, labels = axA.get_legend_handles_labels()
axA.legend(handles[:3], ["Near permanent", "Near seasonal", "Other / weak"], title="", loc="lower right", frameon=True)
axA.set_title("A. High-HDSI distributions of individual stress components", loc="left", fontsize=16)
axA.set_xlabel("Standardized anomaly (larger = worse)")
axA.set_ylabel("")
axA.axvline(0, color="k", lw=1, ls="--", alpha=0.7)

# Panel B
axB = fig.add_subplot(gs[0, 1])
plot_b = tab_comp_diff.copy()
plot_b["component_label"] = pd.Categorical(plot_b["component_label"], categories=[comp_map[c] for c in comp_order], ordered=True)
plot_b = plot_b.sort_values(["component_label", "contrast"])

y_positions = []
y_labels = []
y = 0
for comp in [comp_map[c] for c in comp_order]:
    sub = plot_b[plot_b["component_label"] == comp]
    for _, row in sub.iterrows():
        y_positions.append(y)
        y_labels.append(comp if row["contrast"] == "near_seasonal - near_permanent" else "")
        color = COLORS["near_seasonal"] if "near_seasonal" in row["contrast"] else COLORS["other_weak"]
        axB.errorbar(
            row["difference"], y,
            xerr=[[row["difference"] - row["ci_low"]], [row["ci_high"] - row["difference"]]],
            fmt="o", color=color, ecolor=color, capsize=4, lw=2, ms=8
        )
        y += 1
    y += 0.6

axB.axvline(0, color="k", lw=1, ls="--")
axB.set_yticks(y_positions)
axB.set_yticklabels(y_labels)
axB.set_title("B. Contrast against near-permanent water (bootstrap CI)", loc="left", fontsize=16)
axB.set_xlabel("Difference in component severity")
axB.set_ylabel("")
# custom legend
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color=COLORS["near_seasonal"], label='Near seasonal - Near permanent', lw=2),
    Line2D([0], [0], marker='o', color=COLORS["other_weak"], label='Other / weak - Near permanent', lw=2)
]
axB.legend(handles=legend_elements, loc="lower right", frameon=True)

# Panel C
axC = fig.add_subplot(gs[1, 0])
sns.violinplot(
    data=dlai_plot,
    x="aquatic_context", y="delta_LAI",
    hue="HDSI_group",
    order=ctx_order,
    hue_order=["low_HDSI", "high_HDSI"],
    palette=[COLORS["low_HDSI"], COLORS["high_HDSI"]],
    cut=0,
    inner=None,
    linewidth=0.8,
    ax=axC
)
# overlay box
sns.boxplot(
    data=dlai_plot,
    x="aquatic_context", y="delta_LAI",
    hue="HDSI_group",
    order=ctx_order,
    hue_order=["low_HDSI", "high_HDSI"],
    palette=[COLORS["low_HDSI"], COLORS["high_HDSI"]],
    width=0.22,
    showcaps=True,
    boxprops={"facecolor": "white", "alpha": 0.7},
    medianprops={"color": "black"},
    whiskerprops={"linewidth": 1},
    flierprops={"marker": "", "markersize": 0},
    ax=axC
)
handles, labels = axC.get_legend_handles_labels()
axC.legend(handles[:2], ["Low HDSI", "High HDSI"], title="", loc="lower left", frameon=True)
axC.axhline(0, color="k", lw=1, ls="--", alpha=0.7)
axC.set_title("C. Seasonal LAI deterioration under low vs high HDSI", loc="left", fontsize=16)
axC.set_xlabel("")
axC.set_ylabel("delta_LAI = late summer LAI_z - spring LAI_z")
axC.set_xticklabels(["Near permanent", "Near seasonal", "Other / weak"], rotation=0)

# Panel D
axD = fig.add_subplot(gs[1, 1])
# dumbbell plot for risk amplification
plot_d = tab_risk_amp.copy()
plot_d["event_label"] = pd.Categorical(plot_d["event_label"], categories=["LAI loss", "Joint loss"], ordered=True)
plot_d = plot_d.sort_values(["event_label", "aquatic_context"])

ypos = []
ytlab = []
yy = 0
for evt in ["LAI loss", "Joint loss"]:
    sub_evt = plot_d[plot_d["event_label"] == evt]
    for ctx in ctx_order:
        row = sub_evt[sub_evt["aquatic_context"] == ctx].iloc[0]
        axD.plot([row["p_low"], row["p_high"]], [yy, yy], color=COLORS[ctx], lw=3, alpha=0.85)
        axD.errorbar(row["p_low"], yy, xerr=[[row["p_low"] - row["p_low_ci_low"]], [row["p_low_ci_high"] - row["p_low"]]],
                     fmt="o", color=COLORS["low_HDSI"], ecolor=COLORS["low_HDSI"], capsize=3, ms=7)
        axD.errorbar(row["p_high"], yy, xerr=[[row["p_high"] - row["p_high_ci_low"]], [row["p_high_ci_high"] - row["p_high"]]],
                     fmt="s", color=COLORS["high_HDSI"], ecolor=COLORS["high_HDSI"], capsize=3, ms=7)
        ypos.append(yy)
        ytlab.append(f"{evt} | {ctx.replace('_', ' ')}")
        yy += 1
    yy += 0.8

axD.set_yticks(ypos)
axD.set_yticklabels(ytlab, fontsize=11)
axD.set_xlabel("Adjusted loss probability")
axD.set_title("D. Propagation from HDSI to vegetation loss", loc="left", fontsize=16)
axD.axvline(0, color="k", lw=0.8)
legend_elems = [
    Line2D([0], [0], marker='o', color=COLORS["low_HDSI"], label='Low HDSI', lw=0, markersize=8),
    Line2D([0], [0], marker='s', color=COLORS["high_HDSI"], label='High HDSI', lw=0, markersize=8),
]
axD.legend(handles=legend_elems, loc="lower right", frameon=True)

fig.suptitle("Figure 4. Clearer mechanism diagnostics of aquatic buffering", fontsize=20, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure4_mechanism_improved.png"), bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "Figure4_mechanism_improved.pdf"), bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 9. Figure 5: Regional boundary conditions (forest plots + CI)
# ------------------------------------------------------------
regions = (
    d[region_col]
    .dropna()
    .astype(str)
    .value_counts()
    .index
    .tolist()
)

# if too many region ids, keep regions with sufficient grids
region_grid_counts = d.groupby(region_col)[grid_col].nunique().sort_values(ascending=False)
regions_keep = region_grid_counts[region_grid_counts >= 100].index.astype(str).tolist()

d_reg = d.copy()
d_reg[region_col] = d_reg[region_col].astype(str)
d_reg = d_reg[d_reg[region_col].isin(regions_keep)].copy()

def compute_region_afb(df_region, event_col, n_boot=400):
    """
    AFB = fragility(other/weak) - fragility(near_permanent)
    fragility = P(high HDSI loss) - P(low HDSI loss)
    """
    out = None
    ctx_perm = df_region[df_region["aquatic_context"] == "near_permanent"]
    ctx_other = df_region[df_region["aquatic_context"] == "other_weak"]

    lp = ctx_perm.loc[ctx_perm["HDSI_group"] == "low_HDSI", event_col].dropna().values
    hp = ctx_perm.loc[ctx_perm["HDSI_group"] == "high_HDSI", event_col].dropna().values
    lo = ctx_other.loc[ctx_other["HDSI_group"] == "low_HDSI", event_col].dropna().values
    ho = ctx_other.loc[ctx_other["HDSI_group"] == "high_HDSI", event_col].dropna().values

    if min(len(lp), len(hp), len(lo), len(ho)) < 20:
        return np.nan, np.nan, np.nan, len(lp), len(hp), len(lo), len(ho)

    rng = np.random.default_rng(42)
    boots = []
    for _ in range(n_boot):
        slp = rng.choice(lp, size=len(lp), replace=True)
        shp = rng.choice(hp, size=len(hp), replace=True)
        slo = rng.choice(lo, size=len(lo), replace=True)
        sho = rng.choice(ho, size=len(ho), replace=True)

        frag_perm = shp.mean() - slp.mean()
        frag_other = sho.mean() - slo.mean()
        boots.append(frag_other - frag_perm)

    est = np.mean(boots)
    lo_ci = np.percentile(boots, 2.5)
    hi_ci = np.percentile(boots, 97.5)
    return est, lo_ci, hi_ci, len(lp), len(hp), len(lo), len(ho)

reg_rows = []
for reg in regions_keep:
    sub = d_reg[d_reg[region_col].astype(str) == reg].copy()
    for evt, metric in [("EVI_neg_z05", "EVI"), ("LAI_neg_z05", "LAI"), ("Joint_neg_z05", "Joint")]:
        est, lo_ci, hi_ci, nlp, nhp, nlo, nho = compute_region_afb(sub, evt, n_boot=500)
        reg_rows.append({
            "region": reg,
            "metric": metric,
            "AFB": est,
            "ci_low": lo_ci,
            "ci_high": hi_ci,
            "n_low_perm": nlp,
            "n_high_perm": nhp,
            "n_low_other": nlo,
            "n_high_other": nho,
            "n_grids": sub[grid_col].nunique(),
            "n_samples": len(sub)
        })

tab_region_afb = pd.DataFrame(reg_rows)
tab_region_afb.to_csv(os.path.join(TAB_DIR, "Figure5_regional_AFB_CI.csv"), index=False)

# draw Figure 5
fig, axes = plt.subplots(1, 3, figsize=(19, 8), sharex=False)
for ax, metric, color in zip(axes, ["EVI", "LAI", "Joint"], [COLORS["EVI"], COLORS["LAI"], COLORS["Joint"]]):
    sub = tab_region_afb[tab_region_afb["metric"] == metric].copy()
    sub = sub.sort_values("AFB")
    y = np.arange(len(sub))

    ax.errorbar(
        sub["AFB"], y,
        xerr=[sub["AFB"] - sub["ci_low"], sub["ci_high"] - sub["AFB"]],
        fmt="o", color=color, ecolor=color, capsize=4, lw=2, ms=8
    )
    ax.axvline(0, color="k", lw=1, ls="--")
    ax.set_yticks(y)
    ax.set_yticklabels(sub["region"])
    ax.set_title(metric, fontsize=16)
    ax.set_xlabel("AFB = Fragility(other/weak) - Fragility(near permanent)")
    # annotate sample size
    for xi, yi, ng in zip(sub["AFB"], y, sub["n_grids"]):
        if np.isfinite(xi):
            ax.text(xi, yi + 0.18, f"n={ng}", fontsize=9, color="0.35")

axes[0].set_ylabel("Region")
fig.suptitle("Figure 5. Regional boundary conditions of aquatic buffering", fontsize=20, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "Figure5_regional_boundary_conditions_improved.png"), bbox_inches="tight")
fig.savefig(os.path.join(FIG_DIR, "Figure5_regional_boundary_conditions_improved.pdf"), bbox_inches="tight")
plt.close(fig)

# ------------------------------------------------------------
# 10. Extra table: detailed component summary under high HDSI
# ------------------------------------------------------------
comp_summary_rows = []
for c in comp_order:
    for ctx in ctx_order:
        x = high.loc[high["aquatic_context"] == ctx, c].dropna().values
        est, lo_ci, hi_ci = bootstrap_ci(x, func=np.mean, n_boot=500)
        med = np.median(x) if len(x) > 0 else np.nan
        q1 = np.percentile(x, 25) if len(x) > 0 else np.nan
        q3 = np.percentile(x, 75) if len(x) > 0 else np.nan
        comp_summary_rows.append({
            "component": c,
            "component_label": comp_map[c],
            "aquatic_context": ctx,
            "mean": est,
            "ci_low": lo_ci,
            "ci_high": hi_ci,
            "median": med,
            "q25": q1,
            "q75": q3,
            "n": len(x),
            "grids": high.loc[high["aquatic_context"] == ctx, grid_col].nunique()
        })
tab_comp_summary = pd.DataFrame(comp_summary_rows)
tab_comp_summary.to_csv(os.path.join(TAB_DIR, "component_summary_under_high_HDSI.csv"), index=False)

# ------------------------------------------------------------
# 11. Done
# ------------------------------------------------------------
print("\nDone.")
print("Figures saved to:", FIG_DIR)
print("Tables saved to:", TAB_DIR)

In [ ]:
# ============================================================
# Improved direct plotting only
# - Legends outside
# - Unified font and figure size
# - Better panel layout
# - No files saved
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SNS = True
except Exception:
    HAS_SNS = False

# ============================================================
# 0. Load data
# ============================================================

PANEL_FP = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/_aquatic_buffering/china_aquatic_buffering_panel_v2.parquet"

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

# ============================================================
# 1. Global figure style
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COL = {
    "near_permanent": "#3B6EA8",
    "near_seasonal": "#E69F00",
    "other_weak": "#666666",
    "EVI": "#4E9A51",
    "LAI": "#7E62A3",
    "Joint": "#B2182B",
    "low": "#72A9D5",
    "high": "#D95F5F",
    "dark": "#222222",
    "gray": "#666666",
}

ctx_order = ["near_permanent", "near_seasonal", "other_weak"]
ctx_label = {
    "near_permanent": "Near permanent",
    "near_seasonal": "Near seasonal",
    "other_weak": "Other / weak",
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.16, linewidth=0.45)

def panel_label(ax, lab, x=-0.08, y=1.04):
    ax.text(
        x, y, lab,
        transform=ax.transAxes,
        fontsize=13,
        fontweight="bold",
        ha="left",
        va="top"
    )

def boot_mean_ci(x, n_boot=400, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    if len(x) <= 2:
        return m, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = [
        rng.choice(x, size=len(x), replace=True).mean()
        for _ in range(n_boot)
    ]
    return m, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def boot_diff_ci(a, b, n_boot=400, seed=42):
    # mean(a) - mean(b)
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    d0 = a.mean() - b.mean()
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots.append(aa.mean() - bb.mean())
    return d0, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def sample_for_plot(x, n=2500, seed=42):
    if len(x) <= n:
        return x
    return x.sample(n=n, random_state=seed)

# ============================================================
# 2. Prepare variables
# ============================================================

num_cols = [
    "lon", "lat", "year", "elevation", "slope",
    "HDSI",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_SM_rootproxy_z", "summer_Precip_z",
    "summer_LST_Day_z", "summer_VPD_z", "summer_Tair_z"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )

# Aquatic context
df["aquatic_context"] = "other_weak"
df.loc[df["dist_perm_water_km"] < 1.0, "aquatic_context"] = "near_permanent"
df.loc[
    (df["dist_perm_water_km"] >= 1.0) &
    (df["dist_seasonal_water_km"] < 1.0),
    "aquatic_context"
] = "near_seasonal"

# HDSI group
q25 = df["HDSI"].quantile(0.25)
q75 = df["HDSI"].quantile(0.75)

df["HDSI_group"] = "mid"
df.loc[df["HDSI"] <= q25, "HDSI_group"] = "low"
df.loc[df["HDSI"] >= q75, "HDSI_group"] = "high"

# Loss events
df["EVI_loss"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_loss"] = (df["late_summer_LAI_z"] < -0.5).astype(int)

# Important: Joint loss = EVI OR LAI
df["Joint_loss"] = ((df["EVI_loss"] == 1) | (df["LAI_loss"] == 1)).astype(int)

# Seasonal deterioration
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# Stress components; larger = worse
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

# Macro region fallback
def assign_macro_region(row):
    lon = row.get("lon", np.nan)
    lat = row.get("lat", np.nan)
    elev = row.get("elevation", np.nan)

    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

required = [
    "HDSI", "HDSI_group", "aquatic_context",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "spring_EVI_z", "spring_LAI_z",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "EVI_loss", "LAI_loss", "Joint_loss", "delta_LAI",
    "macro_region"
]

d = df.dropna(subset=required).copy()
d = d[d["aquatic_context"].isin(ctx_order)].copy()

print("Valid sample:", d.shape)
display(d["aquatic_context"].value_counts())

# ============================================================
# 3. Supplementary sample-size table
# ============================================================

dist_bins = [0, 0.25, 0.5, 1, 2, 3, 5, 7.5, 10, 15, np.inf]
dist_labels = ["0-0.25", "0.25-0.5", "0.5-1", "1-2", "2-3", "3-5", "5-7.5", "7.5-10", "10-15", ">15"]

d["dist_perm_band"] = pd.cut(
    d["dist_perm_water_km"],
    bins=dist_bins,
    labels=dist_labels,
    include_lowest=True,
    right=False
)

sample_table = (
    d.groupby(["aquatic_context", "dist_perm_band"], observed=True)
    .agg(
        n_samples=("HDSI", "size"),
        mean_HDSI=("HDSI", "mean"),
        EVI_loss_rate=("EVI_loss", "mean"),
        LAI_loss_rate=("LAI_loss", "mean"),
        Joint_loss_rate=("Joint_loss", "mean"),
    )
    .reset_index()
)

display(sample_table)

# ============================================================
# 4. Improved Figure 4
# ============================================================

comp_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
comp_label = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature",
}

high = d[d["HDSI_group"] == "high"].copy()

# ------------------------------------------------------------
# 4.1 Panel A: high-HDSI distributions of stress components
# ------------------------------------------------------------

long_list = []
for comp in comp_order:
    tmp = high[["aquatic_context", comp]].dropna().copy()
    tmp = tmp.rename(columns={comp: "value"})
    tmp["component"] = comp
    tmp["component_label"] = comp_label[comp]
    long_list.append(tmp)

long_high = pd.concat(long_list, axis=0, ignore_index=True)

long_high_plot = (
    long_high.groupby(["component", "aquatic_context"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4.2 Panel B: component differences
# ------------------------------------------------------------

component_diff_rows = []

for comp in comp_order:
    near = high.loc[high["aquatic_context"] == "near_permanent", comp].dropna().values

    for group in ["near_seasonal", "other_weak"]:
        vals = high.loc[high["aquatic_context"] == group, comp].dropna().values
        est, lo, hi = boot_diff_ci(vals, near, n_boot=500)

        component_diff_rows.append({
            "component": comp,
            "component_label": comp_label[comp],
            "group": group,
            "contrast": f"{ctx_label[group]} − Near permanent",
            "difference": est,
            "ci_low": lo,
            "ci_high": hi,
            "n_group": len(vals),
            "n_near": len(near)
        })

component_diff = pd.DataFrame(component_diff_rows)
display(component_diff)

# ------------------------------------------------------------
# 4.3 Panel C: delta LAI distribution
# ------------------------------------------------------------

dlai_plot = (
    d[d["HDSI_group"].isin(["low", "high"])]
    [["aquatic_context", "HDSI_group", "delta_LAI"]]
    .dropna()
    .groupby(["aquatic_context", "HDSI_group"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4.4 Panel D: risk propagation
# ------------------------------------------------------------

risk_rows = []

for event_col, event_name in [
    ("LAI_loss", "LAI loss"),
    ("Joint_loss", "Joint loss")
]:
    for ctx in ctx_order:
        sub = d[d["aquatic_context"] == ctx]

        low_vals = sub.loc[sub["HDSI_group"] == "low", event_col].values
        high_vals = sub.loc[sub["HDSI_group"] == "high", event_col].values

        low_m, low_lo, low_hi = boot_mean_ci(low_vals, n_boot=500)
        high_m, high_lo, high_hi = boot_mean_ci(high_vals, n_boot=500)
        amp, amp_lo, amp_hi = boot_diff_ci(high_vals, low_vals, n_boot=500)

        risk_rows.append({
            "event": event_name,
            "aquatic_context": ctx,
            "low_risk": low_m,
            "low_ci_low": low_lo,
            "low_ci_high": low_hi,
            "high_risk": high_m,
            "high_ci_low": high_lo,
            "high_ci_high": high_hi,
            "amplification": amp,
            "amp_ci_low": amp_lo,
            "amp_ci_high": amp_hi,
            "n_low": len(low_vals),
            "n_high": len(high_vals)
        })

risk_prop = pd.DataFrame(risk_rows)
display(risk_prop)

# ------------------------------------------------------------
# 4.5 Draw Figure 4
# ------------------------------------------------------------

fig = plt.figure(figsize=(17.5, 11.5), constrained_layout=False)
gs = fig.add_gridspec(
    2, 2,
    height_ratios=[1.12, 1.0],
    width_ratios=[1.10, 0.90],
    hspace=0.34,
    wspace=0.34
)

# ---------- Panel A ----------
axA = fig.add_subplot(gs[0, 0])

if HAS_SNS:
    sns.violinplot(
        data=long_high_plot,
        x="value",
        y="component_label",
        hue="aquatic_context",
        order=[comp_label[c] for c in comp_order],
        hue_order=ctx_order,
        palette=[COL[c] for c in ctx_order],
        cut=0,
        inner="quartile",
        linewidth=0.75,
        scale="width",
        ax=axA
    )

    # Put legend outside
    handles, labels = axA.get_legend_handles_labels()
    axA.legend(
        handles,
        [ctx_label[c] for c in ctx_order],
        title="Aquatic context",
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.50),
        borderaxespad=0
    )
else:
    # Simple fallback
    for i, comp in enumerate(comp_order):
        for j, ctx in enumerate(ctx_order):
            vals = high.loc[high["aquatic_context"] == ctx, comp].dropna().values
            if len(vals) > 2500:
                vals = np.random.default_rng(42).choice(vals, 2500, replace=False)
            pos = i + (j - 1) * 0.22
            vp = axA.violinplot(vals, positions=[pos], vert=False, widths=0.18, showextrema=False)
            for body in vp["bodies"]:
                body.set_facecolor(COL[ctx])
                body.set_alpha(0.55)

    axA.set_yticks(range(len(comp_order)))
    axA.set_yticklabels([comp_label[c] for c in comp_order])

axA.axvline(0, color="black", linestyle="--", linewidth=0.9)
axA.set_title("A. High-HDSI stress-component distributions", loc="left", pad=8)
axA.set_xlabel("Standardized anomaly; larger = more stressful")
axA.set_ylabel("")
panel_label(axA, "A")
clean_ax(axA)

# ---------- Panel B ----------
axB = fig.add_subplot(gs[0, 1])

plotB = component_diff.copy()
plotB["component_label"] = pd.Categorical(
    plotB["component_label"],
    categories=[comp_label[c] for c in comp_order],
    ordered=True
)
plotB = plotB.sort_values(["component_label", "group"])

y_positions = []
y_labels = []
y = 0

for comp_lab in [comp_label[c] for c in comp_order]:
    sub = plotB[plotB["component_label"] == comp_lab]
    for _, row in sub.iterrows():
        color = COL[row["group"]]

        axB.errorbar(
            row["difference"], y,
            xerr=[
                [row["difference"] - row["ci_low"]],
                [row["ci_high"] - row["difference"]]
            ],
            fmt="o",
            color=color,
            ecolor=color,
            capsize=3.5,
            markersize=6,
            linewidth=1.7
        )

        y_positions.append(y)
        y_labels.append(comp_lab if row["group"] == "near_seasonal" else "")
        y += 1

    y += 0.60

axB.axvline(0, color="black", linestyle="--", linewidth=0.9)
axB.set_yticks(y_positions)
axB.set_yticklabels(y_labels)
axB.set_xlabel("Difference from near-permanent water")
axB.set_title("B. Exposure contrasts under high HDSI", loc="left", pad=8)
panel_label(axB, "B")
clean_ax(axB)

from matplotlib.lines import Line2D

axB.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["near_seasonal"],
               label="Near seasonal − Near permanent", linewidth=1.7),
        Line2D([0], [0], marker="o", color=COL["other_weak"],
               label="Other/weak − Near permanent", linewidth=1.7),
    ],
    title="Contrast",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.03, 0.50),
    borderaxespad=0
)

# ---------- Panel C ----------
axC = fig.add_subplot(gs[1, 0])

if HAS_SNS:
    sns.violinplot(
        data=dlai_plot,
        x="aquatic_context",
        y="delta_LAI",
        hue="HDSI_group",
        order=ctx_order,
        hue_order=["low", "high"],
        palette=[COL["low"], COL["high"]],
        cut=0,
        inner=None,
        linewidth=0.75,
        scale="width",
        ax=axC
    )

    sns.boxplot(
        data=dlai_plot,
        x="aquatic_context",
        y="delta_LAI",
        hue="HDSI_group",
        order=ctx_order,
        hue_order=["low", "high"],
        width=0.22,
        showcaps=True,
        boxprops={"facecolor": "white", "alpha": 0.85, "linewidth": 0.7},
        whiskerprops={"linewidth": 0.7},
        medianprops={"color": "black", "linewidth": 1.0},
        showfliers=False,
        ax=axC
    )

    handles, labels = axC.get_legend_handles_labels()
    axC.legend(
        handles[:2],
        ["Low HDSI", "High HDSI"],
        title="Stress level",
        frameon=False,
        loc="center left",
        bbox_to_anchor=(1.02, 0.50),
        borderaxespad=0
    )

axC.axhline(0, color="black", linestyle="--", linewidth=0.9)
axC.set_xticks(range(len(ctx_order)))
axC.set_xticklabels([ctx_label[c] for c in ctx_order], rotation=0)
axC.set_ylabel("ΔLAI = late-summer LAI_z − spring LAI_z")
axC.set_xlabel("")
axC.set_title("C. Seasonal LAI deterioration", loc="left", pad=8)
panel_label(axC, "C")
clean_ax(axC)

# ---------- Panel D ----------
axD = fig.add_subplot(gs[1, 1])

plotD = risk_prop.copy()
y = 0
yticks = []
yticklabels = []

for event in ["LAI loss", "Joint loss"]:
    sub_event = plotD[plotD["event"] == event]

    for ctx in ctx_order:
        row = sub_event[sub_event["aquatic_context"] == ctx].iloc[0]

        axD.plot(
            [row["low_risk"], row["high_risk"]],
            [y, y],
            color=COL[ctx],
            linewidth=2.4,
            alpha=0.82
        )

        axD.errorbar(
            row["low_risk"], y,
            xerr=[
                [row["low_risk"] - row["low_ci_low"]],
                [row["low_ci_high"] - row["low_risk"]]
            ],
            fmt="o",
            color=COL["low"],
            ecolor=COL["low"],
            capsize=3,
            markersize=5.8
        )

        axD.errorbar(
            row["high_risk"], y,
            xerr=[
                [row["high_risk"] - row["high_ci_low"]],
                [row["high_ci_high"] - row["high_risk"]]
            ],
            fmt="s",
            color=COL["high"],
            ecolor=COL["high"],
            capsize=3,
            markersize=5.8
        )

        yticks.append(y)
        yticklabels.append(f"{event} | {ctx_label[ctx]}")
        y += 1

    y += 0.72

axD.set_yticks(yticks)
axD.set_yticklabels(yticklabels)
axD.set_xlabel("Loss probability")
axD.set_title("D. HDSI-to-loss propagation", loc="left", pad=8)
panel_label(axD, "D")
clean_ax(axD)

axD.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["low"],
               label="Low HDSI", linestyle="None", markersize=6.5),
        Line2D([0], [0], marker="s", color=COL["high"],
               label="High HDSI", linestyle="None", markersize=6.5),
    ],
    title="Stress level",
    frameon=False,
    loc="center left",
    bbox_to_anchor=(1.03, 0.50),
    borderaxespad=0
)

fig.suptitle(
    "Figure 4. Mechanism diagnostics: exposure contrast and stress-to-loss propagation",
    fontsize=15,
    fontweight="bold",
    y=0.995
)

plt.subplots_adjust(left=0.08, right=0.82, top=0.93, bottom=0.08)
plt.show()

# ============================================================
# 5. Improved Figure 5: regional boundary conditions
# ============================================================

def regional_afb(sub, event_col, n_boot=500, seed=42):
    near = sub[sub["aquatic_context"] == "near_permanent"]
    other = sub[sub["aquatic_context"] == "other_weak"]

    near_low = near.loc[near["HDSI_group"] == "low", event_col].dropna().values
    near_high = near.loc[near["HDSI_group"] == "high", event_col].dropna().values
    other_low = other.loc[other["HDSI_group"] == "low", event_col].dropna().values
    other_high = other.loc[other["HDSI_group"] == "high", event_col].dropna().values

    if min(len(near_low), len(near_high), len(other_low), len(other_high)) < 20:
        return np.nan, np.nan, np.nan, len(near_low), len(near_high), len(other_low), len(other_high)

    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        nl = rng.choice(near_low, size=len(near_low), replace=True)
        nh = rng.choice(near_high, size=len(near_high), replace=True)
        ol = rng.choice(other_low, size=len(other_low), replace=True)
        oh = rng.choice(other_high, size=len(other_high), replace=True)

        frag_near = nh.mean() - nl.mean()
        frag_other = oh.mean() - ol.mean()
        boots.append(frag_other - frag_near)

    boots = np.asarray(boots)

    return (
        boots.mean(),
        np.percentile(boots, 2.5),
        np.percentile(boots, 97.5),
        len(near_low),
        len(near_high),
        len(other_low),
        len(other_high)
    )

region_rows = []

for region, sub in d.groupby("macro_region"):
    if len(sub) < 1000:
        continue

    for event_col, response in [
        ("EVI_loss", "EVI"),
        ("LAI_loss", "LAI"),
        ("Joint_loss", "Joint")
    ]:
        est, lo, hi, n_nl, n_nh, n_ol, n_oh = regional_afb(sub, event_col, n_boot=500)

        region_rows.append({
            "region": region,
            "response": response,
            "AFB": est,
            "ci_low": lo,
            "ci_high": hi,
            "n_near_low": n_nl,
            "n_near_high": n_nh,
            "n_other_low": n_ol,
            "n_other_high": n_oh,
            "n_samples": len(sub),
        })

regional_afb_df = pd.DataFrame(region_rows)
display(regional_afb_df)

fig, axes = plt.subplots(
    1, 3,
    figsize=(16.5, 5.8),
    sharey=False,
    constrained_layout=False
)

for ax, response, color, lab in zip(
    axes,
    ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = regional_afb_df[regional_afb_df["response"] == response].dropna(subset=["AFB"]).copy()
    sub = sub.sort_values("AFB")

    y = np.arange(len(sub))

    ax.errorbar(
        sub["AFB"],
        y,
        xerr=[
            sub["AFB"] - sub["ci_low"],
            sub["ci_high"] - sub["AFB"]
        ],
        fmt="o",
        color=color,
        ecolor=color,
        capsize=3.5,
        markersize=6,
        linewidth=1.8
    )

    ax.axvline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["region"])
    ax.set_title(f"{response} regional AFB", loc="left", pad=8)
    ax.set_xlabel("AFB = Fragility(other/weak) − Fragility(near permanent)")
    panel_label(ax, lab)
    clean_ax(ax)

fig.suptitle(
    "Figure 5. Regional boundary conditions of aquatic buffering",
    fontsize=15,
    fontweight="bold",
    y=1.02
)

plt.subplots_adjust(left=0.08, right=0.98, top=0.86, bottom=0.14, wspace=0.36)
plt.show()

# ============================================================
# 6. Threshold sensitivity figure
# ============================================================

thresholds = [-0.25, -0.5, -0.75, -1.0]
sens_rows = []

for thr in thresholds:
    tmp = d.copy()

    tmp["EVI_evt"] = (tmp["late_summer_EVI_z"] < thr).astype(int)
    tmp["LAI_evt"] = (tmp["late_summer_LAI_z"] < thr).astype(int)
    tmp["Joint_evt"] = ((tmp["EVI_evt"] == 1) | (tmp["LAI_evt"] == 1)).astype(int)

    for event_col, response in [
        ("EVI_evt", "EVI"),
        ("LAI_evt", "LAI"),
        ("Joint_evt", "Joint")
    ]:
        near = tmp[tmp["aquatic_context"] == "near_permanent"]
        other = tmp[tmp["aquatic_context"] == "other_weak"]

        near_low = near.loc[near["HDSI_group"] == "low", event_col].values
        near_high = near.loc[near["HDSI_group"] == "high", event_col].values
        other_low = other.loc[other["HDSI_group"] == "low", event_col].values
        other_high = other.loc[other["HDSI_group"] == "high", event_col].values

        rng = np.random.default_rng(42)
        boots = []

        if min(len(near_low), len(near_high), len(other_low), len(other_high)) > 0:
            for _ in range(500):
                nl = rng.choice(near_low, len(near_low), replace=True)
                nh = rng.choice(near_high, len(near_high), replace=True)
                ol = rng.choice(other_low, len(other_low), replace=True)
                oh = rng.choice(other_high, len(other_high), replace=True)

                frag_near = nh.mean() - nl.mean()
                frag_other = oh.mean() - ol.mean()
                boots.append(frag_other - frag_near)

            boots = np.asarray(boots)
            est = boots.mean()
            lo = np.percentile(boots, 2.5)
            hi = np.percentile(boots, 97.5)
        else:
            est, lo, hi = np.nan, np.nan, np.nan

        sens_rows.append({
            "threshold": thr,
            "response": response,
            "AFB": est,
            "ci_low": lo,
            "ci_high": hi
        })

sens_df = pd.DataFrame(sens_rows)
display(sens_df)

fig, axes = plt.subplots(
    1, 3,
    figsize=(13.5, 4.0),
    sharey=True,
    constrained_layout=False
)

for ax, response, color, lab in zip(
    axes,
    ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = sens_df[sens_df["response"] == response].copy()

    ax.errorbar(
        sub["threshold"],
        sub["AFB"],
        yerr=[
            sub["AFB"] - sub["ci_low"],
            sub["ci_high"] - sub["AFB"]
        ],
        fmt="o-",
        color=color,
        capsize=3.5,
        linewidth=1.8,
        markersize=6
    )

    ax.axhline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_title(response, loc="left", pad=8)
    ax.set_xlabel("Loss threshold")
    panel_label(ax, lab)
    clean_ax(ax)

axes[0].set_ylabel("AFB = Fragility(other/weak) − Fragility(near permanent)")

fig.suptitle(
    "Threshold sensitivity of aquatic buffering",
    fontsize=14,
    fontweight="bold",
    y=1.04
)

plt.subplots_adjust(left=0.08, right=0.98, top=0.80, bottom=0.20, wspace=0.25)
plt.show()

In [ ]:
# ============================================================
# Complete plotting code for:
#   Fig 4 (mechanism diagnosis, improved layout)
#   Fig 5 (regional boundary conditions)
#   Threshold sensitivity figure
# Directly shows figures; does NOT save.
# Fixed: no overlapping, unified fonts, legends outside.
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

# ------------------------------------------------------------
# 0. Paths and data loading
# ------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/_aquatic_buffering"
PANEL_FP = os.path.join(BASE_DIR, "china_aquatic_buffering_panel_v2.parquet")

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

# ------------------------------------------------------------
# 1. Global figure style (unified fonts)
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Color palette
COL = {
    "near_permanent": "#3B6EA8",
    "near_seasonal": "#E69F00",
    "other_weak": "#666666",
    "EVI": "#4E9A51",
    "LAI": "#7E62A3",
    "Joint": "#B2182B",
    "low": "#72A9D5",
    "high": "#D95F5F",
    "dark": "#222222",
    "gray": "#666666",
}

ctx_order = ["near_permanent", "near_seasonal", "other_weak"]
ctx_label = {
    "near_permanent": "Near permanent",
    "near_seasonal": "Near seasonal",
    "other_weak": "Other / weak",
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.16, linewidth=0.45)

def panel_label(ax, lab, x=-0.08, y=1.04, fontsize=12):
    ax.text(
        x, y, lab,
        transform=ax.transAxes,
        fontsize=fontsize,
        fontweight="bold",
        ha="left",
        va="top"
    )

def boot_mean_ci(x, n_boot=400, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    if len(x) <= 2:
        return m, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = [rng.choice(x, size=len(x), replace=True).mean() for _ in range(n_boot)]
    return m, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def boot_diff_ci(a, b, n_boot=400, seed=42):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    d0 = a.mean() - b.mean()
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots.append(aa.mean() - bb.mean())
    return d0, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def sample_for_plot(x, n=2500, seed=42):
    if len(x) <= n:
        return x
    return x.sample(n=n, random_state=seed)

# ------------------------------------------------------------
# 2. Prepare variables (same as original)
# ------------------------------------------------------------
num_cols = [
    "lon", "lat", "year", "elevation", "slope",
    "HDSI",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_SM_rootproxy_z", "summer_Precip_z",
    "summer_LST_Day_z", "summer_VPD_z", "summer_Tair_z"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )

# Aquatic context
df["aquatic_context"] = "other_weak"
df.loc[df["dist_perm_water_km"] < 1.0, "aquatic_context"] = "near_permanent"
df.loc[
    (df["dist_perm_water_km"] >= 1.0) & (df["dist_seasonal_water_km"] < 1.0),
    "aquatic_context"
] = "near_seasonal"

# HDSI groups
q25 = df["HDSI"].quantile(0.25)
q75 = df["HDSI"].quantile(0.75)
df["HDSI_group"] = "mid"
df.loc[df["HDSI"] <= q25, "HDSI_group"] = "low"
df.loc[df["HDSI"] >= q75, "HDSI_group"] = "high"

# Loss events
df["EVI_loss"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_loss"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
df["Joint_loss"] = ((df["EVI_loss"] == 1) | (df["LAI_loss"] == 1)).astype(int)

# Seasonal deterioration
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# Stress components (larger = worse)
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

def assign_macro_region(row):
    lon = row.get("lon", np.nan)
    lat = row.get("lat", np.nan)
    elev = row.get("elevation", np.nan)
    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

required = [
    "HDSI", "HDSI_group", "aquatic_context",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "spring_EVI_z", "spring_LAI_z",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "EVI_loss", "LAI_loss", "Joint_loss", "delta_LAI",
    "macro_region"
]

d = df.dropna(subset=required).copy()
d = d[d["aquatic_context"].isin(ctx_order)].copy()
print("Valid sample:", d.shape)

# ------------------------------------------------------------
# 3. Figure 4: Mechanism diagnosis (fixed layout)
# ------------------------------------------------------------
comp_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
comp_label = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature",
}

high = d[d["HDSI_group"] == "high"].copy()

# ---- Panel A data: long format for violin plots ----
long_list = []
for comp in comp_order:
    tmp = high[["aquatic_context", comp]].dropna().copy()
    tmp = tmp.rename(columns={comp: "value"})
    tmp["component"] = comp
    tmp["component_label"] = comp_label[comp]
    long_list.append(tmp)
long_high = pd.concat(long_list, axis=0, ignore_index=True)
long_high_plot = (
    long_high.groupby(["component", "aquatic_context"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ---- Panel B: component differences ----
component_diff_rows = []
for comp in comp_order:
    near = high.loc[high["aquatic_context"] == "near_permanent", comp].dropna().values
    for group in ["near_seasonal", "other_weak"]:
        vals = high.loc[high["aquatic_context"] == group, comp].dropna().values
        est, lo, hi = boot_diff_ci(vals, near, n_boot=500)
        component_diff_rows.append({
            "component": comp,
            "component_label": comp_label[comp],
            "group": group,
            "contrast": f"{ctx_label[group]} − Near permanent",
            "difference": est,
            "ci_low": lo,
            "ci_high": hi,
        })
component_diff = pd.DataFrame(component_diff_rows)

# ---- Panel C: delta LAI ----
dlai_plot = (
    d[d["HDSI_group"].isin(["low", "high"])]
    [["aquatic_context", "HDSI_group", "delta_LAI"]]
    .dropna()
    .groupby(["aquatic_context", "HDSI_group"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ---- Panel D: risk propagation ----
risk_rows = []
for event_col, event_name in [("LAI_loss", "LAI loss"), ("Joint_loss", "Joint loss")]:
    for ctx in ctx_order:
        sub = d[d["aquatic_context"] == ctx]
        low_vals = sub.loc[sub["HDSI_group"] == "low", event_col].values
        high_vals = sub.loc[sub["HDSI_group"] == "high", event_col].values
        low_m, low_lo, low_hi = boot_mean_ci(low_vals, n_boot=500)
        high_m, high_lo, high_hi = boot_mean_ci(high_vals, n_boot=500)
        risk_rows.append({
            "event": event_name,
            "aquatic_context": ctx,
            "low_risk": low_m,
            "low_ci_low": low_lo,
            "low_ci_high": low_hi,
            "high_risk": high_m,
            "high_ci_low": high_lo,
            "high_ci_high": high_hi,
        })
risk_prop = pd.DataFrame(risk_rows)

# ---- Draw Figure 4 with fixed margins ----
fig = plt.figure(figsize=(17.5, 11.5), constrained_layout=False)
gs = fig.add_gridspec(2, 2, height_ratios=[1.12, 1.0],
                      width_ratios=[1.10, 0.90], hspace=0.30, wspace=0.30)

# Panel A
axA = fig.add_subplot(gs[0, 0])
if HAS_SNS:
    sns.violinplot(
        data=long_high_plot,
        x="value", y="component_label",
        hue="aquatic_context",
        order=[comp_label[c] for c in comp_order],
        hue_order=ctx_order,
        palette=[COL[c] for c in ctx_order],
        cut=0, inner="quartile", linewidth=0.75, scale="width",
        ax=axA
    )
    handles, labels = axA.get_legend_handles_labels()
    axA.legend(
        handles, [ctx_label[c] for c in ctx_order],
        title="Aquatic context", frameon=False,
        loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0
    )
axA.axvline(0, color="black", linestyle="--", linewidth=0.9)
axA.set_title("A. High‑HDSI stress‑component distributions", loc="left", pad=8)
axA.set_xlabel("Standardized anomaly; larger = more stressful")
panel_label(axA, "A", fontsize=12)
clean_ax(axA)

# Panel B
axB = fig.add_subplot(gs[0, 1])
plotB = component_diff.copy()
plotB["component_label"] = pd.Categorical(plotB["component_label"],
                                          categories=[comp_label[c] for c in comp_order],
                                          ordered=True)
plotB = plotB.sort_values(["component_label", "group"])
y_pos, y_labs = [], []
y = 0
for comp_lab in [comp_label[c] for c in comp_order]:
    sub = plotB[plotB["component_label"] == comp_lab]
    for _, row in sub.iterrows():
        color = COL[row["group"]]
        axB.errorbar(row["difference"], y,
                     xerr=[[row["difference"] - row["ci_low"]],
                           [row["ci_high"] - row["difference"]]],
                     fmt="o", color=color, ecolor=color,
                     capsize=3.5, markersize=6, linewidth=1.7)
        y_pos.append(y)
        y_labs.append(comp_lab if row["group"] == "near_seasonal" else "")
        y += 1
    y += 0.6
axB.axvline(0, color="black", linestyle="--", linewidth=0.9)
axB.set_yticks(y_pos)
axB.set_yticklabels(y_labs)
axB.set_xlabel("Difference from near‑permanent water")
axB.set_title("B. Exposure contrasts under high HDSI", loc="left", pad=8)
panel_label(axB, "B", fontsize=12)
clean_ax(axB)
axB.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["near_seasonal"],
               label="Near seasonal − Near permanent"),
        Line2D([0], [0], marker="o", color=COL["other_weak"],
               label="Other/weak − Near permanent"),
    ],
    title="Contrast", frameon=False,
    loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0
)

# Panel C
axC = fig.add_subplot(gs[1, 0])
if HAS_SNS:
    sns.violinplot(
        data=dlai_plot, x="aquatic_context", y="delta_LAI",
        hue="HDSI_group", order=ctx_order, hue_order=["low", "high"],
        palette=[COL["low"], COL["high"]], cut=0, inner=None,
        linewidth=0.75, scale="width", ax=axC
    )
    sns.boxplot(
        data=dlai_plot, x="aquatic_context", y="delta_LAI",
        hue="HDSI_group", order=ctx_order, hue_order=["low", "high"],
        width=0.22, showcaps=True,
        boxprops={"facecolor": "white", "alpha": 0.85, "linewidth": 0.7},
        whiskerprops={"linewidth": 0.7}, medianprops={"color": "black", "linewidth": 1.0},
        showfliers=False, ax=axC
    )
    handles, labels = axC.get_legend_handles_labels()
    axC.legend(handles[:2], ["Low HDSI", "High HDSI"],
               title="Stress level", frameon=False,
               loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0)
axC.axhline(0, color="black", linestyle="--", linewidth=0.9)
axC.set_xticks(range(len(ctx_order)))
axC.set_xticklabels([ctx_label[c] for c in ctx_order])
axC.set_ylabel("ΔLAI = late‑summer LAI_z − spring LAI_z")
axC.set_title("C. Seasonal LAI deterioration", loc="left", pad=8)
panel_label(axC, "C", fontsize=12)
clean_ax(axC)

# Panel D
axD = fig.add_subplot(gs[1, 1])
plotD = risk_prop.copy()
y = 0
yticks, yticklabels = [], []
for event in ["LAI loss", "Joint loss"]:
    sub_event = plotD[plotD["event"] == event]
    for ctx in ctx_order:
        row = sub_event[sub_event["aquatic_context"] == ctx].iloc[0]
        axD.plot([row["low_risk"], row["high_risk"]], [y, y],
                 color=COL[ctx], linewidth=2.4, alpha=0.82)
        axD.errorbar(row["low_risk"], y,
                     xerr=[[row["low_risk"] - row["low_ci_low"]],
                           [row["low_ci_high"] - row["low_risk"]]],
                     fmt="o", color=COL["low"], ecolor=COL["low"],
                     capsize=3, markersize=5.8)
        axD.errorbar(row["high_risk"], y,
                     xerr=[[row["high_risk"] - row["high_ci_low"]],
                           [row["high_ci_high"] - row["high_risk"]]],
                     fmt="s", color=COL["high"], ecolor=COL["high"],
                     capsize=3, markersize=5.8)
        yticks.append(y)
        yticklabels.append(f"{event} | {ctx_label[ctx]}")
        y += 1
    y += 0.72
axD.set_yticks(yticks)
axD.set_yticklabels(yticklabels)
axD.set_xlabel("Loss probability")
axD.set_title("D. HDSI‑to‑loss propagation", loc="left", pad=8)
panel_label(axD, "D", fontsize=12)
clean_ax(axD)
axD.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["low"], label="Low HDSI", linestyle="None", markersize=6.5),
        Line2D([0], [0], marker="s", color=COL["high"], label="High HDSI", linestyle="None", markersize=6.5),
    ],
    title="Stress level", frameon=False,
    loc="center left", bbox_to_anchor=(1.02, 0.5), borderaxespad=0
)

fig.suptitle("Figure 4. Mechanism diagnostics: exposure contrast and stress‑to‑loss propagation",
             fontsize=14, fontweight="bold", y=0.98)

# Important: margins adjusted to avoid legend clipping
plt.subplots_adjust(left=0.10, right=0.86, top=0.92, bottom=0.08)
plt.show()

# ------------------------------------------------------------
# 4. Figure 5: Regional boundary conditions (fixed layout)
# ------------------------------------------------------------
def regional_afb(sub, event_col, n_boot=500, seed=42):
    near = sub[sub["aquatic_context"] == "near_permanent"]
    other = sub[sub["aquatic_context"] == "other_weak"]
    near_low = near.loc[near["HDSI_group"] == "low", event_col].dropna().values
    near_high = near.loc[near["HDSI_group"] == "high", event_col].dropna().values
    other_low = other.loc[other["HDSI_group"] == "low", event_col].dropna().values
    other_high = other.loc[other["HDSI_group"] == "high", event_col].dropna().values
    if min(len(near_low), len(near_high), len(other_low), len(other_high)) < 20:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        nl = rng.choice(near_low, size=len(near_low), replace=True)
        nh = rng.choice(near_high, size=len(near_high), replace=True)
        ol = rng.choice(other_low, size=len(other_low), replace=True)
        oh = rng.choice(other_high, size=len(other_high), replace=True)
        frag_near = nh.mean() - nl.mean()
        frag_other = oh.mean() - ol.mean()
        boots.append(frag_other - frag_near)
    boots = np.asarray(boots)
    return boots.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

region_rows = []
for region, sub in d.groupby("macro_region"):
    if len(sub) < 1000:
        continue
    for event_col, response in [("EVI_loss", "EVI"), ("LAI_loss", "LAI"), ("Joint_loss", "Joint")]:
        est, lo, hi = regional_afb(sub, event_col, n_boot=500)
        if not np.isnan(est):
            region_rows.append({
                "region": region,
                "response": response,
                "AFB": est,
                "ci_low": lo,
                "ci_high": hi
            })
regional_afb_df = pd.DataFrame(region_rows)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.8), sharey=False, constrained_layout=False)
for ax, response, color, lab in zip(
    axes, ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = regional_afb_df[regional_afb_df["response"] == response].dropna(subset=["AFB"]).copy()
    sub = sub.sort_values("AFB")
    y = np.arange(len(sub))
    ax.errorbar(sub["AFB"], y,
                xerr=[sub["AFB"] - sub["ci_low"], sub["ci_high"] - sub["AFB"]],
                fmt="o", color=color, ecolor=color,
                capsize=3.5, markersize=6, linewidth=1.8)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["region"])
    ax.set_title(f"{response} regional AFB", loc="left", pad=8)
    ax.set_xlabel("AFB = Fragility(other/weak) − Fragility(near permanent)")
    panel_label(ax, lab, fontsize=12)
    clean_ax(ax)

fig.suptitle("Figure 5. Regional boundary conditions of aquatic buffering",
             fontsize=14, fontweight="bold", y=0.98)
plt.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14, wspace=0.36)
plt.show()

# ------------------------------------------------------------
# 5. Threshold sensitivity figure (fixed layout)
# ------------------------------------------------------------
thresholds = [-0.25, -0.5, -0.75, -1.0]
sens_rows = []
for thr in thresholds:
    tmp = d.copy()
    tmp["EVI_evt"] = (tmp["late_summer_EVI_z"] < thr).astype(int)
    tmp["LAI_evt"] = (tmp["late_summer_LAI_z"] < thr).astype(int)
    tmp["Joint_evt"] = ((tmp["EVI_evt"] == 1) | (tmp["LAI_evt"] == 1)).astype(int)
    for event_col, response in [("EVI_evt", "EVI"), ("LAI_evt", "LAI"), ("Joint_evt", "Joint")]:
        near = tmp[tmp["aquatic_context"] == "near_permanent"]
        other = tmp[tmp["aquatic_context"] == "other_weak"]
        near_low = near.loc[near["HDSI_group"] == "low", event_col].values
        near_high = near.loc[near["HDSI_group"] == "high", event_col].values
        other_low = other.loc[other["HDSI_group"] == "low", event_col].values
        other_high = other.loc[other["HDSI_group"] == "high", event_col].values
        rng = np.random.default_rng(42)
        boots = []
        if min(len(near_low), len(near_high), len(other_low), len(other_high)) > 0:
            for _ in range(500):
                nl = rng.choice(near_low, len(near_low), replace=True)
                nh = rng.choice(near_high, len(near_high), replace=True)
                ol = rng.choice(other_low, len(other_low), replace=True)
                oh = rng.choice(other_high, len(other_high), replace=True)
                boots.append((oh.mean() - ol.mean()) - (nh.mean() - nl.mean()))
            boots = np.asarray(boots)
            est = boots.mean()
            lo = np.percentile(boots, 2.5)
            hi = np.percentile(boots, 97.5)
        else:
            est, lo, hi = np.nan, np.nan, np.nan
        sens_rows.append({"threshold": thr, "response": response, "AFB": est, "ci_low": lo, "ci_high": hi})
sens_df = pd.DataFrame(sens_rows)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True, constrained_layout=False)
for ax, response, color, lab in zip(
    axes, ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = sens_df[sens_df["response"] == response].copy()
    ax.errorbar(sub["threshold"], sub["AFB"],
                yerr=[sub["AFB"] - sub["ci_low"], sub["ci_high"] - sub["AFB"]],
                fmt="o-", color=color, capsize=3.5, linewidth=1.8, markersize=6)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_title(response, loc="left", pad=8)
    ax.set_xlabel("Loss threshold")
    panel_label(ax, lab, fontsize=12)
    clean_ax(ax)
axes[0].set_ylabel("AFB = Fragility(other/weak) − Fragility(near permanent)")
fig.suptitle("Threshold sensitivity of aquatic buffering",
             fontsize=14, fontweight="bold", y=0.98)
plt.subplots_adjust(left=0.08, right=0.98, top=0.85, bottom=0.20, wspace=0.25)
plt.show()

In [ ]:
# ============================================================
# Complete plotting code for:
#   Fig 4 (mechanism diagnosis) – legends below subplots
#   Fig 5 (regional boundary conditions)
#   Threshold sensitivity figure
# Directly shows figures; does NOT save.
# ============================================================

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

try:
    import seaborn as sns
    HAS_SNS = True
except ImportError:
    HAS_SNS = False

# ------------------------------------------------------------
# 0. Paths and data loading
# ------------------------------------------------------------
BASE_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/_aquatic_buffering"
PANEL_FP = os.path.join(BASE_DIR, "china_aquatic_buffering_panel_v2.parquet")

df = pd.read_parquet(PANEL_FP)
print("Raw panel:", df.shape)

# ------------------------------------------------------------
# 1. Global figure style (unified fonts)
# ------------------------------------------------------------
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 9.5,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# Color palette
COL = {
    "near_permanent": "#3B6EA8",
    "near_seasonal": "#E69F00",
    "other_weak": "#666666",
    "EVI": "#4E9A51",
    "LAI": "#7E62A3",
    "Joint": "#B2182B",
    "low": "#72A9D5",
    "high": "#D95F5F",
    "dark": "#222222",
    "gray": "#666666",
}

ctx_order = ["near_permanent", "near_seasonal", "other_weak"]
ctx_label = {
    "near_permanent": "Near permanent",
    "near_seasonal": "Near seasonal",
    "other_weak": "Other / weak",
}

def clean_ax(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.16, linewidth=0.45)

def panel_label(ax, lab, x=-0.08, y=1.04, fontsize=12):
    ax.text(
        x, y, lab,
        transform=ax.transAxes,
        fontsize=fontsize,
        fontweight="bold",
        ha="left",
        va="top"
    )

def boot_mean_ci(x, n_boot=400, seed=42):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    m = x.mean()
    if len(x) <= 2:
        return m, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = [rng.choice(x, size=len(x), replace=True).mean() for _ in range(n_boot)]
    return m, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def boot_diff_ci(a, b, n_boot=400, seed=42):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]
    if len(a) == 0 or len(b) == 0:
        return np.nan, np.nan, np.nan
    d0 = a.mean() - b.mean()
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        aa = rng.choice(a, size=len(a), replace=True)
        bb = rng.choice(b, size=len(b), replace=True)
        boots.append(aa.mean() - bb.mean())
    return d0, np.percentile(boots, 2.5), np.percentile(boots, 97.5)

def sample_for_plot(x, n=2500, seed=42):
    if len(x) <= n:
        return x
    return x.sample(n=n, random_state=seed)

# ------------------------------------------------------------
# 2. Prepare variables (same as original)
# ------------------------------------------------------------
num_cols = [
    "lon", "lat", "year", "elevation", "slope",
    "HDSI",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "summer_SM_rootproxy_z", "summer_Precip_z",
    "summer_LST_Day_z", "summer_VPD_z", "summer_Tair_z"
]

for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

if "HDSI" not in df.columns:
    df["HDSI"] = np.nanmean(
        np.vstack([
            df["summer_VPD_z"].values,
            df["summer_LST_Day_z"].values,
            -df["summer_SM_rootproxy_z"].values,
            -df["summer_Precip_z"].values,
        ]),
        axis=0
    )

# Aquatic context
df["aquatic_context"] = "other_weak"
df.loc[df["dist_perm_water_km"] < 1.0, "aquatic_context"] = "near_permanent"
df.loc[
    (df["dist_perm_water_km"] >= 1.0) & (df["dist_seasonal_water_km"] < 1.0),
    "aquatic_context"
] = "near_seasonal"

# HDSI groups
q25 = df["HDSI"].quantile(0.25)
q75 = df["HDSI"].quantile(0.75)
df["HDSI_group"] = "mid"
df.loc[df["HDSI"] <= q25, "HDSI_group"] = "low"
df.loc[df["HDSI"] >= q75, "HDSI_group"] = "high"

# Loss events
df["EVI_loss"] = (df["late_summer_EVI_z"] < -0.5).astype(int)
df["LAI_loss"] = (df["late_summer_LAI_z"] < -0.5).astype(int)
df["Joint_loss"] = ((df["EVI_loss"] == 1) | (df["LAI_loss"] == 1)).astype(int)

# Seasonal deterioration
df["delta_EVI"] = df["late_summer_EVI_z"] - df["spring_EVI_z"]
df["delta_LAI"] = df["late_summer_LAI_z"] - df["spring_LAI_z"]

# Stress components (larger = worse)
df["SM_deficit"] = -df["summer_SM_rootproxy_z"]
df["Precip_deficit"] = -df["summer_Precip_z"]
df["LST_Day"] = df["summer_LST_Day_z"]
df["VPD"] = df["summer_VPD_z"]
df["Tair"] = df["summer_Tair_z"]

def assign_macro_region(row):
    lon = row.get("lon", np.nan)
    lat = row.get("lat", np.nan)
    elev = row.get("elevation", np.nan)
    if pd.notna(elev) and elev >= 3000 and 75 <= lon <= 105 and 25 <= lat <= 42:
        return "Tibetan_Plateau"
    if lon >= 118 and lat >= 40:
        return "Northeast"
    if lon < 105 and lat >= 35:
        return "Northwest_Arid"
    if 105 <= lon < 122 and 34 <= lat < 40:
        return "North_China"
    if 110 <= lon < 123 and 27 <= lat < 34:
        return "East_China"
    if lon >= 105 and lat < 27:
        return "South_China"
    if lon < 110 and lat < 35:
        return "Southwest"
    return "Other"

if "macro_region" not in df.columns:
    df["macro_region"] = df.apply(assign_macro_region, axis=1)
else:
    df["macro_region"] = df["macro_region"].astype(str)

required = [
    "HDSI", "HDSI_group", "aquatic_context",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "spring_EVI_z", "spring_LAI_z",
    "dist_perm_water_km", "dist_seasonal_water_km",
    "SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair",
    "EVI_loss", "LAI_loss", "Joint_loss", "delta_LAI",
    "macro_region"
]

d = df.dropna(subset=required).copy()
d = d[d["aquatic_context"].isin(ctx_order)].copy()
print("Valid sample:", d.shape)

# ------------------------------------------------------------
# 3. Figure 4: Mechanism diagnosis (legends below subplots)
# ------------------------------------------------------------
comp_order = ["SM_deficit", "Precip_deficit", "LST_Day", "VPD", "Tair"]
comp_label = {
    "SM_deficit": "Water deficit",
    "Precip_deficit": "Precipitation deficit",
    "LST_Day": "Surface heat",
    "VPD": "Atmospheric dryness",
    "Tair": "Air temperature",
}

high = d[d["HDSI_group"] == "high"].copy()

# ---- Panel A data: long format for violin plots ----
long_list = []
for comp in comp_order:
    tmp = high[["aquatic_context", comp]].dropna().copy()
    tmp = tmp.rename(columns={comp: "value"})
    tmp["component"] = comp
    tmp["component_label"] = comp_label[comp]
    long_list.append(tmp)
long_high = pd.concat(long_list, axis=0, ignore_index=True)
long_high_plot = (
    long_high.groupby(["component", "aquatic_context"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ---- Panel B: component differences ----
component_diff_rows = []
for comp in comp_order:
    near = high.loc[high["aquatic_context"] == "near_permanent", comp].dropna().values
    for group in ["near_seasonal", "other_weak"]:
        vals = high.loc[high["aquatic_context"] == group, comp].dropna().values
        est, lo, hi = boot_diff_ci(vals, near, n_boot=500)
        component_diff_rows.append({
            "component": comp,
            "component_label": comp_label[comp],
            "group": group,
            "contrast": f"{ctx_label[group]} − Near permanent",
            "difference": est,
            "ci_low": lo,
            "ci_high": hi,
        })
component_diff = pd.DataFrame(component_diff_rows)

# ---- Panel C: delta LAI ----
dlai_plot = (
    d[d["HDSI_group"].isin(["low", "high"])]
    [["aquatic_context", "HDSI_group", "delta_LAI"]]
    .dropna()
    .groupby(["aquatic_context", "HDSI_group"], group_keys=False)
    .apply(lambda x: sample_for_plot(x, n=2200))
    .reset_index(drop=True)
)

# ---- Panel D: risk propagation ----
risk_rows = []
for event_col, event_name in [("LAI_loss", "LAI loss"), ("Joint_loss", "Joint loss")]:
    for ctx in ctx_order:
        sub = d[d["aquatic_context"] == ctx]
        low_vals = sub.loc[sub["HDSI_group"] == "low", event_col].values
        high_vals = sub.loc[sub["HDSI_group"] == "high", event_col].values
        low_m, low_lo, low_hi = boot_mean_ci(low_vals, n_boot=500)
        high_m, high_lo, high_hi = boot_mean_ci(high_vals, n_boot=500)
        risk_rows.append({
            "event": event_name,
            "aquatic_context": ctx,
            "low_risk": low_m,
            "low_ci_low": low_lo,
            "low_ci_high": low_hi,
            "high_risk": high_m,
            "high_ci_low": high_lo,
            "high_ci_high": high_hi,
        })
risk_prop = pd.DataFrame(risk_rows)

# ---- Draw Figure 4 with legends below each subplot ----
fig = plt.figure(figsize=(17.5, 13.0), constrained_layout=False)  # taller to fit legends
gs = fig.add_gridspec(2, 2, height_ratios=[1.12, 1.2],
                      width_ratios=[1.10, 0.90], hspace=0.50, wspace=0.30)

# Panel A
axA = fig.add_subplot(gs[0, 0])
if HAS_SNS:
    sns.violinplot(
        data=long_high_plot,
        x="value", y="component_label",
        hue="aquatic_context",
        order=[comp_label[c] for c in comp_order],
        hue_order=ctx_order,
        palette=[COL[c] for c in ctx_order],
        cut=0, inner="quartile", linewidth=0.75, scale="width",
        ax=axA
    )
    # Legend below the subplot
    axA.legend(
        loc="upper center", bbox_to_anchor=(0.5, -0.18),
        title="Aquatic context", frameon=False, ncol=3
    )
axA.axvline(0, color="black", linestyle="--", linewidth=0.9)
axA.set_title("A. High‑HDSI stress‑component distributions", loc="left", pad=8)
axA.set_xlabel("Standardized anomaly; larger = more stressful")
panel_label(axA, "A", fontsize=12)
clean_ax(axA)

# Panel B
axB = fig.add_subplot(gs[0, 1])
plotB = component_diff.copy()
plotB["component_label"] = pd.Categorical(plotB["component_label"],
                                          categories=[comp_label[c] for c in comp_order],
                                          ordered=True)
plotB = plotB.sort_values(["component_label", "group"])
y_pos, y_labs = [], []
y = 0
for comp_lab in [comp_label[c] for c in comp_order]:
    sub = plotB[plotB["component_label"] == comp_lab]
    for _, row in sub.iterrows():
        color = COL[row["group"]]
        axB.errorbar(row["difference"], y,
                     xerr=[[row["difference"] - row["ci_low"]],
                           [row["ci_high"] - row["difference"]]],
                     fmt="o", color=color, ecolor=color,
                     capsize=3.5, markersize=6, linewidth=1.7)
        y_pos.append(y)
        y_labs.append(comp_lab if row["group"] == "near_seasonal" else "")
        y += 1
    y += 0.6
axB.axvline(0, color="black", linestyle="--", linewidth=0.9)
axB.set_yticks(y_pos)
axB.set_yticklabels(y_labs)
axB.set_xlabel("Difference from near‑permanent water")
axB.set_title("B. Exposure contrasts under high HDSI", loc="left", pad=8)
panel_label(axB, "B", fontsize=12)
clean_ax(axB)
# Legend below Panel B
axB.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["near_seasonal"],
               label="Near seasonal − Near permanent"),
        Line2D([0], [0], marker="o", color=COL["other_weak"],
               label="Other/weak − Near permanent"),
    ],
    title="Contrast", frameon=False,
    loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2
)

# Panel C
axC = fig.add_subplot(gs[1, 0])
if HAS_SNS:
    sns.violinplot(
        data=dlai_plot, x="aquatic_context", y="delta_LAI",
        hue="HDSI_group", order=ctx_order, hue_order=["low", "high"],
        palette=[COL["low"], COL["high"]], cut=0, inner=None,
        linewidth=0.75, scale="width", ax=axC
    )
    sns.boxplot(
        data=dlai_plot, x="aquatic_context", y="delta_LAI",
        hue="HDSI_group", order=ctx_order, hue_order=["low", "high"],
        width=0.22, showcaps=True,
        boxprops={"facecolor": "white", "alpha": 0.85, "linewidth": 0.7},
        whiskerprops={"linewidth": 0.7}, medianprops={"color": "black", "linewidth": 1.0},
        showfliers=False, ax=axC
    )
    # Legend below Panel C
    axC.legend(
        ["Low HDSI", "High HDSI"], title="Stress level", frameon=False,
        loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2
    )
axC.axhline(0, color="black", linestyle="--", linewidth=0.9)
axC.set_xticks(range(len(ctx_order)))
axC.set_xticklabels([ctx_label[c] for c in ctx_order])
axC.set_ylabel("ΔLAI = late‑summer LAI_z − spring LAI_z")
axC.set_title("C. Seasonal LAI deterioration", loc="left", pad=8)
panel_label(axC, "C", fontsize=12)
clean_ax(axC)

# Panel D
axD = fig.add_subplot(gs[1, 1])
plotD = risk_prop.copy()
y = 0
yticks, yticklabels = [], []
for event in ["LAI loss", "Joint loss"]:
    sub_event = plotD[plotD["event"] == event]
    for ctx in ctx_order:
        row = sub_event[sub_event["aquatic_context"] == ctx].iloc[0]
        axD.plot([row["low_risk"], row["high_risk"]], [y, y],
                 color=COL[ctx], linewidth=2.4, alpha=0.82)
        axD.errorbar(row["low_risk"], y,
                     xerr=[[row["low_risk"] - row["low_ci_low"]],
                           [row["low_ci_high"] - row["low_risk"]]],
                     fmt="o", color=COL["low"], ecolor=COL["low"],
                     capsize=3, markersize=5.8)
        axD.errorbar(row["high_risk"], y,
                     xerr=[[row["high_risk"] - row["high_ci_low"]],
                           [row["high_ci_high"] - row["high_risk"]]],
                     fmt="s", color=COL["high"], ecolor=COL["high"],
                     capsize=3, markersize=5.8)
        yticks.append(y)
        yticklabels.append(f"{event} | {ctx_label[ctx]}")
        y += 1
    y += 0.72
axD.set_yticks(yticks)
axD.set_yticklabels(yticklabels)
axD.set_xlabel("Loss probability")
axD.set_title("D. HDSI‑to‑loss propagation", loc="left", pad=8)
panel_label(axD, "D", fontsize=12)
clean_ax(axD)
# Legend below Panel D
axD.legend(
    handles=[
        Line2D([0], [0], marker="o", color=COL["low"], label="Low HDSI", linestyle="None", markersize=6.5),
        Line2D([0], [0], marker="s", color=COL["high"], label="High HDSI", linestyle="None", markersize=6.5),
    ],
    title="Stress level", frameon=False,
    loc="upper center", bbox_to_anchor=(0.5, -0.18), ncol=2
)

fig.suptitle("Figure 4. Mechanism diagnostics: exposure contrast and stress‑to‑loss propagation",
             fontsize=14, fontweight="bold", y=0.98)

# Adjust margins to make room for bottom legends
plt.subplots_adjust(left=0.10, right=0.95, top=0.92, bottom=0.12)
plt.show()

# ------------------------------------------------------------
# 4. Figure 5: Regional boundary conditions (no legend needed)
# ------------------------------------------------------------
def regional_afb(sub, event_col, n_boot=500, seed=42):
    near = sub[sub["aquatic_context"] == "near_permanent"]
    other = sub[sub["aquatic_context"] == "other_weak"]
    near_low = near.loc[near["HDSI_group"] == "low", event_col].dropna().values
    near_high = near.loc[near["HDSI_group"] == "high", event_col].dropna().values
    other_low = other.loc[other["HDSI_group"] == "low", event_col].dropna().values
    other_high = other.loc[other["HDSI_group"] == "high", event_col].dropna().values
    if min(len(near_low), len(near_high), len(other_low), len(other_high)) < 20:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        nl = rng.choice(near_low, size=len(near_low), replace=True)
        nh = rng.choice(near_high, size=len(near_high), replace=True)
        ol = rng.choice(other_low, size=len(other_low), replace=True)
        oh = rng.choice(other_high, size=len(other_high), replace=True)
        frag_near = nh.mean() - nl.mean()
        frag_other = oh.mean() - ol.mean()
        boots.append(frag_other - frag_near)
    boots = np.asarray(boots)
    return boots.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)

region_rows = []
for region, sub in d.groupby("macro_region"):
    if len(sub) < 1000:
        continue
    for event_col, response in [("EVI_loss", "EVI"), ("LAI_loss", "LAI"), ("Joint_loss", "Joint")]:
        est, lo, hi = regional_afb(sub, event_col, n_boot=500)
        if not np.isnan(est):
            region_rows.append({
                "region": region,
                "response": response,
                "AFB": est,
                "ci_low": lo,
                "ci_high": hi
            })
regional_afb_df = pd.DataFrame(region_rows)

fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.8), sharey=False, constrained_layout=False)
for ax, response, color, lab in zip(
    axes, ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = regional_afb_df[regional_afb_df["response"] == response].dropna(subset=["AFB"]).copy()
    sub = sub.sort_values("AFB")
    y = np.arange(len(sub))
    ax.errorbar(sub["AFB"], y,
                xerr=[sub["AFB"] - sub["ci_low"], sub["ci_high"] - sub["AFB"]],
                fmt="o", color=color, ecolor=color,
                capsize=3.5, markersize=6, linewidth=1.8)
    ax.axvline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_yticks(y)
    ax.set_yticklabels(sub["region"])
    ax.set_title(f"{response} regional AFB", loc="left", pad=8)
    ax.set_xlabel("AFB = Fragility(other/weak) − Fragility(near permanent)")
    panel_label(ax, lab, fontsize=12)
    clean_ax(ax)

fig.suptitle("Figure 5. Regional boundary conditions of aquatic buffering",
             fontsize=14, fontweight="bold", y=0.98)
plt.subplots_adjust(left=0.08, right=0.98, top=0.88, bottom=0.14, wspace=0.36)
plt.show()

# ------------------------------------------------------------
# 5. Threshold sensitivity figure (no legend needed)
# ------------------------------------------------------------
thresholds = [-0.25, -0.5, -0.75, -1.0]
sens_rows = []
for thr in thresholds:
    tmp = d.copy()
    tmp["EVI_evt"] = (tmp["late_summer_EVI_z"] < thr).astype(int)
    tmp["LAI_evt"] = (tmp["late_summer_LAI_z"] < thr).astype(int)
    tmp["Joint_evt"] = ((tmp["EVI_evt"] == 1) | (tmp["LAI_evt"] == 1)).astype(int)
    for event_col, response in [("EVI_evt", "EVI"), ("LAI_evt", "LAI"), ("Joint_evt", "Joint")]:
        near = tmp[tmp["aquatic_context"] == "near_permanent"]
        other = tmp[tmp["aquatic_context"] == "other_weak"]
        near_low = near.loc[near["HDSI_group"] == "low", event_col].values
        near_high = near.loc[near["HDSI_group"] == "high", event_col].values
        other_low = other.loc[other["HDSI_group"] == "low", event_col].values
        other_high = other.loc[other["HDSI_group"] == "high", event_col].values
        rng = np.random.default_rng(42)
        boots = []
        if min(len(near_low), len(near_high), len(other_low), len(other_high)) > 0:
            for _ in range(500):
                nl = rng.choice(near_low, len(near_low), replace=True)
                nh = rng.choice(near_high, len(near_high), replace=True)
                ol = rng.choice(other_low, len(other_low), replace=True)
                oh = rng.choice(other_high, len(other_high), replace=True)
                boots.append((oh.mean() - ol.mean()) - (nh.mean() - nl.mean()))
            boots = np.asarray(boots)
            est = boots.mean()
            lo = np.percentile(boots, 2.5)
            hi = np.percentile(boots, 97.5)
        else:
            est, lo, hi = np.nan, np.nan, np.nan
        sens_rows.append({"threshold": thr, "response": response, "AFB": est, "ci_low": lo, "ci_high": hi})
sens_df = pd.DataFrame(sens_rows)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True, constrained_layout=False)
for ax, response, color, lab in zip(
    axes, ["EVI", "LAI", "Joint"],
    [COL["EVI"], COL["LAI"], COL["Joint"]],
    ["A", "B", "C"]
):
    sub = sens_df[sens_df["response"] == response].copy()
    ax.errorbar(sub["threshold"], sub["AFB"],
                yerr=[sub["AFB"] - sub["ci_low"], sub["ci_high"] - sub["AFB"]],
                fmt="o-", color=color, capsize=3.5, linewidth=1.8, markersize=6)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.9)
    ax.set_title(response, loc="left", pad=8)
    ax.set_xlabel("Loss threshold")
    panel_label(ax, lab, fontsize=12)
    clean_ax(ax)
axes[0].set_ylabel("AFB = Fragility(other/weak) − Fragility(near permanent)")
fig.suptitle("Threshold sensitivity of aquatic buffering",
             fontsize=14, fontweight="bold", y=0.98)
plt.subplots_adjust(left=0.08, right=0.98, top=0.85, bottom=0.20, wspace=0.25)
plt.show()